# F1 Pantheon v2.2 LIVE — Clean Production Notebook

This notebook contains the **current production path only** for the 1982–2026 LIVE Pantheon product.

It deliberately keeps the statistical specification unchanged. The cleanup is structural:

- the frozen 1982–2025 development/validation history is removed from the main execution path;
- superseded tenure experiments, predictive-CV selection work and one-off forensic audits are moved to the validation archive;
- interrupted-fit recovery cells are removed from the ordinary path;
- the 2015 Marussia/Manor familiarity alias is applied **before** the observation-safe familiarity rebuild;
- the final dashboard uses the extracted `dashboard_base_stage20.py` source directly rather than rediscovering the old notebook;
- outputs and execution counts are cleared to avoid stale results.

The original notebook remains the provenance record. Do not delete it.


## Part I — Configuration and source data

The original Ergast-format files remain untouched. The notebook augments them in memory with complete 2025 data and the current 2026 LIVE snapshot.


In [ ]:
# ==========================================================
# STAGE 1 -- CONFIGURATION
# ==========================================================

import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("f1_pantheon")


@dataclass(frozen=True)
class Config:

    data_dir: str = "ergast_data"

    # IMPORTANT:
    # Separate output directory: the validated 1982–2024
    # production files remain untouched.
    output_dir: str = "model_outputs/" "joint_driver_car_v2_1_1982_2025"

    historical_floor_year: int = 1982
    historical_ceiling_year: int = 1995

    modern_floor_year: int = 1996

    # ------------------------------------------------------
    # Final coverage endpoint:
    # ------------------------------------------------------
    ceiling_year: int = 2025

    # Modern race-pace construction.
    min_clean_laps: int = 12
    min_drivers_per_lap: int = 8
    min_field_drivers: int = 10

    # Context correction.
    tenure_scale_races: float = 18.0
    gamma_tenure_sigma: float = 0.15

    # Driver state space.
    rho_driver: float = 0.85
    sigma_driver_base_prior: float = 0.75
    sigma_driver_drift_prior: float = 0.25

    # Constructor-season effect.
    sigma_car_prior: float = 0.75

    # Modern observation likelihoods.
    obs_sigma_prior: float = 0.75
    student_t_nu: int = 4

    # Historical Plackett-Luce temperatures.
    hist_quali_tau_median: float = 1.5
    hist_race_tau_median: float = 2.0
    hist_tau_log_sigma: float = 0.5


CFG = Config()


Path(CFG.output_dir).mkdir(
    parents=True,
    exist_ok=True,
)


CFG

In [ ]:
@dataclass
class RawTables:
    races: pd.DataFrame
    drivers: pd.DataFrame
    results: pd.DataFrame
    laps: pd.DataFrame
    qualifying: pd.DataFrame
    constructors: pd.DataFrame
    status: pd.DataFrame

    driver_ref: dict
    driver_id: dict
    constructor_ref: dict
    status_map: dict


def parse_lap_time(t: Optional[str]) -> float:
    if not isinstance(t, str) or not t.strip() or t == r"\N":
        return np.nan

    try:
        parts = t.strip().split(":")
        if len(parts) == 2:
            return float(parts[0]) * 60.0 + float(parts[1])
        return float(parts[0])
    except (ValueError, IndexError):
        return np.nan


def robust_scale(x) -> float:
    x = np.asarray(x, dtype=float)

    med = np.median(x)
    mad = np.median(np.abs(x - med))
    scale = 1.4826 * mad

    if not np.isfinite(scale) or scale < 1e-8:
        scale = np.std(x, ddof=0)

    return max(float(scale), 1e-8)


def load_raw_tables(cfg=CFG) -> RawTables:
    d = Path(cfg.data_dir)

    races = pd.read_csv(d / "races.csv")
    drivers = pd.read_csv(d / "drivers.csv")
    results = pd.read_csv(d / "results.csv")
    laps = pd.read_csv(d / "lap_times.csv")
    qualifying = pd.read_csv(d / "qualifying.csv")
    status = pd.read_csv(d / "status.csv")

    constructors_path = d / "constructors.csv"

    if constructors_path.exists():
        constructors = pd.read_csv(constructors_path)
    else:
        constructors = pd.DataFrame(columns=["constructorId", "constructorRef", "name"])

    driver_ref = dict(zip(drivers["driverId"], drivers["driverRef"]))

    driver_id = dict(zip(drivers["driverRef"], drivers["driverId"]))

    if "constructorRef" in constructors.columns:
        constructor_ref = dict(
            zip(
                constructors["constructorId"],
                constructors["constructorRef"],
            )
        )
    elif "name" in constructors.columns:
        constructor_ref = dict(
            zip(
                constructors["constructorId"],
                constructors["name"],
            )
        )
    else:
        constructor_ref = {}

    status_map = dict(
        zip(
            status["statusId"],
            status["status"],
        )
    )

    return RawTables(
        races=races,
        drivers=drivers,
        results=results,
        laps=laps,
        qualifying=qualifying,
        constructors=constructors,
        status=status,
        driver_ref=driver_ref,
        driver_id=driver_id,
        constructor_ref=constructor_ref,
        status_map=status_map,
    )


raw = load_raw_tables(CFG)

print("Races:", len(raw.races))
print("Results:", len(raw.results))
print("Lap rows:", len(raw.laps))
print("Qualifying rows:", len(raw.qualifying))
print("Status rows:", len(raw.status))

## Part II — Data augmentation

In [ ]:
# ==========================================================
# STAGE 2B — AUGMENT ERGAST ARCHIVE WITH COMPLETE 2025 DATA
# ==========================================================
#
# Source:
#   Jolpica F1 Ergast-compatible API
#
# IMPORTANT:
#   - Keeps the original ergast_data/*.csv untouched.
#   - Reuses every JSON page already downloaded.
#   - Uses conservative sustained rate limiting.
#   - Can be interrupted and safely rerun.
# ==========================================================

from pathlib import Path

import hashlib
import json
import sys
import time

import numpy as np
import pandas as pd

try:
    import requests
except ImportError as exc:
    raise RuntimeError(
        "This cell requires requests. Install it with:\n\n" "    %pip install requests\n"
    ) from exc


# ==========================================================
# CONFIGURATION
# ==========================================================

JOLPICA_YEAR = 2025

JOLPICA_BASE = "https://api.jolpi.ca/ergast/f1"

JOLPICA_USER_AGENT = (
    "F1Pantheon/2.1 " f"Python/{sys.version_info.major}." f"{sys.version_info.minor}"
)

JOLPICA_PAGE_LIMIT = 100


# ----------------------------------------------------------
# IMPORTANT CHANGE
#
# 8 seconds corresponds to at most ~450 live requests/hour,
# leaving some headroom below the sustained API budget.
#
# Cached pages do NOT sleep.
# ----------------------------------------------------------

MIN_LIVE_REQUEST_INTERVAL = 8.0


# Do NOT set this True unless you deliberately want to
# redownload everything.
FORCE_REFRESH_2025 = False


JOLPICA_CACHE_DIR = Path("jolpica_cache") / "2025"

JSON_CACHE_DIR = JOLPICA_CACHE_DIR / "json"

TABLE_CACHE_DIR = JOLPICA_CACHE_DIR / "ergast_compatible"


JSON_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TABLE_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


SNAPSHOT_PATHS = {
    "races": TABLE_CACHE_DIR / "races_2025.csv",
    "results": TABLE_CACHE_DIR / "results_2025.csv",
    "qualifying": TABLE_CACHE_DIR / "qualifying_2025.csv",
    "laps": TABLE_CACHE_DIR / "lap_times_2025.csv",
    "drivers": TABLE_CACHE_DIR / "new_drivers_2025.csv",
    "constructors": TABLE_CACHE_DIR / "new_constructors_2025.csv",
    "status": TABLE_CACHE_DIR / "new_status_2025.csv",
}


# ==========================================================
# HTTP SESSION
# ==========================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent": JOLPICA_USER_AGENT,
    }
)


_last_live_request = 0.0


# ==========================================================
# JSON CACHE
# ==========================================================


def _cache_path(
    endpoint,
    offset,
    limit,
):

    identity = f"{endpoint}|" f"{offset}|" f"{limit}"

    digest = hashlib.sha1(identity.encode("utf-8")).hexdigest()[:12]

    readable = endpoint.strip("/").replace("/", "__")

    return JSON_CACHE_DIR / (f"{readable}" f"__offset_{offset}" f"__{digest}.json")


# ==========================================================
# RATE-LIMIT HELPERS
# ==========================================================


def _parse_retry_after(
    response,
):
    """
    Return Retry-After in seconds if supplied.
    """

    value = response.headers.get("Retry-After")

    if value is None:
        return None

    try:
        return max(
            float(value),
            1.0,
        )

    except (
        TypeError,
        ValueError,
    ):
        return None


def _parse_rate_limit_reset(
    response,
):
    """
    Try common reset headers.

    Some APIs return a Unix epoch; others return seconds
    remaining. We handle either interpretation conservatively.
    """

    possible_headers = [
        "X-RateLimit-Reset",
        "RateLimit-Reset",
        "X-Ratelimit-Reset",
    ]

    now = time.time()

    for header in possible_headers:

        value = response.headers.get(header)

        if value is None:
            continue

        try:

            x = float(value)

        except (
            TypeError,
            ValueError,
        ):
            continue

        # Looks like a Unix timestamp.
        if x > now:

            return max(
                x - now,
                1.0,
            )

        # Otherwise interpret a plausible value as seconds.
        if x > 0 and x < 86400:

            return x

    return None


def _rate_limit_delay(
    response,
    consecutive_429s,
):
    """
    Decide how long to wait after HTTP 429.

    Priority:
      1. Retry-After
      2. explicit rate-limit reset header
      3. conservative increasing fallback
    """

    retry_after = _parse_retry_after(response)

    if retry_after is not None:

        return retry_after + 1.0

    reset_delay = _parse_rate_limit_reset(response)

    if reset_delay is not None:

        return reset_delay + 2.0

    # Sustained-limit fallback.
    #
    # Avoid the previous behaviour of retrying every
    # two seconds while still inside the same quota window.
    fallback = min(
        600.0,
        60.0 * consecutive_429s,
    )

    return fallback


# ==========================================================
# CACHED + RATE-LIMITED REQUEST
# ==========================================================


def jolpica_get(
    endpoint,
    *,
    offset=0,
    limit=JOLPICA_PAGE_LIMIT,
):

    global _last_live_request

    endpoint = endpoint.strip("/")

    cache_path = _cache_path(
        endpoint,
        offset,
        limit,
    )

    # ------------------------------------------------------
    # CACHE HIT
    # ------------------------------------------------------

    if cache_path.exists() and not FORCE_REFRESH_2025:

        with cache_path.open(
            "r",
            encoding="utf-8",
        ) as f:

            return json.load(f)

    url = f"{JOLPICA_BASE}/" f"{endpoint}/"

    params = {
        "limit": int(limit),
        "offset": int(offset),
    }

    consecutive_429s = 0
    other_failures = 0

    # ------------------------------------------------------
    # Keep trying until a page succeeds.
    #
    # Ctrl+C remains safe because completed pages are cached.
    # ------------------------------------------------------

    while True:

        elapsed = time.monotonic() - _last_live_request

        if elapsed < MIN_LIVE_REQUEST_INTERVAL:

            time.sleep(MIN_LIVE_REQUEST_INTERVAL - elapsed)

        try:

            response = session.get(
                url,
                params=params,
                timeout=60,
            )

            _last_live_request = time.monotonic()

            # ==================================================
            # RATE LIMITED
            # ==================================================

            if response.status_code == 429:

                consecutive_429s += 1

                delay = _rate_limit_delay(
                    response,
                    consecutive_429s,
                )

                print("\nJolpica rate limit hit.")

                print(
                    "  endpoint:",
                    endpoint,
                )

                print(
                    "  offset:",
                    offset,
                )

                print(
                    "  consecutive 429s:",
                    consecutive_429s,
                )

                print(
                    "  waiting:",
                    f"{delay:.1f}s",
                )

                # Useful diagnostic if Jolpica exposes quota
                # metadata.
                relevant_headers = {
                    k: v
                    for k, v in response.headers.items()
                    if ("rate" in k.lower() or "retry" in k.lower())
                }

                if relevant_headers:

                    print(
                        "  rate-limit headers:",
                        relevant_headers,
                    )

                time.sleep(delay)

                continue

            # Success or different failure resets 429 streak.
            consecutive_429s = 0

            # ==================================================
            # TEMPORARY SERVER ERROR
            # ==================================================

            if 500 <= response.status_code < 600:

                other_failures += 1

                delay = min(
                    120.0,
                    2.0
                    ** min(
                        other_failures,
                        6,
                    ),
                )

                print("\nTemporary Jolpica server error " f"{response.status_code}.")

                print(f"Retrying after {delay:.1f}s")

                time.sleep(delay)

                continue

            response.raise_for_status()

            payload = response.json()

            # ==================================================
            # SAVE SUCCESSFUL PAGE IMMEDIATELY
            # ==================================================

            with cache_path.open(
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    payload,
                    f,
                )

            return payload

        except KeyboardInterrupt:

            print("\nDownload interrupted.")

            print("All previously completed pages remain cached.")

            raise

        except (
            requests.RequestException,
            ValueError,
        ) as exc:

            other_failures += 1

            if other_failures > 12:

                raise RuntimeError(
                    "Repeated Jolpica request failure for " f"{endpoint}, offset={offset}"
                ) from exc

            delay = min(
                120.0,
                2.0
                ** min(
                    other_failures,
                    6,
                ),
            )

            print(
                "\nRequest failed:",
                repr(exc),
            )

            print(f"Retrying after {delay:.1f}s")

            time.sleep(delay)


# ==========================================================
# PAGINATED ENDPOINT
# ==========================================================


def jolpica_pages(
    endpoint,
):
    """
    Retrieve every page for an endpoint.

    Cached pages return immediately.

    If the run is interrupted, restarting this cell resumes
    from the first uncached page.
    """

    pages = []

    offset = 0

    while True:

        payload = jolpica_get(
            endpoint,
            offset=offset,
            limit=JOLPICA_PAGE_LIMIT,
        )

        pages.append(payload)

        mr = payload["MRData"]

        total = int(
            mr.get(
                "total",
                0,
            )
        )

        returned_limit = int(
            mr.get(
                "limit",
                JOLPICA_PAGE_LIMIT,
            )
        )

        returned_limit = max(
            returned_limit,
            1,
        )

        if offset + returned_limit >= total:

            break

        offset += returned_limit

    return pages


# ==========================================================
# GENERAL HELPERS
# ==========================================================


def finite_int(
    value,
    default=None,
):

    try:

        if value is None or value == "":
            return default

        return int(float(value))

    except (
        TypeError,
        ValueError,
    ):

        return default


def lap_time_to_ms(
    time_string,
):

    if (
        not isinstance(
            time_string,
            str,
        )
        or not time_string.strip()
    ):

        return np.nan

    try:

        parts = [float(x) for x in (time_string.strip().split(":"))]

        if len(parts) == 2:

            seconds = 60.0 * parts[0] + parts[1]

        elif len(parts) == 3:

            seconds = 3600.0 * parts[0] + 60.0 * parts[1] + parts[2]

        else:

            return np.nan

        return int(round(seconds * 1000.0))

    except ValueError:

        return np.nan


def max_numeric_id(
    df,
    column,
):

    if column not in df.columns or len(df) == 0:

        return 0

    x = pd.to_numeric(
        df[column],
        errors="coerce",
    )

    if not x.notna().any():

        return 0

    return int(x.max())


def align_columns(
    rows,
    reference_df,
):

    df = pd.DataFrame(rows)

    for column in reference_df.columns:

        if column not in (df.columns):

            df[column] = np.nan

    return df[reference_df.columns].copy()


def append_unique(
    base,
    addition,
    subset,
):

    if len(addition) == 0:

        return base.copy()

    out = pd.concat(
        [
            base,
            addition,
        ],
        ignore_index=True,
        sort=False,
    )

    out = out.drop_duplicates(
        subset=subset,
        keep="first",
    ).reset_index(drop=True)

    return out


# ==========================================================
# LOAD COMPLETE CONVERTED SNAPSHOT IF ALREADY AVAILABLE
# ==========================================================

have_snapshot = all(path.exists() for path in SNAPSHOT_PATHS.values())


if have_snapshot and not FORCE_REFRESH_2025:

    print("Loading cached Ergast-compatible " "2025 snapshot.")

    new_races = pd.read_csv(SNAPSHOT_PATHS["races"])

    new_results = pd.read_csv(SNAPSHOT_PATHS["results"])

    new_qualifying = pd.read_csv(SNAPSHOT_PATHS["qualifying"])

    new_laps = pd.read_csv(SNAPSHOT_PATHS["laps"])

    new_drivers = pd.read_csv(SNAPSHOT_PATHS["drivers"])

    new_constructors = pd.read_csv(SNAPSHOT_PATHS["constructors"])

    new_status = pd.read_csv(SNAPSHOT_PATHS["status"])


else:

    # ======================================================
    # 2025 RACE SCHEDULE
    # ======================================================

    print("Downloading 2025 race schedule...")

    race_pages = jolpica_pages(f"{JOLPICA_YEAR}/races")

    api_races = []

    for page in race_pages:

        api_races.extend(
            page["MRData"]["RaceTable"].get(
                "Races",
                [],
            )
        )

    race_by_round = {}

    for race in api_races:

        rr = finite_int(race.get("round"))

        if rr is not None:

            race_by_round[rr] = race

    rounds = sorted(race_by_round)

    if not rounds:

        raise RuntimeError("Jolpica returned no 2025 races.")

    print(
        "2025 rounds returned:",
        len(rounds),
    )

    # ======================================================
    # LOCAL RACE IDs
    # ======================================================

    next_race_id = (
        max_numeric_id(
            raw.races,
            "raceId",
        )
        + 1
    )

    race_id_by_round = {rr: next_race_id + i for i, rr in enumerate(rounds)}

    # ======================================================
    # CIRCUIT IDs
    # ======================================================

    circuit_ref_to_id = {}

    circuits_path = Path(CFG.data_dir) / "circuits.csv"

    if circuits_path.exists():

        circuits_local = pd.read_csv(circuits_path)

        if {
            "circuitId",
            "circuitRef",
        }.issubset(circuits_local.columns):

            circuit_ref_to_id = {
                str(ref): int(cid)
                for cid, ref in zip(
                    circuits_local["circuitId"],
                    circuits_local["circuitRef"],
                )
            }

    # ======================================================
    # 2025 RESULTS
    # ======================================================

    print("Downloading 2025 race results...")

    result_pages = jolpica_pages(f"{JOLPICA_YEAR}/results")

    api_results = []

    driver_meta = {}

    constructor_meta = {}

    def remember_driver(
        driver,
    ):

        if not driver:

            return

        ref = driver.get("driverId")

        if ref:

            driver_meta[str(ref)] = driver

    def remember_constructor(
        constructor,
    ):

        if not constructor:

            return

        ref = constructor.get("constructorId")

        if ref:

            constructor_meta[str(ref)] = constructor

    for page in result_pages:

        races_here = page["MRData"]["RaceTable"].get(
            "Races",
            [],
        )

        for race in races_here:

            rr = finite_int(race.get("round"))

            if rr not in (race_id_by_round):

                continue

            for result in race.get(
                "Results",
                [],
            ):

                driver = result.get(
                    "Driver",
                    {},
                )

                constructor = result.get(
                    "Constructor",
                    {},
                )

                remember_driver(driver)

                remember_constructor(constructor)

                api_results.append(
                    {
                        "round": rr,
                        "driverRef": str(driver.get("driverId")),
                        "constructorRef": str(constructor.get("constructorId")),
                        "number": result.get("number"),
                        "position": result.get("position"),
                        "positionText": result.get("positionText"),
                        "points": result.get("points"),
                        "grid": result.get("grid"),
                        "laps": result.get("laps"),
                        "status": result.get(
                            "status",
                            "",
                        ),
                        "time": (
                            result.get(
                                "Time",
                                {},
                            ).get("time")
                        ),
                        "milliseconds": (
                            result.get(
                                "Time",
                                {},
                            ).get("millis")
                        ),
                        "fastestLap": (
                            result.get(
                                "FastestLap",
                                {},
                            ).get("lap")
                        ),
                        "rank": (
                            result.get(
                                "FastestLap",
                                {},
                            ).get("rank")
                        ),
                        "fastestLapTime": (
                            result.get(
                                "FastestLap",
                                {},
                            )
                            .get(
                                "Time",
                                {},
                            )
                            .get("time")
                        ),
                        "fastestLapSpeed": (
                            result.get(
                                "FastestLap",
                                {},
                            )
                            .get(
                                "AverageSpeed",
                                {},
                            )
                            .get("speed")
                        ),
                    }
                )

    api_results = (
        pd.DataFrame(api_results)
        .drop_duplicates(
            subset=[
                "round",
                "driverRef",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )

    # ======================================================
    # 2025 QUALIFYING
    # ======================================================

    print("Downloading 2025 qualifying...")

    qualifying_pages = jolpica_pages(f"{JOLPICA_YEAR}/qualifying")

    api_qualifying = []

    for page in qualifying_pages:

        races_here = page["MRData"]["RaceTable"].get(
            "Races",
            [],
        )

        for race in races_here:

            rr = finite_int(race.get("round"))

            if rr not in (race_id_by_round):

                continue

            for result in race.get(
                "QualifyingResults",
                [],
            ):

                driver = result.get(
                    "Driver",
                    {},
                )

                constructor = result.get(
                    "Constructor",
                    {},
                )

                remember_driver(driver)

                remember_constructor(constructor)

                api_qualifying.append(
                    {
                        "round": rr,
                        "driverRef": str(driver.get("driverId")),
                        "constructorRef": str(constructor.get("constructorId")),
                        "number": result.get("number"),
                        "position": result.get("position"),
                        "q1": result.get("Q1"),
                        "q2": result.get("Q2"),
                        "q3": result.get("Q3"),
                    }
                )

    api_qualifying = (
        pd.DataFrame(api_qualifying)
        .drop_duplicates(
            subset=[
                "round",
                "driverRef",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )

    # ======================================================
    # 2025 LAP TIMES
    # ======================================================

    print("Downloading 2025 race lap timing...")

    api_laps = []

    for i, rr in enumerate(
        rounds,
        start=1,
    ):

        race_name = race_by_round[rr].get(
            "raceName",
            f"Round {rr}",
        )

        print(f"  [{i:02d}/{len(rounds):02d}] " f"Round {rr}: {race_name}")

        lap_pages = jolpica_pages(f"{JOLPICA_YEAR}/{rr}/laps")

        for page in lap_pages:

            races_here = page["MRData"]["RaceTable"].get(
                "Races",
                [],
            )

            for race in races_here:

                for lap_object in race.get(
                    "Laps",
                    [],
                ):

                    lap_number = finite_int(lap_object.get("number"))

                    for timing in lap_object.get(
                        "Timings",
                        [],
                    ):

                        driver_ref = str(timing.get("driverId"))

                        api_laps.append(
                            {
                                "round": rr,
                                "lap": lap_number,
                                "driverRef": driver_ref,
                                "position": timing.get("position"),
                                "time": timing.get("time"),
                            }
                        )

    api_laps = (
        pd.DataFrame(api_laps)
        .drop_duplicates(
            subset=[
                "round",
                "lap",
                "driverRef",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )

    # ======================================================
    # EXISTING DRIVER IDS
    # ======================================================

    existing_driver_ref_to_id = {
        str(ref): int(driver_id)
        for driver_id, ref in zip(
            raw.drivers["driverId"],
            raw.drivers["driverRef"],
        )
    }

    # ======================================================
    # EXISTING CONSTRUCTOR IDS
    # ======================================================

    if not {
        "constructorId",
        "constructorRef",
    }.issubset(raw.constructors.columns):

        raise RuntimeError(
            "constructors.csv with constructorRef is "
            "required to bridge 2024 -> 2025 constructor "
            "identities safely."
        )

    existing_constructor_ref_to_id = {
        str(ref): int(constructor_id)
        for constructor_id, ref in zip(
            raw.constructors["constructorId"],
            raw.constructors["constructorRef"],
        )
    }

    # ======================================================
    # ALL 2025 DRIVER REFS
    # ======================================================

    api_driver_refs = set(api_results["driverRef"].dropna())

    api_driver_refs.update(api_qualifying["driverRef"].dropna())

    api_driver_refs.update(api_laps["driverRef"].dropna())

    # ======================================================
    # ALL 2025 CONSTRUCTOR REFS
    # ======================================================

    api_constructor_refs = set(api_results["constructorRef"].dropna())

    api_constructor_refs.update(api_qualifying["constructorRef"].dropna())

    # ======================================================
    # CONSTRUCTOR CONTINUITY
    # ======================================================

    unknown_constructor_refs = sorted(
        api_constructor_refs - set(existing_constructor_ref_to_id)
    )

    print(
        "\n2025 constructor refs:",
        sorted(api_constructor_refs),
    )

    print(
        "Constructor refs not present in " "the 2024 archive:",
        unknown_constructor_refs,
    )

    if unknown_constructor_refs:

        raise RuntimeError(
            "Constructor identity mismatch detected. "
            "Do NOT continue to Stage 3. "
            "Paste the constructor-ref output here."
        )

    constructor_ref_to_id = existing_constructor_ref_to_id.copy()

    # ======================================================
    # NEW DRIVERS
    # ======================================================

    driver_ref_to_id = existing_driver_ref_to_id.copy()

    new_driver_refs = sorted(api_driver_refs - set(existing_driver_ref_to_id))

    next_driver_id = (
        max_numeric_id(
            raw.drivers,
            "driverId",
        )
        + 1
    )

    for i, ref in enumerate(new_driver_refs):

        driver_ref_to_id[ref] = next_driver_id + i

    print(
        "\nNew 2025 driver refs:",
        new_driver_refs,
    )

    # ======================================================
    # STATUS IDS
    # ======================================================

    existing_status_text_to_id = {
        str(status).strip().casefold(): int(status_id)
        for status_id, status in zip(
            raw.status["statusId"],
            raw.status["status"],
        )
    }

    status_text_to_id = existing_status_text_to_id.copy()

    statuses_2025 = sorted(
        {str(x).strip() for x in api_results["status"].dropna() if str(x).strip()}
    )

    unseen_statuses = [
        status_text
        for status_text in statuses_2025
        if (status_text.casefold() not in status_text_to_id)
    ]

    next_status_id = (
        max_numeric_id(
            raw.status,
            "statusId",
        )
        + 1
    )

    for i, status_text in enumerate(unseen_statuses):

        status_text_to_id[status_text.casefold()] = next_status_id + i

    # ======================================================
    # RACES TABLE
    # ======================================================

    race_rows = []

    for rr in rounds:

        race = race_by_round[rr]

        circuit = race.get(
            "Circuit",
            {},
        )

        circuit_ref = circuit.get("circuitId")

        fp1 = race.get(
            "FirstPractice",
            {},
        )

        fp2 = race.get(
            "SecondPractice",
            {},
        )

        fp3 = race.get(
            "ThirdPractice",
            {},
        )

        qualifying_sched = race.get(
            "Qualifying",
            {},
        )

        sprint = race.get(
            "Sprint",
            {},
        )

        race_rows.append(
            {
                "raceId": race_id_by_round[rr],
                "year": JOLPICA_YEAR,
                "round": rr,
                "circuitId": (
                    circuit_ref_to_id.get(str(circuit_ref))
                    if (circuit_ref is not None)
                    else np.nan
                ),
                "name": race.get("raceName"),
                "date": race.get("date"),
                "time": race.get("time"),
                "url": race.get("url"),
                "fp1_date": fp1.get("date"),
                "fp1_time": fp1.get("time"),
                "fp2_date": fp2.get("date"),
                "fp2_time": fp2.get("time"),
                "fp3_date": fp3.get("date"),
                "fp3_time": fp3.get("time"),
                "quali_date": qualifying_sched.get("date"),
                "quali_time": qualifying_sched.get("time"),
                "sprint_date": sprint.get("date"),
                "sprint_time": sprint.get("time"),
            }
        )

    new_races = align_columns(
        race_rows,
        raw.races,
    )

    # ======================================================
    # NEW DRIVER TABLE
    # ======================================================

    driver_rows = []

    for ref in new_driver_refs:

        meta = driver_meta.get(
            ref,
            {},
        )

        driver_rows.append(
            {
                "driverId": driver_ref_to_id[ref],
                "driverRef": ref,
                "number": meta.get("permanentNumber"),
                "code": meta.get("code"),
                "forename": meta.get("givenName"),
                "surname": meta.get("familyName"),
                "dob": meta.get("dateOfBirth"),
                "nationality": meta.get("nationality"),
                "url": meta.get("url"),
            }
        )

    new_drivers = align_columns(
        driver_rows,
        raw.drivers,
    )

    # No new constructor identity is expected for 2025.
    new_constructors = align_columns(
        [],
        raw.constructors,
    )

    # ======================================================
    # NEW STATUS TABLE
    # ======================================================

    status_rows = []

    for status_text in unseen_statuses:

        status_rows.append(
            {
                "statusId": status_text_to_id[status_text.casefold()],
                "status": status_text,
            }
        )

    new_status = align_columns(
        status_rows,
        raw.status,
    )

    # ======================================================
    # RESULTS TABLE
    # ======================================================

    next_result_id = (
        max_numeric_id(
            raw.results,
            "resultId",
        )
        + 1
    )

    result_rows = []

    for row_number, r in enumerate(api_results.itertuples(index=False)):

        rr = int(r.round)

        position_order = finite_int(r.position)

        result_rows.append(
            {
                "resultId": next_result_id + row_number,
                "raceId": race_id_by_round[rr],
                "driverId": driver_ref_to_id[r.driverRef],
                "constructorId": constructor_ref_to_id[r.constructorRef],
                "number": finite_int(r.number),
                "grid": finite_int(
                    r.grid,
                    0,
                ),
                "position": position_order,
                "positionText": r.positionText,
                "positionOrder": position_order,
                "points": pd.to_numeric(
                    r.points,
                    errors="coerce",
                ),
                "laps": finite_int(
                    r.laps,
                    0,
                ),
                "time": r.time,
                "milliseconds": pd.to_numeric(
                    r.milliseconds,
                    errors="coerce",
                ),
                "fastestLap": finite_int(r.fastestLap),
                "rank": finite_int(r.rank),
                "fastestLapTime": r.fastestLapTime,
                "fastestLapSpeed": pd.to_numeric(
                    r.fastestLapSpeed,
                    errors="coerce",
                ),
                "statusId": status_text_to_id[str(r.status).strip().casefold()],
            }
        )

    new_results = align_columns(
        result_rows,
        raw.results,
    )

    # ======================================================
    # QUALIFYING TABLE
    # ======================================================

    next_qualify_id = (
        max_numeric_id(
            raw.qualifying,
            "qualifyId",
        )
        + 1
    )

    qualifying_rows = []

    for row_number, q in enumerate(api_qualifying.itertuples(index=False)):

        qualifying_rows.append(
            {
                "qualifyId": next_qualify_id + row_number,
                "raceId": race_id_by_round[int(q.round)],
                "driverId": driver_ref_to_id[q.driverRef],
                "constructorId": constructor_ref_to_id[q.constructorRef],
                "number": finite_int(q.number),
                "position": finite_int(q.position),
                "q1": q.q1,
                "q2": q.q2,
                "q3": q.q3,
            }
        )

    new_qualifying = align_columns(
        qualifying_rows,
        raw.qualifying,
    )

    # ======================================================
    # LAP TIMES TABLE
    # ======================================================

    lap_rows = []

    for l in api_laps.itertuples(index=False):

        if l.driverRef not in driver_ref_to_id:

            raise RuntimeError("Lap data contains unknown driverRef: " f"{l.driverRef}")

        lap_rows.append(
            {
                "raceId": race_id_by_round[int(l.round)],
                "driverId": driver_ref_to_id[l.driverRef],
                "lap": finite_int(l.lap),
                "position": finite_int(l.position),
                "time": l.time,
                "milliseconds": lap_time_to_ms(l.time),
            }
        )

    new_laps = align_columns(
        lap_rows,
        raw.laps,
    )

    # ======================================================
    # SAVE COMPLETE CONVERTED 2025 SNAPSHOT
    # ======================================================

    snapshot_frames = {
        "races": new_races,
        "results": new_results,
        "qualifying": new_qualifying,
        "laps": new_laps,
        "drivers": new_drivers,
        "constructors": new_constructors,
        "status": new_status,
    }

    for name, df in snapshot_frames.items():

        df.to_csv(
            SNAPSHOT_PATHS[name],
            index=False,
        )

    print(
        "\nSaved converted 2025 snapshot ->",
        TABLE_CACHE_DIR,
    )


# ==========================================================
# APPEND 2025 TABLES TO RAW IN MEMORY
# ==========================================================

raw.drivers = append_unique(
    raw.drivers,
    new_drivers,
    subset=["driverId"],
)


raw.constructors = append_unique(
    raw.constructors,
    new_constructors,
    subset=["constructorId"],
)


raw.status = append_unique(
    raw.status,
    new_status,
    subset=["statusId"],
)


raw.races = append_unique(
    raw.races,
    new_races,
    subset=["raceId"],
)


raw.results = append_unique(
    raw.results,
    new_results,
    subset=[
        "raceId",
        "driverId",
        "constructorId",
    ],
)


raw.qualifying = append_unique(
    raw.qualifying,
    new_qualifying,
    subset=[
        "raceId",
        "driverId",
        "constructorId",
    ],
)


raw.laps = append_unique(
    raw.laps,
    new_laps,
    subset=[
        "raceId",
        "driverId",
        "lap",
    ],
)


# ==========================================================
# REBUILD RAW LOOKUP DICTIONARIES
# ==========================================================

raw.driver_ref = dict(
    zip(
        pd.to_numeric(raw.drivers["driverId"]).astype(int),
        raw.drivers["driverRef"].astype(str),
    )
)


raw.driver_id = dict(
    zip(
        raw.drivers["driverRef"].astype(str),
        pd.to_numeric(raw.drivers["driverId"]).astype(int),
    )
)


raw.constructor_ref = dict(
    zip(
        pd.to_numeric(raw.constructors["constructorId"]).astype(int),
        raw.constructors["constructorRef"].astype(str),
    )
)


raw.status_map = dict(
    zip(
        pd.to_numeric(raw.status["statusId"]).astype(int),
        raw.status["status"].astype(str),
    )
)


# ==========================================================
# FINAL 2025 COVERAGE AUDIT
# ==========================================================

races_augmented_2025 = raw.races[
    pd.to_numeric(
        raw.races["year"],
        errors="coerce",
    )
    == JOLPICA_YEAR
].copy()


race_ids_augmented_2025 = set(
    pd.to_numeric(
        races_augmented_2025["raceId"],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)


results_augmented_2025 = raw.results[
    pd.to_numeric(
        raw.results["raceId"],
        errors="coerce",
    ).isin(race_ids_augmented_2025)
]


qualifying_augmented_2025 = raw.qualifying[
    pd.to_numeric(
        raw.qualifying["raceId"],
        errors="coerce",
    ).isin(race_ids_augmented_2025)
]


laps_augmented_2025 = raw.laps[
    pd.to_numeric(
        raw.laps["raceId"],
        errors="coerce",
    ).isin(race_ids_augmented_2025)
]


coverage_2025 = pd.DataFrame(
    {
        "table": [
            "races",
            "results",
            "qualifying",
            "lap_times",
        ],
        "rows_2025": [
            len(races_augmented_2025),
            len(results_augmented_2025),
            len(qualifying_augmented_2025),
            len(laps_augmented_2025),
        ],
        "races_covered": [
            races_augmented_2025["raceId"].nunique(),
            results_augmented_2025["raceId"].nunique(),
            qualifying_augmented_2025["raceId"].nunique(),
            laps_augmented_2025["raceId"].nunique(),
        ],
    }
)


print("\n========================================")

print("AUGMENTED 2025 DATA COVERAGE")

print("========================================")


display(coverage_2025)


# ==========================================================
# RACE-LEVEL COVERAGE
# ==========================================================

race_coverage = races_augmented_2025[
    [
        "raceId",
        "round",
        "name",
    ]
].copy()


race_coverage["has_results"] = race_coverage["raceId"].isin(results_augmented_2025["raceId"])


race_coverage["has_qualifying"] = race_coverage["raceId"].isin(
    qualifying_augmented_2025["raceId"]
)


race_coverage["has_laps"] = race_coverage["raceId"].isin(laps_augmented_2025["raceId"])


print("\nRace-level 2025 coverage:")


display(race_coverage.sort_values("round"))


# ==========================================================
# 2025 DRIVERS
# ==========================================================

driver_ids_2025 = set(
    pd.to_numeric(
        results_augmented_2025["driverId"],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)


drivers_2025_table = raw.drivers[
    pd.to_numeric(
        raw.drivers["driverId"],
        errors="coerce",
    ).isin(driver_ids_2025)
][
    [
        col
        for col in [
            "driverId",
            "driverRef",
            "forename",
            "surname",
        ]
        if col in raw.drivers.columns
    ]
].sort_values(
    "driverRef"
)


print("\n2025 drivers:")


display(drivers_2025_table)


# ==========================================================
# 2025 CONSTRUCTORS
# ==========================================================

constructor_ids_2025 = set(
    pd.to_numeric(
        results_augmented_2025["constructorId"],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)


constructors_2025_table = raw.constructors[
    pd.to_numeric(
        raw.constructors["constructorId"],
        errors="coerce",
    ).isin(constructor_ids_2025)
][
    [
        col
        for col in [
            "constructorId",
            "constructorRef",
            "name",
        ]
        if col in raw.constructors.columns
    ]
].sort_values(
    "constructorRef"
)


print("\n2025 constructors:")


display(constructors_2025_table)


# ==========================================================
# HARD VALIDATION
# ==========================================================

expected_races = races_augmented_2025["raceId"].nunique()


for label, frame in [
    (
        "results",
        results_augmented_2025,
    ),
    (
        "qualifying",
        qualifying_augmented_2025,
    ),
    (
        "lap_times",
        laps_augmented_2025,
    ),
]:

    n_covered = frame["raceId"].nunique()

    if n_covered != expected_races:

        raise RuntimeError(
            f"2025 {label} covers "
            f"{n_covered}/{expected_races} races. "
            "Do NOT continue to Stage 3."
        )


if not (
    race_coverage[
        [
            "has_results",
            "has_qualifying",
            "has_laps",
        ]
    ]
    .all()
    .all()
):

    raise RuntimeError("At least one 2025 race is missing " "results/qualifying/lap coverage.")


print("\n2025 augmentation PASSED.")


print(
    "Maximum year now available:",
    pd.to_numeric(
        raw.races["year"],
        errors="coerce",
    ).max(),
)


print("\nOriginal ergast_data CSV files " "have NOT been modified.")

In [ ]:
# ==========================================================
# STAGE 2C — PATCH 2025 MIAMI QUALIFYING TIMES FROM FIA
# ==========================================================
#
# Jolpica contains the 20 Miami qualifying rows but the
# Q1/Q2/Q3 timing fields are missing.
#
# Source:
# FIA 2025 Miami GP Provisional Qualifying Classification
# Document 58, 03 May 2025.
#
# Only q1/q2/q3 are patched. Driver/constructor/race IDs
# remain exactly as already imported.
# ==========================================================

MIAMI_YEAR = 2025
MIAMI_ROUND = 6


# ==========================================================
# 1. FIND LOCAL MIAMI RACE ID
# ==========================================================

miami_race_rows = raw.races[
    (
        pd.to_numeric(
            raw.races["year"],
            errors="coerce",
        )
        == MIAMI_YEAR
    )
    & (
        pd.to_numeric(
            raw.races["round"],
            errors="coerce",
        )
        == MIAMI_ROUND
    )
].copy()


if len(miami_race_rows) != 1:

    raise RuntimeError(
        "Expected exactly one 2025 Miami race row, " f"found {len(miami_race_rows)}."
    )


MIAMI_RACE_ID = int(miami_race_rows["raceId"].iloc[0])


print(
    "Miami raceId:",
    MIAMI_RACE_ID,
)


# ==========================================================
# 2. FIA QUALIFYING TIMES
# ==========================================================

miami_fia_times = {
    "max_verstappen": {
        "q1": "1:26.870",
        "q2": "1:26.643",
        "q3": "1:26.204",
    },
    "norris": {
        "q1": "1:26.955",
        "q2": "1:26.499",
        "q3": "1:26.269",
    },
    "antonelli": {
        "q1": "1:27.077",
        "q2": "1:26.606",
        "q3": "1:26.271",
    },
    "piastri": {
        "q1": "1:27.006",
        "q2": "1:26.269",
        "q3": "1:26.375",
    },
    "russell": {
        "q1": "1:27.014",
        "q2": "1:26.575",
        "q3": "1:26.385",
    },
    "sainz": {
        "q1": "1:27.098",
        "q2": "1:26.847",
        "q3": "1:26.569",
    },
    "albon": {
        "q1": "1:27.042",
        "q2": "1:26.855",
        "q3": "1:26.682",
    },
    "leclerc": {
        "q1": "1:27.417",
        "q2": "1:26.948",
        "q3": "1:26.754",
    },
    "ocon": {
        "q1": "1:27.450",
        "q2": "1:26.967",
        "q3": "1:26.824",
    },
    "tsunoda": {
        "q1": "1:27.298",
        "q2": "1:26.959",
        "q3": "1:26.943",
    },
    "hadjar": {
        "q1": "1:27.301",
        "q2": "1:26.987",
        "q3": np.nan,
    },
    "hamilton": {
        "q1": "1:27.279",
        "q2": "1:27.006",
        "q3": np.nan,
    },
    "bortoleto": {
        "q1": "1:27.343",
        "q2": "1:27.151",
        "q3": np.nan,
    },
    "doohan": {
        "q1": "1:27.422",
        "q2": "1:27.186",
        "q3": np.nan,
    },
    "lawson": {
        "q1": "1:27.444",
        "q2": "1:27.363",
        "q3": np.nan,
    },
    "hulkenberg": {
        "q1": "1:27.473",
        "q2": np.nan,
        "q3": np.nan,
    },
    "alonso": {
        "q1": "1:27.604",
        "q2": np.nan,
        "q3": np.nan,
    },
    "gasly": {
        "q1": "1:27.710",
        "q2": np.nan,
        "q3": np.nan,
    },
    "stroll": {
        "q1": "1:27.830",
        "q2": np.nan,
        "q3": np.nan,
    },
    "bearman": {
        "q1": "1:27.999",
        "q2": np.nan,
        "q3": np.nan,
    },
}


# ==========================================================
# 3. MAP DRIVER REFS -> DRIVER IDs
# ==========================================================

driver_ref_to_id = dict(
    zip(
        raw.drivers["driverRef"].astype(str),
        pd.to_numeric(
            raw.drivers["driverId"],
            errors="coerce",
        ).astype(int),
    )
)


missing_driver_refs = sorted(set(miami_fia_times) - set(driver_ref_to_id))


if missing_driver_refs:

    raise RuntimeError(
        "Miami FIA patch contains unknown driver refs: " + ", ".join(missing_driver_refs)
    )


# ==========================================================
# 4. SHOW CURRENT MIAMI QUALIFYING BEFORE PATCH
# ==========================================================

miami_before = raw.qualifying[raw.qualifying["raceId"] == MIAMI_RACE_ID].copy()


miami_before["driver"] = miami_before["driverId"].map(raw.driver_ref)


print("\nMiami qualifying BEFORE patch:")


display(
    miami_before[
        [
            "driver",
            "q1",
            "q2",
            "q3",
        ]
    ].sort_values("driver")
)


# ==========================================================
# 5. APPLY PATCH
# ==========================================================

patched_count = 0


for driver_ref, times in miami_fia_times.items():

    driver_id = driver_ref_to_id[driver_ref]

    mask = (raw.qualifying["raceId"] == MIAMI_RACE_ID) & (
        raw.qualifying["driverId"] == driver_id
    )

    n_rows = int(mask.sum())

    if n_rows != 1:

        raise RuntimeError(
            f"Expected one Miami qualifying row for " f"{driver_ref}; found {n_rows}."
        )

    raw.qualifying.loc[
        mask,
        "q1",
    ] = times["q1"]

    raw.qualifying.loc[
        mask,
        "q2",
    ] = times["q2"]

    raw.qualifying.loc[
        mask,
        "q3",
    ] = times["q3"]

    patched_count += 1


print(
    "\nPatched Miami drivers:",
    patched_count,
)


# ==========================================================
# 6. VERIFY PATCH
# ==========================================================

miami_after = raw.qualifying[raw.qualifying["raceId"] == MIAMI_RACE_ID].copy()


miami_after["driver"] = miami_after["driverId"].map(raw.driver_ref)


print("\nMiami qualifying AFTER patch:")


display(
    miami_after[
        [
            "position",
            "driver",
            "q1",
            "q2",
            "q3",
        ]
    ].sort_values("position")
)


# Expected session participation:
#
# Q1 = 20
# Q2 = 15
# Q3 = 10

miami_counts = {
    "q1": int(miami_after["q1"].notna().sum()),
    "q2": int(miami_after["q2"].notna().sum()),
    "q3": int(miami_after["q3"].notna().sum()),
}


print(
    "\nMiami populated session counts:",
    miami_counts,
)


assert miami_counts["q1"] == 20

assert miami_counts["q2"] == 15

assert miami_counts["q3"] == 10


# ==========================================================
# 7. SAVE PATCHED 2025 QUALIFYING SNAPSHOT
#
# This updates only our separate Jolpica-derived cache,
# never the original ergast_data/qualifying.csv.
# ==========================================================

patched_2025_qualifying = raw.qualifying[
    raw.qualifying["raceId"].isin(races_augmented_2025["raceId"])
].copy()


patched_snapshot_path = TABLE_CACHE_DIR / "qualifying_2025.csv"


patched_2025_qualifying.to_csv(
    patched_snapshot_path,
    index=False,
)


print("\nSaved patched 2025 qualifying snapshot ->")

print(patched_snapshot_path)


print("\nMiami FIA qualifying patch PASSED.")

In [ ]:
# ==========================================================
# STAGE 2D / STAGE 32A — APPEND LIVE 2026 DATA SO FAR
# ==========================================================
#
# INSERT THIS CELL:
#   immediately AFTER the existing 2025 augmentation / Miami patch
#   and BEFORE Stage 3.
#
# PURPOSE
# -------
# Extend the already-augmented Ergast-format tables from 2025 into
# the CURRENT, INCOMPLETE 2026 season without touching the original
# ergast_data/*.csv files.
#
# Important differences from the old 2025 cell:
#
#   1. 2026 is still in progress, so the full schedule contains
#      FUTURE races. We therefore define the usable season from
#      completed Grand Prix RESULT rounds, not the schedule.
#
#   2. New constructor identities are allowed. This is necessary for
#      new/rebranded 2026 entries (rather than aborting when an unknown
#      constructorRef appears).
#
#   3. The mutable season endpoints (schedule/results/qualifying) are
#      refreshed every time this cell is run. Completed old lap pages
#      are cached, while the latest completed round is refreshed.
#
#   4. The cell is safe to rerun in the same kernel: any previously
#      appended 2026 event rows are stripped and rebuilt first.
#
#   5. At the end, CFG is changed to a SEPARATE live-2026 output
#      directory and ceiling_year=2026. The frozen 1982–2025 outputs
#      are never overwritten.
#
# DATA INCLUDED
# -------------
# Only Grand Prix weekends for which the API currently has:
#
#   - race results
#   - Grand Prix qualifying
#   - race lap timing
#
# Future scheduled races are ignored.
#
# ==========================================================

from dataclasses import replace
from pathlib import Path

import hashlib
import json
import sys
import time

import numpy as np
import pandas as pd

try:
    import requests
except ImportError as exc:
    raise RuntimeError(
        "This cell requires requests. Install it with:\n\n"
        "    %pip install requests"
    ) from exc


# ==========================================================
# 0. REQUIRE THE EXISTING 2025 AUGMENTATION ENVIRONMENT
# ==========================================================

required = [
    "CFG",
    "raw",
    "finite_int",
    "lap_time_to_ms",
    "max_numeric_id",
    "align_columns",
    "append_unique",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run Stage 2, Stage 2B and Stage 2C first. Missing:\n  "
        + "\n  ".join(missing)
    )


# ==========================================================
# 1. CONFIGURATION
# ==========================================================

LIVE_YEAR = 2026

JOLPICA_2026_BASE = "https://api.jolpi.ca/ergast/f1"

JOLPICA_2026_USER_AGENT = (
    "F1Pantheon/2.1-live-2026 "
    f"Python/{sys.version_info.major}.{sys.version_info.minor}"
)

JOLPICA_2026_PAGE_LIMIT = 100

# Same conservative sustained rate as the completed-2025 ingestion.
MIN_LIVE_REQUEST_INTERVAL_2026 = 8.0

JOLPICA_2026_CACHE_DIR = (
    Path("jolpica_cache")
    /
    "2026_live"
)

JSON_CACHE_DIR_2026 = (
    JOLPICA_2026_CACHE_DIR
    /
    "json"
)

TABLE_CACHE_DIR_2026 = (
    JOLPICA_2026_CACHE_DIR
    /
    "ergast_compatible"
)

JSON_CACHE_DIR_2026.mkdir(
    parents=True,
    exist_ok=True,
)

TABLE_CACHE_DIR_2026.mkdir(
    parents=True,
    exist_ok=True,
)

SNAPSHOT_PATHS_2026 = {
    "races":
        TABLE_CACHE_DIR_2026
        /
        "races_2026_live.csv",

    "results":
        TABLE_CACHE_DIR_2026
        /
        "results_2026_live.csv",

    "qualifying":
        TABLE_CACHE_DIR_2026
        /
        "qualifying_2026_live.csv",

    "laps":
        TABLE_CACHE_DIR_2026
        /
        "lap_times_2026_live.csv",

    "drivers":
        TABLE_CACHE_DIR_2026
        /
        "new_drivers_2026_live.csv",

    "constructors":
        TABLE_CACHE_DIR_2026
        /
        "new_constructors_2026_live.csv",

    "status":
        TABLE_CACHE_DIR_2026
        /
        "new_status_2026_live.csv",
}


# ==========================================================
# 2. HTTP SESSION + CACHE
# ==========================================================

session_2026 = requests.Session()

session_2026.headers.update(
    {
        "User-Agent":
            JOLPICA_2026_USER_AGENT,
    }
)

_last_live_request_2026 = 0.0


def _cache_path_2026(
    endpoint,
    offset,
    limit,
):
    identity = (
        f"{endpoint}|"
        f"{offset}|"
        f"{limit}"
    )

    digest = (
        hashlib.sha1(
            identity.encode(
                "utf-8"
            )
        )
        .hexdigest()[
            :12
        ]
    )

    readable = (
        endpoint
        .strip("/")
        .replace(
            "/",
            "__",
        )
    )

    return (
        JSON_CACHE_DIR_2026
        /
        (
            f"{readable}"
            f"__offset_{offset}"
            f"__{digest}.json"
        )
    )


def _retry_after_2026(
    response,
):
    value = response.headers.get(
        "Retry-After"
    )

    if value is not None:
        try:
            return max(
                float(
                    value
                ),
                1.0,
            )
        except (
            TypeError,
            ValueError,
        ):
            pass

    now = time.time()

    for header in [
        "X-RateLimit-Reset",
        "RateLimit-Reset",
        "X-Ratelimit-Reset",
    ]:
        value = response.headers.get(
            header
        )

        if value is None:
            continue

        try:
            x = float(
                value
            )
        except (
            TypeError,
            ValueError,
        ):
            continue

        if x > now:
            return max(
                x
                -
                now,
                1.0,
            )

        if 0 < x < 86400:
            return x

    return None


def jolpica_get_2026(
    endpoint,
    *,
    offset=0,
    limit=JOLPICA_2026_PAGE_LIMIT,
    force_live=False,
):
    """
    Cached, rate-limited request.

    force_live=True:
        bypass any cached copy and replace it with a fresh response.

    We use force_live for the season-level mutable endpoints and for
    the latest completed race's lap pages.
    """

    global _last_live_request_2026

    endpoint = endpoint.strip("/")

    cache_path = _cache_path_2026(
        endpoint,
        offset,
        limit,
    )

    if (
        cache_path.exists()
        and
        not force_live
    ):
        with cache_path.open(
            "r",
            encoding="utf-8",
        ) as f:
            return json.load(
                f
            )

    url = (
        f"{JOLPICA_2026_BASE}/"
        f"{endpoint}/"
    )

    params = {
        "limit":
            int(
                limit
            ),
        "offset":
            int(
                offset
            ),
    }

    consecutive_429s = 0
    other_failures = 0

    while True:

        elapsed = (
            time.monotonic()
            -
            _last_live_request_2026
        )

        if (
            elapsed
            <
            MIN_LIVE_REQUEST_INTERVAL_2026
        ):
            time.sleep(
                MIN_LIVE_REQUEST_INTERVAL_2026
                -
                elapsed
            )

        try:
            response = session_2026.get(
                url,
                params=
                    params,
                timeout=
                    60,
            )

            _last_live_request_2026 = (
                time.monotonic()
            )

            if response.status_code == 429:
                consecutive_429s += 1

                delay = _retry_after_2026(
                    response
                )

                if delay is None:
                    delay = min(
                        600.0,
                        60.0
                        *
                        consecutive_429s,
                    )

                delay += 1.0

                print(
                    "\nJolpica rate limit hit."
                )

                print(
                    "  endpoint:",
                    endpoint,
                )

                print(
                    "  offset:",
                    offset,
                )

                print(
                    "  waiting:",
                    f"{delay:.1f}s",
                )

                time.sleep(
                    delay
                )

                continue

            consecutive_429s = 0

            if (
                500
                <=
                response.status_code
                <
                600
            ):
                other_failures += 1

                delay = min(
                    120.0,
                    2.0
                    **
                    min(
                        other_failures,
                        6,
                    ),
                )

                print(
                    "\nTemporary Jolpica "
                    f"{response.status_code}."
                )

                print(
                    f"Retrying after {delay:.1f}s"
                )

                time.sleep(
                    delay
                )

                continue

            response.raise_for_status()

            payload = response.json()

            with cache_path.open(
                "w",
                encoding="utf-8",
            ) as f:
                json.dump(
                    payload,
                    f,
                )

            return payload

        except KeyboardInterrupt:
            print(
                "\nDownload interrupted."
            )

            print(
                "Completed JSON pages remain cached."
            )

            raise

        except (
            requests.RequestException,
            ValueError,
        ) as exc:
            other_failures += 1

            if other_failures > 12:
                raise RuntimeError(
                    "Repeated Jolpica failure for "
                    f"{endpoint}, offset={offset}"
                ) from exc

            delay = min(
                120.0,
                2.0
                **
                min(
                    other_failures,
                    6,
                ),
            )

            print(
                "\nRequest failed:",
                repr(
                    exc
                ),
            )

            print(
                f"Retrying after {delay:.1f}s"
            )

            time.sleep(
                delay
            )


def jolpica_pages_2026(
    endpoint,
    *,
    force_live=False,
):
    pages = []
    offset = 0

    while True:
        payload = jolpica_get_2026(
            endpoint,
            offset=
                offset,
            limit=
                JOLPICA_2026_PAGE_LIMIT,
            force_live=
                force_live,
        )

        pages.append(
            payload
        )

        mr = payload[
            "MRData"
        ]

        total = int(
            mr.get(
                "total",
                0,
            )
        )

        returned_limit = max(
            int(
                mr.get(
                    "limit",
                    JOLPICA_2026_PAGE_LIMIT,
                )
            ),
            1,
        )

        if (
            offset
            +
            returned_limit
            >=
            total
        ):
            break

        offset += returned_limit

    return pages


# ==========================================================
# 3. MAKE RERUNS SAFE — REMOVE OLD 2026 EVENT ROWS
# ==========================================================
#
# Keep any already-added driver / constructor / status metadata so
# local IDs remain stable inside a long-running kernel. Rebuild only
# the season event tables.
# ==========================================================

existing_2026_race_ids = set(
    pd.to_numeric(
        raw.races.loc[
            pd.to_numeric(
                raw.races[
                    "year"
                ],
                errors="coerce",
            )
            ==
            LIVE_YEAR,
            "raceId",
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)


if existing_2026_race_ids:

    print(
        "Removing previously appended 2026 "
        "event rows before refresh..."
    )

    raw.results = raw.results[
        ~pd.to_numeric(
            raw.results[
                "raceId"
            ],
            errors="coerce",
        )
        .isin(
            existing_2026_race_ids
        )
    ].copy()

    raw.qualifying = raw.qualifying[
        ~pd.to_numeric(
            raw.qualifying[
                "raceId"
            ],
            errors="coerce",
        )
        .isin(
            existing_2026_race_ids
        )
    ].copy()

    raw.laps = raw.laps[
        ~pd.to_numeric(
            raw.laps[
                "raceId"
            ],
            errors="coerce",
        )
        .isin(
            existing_2026_race_ids
        )
    ].copy()

    raw.races = raw.races[
        ~pd.to_numeric(
            raw.races[
                "raceId"
            ],
            errors="coerce",
        )
        .isin(
            existing_2026_race_ids
        )
    ].copy()


# ==========================================================
# 4. RESTORE PRIOR 2026 METADATA IDS IF A SNAPSHOT EXISTS
# ==========================================================
#
# This makes local driver / constructor / status IDs stable across
# clean-kernel refreshes later in the season.
# ==========================================================

for label, attr, subset in [
    (
        "drivers",
        "drivers",
        [
            "driverId",
        ],
    ),
    (
        "constructors",
        "constructors",
        [
            "constructorId",
        ],
    ),
    (
        "status",
        "status",
        [
            "statusId",
        ],
    ),
]:

    path = SNAPSHOT_PATHS_2026[
        label
    ]

    if path.exists():

        previous = pd.read_csv(
            path
        )

        setattr(
            raw,
            attr,
            append_unique(
                getattr(
                    raw,
                    attr,
                ),
                previous,
                subset=
                    subset,
            ),
        )


# ==========================================================
# 5. REFRESH THE LIVE SEASON-LEVEL ENDPOINTS
# ==========================================================

print(
    "\nRefreshing 2026 race schedule..."
)

race_pages = jolpica_pages_2026(
    f"{LIVE_YEAR}/races",
    force_live=
        True,
)

api_races = []

for page in race_pages:
    api_races.extend(
        page[
            "MRData"
        ][
            "RaceTable"
        ].get(
            "Races",
            [],
        )
    )


race_by_round = {}

for race in api_races:
    rr = finite_int(
        race.get(
            "round"
        )
    )

    if rr is not None:
        race_by_round[
            rr
        ] = race


print(
    "Refreshing 2026 Grand Prix results..."
)

result_pages = jolpica_pages_2026(
    f"{LIVE_YEAR}/results",
    force_live=
        True,
)

api_results_rows = []

driver_meta = {}
constructor_meta = {}


def remember_driver_2026(
    driver,
):
    if not driver:
        return

    ref = driver.get(
        "driverId"
    )

    if ref:
        driver_meta[
            str(
                ref
            )
        ] = driver


def remember_constructor_2026(
    constructor,
):
    if not constructor:
        return

    ref = constructor.get(
        "constructorId"
    )

    if ref:
        constructor_meta[
            str(
                ref
            )
        ] = constructor


for page in result_pages:

    races_here = (
        page[
            "MRData"
        ][
            "RaceTable"
        ]
        .get(
            "Races",
            [],
        )
    )

    for race in races_here:

        rr = finite_int(
            race.get(
                "round"
            )
        )

        if rr is None:
            continue

        for result in race.get(
            "Results",
            [],
        ):

            driver = result.get(
                "Driver",
                {},
            )

            constructor = result.get(
                "Constructor",
                {},
            )

            remember_driver_2026(
                driver
            )

            remember_constructor_2026(
                constructor
            )

            api_results_rows.append(
                {
                    "round":
                        rr,

                    "driverRef":
                        str(
                            driver.get(
                                "driverId"
                            )
                        ),

                    "constructorRef":
                        str(
                            constructor.get(
                                "constructorId"
                            )
                        ),

                    "number":
                        result.get(
                            "number"
                        ),

                    "position":
                        result.get(
                            "position"
                        ),

                    "positionText":
                        result.get(
                            "positionText"
                        ),

                    "points":
                        result.get(
                            "points"
                        ),

                    "grid":
                        result.get(
                            "grid"
                        ),

                    "laps":
                        result.get(
                            "laps"
                        ),

                    "status":
                        result.get(
                            "status",
                            "",
                        ),

                    "time":
                        result.get(
                            "Time",
                            {},
                        ).get(
                            "time"
                        ),

                    "milliseconds":
                        result.get(
                            "Time",
                            {},
                        ).get(
                            "millis"
                        ),

                    "fastestLap":
                        result.get(
                            "FastestLap",
                            {},
                        ).get(
                            "lap"
                        ),

                    "rank":
                        result.get(
                            "FastestLap",
                            {},
                        ).get(
                            "rank"
                        ),

                    "fastestLapTime":
                        result.get(
                            "FastestLap",
                            {},
                        ).get(
                            "Time",
                            {},
                        ).get(
                            "time"
                        ),

                    "fastestLapSpeed":
                        result.get(
                            "FastestLap",
                            {},
                        ).get(
                            "AverageSpeed",
                            {},
                        ).get(
                            "speed"
                        ),
                }
            )


api_results = pd.DataFrame(
    api_results_rows
)


if api_results.empty:
    raise RuntimeError(
        "Jolpica currently returned no completed 2026 "
        "Grand Prix results."
    )


api_results = (
    api_results
    .drop_duplicates(
        subset=[
            "round",
            "driverRef",
        ],
        keep="last",
    )
    .reset_index(
        drop=True
    )
)


completed_rounds = sorted(
    pd.to_numeric(
        api_results[
            "round"
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
    .unique()
)


print(
    "\nCompleted 2026 GP rounds returned:",
    completed_rounds,
)

print(
    "Number of completed rounds:",
    len(
        completed_rounds
    ),
)


missing_schedule_rounds = [
    rr
    for rr in completed_rounds
    if rr not in race_by_round
]

if missing_schedule_rounds:
    raise RuntimeError(
        "Results exist for rounds not present in the "
        "2026 schedule endpoint: "
        f"{missing_schedule_rounds}"
    )


# ==========================================================
# 6. REFRESH GRAND PRIX QUALIFYING
# ==========================================================

print(
    "\nRefreshing 2026 qualifying..."
)

qualifying_pages = jolpica_pages_2026(
    f"{LIVE_YEAR}/qualifying",
    force_live=
        True,
)

api_qualifying_rows = []


for page in qualifying_pages:

    races_here = (
        page[
            "MRData"
        ][
            "RaceTable"
        ]
        .get(
            "Races",
            [],
        )
    )

    for race in races_here:

        rr = finite_int(
            race.get(
                "round"
            )
        )

        if (
            rr is None
            or
            rr not in completed_rounds
        ):
            continue

        for result in race.get(
            "QualifyingResults",
            [],
        ):

            driver = result.get(
                "Driver",
                {},
            )

            constructor = result.get(
                "Constructor",
                {},
            )

            remember_driver_2026(
                driver
            )

            remember_constructor_2026(
                constructor
            )

            api_qualifying_rows.append(
                {
                    "round":
                        rr,

                    "driverRef":
                        str(
                            driver.get(
                                "driverId"
                            )
                        ),

                    "constructorRef":
                        str(
                            constructor.get(
                                "constructorId"
                            )
                        ),

                    "number":
                        result.get(
                            "number"
                        ),

                    "position":
                        result.get(
                            "position"
                        ),

                    "q1":
                        result.get(
                            "Q1"
                        ),

                    "q2":
                        result.get(
                            "Q2"
                        ),

                    "q3":
                        result.get(
                            "Q3"
                        ),
                }
            )


api_qualifying = pd.DataFrame(
    api_qualifying_rows
)


if api_qualifying.empty:
    raise RuntimeError(
        "No 2026 qualifying data returned."
    )


api_qualifying = (
    api_qualifying
    .drop_duplicates(
        subset=[
            "round",
            "driverRef",
        ],
        keep="last",
    )
    .reset_index(
        drop=True
    )
)


qualifying_rounds = set(
    pd.to_numeric(
        api_qualifying[
            "round"
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)


missing_quali_rounds = sorted(
    set(
        completed_rounds
    )
    -
    qualifying_rounds
)


if missing_quali_rounds:
    raise RuntimeError(
        "Completed 2026 race result(s) exist but Grand Prix "
        "qualifying is missing for round(s): "
        f"{missing_quali_rounds}\n\n"
        "This is likely an API-lag issue. Rerun later rather "
        "than silently fitting an inconsistent latest round."
    )


# ==========================================================
# 7. RACE LAP TIMING
# ==========================================================
#
# Stable old completed rounds are cached.
# The latest completed round is always refreshed.
# ==========================================================

print(
    "\nDownloading / loading 2026 race lap timing..."
)

api_laps_rows = []

latest_completed_round = max(
    completed_rounds
)


for i, rr in enumerate(
    completed_rounds,
    start=1,
):

    race_name = race_by_round[
        rr
    ].get(
        "raceName",
        f"Round {rr}",
    )

    force_latest = (
        rr
        ==
        latest_completed_round
    )

    print(
        f"  [{i:02d}/{len(completed_rounds):02d}] "
        f"Round {rr}: {race_name}"
        +
        (
            " [refreshing latest]"
            if force_latest
            else
            " [cache allowed]"
        )
    )

    lap_pages = jolpica_pages_2026(
        f"{LIVE_YEAR}/{rr}/laps",
        force_live=
            force_latest,
    )

    for page in lap_pages:

        races_here = (
            page[
                "MRData"
            ][
                "RaceTable"
            ]
            .get(
                "Races",
                [],
            )
        )

        for race in races_here:

            for lap_object in race.get(
                "Laps",
                [],
            ):

                lap_number = finite_int(
                    lap_object.get(
                        "number"
                    )
                )

                for timing in lap_object.get(
                    "Timings",
                    [],
                ):

                    api_laps_rows.append(
                        {
                            "round":
                                rr,

                            "lap":
                                lap_number,

                            "driverRef":
                                str(
                                    timing.get(
                                        "driverId"
                                    )
                                ),

                            "position":
                                timing.get(
                                    "position"
                                ),

                            "time":
                                timing.get(
                                    "time"
                                ),
                        }
                    )


api_laps = pd.DataFrame(
    api_laps_rows
)


if api_laps.empty:
    raise RuntimeError(
        "No 2026 race lap timing returned."
    )


api_laps = (
    api_laps
    .drop_duplicates(
        subset=[
            "round",
            "lap",
            "driverRef",
        ],
        keep="last",
    )
    .reset_index(
        drop=True
    )
)


lap_rounds = set(
    pd.to_numeric(
        api_laps[
            "round"
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)


missing_lap_rounds = sorted(
    set(
        completed_rounds
    )
    -
    lap_rounds
)


if missing_lap_rounds:
    raise RuntimeError(
        "Completed 2026 race result(s) exist but lap timing "
        "is missing for round(s): "
        f"{missing_lap_rounds}\n\n"
        "Rerun later rather than silently fitting an "
        "incomplete latest round."
    )


# ==========================================================
# 8. STABLE LOCAL RACE IDS
# ==========================================================

prior_races_2026 = None

if SNAPSHOT_PATHS_2026[
    "races"
].exists():

    try:
        prior_races_2026 = pd.read_csv(
            SNAPSHOT_PATHS_2026[
                "races"
            ]
        )
    except Exception:
        prior_races_2026 = None


race_id_by_round = {}


if (
    prior_races_2026 is not None
    and
    {
        "raceId",
        "round",
    }.issubset(
        prior_races_2026.columns
    )
):

    prior_map = (
        prior_races_2026[
            [
                "raceId",
                "round",
            ]
        ]
        .dropna()
        .copy()
    )

    for row in prior_map.itertuples(
        index=False
    ):
        race_id_by_round[
            int(
                row.round
            )
        ] = int(
            row.raceId
        )


next_race_id = (
    max(
        max_numeric_id(
            raw.races,
            "raceId",
        ),
        max(
            race_id_by_round.values(),
            default=0,
        ),
    )
    +
    1
)


for rr in completed_rounds:

    if rr not in race_id_by_round:
        race_id_by_round[
            rr
        ] = next_race_id

        next_race_id += 1


# ==========================================================
# 9. CIRCUIT LOOKUP
# ==========================================================

circuit_ref_to_id = {}

circuits_path = (
    Path(
        CFG.data_dir
    )
    /
    "circuits.csv"
)


if circuits_path.exists():

    circuits_local = pd.read_csv(
        circuits_path
    )

    if {
        "circuitId",
        "circuitRef",
    }.issubset(
        circuits_local.columns
    ):

        circuit_ref_to_id = {
            str(
                ref
            ):
                int(
                    cid
                )
            for cid, ref
            in zip(
                circuits_local[
                    "circuitId"
                ],
                circuits_local[
                    "circuitRef"
                ],
            )
        }


unknown_circuit_refs = sorted(
    {
        str(
            race_by_round[
                rr
            ].get(
                "Circuit",
                {},
            ).get(
                "circuitId"
            )
        )
        for rr in completed_rounds
        if (
            race_by_round[
                rr
            ].get(
                "Circuit",
                {},
            ).get(
                "circuitId"
            )
            is not None
            and
            str(
                race_by_round[
                    rr
                ].get(
                    "Circuit",
                    {},
                ).get(
                    "circuitId"
                )
            )
            not in circuit_ref_to_id
        )
    }
)


if unknown_circuit_refs:

    print(
        "\nNOTE — new circuitRef(s) not in the old local "
        "circuits.csv:"
    )

    print(
        unknown_circuit_refs
    )

    print(
        "Their race.circuitId will be NaN. The current Pantheon "
        "likelihood does not use circuitId, so this is not a "
        "model blocker."
    )


# ==========================================================
# 10. DRIVER IDS
# ==========================================================

existing_driver_ref_to_id = {
    str(
        ref
    ):
        int(
            driver_id
        )
    for driver_id, ref
    in zip(
        raw.drivers[
            "driverId"
        ],
        raw.drivers[
            "driverRef"
        ],
    )
}


api_driver_refs = set(
    api_results[
        "driverRef"
    ].dropna()
)

api_driver_refs.update(
    api_qualifying[
        "driverRef"
    ].dropna()
)

api_driver_refs.update(
    api_laps[
        "driverRef"
    ].dropna()
)


driver_ref_to_id = (
    existing_driver_ref_to_id.copy()
)


new_driver_refs = sorted(
    api_driver_refs
    -
    set(
        existing_driver_ref_to_id
    )
)


next_driver_id = (
    max_numeric_id(
        raw.drivers,
        "driverId",
    )
    +
    1
)


for i, ref in enumerate(
    new_driver_refs
):
    driver_ref_to_id[
        ref
    ] = (
        next_driver_id
        +
        i
    )


print(
    "\nNew 2026 driver refs:",
    new_driver_refs,
)


# ==========================================================
# 11. CONSTRUCTOR IDS — NEW CONSTRUCTORS ARE ALLOWED
# ==========================================================

if not {
    "constructorId",
    "constructorRef",
}.issubset(
    raw.constructors.columns
):
    raise RuntimeError(
        "constructors.csv with constructorRef is required."
    )


existing_constructor_ref_to_id = {
    str(
        ref
    ):
        int(
            constructor_id
        )
    for constructor_id, ref
    in zip(
        raw.constructors[
            "constructorId"
        ],
        raw.constructors[
            "constructorRef"
        ],
    )
}


api_constructor_refs = set(
    api_results[
        "constructorRef"
    ].dropna()
)

api_constructor_refs.update(
    api_qualifying[
        "constructorRef"
    ].dropna()
)


constructor_ref_to_id = (
    existing_constructor_ref_to_id.copy()
)


new_constructor_refs = sorted(
    api_constructor_refs
    -
    set(
        existing_constructor_ref_to_id
    )
)


next_constructor_id = (
    max_numeric_id(
        raw.constructors,
        "constructorId",
    )
    +
    1
)


for i, ref in enumerate(
    new_constructor_refs
):
    constructor_ref_to_id[
        ref
    ] = (
        next_constructor_id
        +
        i
    )


print(
    "\n2026 constructor refs:",
    sorted(
        api_constructor_refs
    ),
)

print(
    "New constructor refs:",
    new_constructor_refs,
)


# ==========================================================
# 12. STATUS IDS
# ==========================================================

existing_status_text_to_id = {
    str(
        status
    )
    .strip()
    .casefold():
        int(
            status_id
        )
    for status_id, status
    in zip(
        raw.status[
            "statusId"
        ],
        raw.status[
            "status"
        ],
    )
}


status_text_to_id = (
    existing_status_text_to_id.copy()
)


statuses_2026 = sorted(
    {
        str(
            x
        ).strip()
        for x
        in api_results[
            "status"
        ].dropna()
        if str(
            x
        ).strip()
    }
)


unseen_statuses = [
    status_text
    for status_text in statuses_2026
    if (
        status_text.casefold()
        not in status_text_to_id
    )
]


next_status_id = (
    max_numeric_id(
        raw.status,
        "statusId",
    )
    +
    1
)


for i, status_text in enumerate(
    unseen_statuses
):
    status_text_to_id[
        status_text.casefold()
    ] = (
        next_status_id
        +
        i
    )


# ==========================================================
# 13. RACES TABLE — COMPLETED ROUNDS ONLY
# ==========================================================

race_rows = []


for rr in completed_rounds:

    race = race_by_round[
        rr
    ]

    circuit = race.get(
        "Circuit",
        {},
    )

    circuit_ref = circuit.get(
        "circuitId"
    )

    fp1 = race.get(
        "FirstPractice",
        {},
    )

    fp2 = race.get(
        "SecondPractice",
        {},
    )

    fp3 = race.get(
        "ThirdPractice",
        {},
    )

    qualifying_sched = race.get(
        "Qualifying",
        {},
    )

    sprint = race.get(
        "Sprint",
        {},
    )

    race_rows.append(
        {
            "raceId":
                race_id_by_round[
                    rr
                ],

            "year":
                LIVE_YEAR,

            "round":
                rr,

            "circuitId":
                (
                    circuit_ref_to_id.get(
                        str(
                            circuit_ref
                        )
                    )
                    if circuit_ref is not None
                    else np.nan
                ),

            "name":
                race.get(
                    "raceName"
                ),

            "date":
                race.get(
                    "date"
                ),

            "time":
                race.get(
                    "time"
                ),

            "url":
                race.get(
                    "url"
                ),

            "fp1_date":
                fp1.get(
                    "date"
                ),

            "fp1_time":
                fp1.get(
                    "time"
                ),

            "fp2_date":
                fp2.get(
                    "date"
                ),

            "fp2_time":
                fp2.get(
                    "time"
                ),

            "fp3_date":
                fp3.get(
                    "date"
                ),

            "fp3_time":
                fp3.get(
                    "time"
                ),

            "quali_date":
                qualifying_sched.get(
                    "date"
                ),

            "quali_time":
                qualifying_sched.get(
                    "time"
                ),

            "sprint_date":
                sprint.get(
                    "date"
                ),

            "sprint_time":
                sprint.get(
                    "time"
                ),
        }
    )


new_races_2026 = align_columns(
    race_rows,
    raw.races,
)


# ==========================================================
# 14. NEW DRIVER METADATA
# ==========================================================

driver_rows = []


for ref in new_driver_refs:

    meta = driver_meta.get(
        ref,
        {},
    )

    driver_rows.append(
        {
            "driverId":
                driver_ref_to_id[
                    ref
                ],

            "driverRef":
                ref,

            "number":
                meta.get(
                    "permanentNumber"
                ),

            "code":
                meta.get(
                    "code"
                ),

            "forename":
                meta.get(
                    "givenName"
                ),

            "surname":
                meta.get(
                    "familyName"
                ),

            "dob":
                meta.get(
                    "dateOfBirth"
                ),

            "nationality":
                meta.get(
                    "nationality"
                ),

            "url":
                meta.get(
                    "url"
                ),
        }
    )


new_drivers_2026 = align_columns(
    driver_rows,
    raw.drivers,
)


# ==========================================================
# 15. NEW CONSTRUCTOR METADATA
# ==========================================================

constructor_rows = []


for ref in new_constructor_refs:

    meta = constructor_meta.get(
        ref,
        {},
    )

    constructor_rows.append(
        {
            "constructorId":
                constructor_ref_to_id[
                    ref
                ],

            "constructorRef":
                ref,

            "name":
                meta.get(
                    "name"
                ),

            "nationality":
                meta.get(
                    "nationality"
                ),

            "url":
                meta.get(
                    "url"
                ),
        }
    )


new_constructors_2026 = align_columns(
    constructor_rows,
    raw.constructors,
)


# ==========================================================
# 16. NEW STATUS METADATA
# ==========================================================

status_rows = []


for status_text in unseen_statuses:

    status_rows.append(
        {
            "statusId":
                status_text_to_id[
                    status_text.casefold()
                ],

            "status":
                status_text,
        }
    )


new_status_2026 = align_columns(
    status_rows,
    raw.status,
)


# ==========================================================
# 17. RESULTS TABLE
# ==========================================================

next_result_id = (
    max_numeric_id(
        raw.results,
        "resultId",
    )
    +
    1
)


result_rows = []


for row_number, r in enumerate(
    api_results.itertuples(
        index=False
    )
):

    rr = int(
        r.round
    )

    if rr not in completed_rounds:
        continue

    position_order = finite_int(
        r.position
    )

    result_rows.append(
        {
            "resultId":
                next_result_id
                +
                row_number,

            "raceId":
                race_id_by_round[
                    rr
                ],

            "driverId":
                driver_ref_to_id[
                    r.driverRef
                ],

            "constructorId":
                constructor_ref_to_id[
                    r.constructorRef
                ],

            "number":
                finite_int(
                    r.number
                ),

            "grid":
                finite_int(
                    r.grid,
                    0,
                ),

            "position":
                position_order,

            "positionText":
                r.positionText,

            "positionOrder":
                position_order,

            "points":
                pd.to_numeric(
                    r.points,
                    errors="coerce",
                ),

            "laps":
                finite_int(
                    r.laps,
                    0,
                ),

            "time":
                r.time,

            "milliseconds":
                pd.to_numeric(
                    r.milliseconds,
                    errors="coerce",
                ),

            "fastestLap":
                finite_int(
                    r.fastestLap
                ),

            "rank":
                finite_int(
                    r.rank
                ),

            "fastestLapTime":
                r.fastestLapTime,

            "fastestLapSpeed":
                pd.to_numeric(
                    r.fastestLapSpeed,
                    errors="coerce",
                ),

            "statusId":
                status_text_to_id[
                    str(
                        r.status
                    )
                    .strip()
                    .casefold()
                ],
        }
    )


new_results_2026 = align_columns(
    result_rows,
    raw.results,
)


# ==========================================================
# 18. QUALIFYING TABLE
# ==========================================================

next_qualify_id = (
    max_numeric_id(
        raw.qualifying,
        "qualifyId",
    )
    +
    1
)


qualifying_rows = []


for row_number, q in enumerate(
    api_qualifying.itertuples(
        index=False
    )
):

    rr = int(
        q.round
    )

    qualifying_rows.append(
        {
            "qualifyId":
                next_qualify_id
                +
                row_number,

            "raceId":
                race_id_by_round[
                    rr
                ],

            "driverId":
                driver_ref_to_id[
                    q.driverRef
                ],

            "constructorId":
                constructor_ref_to_id[
                    q.constructorRef
                ],

            "number":
                finite_int(
                    q.number
                ),

            "position":
                finite_int(
                    q.position
                ),

            "q1":
                q.q1,

            "q2":
                q.q2,

            "q3":
                q.q3,
        }
    )


new_qualifying_2026 = align_columns(
    qualifying_rows,
    raw.qualifying,
)


# ==========================================================
# 19. LAP TIMES TABLE
# ==========================================================

lap_rows = []


for l in api_laps.itertuples(
    index=False
):

    if (
        l.driverRef
        not in driver_ref_to_id
    ):
        raise RuntimeError(
            "Lap data contains unknown driverRef: "
            f"{l.driverRef}"
        )

    lap_rows.append(
        {
            "raceId":
                race_id_by_round[
                    int(
                        l.round
                    )
                ],

            "driverId":
                driver_ref_to_id[
                    l.driverRef
                ],

            "lap":
                finite_int(
                    l.lap
                ),

            "position":
                finite_int(
                    l.position
                ),

            "time":
                l.time,

            "milliseconds":
                lap_time_to_ms(
                    l.time
                ),
        }
    )


new_laps_2026 = align_columns(
    lap_rows,
    raw.laps,
)


# ==========================================================
# 20. SAVE CURRENT CONVERTED 2026 SNAPSHOT
# ==========================================================

snapshot_frames_2026 = {
    "races":
        new_races_2026,

    "results":
        new_results_2026,

    "qualifying":
        new_qualifying_2026,

    "laps":
        new_laps_2026,

    "drivers":
        new_drivers_2026,

    "constructors":
        new_constructors_2026,

    "status":
        new_status_2026,
}


for name, df in snapshot_frames_2026.items():

    df.to_csv(
        SNAPSHOT_PATHS_2026[
            name
        ],
        index=False,
    )


print(
    "\nSaved current converted 2026 snapshot ->"
)

print(
    TABLE_CACHE_DIR_2026
)


# ==========================================================
# 21. APPEND 2026 METADATA + EVENTS TO RAW IN MEMORY
# ==========================================================

raw.drivers = append_unique(
    raw.drivers,
    new_drivers_2026,
    subset=[
        "driverId",
    ],
)


raw.constructors = append_unique(
    raw.constructors,
    new_constructors_2026,
    subset=[
        "constructorId",
    ],
)


raw.status = append_unique(
    raw.status,
    new_status_2026,
    subset=[
        "statusId",
    ],
)


raw.races = append_unique(
    raw.races,
    new_races_2026,
    subset=[
        "raceId",
    ],
)


raw.results = append_unique(
    raw.results,
    new_results_2026,
    subset=[
        "raceId",
        "driverId",
        "constructorId",
    ],
)


raw.qualifying = append_unique(
    raw.qualifying,
    new_qualifying_2026,
    subset=[
        "raceId",
        "driverId",
        "constructorId",
    ],
)


raw.laps = append_unique(
    raw.laps,
    new_laps_2026,
    subset=[
        "raceId",
        "driverId",
        "lap",
    ],
)


# ==========================================================
# 22. REBUILD LOOKUP DICTIONARIES
# ==========================================================

raw.driver_ref = dict(
    zip(
        pd.to_numeric(
            raw.drivers[
                "driverId"
            ]
        ).astype(int),
        raw.drivers[
            "driverRef"
        ].astype(str),
    )
)


raw.driver_id = dict(
    zip(
        raw.drivers[
            "driverRef"
        ].astype(str),
        pd.to_numeric(
            raw.drivers[
                "driverId"
            ]
        ).astype(int),
    )
)


raw.constructor_ref = dict(
    zip(
        pd.to_numeric(
            raw.constructors[
                "constructorId"
            ]
        ).astype(int),
        raw.constructors[
            "constructorRef"
        ].astype(str),
    )
)


raw.status_map = dict(
    zip(
        pd.to_numeric(
            raw.status[
                "statusId"
            ]
        ).astype(int),
        raw.status[
            "status"
        ].astype(str),
    )
)


# ==========================================================
# 23. 2026 COVERAGE AUDIT
# ==========================================================

races_augmented_2026 = raw.races[
    pd.to_numeric(
        raw.races[
            "year"
        ],
        errors="coerce",
    )
    ==
    LIVE_YEAR
].copy()


race_ids_augmented_2026 = set(
    pd.to_numeric(
        races_augmented_2026[
            "raceId"
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)


results_augmented_2026 = raw.results[
    pd.to_numeric(
        raw.results[
            "raceId"
        ],
        errors="coerce",
    )
    .isin(
        race_ids_augmented_2026
    )
].copy()


qualifying_augmented_2026 = raw.qualifying[
    pd.to_numeric(
        raw.qualifying[
            "raceId"
        ],
        errors="coerce",
    )
    .isin(
        race_ids_augmented_2026
    )
].copy()


laps_augmented_2026 = raw.laps[
    pd.to_numeric(
        raw.laps[
            "raceId"
        ],
        errors="coerce",
    )
    .isin(
        race_ids_augmented_2026
    )
].copy()


coverage_2026 = pd.DataFrame(
    {
        "table":
            [
                "races",
                "results",
                "qualifying",
                "lap_times",
            ],

        "rows_2026_so_far":
            [
                len(
                    races_augmented_2026
                ),
                len(
                    results_augmented_2026
                ),
                len(
                    qualifying_augmented_2026
                ),
                len(
                    laps_augmented_2026
                ),
            ],

        "races_covered":
            [
                races_augmented_2026[
                    "raceId"
                ].nunique(),

                results_augmented_2026[
                    "raceId"
                ].nunique(),

                qualifying_augmented_2026[
                    "raceId"
                ].nunique(),

                laps_augmented_2026[
                    "raceId"
                ].nunique(),
            ],
    }
)


print(
    "\n========================================"
)

print(
    "LIVE 2026 DATA COVERAGE"
)

print(
    "========================================"
)


display(
    coverage_2026
)


race_coverage_2026 = (
    races_augmented_2026[
        [
            "raceId",
            "round",
            "name",
            "date",
        ]
    ]
    .copy()
)


race_coverage_2026[
    "has_results"
] = (
    race_coverage_2026[
        "raceId"
    ]
    .isin(
        results_augmented_2026[
            "raceId"
        ]
    )
)


race_coverage_2026[
    "has_qualifying"
] = (
    race_coverage_2026[
        "raceId"
    ]
    .isin(
        qualifying_augmented_2026[
            "raceId"
        ]
    )
)


race_coverage_2026[
    "has_laps"
] = (
    race_coverage_2026[
        "raceId"
    ]
    .isin(
        laps_augmented_2026[
            "raceId"
        ]
    )
)


print(
    "\nRace-level 2026 coverage:"
)


display(
    race_coverage_2026.sort_values(
        "round"
    )
)


# Drivers
driver_ids_2026 = set(
    pd.to_numeric(
        results_augmented_2026[
            "driverId"
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)


drivers_2026_table = (
    raw.drivers[
        pd.to_numeric(
            raw.drivers[
                "driverId"
            ],
            errors="coerce",
        )
        .isin(
            driver_ids_2026
        )
    ][
        [
            col
            for col
            in [
                "driverId",
                "driverRef",
                "forename",
                "surname",
            ]
            if col in raw.drivers.columns
        ]
    ]
    .sort_values(
        "driverRef"
    )
)


print(
    "\n2026 drivers:"
)


display(
    drivers_2026_table
)


# Constructors
constructor_ids_2026 = set(
    pd.to_numeric(
        results_augmented_2026[
            "constructorId"
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)


constructors_2026_table = (
    raw.constructors[
        pd.to_numeric(
            raw.constructors[
                "constructorId"
            ],
            errors="coerce",
        )
        .isin(
            constructor_ids_2026
        )
    ][
        [
            col
            for col
            in [
                "constructorId",
                "constructorRef",
                "name",
            ]
            if col in raw.constructors.columns
        ]
    ]
    .sort_values(
        "constructorRef"
    )
)


print(
    "\n2026 constructors:"
)


display(
    constructors_2026_table
)


# ==========================================================
# 24. HARD VALIDATION — COMPLETED ROUNDS ONLY
# ==========================================================

expected_races_2026 = len(
    completed_rounds
)


if (
    races_augmented_2026[
        "raceId"
    ].nunique()
    !=
    expected_races_2026
):
    raise RuntimeError(
        "2026 races table does not contain exactly the "
        "completed-result rounds."
    )


for label, frame in [
    (
        "results",
        results_augmented_2026,
    ),
    (
        "qualifying",
        qualifying_augmented_2026,
    ),
    (
        "lap_times",
        laps_augmented_2026,
    ),
]:

    n_covered = frame[
        "raceId"
    ].nunique()

    if (
        n_covered
        !=
        expected_races_2026
    ):
        raise RuntimeError(
            f"2026 {label} covers "
            f"{n_covered}/{expected_races_2026} "
            "completed races."
        )


if not (
    race_coverage_2026[
        [
            "has_results",
            "has_qualifying",
            "has_laps",
        ]
    ]
    .all()
    .all()
):
    raise RuntimeError(
        "At least one completed 2026 GP is missing "
        "results / qualifying / lap coverage."
    )


# ==========================================================
# 25. CREATE A SEPARATE LIVE-2026 MODEL CONFIG
# ==========================================================
#
# This is intentionally a new output directory.
# The official 1982–2025 production fit remains frozen.
#
# Because this cell belongs BEFORE Stage 3, rerunning Stages 3 onward
# will bind their default cfg argument to this 2026 configuration.
# ==========================================================

CFG = replace(
    CFG,

    ceiling_year=
        LIVE_YEAR,

    output_dir=
        (
            "model_outputs/"
            "joint_driver_car_v2_1_1982_2026_LIVE"
        ),
)


Path(
    CFG.output_dir
).mkdir(
    parents=True,
    exist_ok=True,
)


# ==========================================================
# 26. FINAL SUMMARY
# ==========================================================

print(
    "\n========================================"
)

print(
    "LIVE 2026 AUGMENTATION PASSED"
)

print(
    "========================================"
)


print(
    "Completed GP rounds included:",
    completed_rounds,
)


print(
    "Latest included round:",
    latest_completed_round,
)


print(
    "Maximum year now available in raw.races:",
    int(
        pd.to_numeric(
            raw.races[
                "year"
            ],
            errors="coerce",
        )
        .max()
    ),
)


print(
    "CFG.ceiling_year:",
    CFG.ceiling_year,
)


print(
    "Live-2026 output directory:"
)

print(
    Path(
        CFG.output_dir
    ).resolve()
)


print(
    "\nOriginal ergast_data CSV files have NOT been modified."
)


print(
    "\nNEXT:"
)

print(
    "  Run Stage 3, Stage 4, Stage 5 and Stage 6."
)

print(
    "  Then STOP and inspect the 2026 modern-observation audit "
    "before building / sampling the 1982–2026 model."
)


## Part III — Observation-construction helpers

These are the validated v2.1 observation/model-building functions. The initial literal-constructor familiarity tables are built only to provide the audited inputs used by the locked organisational-familiarity construction. Stage 32H-A later rebuilds the final LIVE likelihood inputs with the v2.2 familiarity specification.


In [ ]:
def build_driver_event_covariates(raw, cfg=CFG):
    races = raw.races[["raceId", "year", "round", "date"]].copy()

    races["date"] = pd.to_datetime(
        races["date"],
        errors="coerce",
    )

    # One row per driver / constructor / race BEFORE cumcount.
    starts = (
        raw.results[["raceId", "driverId", "constructorId"]]
        .drop_duplicates(
            subset=[
                "raceId",
                "driverId",
                "constructorId",
            ]
        )
        .merge(
            races,
            on="raceId",
            how="left",
            validate="many_to_one",
        )
        .sort_values(["date", "raceId", "driverId"])
        .reset_index(drop=True)
    )

    starts["team_starts_before"] = starts.groupby(["driverId", "constructorId"]).cumcount()

    starts["tenure_raw"] = np.tanh(starts["team_starts_before"] / cfg.tenure_scale_races)

    out = starts[
        [
            "raceId",
            "driverId",
            "constructorId",
            "tenure_raw",
        ]
    ].copy()

    key_cols = [
        "raceId",
        "driverId",
        "constructorId",
    ]

    duplicate_mask = out.duplicated(
        key_cols,
        keep=False,
    )

    if duplicate_mask.any():
        display(out.loc[duplicate_mask].sort_values(key_cols))
        raise ValueError(
            "driver_event_cov contains duplicate " "(raceId, driverId, constructorId) keys."
        )

    return out


def add_centered_tenure(
    df,
    driver_event_cov,
    group_cols,
):
    out = df.merge(
        driver_event_cov,
        on=[
            "raceId",
            "driverId",
            "constructorId",
        ],
        how="left",
        validate="many_to_one",
    )

    out["tenure_raw"] = out["tenure_raw"].fillna(0.0)

    out["tenure_c"] = out["tenure_raw"] - out.groupby(group_cols)["tenure_raw"].transform(
        "mean"
    )

    return out


driver_event_cov = build_driver_event_covariates(
    raw,
    CFG,
)

print(
    "Driver-event covariates:",
    f"{len(driver_event_cov):,}",
)

print(
    "Duplicate lookup keys:",
    driver_event_cov.duplicated(
        [
            "raceId",
            "driverId",
            "constructorId",
        ]
    ).sum(),
)

In [ ]:
def build_lap_relative_pace(raw, driver_event_cov, cfg=CFG):
    valid_races = raw.races[
        (raw.races["year"] >= cfg.modern_floor_year) & (raw.races["year"] <= cfg.ceiling_year)
    ][["raceId", "year"]].copy()

    laps = (
        raw.laps[
            [
                "raceId",
                "driverId",
                "lap",
                "milliseconds",
            ]
        ]
        .merge(
            valid_races,
            on="raceId",
            how="inner",
        )
        .merge(
            raw.results[
                [
                    "raceId",
                    "driverId",
                    "constructorId",
                ]
            ].drop_duplicates(
                [
                    "raceId",
                    "driverId",
                    "constructorId",
                ]
            ),
            on=[
                "raceId",
                "driverId",
            ],
            how="inner",
        )
        .copy()
    )

    laps["sec"] = (
        pd.to_numeric(
            laps["milliseconds"],
            errors="coerce",
        )
        / 1000.0
    )

    laps = laps[laps["sec"].notna() & (laps["sec"] > 40.0)].copy()

    # Keep laps where enough of the field is still represented.
    lap_counts = laps.groupby(["raceId", "lap"])["driverId"].transform("nunique")

    laps = laps[lap_counts >= cfg.min_drivers_per_lap].copy()

    laps["field_lap_median"] = laps.groupby(["raceId", "lap"])["sec"].transform("median")

    laps["lap_resid"] = laps["sec"] - laps["field_lap_median"]

    driver_race = laps.groupby(
        [
            "raceId",
            "year",
            "driverId",
            "constructorId",
        ],
        as_index=False,
    ).agg(
        raw_pace=(
            "lap_resid",
            "median",
        ),
        n_comparable_laps=(
            "lap_resid",
            "size",
        ),
    )

    driver_race = driver_race[driver_race["n_comparable_laps"] >= cfg.min_clean_laps].copy()

    rows = []

    for race_id, grp in driver_race.groupby("raceId"):
        if len(grp) < cfg.min_field_drivers:
            continue

        g = grp.copy()

        # Lower residual is better.
        raw_score = -g["raw_pace"].to_numpy(dtype=float)

        scale = robust_scale(raw_score)

        y = (raw_score - np.median(raw_score)) / scale

        # Exact whole-field mean zero within race.
        y = y - y.mean()

        y = np.clip(
            y,
            -4.0,
            4.0,
        )

        reliability = np.clip(
            g["n_comparable_laps"].to_numpy(dtype=float) / 30.0,
            0.60,
            1.40,
        )

        for (_, r), yy, ww in zip(
            g.iterrows(),
            y,
            reliability,
        ):
            rows.append(
                {
                    "raceId": int(r["raceId"]),
                    "year": int(r["year"]),
                    "driverId": int(r["driverId"]),
                    "driver": raw.driver_ref.get(
                        int(r["driverId"]),
                        str(int(r["driverId"])),
                    ),
                    "constructorId": int(r["constructorId"]),
                    "y_pace": float(yy),
                    "pace_weight": float(ww),
                    "n_comparable_laps": int(r["n_comparable_laps"]),
                }
            )

    df = pd.DataFrame(rows)

    df = add_centered_tenure(
        df,
        driver_event_cov,
        group_cols=["raceId"],
    )

    log.info(
        "V2.1 pace: %d driver-race observations across %d races",
        len(df),
        df["raceId"].nunique(),
    )

    return df


df_pace = build_lap_relative_pace(
    raw,
    driver_event_cov,
    CFG,
)

display(df_pace.head())

display(
    df_pace[
        [
            "y_pace",
            "pace_weight",
            "n_comparable_laps",
            "tenure_c",
        ]
    ].describe()
)

In [ ]:
def build_timed_qualifying(
    raw,
    driver_event_cov,
    cfg=CFG,
):
    q = raw.qualifying.copy()

    q = q.merge(
        raw.races[["raceId", "year"]],
        on="raceId",
        how="left",
    )

    q = q[(q["year"] >= cfg.modern_floor_year) & (q["year"] <= cfg.ceiling_year)].copy()

    for session in (
        "q1",
        "q2",
        "q3",
    ):
        col = f"{session}_sec"

        if session in q.columns:
            q[col] = q[session].apply(parse_lap_time)
        else:
            q[col] = np.nan

    min_session_field = {
        "q1": 8,
        "q2": 5,
        "q3": 5,
    }

    rows = []

    for race_id, race_grp in q.groupby("raceId"):
        year = int(race_grp["year"].iloc[0])

        for session in (
            "q1",
            "q2",
            "q3",
        ):
            col = f"{session}_sec"

            g = race_grp[race_grp[col].notna() & (race_grp[col] > 40.0)].copy()

            if len(g) < min_session_field[session]:
                continue

            times = g[col].to_numpy(dtype=float)

            raw_score = -(times - np.median(times))

            scale = robust_scale(raw_score)

            y = raw_score / scale
            y = y - y.mean()
            y = np.clip(
                y,
                -4.0,
                4.0,
            )

            for (_, r), yy in zip(
                g.iterrows(),
                y,
            ):
                rows.append(
                    {
                        "raceId": int(r["raceId"]),
                        "year": year,
                        "session": session,
                        "driverId": int(r["driverId"]),
                        "driver": raw.driver_ref.get(
                            int(r["driverId"]),
                            str(int(r["driverId"])),
                        ),
                        "constructorId": int(r["constructorId"]),
                        "y_quali": float(yy),
                    }
                )

    df = pd.DataFrame(rows)

    # Equalise total qualifying weight approximately by driver/race.
    n_sessions = (
        df.groupby(
            [
                "raceId",
                "driverId",
            ]
        )["session"]
        .transform("count")
        .astype(float)
    )

    df["quali_weight"] = 1.0 / n_sessions

    df = add_centered_tenure(
        df,
        driver_event_cov,
        group_cols=[
            "raceId",
            "session",
        ],
    )

    log.info(
        "V2.1 qualifying: %d driver-session observations across %d races",
        len(df),
        df["raceId"].nunique(),
    )

    return df


df_quali = build_timed_qualifying(
    raw,
    driver_event_cov,
    CFG,
)

display(df_quali.head())

display(
    df_quali.groupby("session").agg(
        observations=(
            "y_quali",
            "size",
        ),
        races=(
            "raceId",
            "nunique",
        ),
    )
)

In [ ]:
def build_pair_and_level_data(
    df,
    *,
    y_col,
    individual_weight_col,
    session_col=None,
):
    group_cols = [
        "raceId",
        "year",
        "constructorId",
    ]

    if session_col is not None:
        group_cols.append(session_col)

    contrast_rows = []
    level_rows = []

    for _, grp in df.groupby(group_cols):
        g = grp.sort_values("driverId").copy()

        if g["driverId"].nunique() != 2 or len(g) != 2:
            continue

        a = g.iloc[0]
        b = g.iloc[1]

        wa = float(a[individual_weight_col])
        wb = float(b[individual_weight_col])

        # Harmonic mean of individual reliability weights.
        pair_reliability = 2.0 / (1.0 / wa + 1.0 / wb)

        base = {
            "raceId": int(a["raceId"]),
            "year": int(a["year"]),
            "constructorId": int(a["constructorId"]),
            "driver_a": a["driver"],
            "driver_b": b["driver"],
            "driverId_a": int(a["driverId"]),
            "driverId_b": int(b["driverId"]),
        }

        if session_col is not None:
            base[session_col] = a[session_col]

        contrast_rows.append(
            {
                **base,
                "y": float(a[y_col] - b[y_col]),
                "tenure_diff": float(a["tenure_c"] - b["tenure_c"]),
                "weight": pair_reliability,
            }
        )

        level_rows.append(
            {
                **base,
                "y": float(0.5 * (a[y_col] + b[y_col])),
                "tenure_mean": float(0.5 * (a["tenure_c"] + b["tenure_c"])),
                "weight": pair_reliability,
            }
        )

    return (
        pd.DataFrame(contrast_rows),
        pd.DataFrame(level_rows),
    )


pace_contrast, pace_level = build_pair_and_level_data(
    df_pace,
    y_col="y_pace",
    individual_weight_col="pace_weight",
)

q_contrast, q_level_all = build_pair_and_level_data(
    df_quali,
    y_col="y_quali",
    individual_weight_col="quali_weight",
    session_col="session",
)

# Common Q1/Q2/Q3 sessions should not triple-count
# a pair's total qualifying contrast information.
n_common_sessions = (
    q_contrast.groupby(
        [
            "raceId",
            "constructorId",
        ]
    )["session"]
    .transform("count")
    .astype(float)
)

q_contrast["weight"] = 1.0 / n_common_sessions

# Q1 anchors the team's whole-grid qualifying level.
q_level = q_level_all[q_level_all["session"] == "q1"].copy()

q_level["weight"] = 1.0

print(
    "Pace contrasts:",
    f"{len(pace_contrast):,}",
)
print(
    "Pace levels:",
    f"{len(pace_level):,}",
)
print(
    "Qualifying contrasts:",
    f"{len(q_contrast):,}",
)
print(
    "Qualifying Q1 levels:",
    f"{len(q_level):,}",
)

display(pace_contrast.head())
display(q_contrast.head())

In [ ]:
@dataclass
class JointIndex:
    active_driver_years: list
    dy_map: dict
    unique_drivers: list
    d_map: dict
    dy_to_driver: np.ndarray

    active_constructor_years: list
    csy_map: dict

    P_driver: np.ndarray
    P_car: np.ndarray


def weighted_centering_matrix(keys, weights):
    """
    Projection matrix that removes the observation-weighted
    mean within each season.
    """
    n = len(keys)
    P = np.eye(n, dtype=float)

    years = np.array(
        [year for _, year in keys],
        dtype=int,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    for year in np.unique(years):
        ids = np.where(years == year)[0]
        w = weights[ids].copy()

        if not np.all(np.isfinite(w)) or w.sum() <= 0:
            w = np.ones(
                len(ids),
                dtype=float,
            )

        w /= w.sum()

        P[np.ix_(ids, ids)] -= np.ones((len(ids), 1)) @ w[None, :]

    return P

In [ ]:
def build_gap_aware_driver_drift_operator(
    idx,
    rho,
):
    n_dy = len(idx.active_driver_years)

    driver_years = {}

    for d, y in idx.active_driver_years:
        driver_years.setdefault(
            d,
            [],
        ).append(int(y))

    for d in driver_years:
        driver_years[d].sort()

    transitions = []

    for d in idx.unique_drivers:
        years = driver_years[d]

        for j in range(
            1,
            len(years),
        ):
            transitions.append((d, j))

    n_steps = len(transitions)

    step_map = {key: p for p, key in enumerate(transitions)}

    L = np.zeros(
        (
            n_dy,
            n_steps,
        ),
        dtype=float,
    )

    for d in idx.unique_drivers:
        years = driver_years[d]

        if len(years) <= 1:
            continue

        coeff_prev = np.zeros(
            n_steps,
            dtype=float,
        )

        for j in range(
            1,
            len(years),
        ):
            gap = years[j] - years[j - 1]

            decay = rho**gap

            if np.isclose(
                rho,
                1.0,
            ):
                gap_scale = np.sqrt(gap)
            else:
                gap_scale = np.sqrt((1.0 - rho ** (2 * gap)) / (1.0 - rho**2))

            coeff_now = decay * coeff_prev

            p = step_map[(d, j)]

            coeff_now[p] += gap_scale

            dy = idx.dy_map[
                (
                    d,
                    years[j],
                )
            ]

            L[
                dy,
                :,
            ] = coeff_now

            coeff_prev = coeff_now

    return L

In [ ]:
def add_historical_tenure(
    df,
    driver_event_cov,
):
    out = df.merge(
        driver_event_cov,
        on=[
            "raceId",
            "driverId",
            "constructorId",
        ],
        how="left",
        validate="many_to_one",
    )

    out["tenure_raw"] = out["tenure_raw"].fillna(0.0)

    out["tenure_c"] = out["tenure_raw"] - out.groupby("raceId")["tenure_raw"].transform("mean")

    return out


def build_historical_rankings(
    raw,
    driver_event_cov,
    cfg=CFG,
):
    results = raw.results.merge(
        raw.races[
            [
                "raceId",
                "year",
                "round",
            ]
        ],
        on="raceId",
        how="left",
        validate="many_to_one",
    ).copy()

    results = results[
        (results["year"] >= cfg.historical_floor_year)
        & (results["year"] <= cfg.historical_ceiling_year)
    ].copy()

    results["grid"] = pd.to_numeric(
        results["grid"],
        errors="coerce",
    )

    results["positionOrder"] = pd.to_numeric(
        results["positionOrder"],
        errors="coerce",
    )

    results["status_str"] = (
        results["statusId"].map(raw.status_map).fillna("").astype(str).str.lower()
    )

    results["driver"] = results["driverId"].map(raw.driver_ref)

    # ======================================================
    # HISTORICAL QUALIFYING
    #
    # Starting-grid order is used as an ordinal qualifying
    # observation. Grid == 0 is excluded.
    # ======================================================

    hist_q = results[results["grid"].notna() & (results["grid"] > 0)][
        [
            "raceId",
            "year",
            "round",
            "driverId",
            "driver",
            "constructorId",
            "grid",
        ]
    ].copy()

    hist_q = hist_q.sort_values(["raceId", "grid"]).drop_duplicates(["raceId", "driverId"])

    q_field_size = hist_q.groupby("raceId")["driverId"].transform("nunique")

    hist_q = hist_q[q_field_size >= 10].copy()

    hist_q["rank_value"] = hist_q["grid"]

    hist_q = add_historical_tenure(
        hist_q,
        driver_event_cov,
    )

    # ======================================================
    # HISTORICAL RACE ORDER
    #
    # Only classified finishers are ranked:
    #   Finished
    #   +1 Lap
    #   +2 Laps
    #   ...
    #
    # We intentionally do not assign an ordinal race result
    # to retirements, because retirement order is a poor
    # proxy for latent race pace.
    # ======================================================

    classified = results["status_str"].str.fullmatch(
        r"finished|\+\d+\s+laps?",
        na=False,
    )

    hist_race = results[
        classified & results["positionOrder"].notna() & (results["positionOrder"] > 0)
    ][
        [
            "raceId",
            "year",
            "round",
            "driverId",
            "driver",
            "constructorId",
            "positionOrder",
            "status_str",
        ]
    ].copy()

    hist_race = hist_race.sort_values(["raceId", "positionOrder"]).drop_duplicates(
        ["raceId", "driverId"]
    )

    race_field_size = hist_race.groupby("raceId")["driverId"].transform("nunique")

    hist_race = hist_race[race_field_size >= 8].copy()

    hist_race["rank_value"] = hist_race["positionOrder"]

    hist_race = add_historical_tenure(
        hist_race,
        driver_event_cov,
    )

    return (
        hist_q.reset_index(drop=True),
        hist_race.reset_index(drop=True),
    )


hist_quali, hist_race = build_historical_rankings(
    raw,
    driver_event_cov,
    CFG,
)


print("Historical qualifying:")
print(f"  observations = {len(hist_quali):,}")
print(f"  races        = {hist_quali['raceId'].nunique():,}")
print(f"  years        = " f"{hist_quali['year'].min()}–" f"{hist_quali['year'].max()}")

print("\nHistorical classified race results:")
print(f"  observations = {len(hist_race):,}")
print(f"  races        = {hist_race['raceId'].nunique():,}")
print(f"  years        = " f"{hist_race['year'].min()}–" f"{hist_race['year'].max()}")

print("\nHistorical qualifying field size:")
display(hist_quali.groupby("raceId").size().describe())

print("Historical classified race field size:")
display(hist_race.groupby("raceId").size().describe())

In [ ]:
# Copies because these numeric index columns will now refer
# to a new, larger state vector.
pace_contrast_all = pace_contrast.copy()

pace_level_all = pace_level.copy()

q_contrast_all = q_contrast.copy()

q_level_all_model = q_level.copy()


def build_combined_index(
    df_pace,
    df_quali,
    modern_pair_tables,
    hist_quali,
    hist_race,
):

    used_dy = set()
    used_csy = set()

    # ------------------------------------------------------
    # Modern states
    # ------------------------------------------------------

    for df in modern_pair_tables:

        used_dy.update(
            (
                d,
                int(y),
            )
            for d, y in zip(
                df["driver_a"],
                df["year"],
            )
        )

        used_dy.update(
            (
                d,
                int(y),
            )
            for d, y in zip(
                df["driver_b"],
                df["year"],
            )
        )

        used_csy.update(
            (
                int(c),
                int(y),
            )
            for c, y in zip(
                df["constructorId"],
                df["year"],
            )
        )

    # ------------------------------------------------------
    # Historical states
    # ------------------------------------------------------

    for df in (
        hist_quali,
        hist_race,
    ):

        used_dy.update(
            (
                d,
                int(y),
            )
            for d, y in zip(
                df["driver"],
                df["year"],
            )
        )

        used_csy.update(
            (
                int(c),
                int(y),
            )
            for c, y in zip(
                df["constructorId"],
                df["year"],
            )
        )

    active_driver_years = sorted(used_dy)

    dy_map = {key: i for i, key in enumerate(active_driver_years)}

    unique_drivers = sorted({d for d, _ in active_driver_years})

    d_map = {d: i for i, d in enumerate(unique_drivers)}

    dy_to_driver = np.array(
        [d_map[d] for d, _ in active_driver_years],
        dtype=np.int64,
    )

    active_constructor_years = sorted(used_csy)

    csy_map = {key: i for i, key in enumerate(active_constructor_years)}

    # ======================================================
    # CENTRING WEIGHTS
    #
    # Modern:
    #   one race-pace contribution
    #   ~one total qualifying contribution
    #
    # Historical:
    #   one qualifying rank
    #   one classified-race rank
    #
    # This approximates occupied-seat / race weighting.
    # ======================================================

    modern_obs = pd.concat(
        [
            df_pace[
                [
                    "driver",
                    "constructorId",
                    "year",
                    "pace_weight",
                ]
            ].rename(columns={"pace_weight": "weight"}),
            df_quali[
                [
                    "driver",
                    "constructorId",
                    "year",
                    "quali_weight",
                ]
            ].rename(columns={"quali_weight": "weight"}),
        ],
        ignore_index=True,
    )

    hist_q_obs = hist_quali[
        [
            "driver",
            "constructorId",
            "year",
        ]
    ].copy()

    hist_q_obs["weight"] = 1.0

    hist_r_obs = hist_race[
        [
            "driver",
            "constructorId",
            "year",
        ]
    ].copy()

    hist_r_obs["weight"] = 1.0

    obs = pd.concat(
        [
            modern_obs,
            hist_q_obs,
            hist_r_obs,
        ],
        ignore_index=True,
    )

    dy_weight = np.zeros(
        len(active_driver_years),
        dtype=float,
    )

    csy_weight = np.zeros(
        len(active_constructor_years),
        dtype=float,
    )

    for row in obs.itertuples(index=False):

        dy_key = (
            row.driver,
            int(row.year),
        )

        csy_key = (
            int(row.constructorId),
            int(row.year),
        )

        if dy_key not in dy_map or csy_key not in csy_map:
            continue

        dy_weight[dy_map[dy_key]] += float(row.weight)

        csy_weight[csy_map[csy_key]] += float(row.weight)

    P_driver = weighted_centering_matrix(
        active_driver_years,
        dy_weight,
    )

    P_car = weighted_centering_matrix(
        active_constructor_years,
        csy_weight,
    )

    combined_idx = JointIndex(
        active_driver_years=active_driver_years,
        dy_map=dy_map,
        unique_drivers=unique_drivers,
        d_map=d_map,
        dy_to_driver=dy_to_driver,
        active_constructor_years=active_constructor_years,
        csy_map=csy_map,
        P_driver=P_driver,
        P_car=P_car,
    )

    # ------------------------------------------------------
    # Re-index modern likelihood tables
    # ------------------------------------------------------

    for df in modern_pair_tables:

        df["dy_a"] = [
            dy_map[
                (
                    d,
                    int(y),
                )
            ]
            for d, y in zip(
                df["driver_a"],
                df["year"],
            )
        ]

        df["dy_b"] = [
            dy_map[
                (
                    d,
                    int(y),
                )
            ]
            for d, y in zip(
                df["driver_b"],
                df["year"],
            )
        ]

        df["csy"] = [
            csy_map[
                (
                    int(c),
                    int(y),
                )
            ]
            for c, y in zip(
                df["constructorId"],
                df["year"],
            )
        ]

    # ------------------------------------------------------
    # Index historical observations
    # ------------------------------------------------------

    for df in (
        hist_quali,
        hist_race,
    ):

        df["dy"] = [
            dy_map[
                (
                    d,
                    int(y),
                )
            ]
            for d, y in zip(
                df["driver"],
                df["year"],
            )
        ]

        df["csy"] = [
            csy_map[
                (
                    int(c),
                    int(y),
                )
            ]
            for c, y in zip(
                df["constructorId"],
                df["year"],
            )
        ]

    return combined_idx


idx_all = build_combined_index(
    df_pace,
    df_quali,
    [
        pace_contrast_all,
        pace_level_all,
        q_contrast_all,
        q_level_all_model,
    ],
    hist_quali,
    hist_race,
)


L_driver_all = build_gap_aware_driver_drift_operator(
    idx_all,
    CFG.rho_driver,
)


print(
    "Combined drivers:",
    len(idx_all.unique_drivers),
)

print(
    "Combined driver-seasons:",
    len(idx_all.active_driver_years),
)

print(
    "Combined constructor-seasons:",
    len(idx_all.active_constructor_years),
)

print(
    "Combined driver transitions:",
    L_driver_all.shape[1],
)

print(
    "\nYear range:",
    min(y for _, y in idx_all.active_driver_years),
    "–",
    max(y for _, y in idx_all.active_driver_years),
)

In [ ]:
import pytensor.tensor as pt


def pack_rankings(df):
    """
    Pack variable-length race rankings into fixed arrays.

    Input rows must contain:
      raceId, year, rank_value, dy, csy, tenure_c

    Within each race, lower rank_value is better.
    """
    groups = []

    for race_id, grp in df.groupby(
        "raceId",
        sort=True,
    ):
        g = grp.sort_values("rank_value").copy()

        if len(g) < 2:
            continue

        groups.append(
            {
                "raceId": int(race_id),
                "year": int(g["year"].iloc[0]),
                "dy": g["dy"].to_numpy(dtype=np.int64),
                "csy": g["csy"].to_numpy(dtype=np.int64),
                "tenure": g["tenure_c"].to_numpy(dtype=float),
            }
        )

    if not groups:
        raise ValueError("No historical ranking groups supplied.")

    n_races = len(groups)

    max_field = max(len(g["dy"]) for g in groups)

    dy = np.zeros(
        (n_races, max_field),
        dtype=np.int64,
    )

    csy = np.zeros(
        (n_races, max_field),
        dtype=np.int64,
    )

    tenure = np.zeros(
        (n_races, max_field),
        dtype=float,
    )

    valid = np.zeros(
        (n_races, max_field),
        dtype=bool,
    )

    choice = np.zeros(
        (n_races, max_field),
        dtype=bool,
    )

    race_ids = np.zeros(
        n_races,
        dtype=np.int64,
    )

    years = np.zeros(
        n_races,
        dtype=np.int64,
    )

    for i, group in enumerate(groups):
        n = len(group["dy"])

        dy[i, :n] = group["dy"]
        csy[i, :n] = group["csy"]
        tenure[i, :n] = group["tenure"]
        valid[i, :n] = True

        # The final remaining competitor contributes
        # probability 1, so only n-1 choices matter.
        choice[i, : n - 1] = True

        race_ids[i] = group["raceId"]

        years[i] = group["year"]

    # candidate_mask[k, j] is True when competitor j
    # remains available when selecting observed position k.
    candidate_mask = np.triu(
        np.ones(
            (
                max_field,
                max_field,
            ),
            dtype=bool,
        )
    )

    return {
        "dy": dy,
        "csy": csy,
        "tenure": tenure,
        "valid": valid,
        "choice": choice,
        "candidate_mask": candidate_mask,
        "raceId": race_ids,
        "year": years,
        "n_races": n_races,
        "max_field": max_field,
    }


hist_q_pack = pack_rankings(hist_quali)

hist_race_pack = pack_rankings(hist_race)

print(
    "Historical qualifying pack:",
    hist_q_pack["n_races"],
    "races ×",
    hist_q_pack["max_field"],
    "max drivers",
)

print(
    "Historical race pack:",
    hist_race_pack["n_races"],
    "races ×",
    hist_race_pack["max_field"],
    "max drivers",
)


def plackett_luce_logp_vectorized(
    packed,
    alpha_dy,
    car_csy,
    gamma_tenure,
    temperature,
):
    """
    Vectorized Plackett-Luce log-likelihood.

    Rows = races.
    Columns = observed order, best -> worst.
    """
    dy = pt.constant(packed["dy"])

    csy = pt.constant(packed["csy"])

    tenure = pt.constant(packed["tenure"])

    valid = pt.constant(packed["valid"])

    choice = pt.constant(packed["choice"])

    candidate_mask = pt.constant(packed["candidate_mask"])

    score = alpha_dy[dy] + car_csy[csy] + gamma_tenure * tenure

    # Multiplication by inverse temperature avoids the
    # problematic vector/scalar division rewrite.
    inv_temperature = 1.0 / temperature

    z = score * inv_temperature

    allowed = (
        candidate_mask[
            None,
            :,
            :,
        ]
        & valid[
            :,
            None,
            :,
        ]
    )

    candidate_scores = pt.where(
        allowed,
        z[:, None, :],
        -1.0e30,
    )

    log_denom = pt.logsumexp(
        candidate_scores,
        axis=2,
    )

    raw_terms = z - log_denom

    terms = pt.where(
        choice,
        raw_terms,
        0.0,
    )

    return pt.sum(terms)

### Frozen model-builder architecture

Only the reusable builder definition is retained here. The intermediate pre-v2.2 model build from the development notebook is intentionally omitted.


In [ ]:
import pymc as pm
import pytensor.tensor as pt
import scipy.sparse as sp
import pytensor.sparse as ps

def sparse_matvec(
    A,
    x,
):
    """
    Sparse matrix-vector multiplication compatible with the
    PyTensor/JAX compilation path.
    """

    return ps.structured_dot(
        A,
        x[:, None],
    )[:, 0]

def build_joint_model_1982_2025(
    pace_contrast,
    pace_level,
    q_contrast,
    q_level,
    hist_q_pack,
    hist_race_pack,
    idx,
    *,
    cfg=CFG,
):
    """
    Build the frozen 1982–2025 Pantheon model.

    No statistical changes relative to the validated
    1982–2024 specification.
    """

    n_drivers = len(idx.unique_drivers)

    n_csy = len(idx.active_constructor_years)

    n_steps = L_driver_sp.shape[1]

    # ======================================================
    # FIXED RELIABILITY SCALE MULTIPLIERS
    # ======================================================

    pace_contrast_scale = 1.0 / np.sqrt(pace_contrast["weight"].to_numpy(dtype=float))

    pace_level_scale = 1.0 / np.sqrt(pace_level["weight"].to_numpy(dtype=float))

    q_contrast_scale = 1.0 / np.sqrt(q_contrast["weight"].to_numpy(dtype=float))

    q_level_scale = 1.0 / np.sqrt(q_level["weight"].to_numpy(dtype=float))

    # ======================================================
    # PRE-COMPUTE FIXED NUMPY ARRAYS
    # ======================================================

    pc_dy_a = pace_contrast["dy_a"].to_numpy(dtype=np.int64)

    pc_dy_b = pace_contrast["dy_b"].to_numpy(dtype=np.int64)

    pc_tenure = pace_contrast["tenure_diff"].to_numpy(dtype=float)

    pc_y = pace_contrast["y"].to_numpy(dtype=float)

    pl_dy_a = pace_level["dy_a"].to_numpy(dtype=np.int64)

    pl_dy_b = pace_level["dy_b"].to_numpy(dtype=np.int64)

    pl_csy = pace_level["csy"].to_numpy(dtype=np.int64)

    pl_tenure = pace_level["tenure_mean"].to_numpy(dtype=float)

    pl_y = pace_level["y"].to_numpy(dtype=float)

    qc_dy_a = q_contrast["dy_a"].to_numpy(dtype=np.int64)

    qc_dy_b = q_contrast["dy_b"].to_numpy(dtype=np.int64)

    qc_tenure = q_contrast["tenure_diff"].to_numpy(dtype=float)

    qc_y = q_contrast["y"].to_numpy(dtype=float)

    ql_dy_a = q_level["dy_a"].to_numpy(dtype=np.int64)

    ql_dy_b = q_level["dy_b"].to_numpy(dtype=np.int64)

    ql_csy = q_level["csy"].to_numpy(dtype=np.int64)

    ql_tenure = q_level["tenure_mean"].to_numpy(dtype=float)

    ql_y = q_level["y"].to_numpy(dtype=float)

    # ======================================================
    # MODEL
    # ======================================================

    with pm.Model() as model:

        # ==================================================
        # DRIVER BASELINE
        # ==================================================

        sigma_driver_base = pm.HalfNormal(
            "sigma_driver_base",
            sigma=cfg.sigma_driver_base_prior,
        )

        theta_z = pm.Normal(
            "theta_z",
            mu=0.0,
            sigma=1.0,
            shape=n_drivers,
        )

        theta_raw_dy = sigma_driver_base * theta_z[idx.dy_to_driver]

        # ==================================================
        # DRIVER SEASONAL DRIFT
        # ==================================================

        sigma_driver_drift = pm.HalfNormal(
            "sigma_driver_drift",
            sigma=cfg.sigma_driver_drift_prior,
        )

        drift_z = pm.Normal(
            "drift_z",
            mu=0.0,
            sigma=1.0,
            shape=n_steps,
        )

        raw_alpha = theta_raw_dy + sparse_matvec(
            L_driver_pt,
            sigma_driver_drift * drift_z,
        )

        alpha_dy = pm.Deterministic(
            "alpha_dy",
            sparse_matvec(
                P_driver_pt,
                raw_alpha,
            ),
        )

        # ==================================================
        # CONSTRUCTOR-SEASON EFFECT
        # ==================================================

        sigma_car = pm.HalfNormal(
            "sigma_car",
            sigma=cfg.sigma_car_prior,
        )

        car_z = pm.Normal(
            "car_z",
            mu=0.0,
            sigma=1.0,
            shape=n_csy,
        )

        car_csy = pm.Deterministic(
            "car_csy",
            sparse_matvec(
                P_car_pt,
                sigma_car * car_z,
            ),
        )

        # ==================================================
        # CONSTRUCTOR TENURE
        # ==================================================

        gamma_tenure = pm.HalfNormal(
            "gamma_tenure",
            sigma=cfg.gamma_tenure_sigma,
        )

        # ==================================================
        # MODERN RESIDUAL SCALES
        # ==================================================

        sigma_pace_contrast = pm.HalfNormal(
            "sigma_pace_contrast",
            sigma=cfg.obs_sigma_prior,
        )

        sigma_pace_level = pm.HalfNormal(
            "sigma_pace_level",
            sigma=cfg.obs_sigma_prior,
        )

        sigma_q_contrast = pm.HalfNormal(
            "sigma_q_contrast",
            sigma=cfg.obs_sigma_prior,
        )

        sigma_q_level = pm.HalfNormal(
            "sigma_q_level",
            sigma=cfg.obs_sigma_prior,
        )

        # ==================================================
        # HISTORICAL TEMPERATURES
        # ==================================================

        tau_hist_quali = pm.LogNormal(
            "tau_hist_quali",
            mu=np.log(cfg.hist_quali_tau_median),
            sigma=cfg.hist_tau_log_sigma,
        )

        tau_hist_race = pm.LogNormal(
            "tau_hist_race",
            mu=np.log(cfg.hist_race_tau_median),
            sigma=cfg.hist_tau_log_sigma,
        )

        # ==================================================
        # MODERN PACE — TEAMMATE CONTRAST
        # ==================================================

        mu_pc = alpha_dy[pc_dy_a] - alpha_dy[pc_dy_b] + gamma_tenure * pc_tenure

        pm.StudentT(
            "like_pace_contrast",
            nu=cfg.student_t_nu,
            mu=mu_pc,
            sigma=(sigma_pace_contrast * pt.constant(pace_contrast_scale)),
            observed=pc_y,
        )

        # ==================================================
        # MODERN PACE — TEAM LEVEL
        # ==================================================

        mu_pl = (
            0.5 * (alpha_dy[pl_dy_a] + alpha_dy[pl_dy_b])
            + car_csy[pl_csy]
            + gamma_tenure * pl_tenure
        )

        pm.StudentT(
            "like_pace_level",
            nu=cfg.student_t_nu,
            mu=mu_pl,
            sigma=(sigma_pace_level * pt.constant(pace_level_scale)),
            observed=pl_y,
        )

        # ==================================================
        # MODERN QUALIFYING — TEAMMATE CONTRAST
        # ==================================================

        mu_qc = alpha_dy[qc_dy_a] - alpha_dy[qc_dy_b] + gamma_tenure * qc_tenure

        pm.StudentT(
            "like_q_contrast",
            nu=cfg.student_t_nu,
            mu=mu_qc,
            sigma=(sigma_q_contrast * pt.constant(q_contrast_scale)),
            observed=qc_y,
        )

        # ==================================================
        # MODERN Q1 — TEAM LEVEL
        # ==================================================

        mu_ql = (
            0.5 * (alpha_dy[ql_dy_a] + alpha_dy[ql_dy_b])
            + car_csy[ql_csy]
            + gamma_tenure * ql_tenure
        )

        pm.StudentT(
            "like_q_level",
            nu=cfg.student_t_nu,
            mu=mu_ql,
            sigma=(sigma_q_level * pt.constant(q_level_scale)),
            observed=ql_y,
        )

        # ==================================================
        # HISTORICAL QUALIFYING
        # ==================================================

        hist_q_logp = plackett_luce_logp_vectorized(
            hist_q_pack,
            alpha_dy,
            car_csy,
            gamma_tenure,
            tau_hist_quali,
        )

        pm.Potential(
            "like_hist_quali",
            hist_q_logp,
        )

        # ==================================================
        # HISTORICAL CLASSIFIED RACE ORDER
        # ==================================================

        hist_race_logp = plackett_luce_logp_vectorized(
            hist_race_pack,
            alpha_dy,
            car_csy,
            gamma_tenure,
            tau_hist_race,
        )

        pm.Potential(
            "like_hist_race",
            hist_race_logp,
        )

    return model


In [ ]:
# Canonical names expected by the familiarity / LIVE rebuild stages.
pace_contrast_all = pace_contrast.copy()
pace_level_all = pace_level.copy()
q_contrast_all = q_contrast.copy()
q_level_all_model = q_level.copy()

print(
    "Pre-familiarity modern likelihood rows:",
    {
        "pace_contrast": len(pace_contrast_all),
        "pace_level": len(pace_level_all),
        "q_contrast": len(q_contrast_all),
        "q1_level": len(q_level_all_model),
    },
)


## Part IV — Locked organisational familiarity

The exploratory half-life selection work is not rerun in the production path. The previously selected **20-GP missed-weekend half-life** remains locked.

The development notebook discovered a 2015 Marussia/Manor source-table alias. In this clean notebook the alias patch is applied before the observation-safe familiarity table is rebuilt, so no manual cell rerun is required.


In [ ]:
# ============================================================
# STAGE 32F — ORGANISATIONAL FAMILIARITY MEMORY AUDIT
# ============================================================
#
# PURPOSE
# -------
# Stage 32E established that the frozen production tenure variable was:
#
#     tanh(previous raw.results appearances for literal constructor / 18)
#
# i.e. it used RESULT APPEARANCES / RACE WEEKENDS, not actual race starts.
#
# The remaining conceptual problem is that literal constructor IDs:
#   1. reset familiarity across genuine organisational rebrands;
#   2. cannot handle short departures / returns sensibly.
#
# This stage therefore constructs alternative familiarity stocks for the
# SAME UNDERLYING ORGANISATION.
#
# Key idea:
#
#     experience stock with organisation L
#
# accumulates by one for every raw.results appearance with L.
#
# While the driver is elsewhere, the stored stock decays according to
# MISSED F1 RACE WEEKENDS:
#
#     E_new = E_old * 0.5 ** (missed_GPs / half_life_GPs)
#
# The familiar production saturation transform is retained:
#
#     tenure = tanh(E / 18)
#
# Candidate definitions:
#
#   literal_production  : frozen-model literal constructor definition
#   strict_stint        : reset after any interrupted appearance streak
#   memory_10gp         : 10 missed-GP half-life  (~0.5 modern season)
#   memory_20gp         : 20 missed-GP half-life  (~1 modern season)
#   memory_40gp         : 40 missed-GP half-life  (~2 modern seasons)
#   memory_80gp         : 80 missed-GP half-life  (~4 modern seasons)
#   no_decay            : all prior experience with organisation retained
#
# IMPORTANT:
#   - Uses raw.results appearances, matching Stage 32E.
#   - Familiarity is measured BEFORE the current weekend.
#   - Rebrands carry familiarity.
#   - A return after a short spell elsewhere does NOT automatically reset.
#   - Long absences naturally decay.
#   - Cadillac remains genuinely new.
#   - RB/Faenza and Red Bull/Milton Keynes remain DIFFERENT organisations.
#
# THIS CELL DOES NOT:
#   - mutate the production tables;
#   - replace Stage 3;
#   - build a PyMC model;
#   - sample anything.
#
# ============================================================

from pathlib import Path
from collections import defaultdict
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. OUTPUT DIRECTORY
# ============================================================

STAGE32F_OUT = Path(
    "model_outputs"
) / "joint_driver_car_v2_1_1982_2026_LIVE" / \
    "validation_organisational_familiarity_memory_audit"

STAGE32F_OUT.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. BASIC CHECKS / FIND MODERN TABLES
# ============================================================

if "raw" not in globals():
    raise RuntimeError(
        "Stage 32F requires the loaded raw F1 data object `raw`."
    )


def _pick_dataframe(label, candidate_names):
    """
    Find the first existing pandas DataFrame from candidate global names.
    """
    for name in candidate_names:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame):
            print(f"{label}: using `{name}` ({len(obj):,} rows)")
            return obj.copy(), name

    raise RuntimeError(
        f"Could not find {label}.\n"
        f"Tried: {candidate_names}"
    )


pace_contrast_32f, pace_contrast_name_32f = _pick_dataframe(
    "Pace contrast table",
    [
        "pace_contrast_all",
        "pace_contrasts_all",
        "pace_contrast",
        "pace_contrasts",
    ],
)

pace_level_32f, pace_level_name_32f = _pick_dataframe(
    "Pace level table",
    [
        "pace_level_all",
        "pace_levels_all",
        "pace_level",
        "pace_levels",
    ],
)

q_contrast_32f, q_contrast_name_32f = _pick_dataframe(
    "Qualifying contrast table",
    [
        "q_contrast_all",
        "q_contrasts_all",
        "quali_contrast_all",
        "qualifying_contrast_all",
        "q_contrast",
        "q_contrasts",
    ],
)

q1_level_32f, q1_level_name_32f = _pick_dataframe(
    "Q1 level table",
    [
        "q_level_all_model",
        "q1_level_all",
        "q_level_all",
        "q1_levels_all",
        "q_level",
        "q1_level",
    ],
)


required_pair_cols = {
    "raceId",
    "driver_a",
    "driver_b",
}

for label, df in [
    ("pace contrast", pace_contrast_32f),
    ("pace level", pace_level_32f),
    ("qualifying contrast", q_contrast_32f),
    ("Q1 level", q1_level_32f),
]:
    missing = required_pair_cols - set(df.columns)

    if missing:
        raise RuntimeError(
            f"{label} table is missing required columns: {missing}"
        )


# ============================================================
# 2. RAW TABLES
# ============================================================

races_32f = raw.races.copy()
results_32f = raw.results.copy()
drivers_32f = raw.drivers.copy()
constructors_32f = raw.constructors.copy()


for col in ["raceId"]:
    races_32f[col] = pd.to_numeric(
        races_32f[col],
        errors="coerce",
    )

for col in ["raceId", "driverId", "constructorId"]:
    results_32f[col] = pd.to_numeric(
        results_32f[col],
        errors="coerce",
    )

drivers_32f["driverId"] = pd.to_numeric(
    drivers_32f["driverId"],
    errors="coerce",
)

constructors_32f["constructorId"] = pd.to_numeric(
    constructors_32f["constructorId"],
    errors="coerce",
)


driver_ref_col = (
    "driverRef"
    if "driverRef" in drivers_32f.columns
    else "driver"
)

constructor_ref_col = (
    "constructorRef"
    if "constructorRef" in constructors_32f.columns
    else "constructor"
)

constructor_name_col = (
    "name"
    if "name" in constructors_32f.columns
    else constructor_ref_col
)


# ============================================================
# 3. GLOBAL RACE ORDER
# ============================================================

race_order_32f = races_32f[
    ["raceId", "year", "round", "name"]
].copy()

race_order_32f["year"] = pd.to_numeric(
    race_order_32f["year"],
    errors="coerce",
)

race_order_32f["round"] = pd.to_numeric(
    race_order_32f["round"],
    errors="coerce",
)

race_order_32f = (
    race_order_32f
    .dropna(subset=["raceId", "year", "round"])
    .sort_values(
        ["year", "round", "raceId"],
        kind="mergesort",
    )
    .drop_duplicates("raceId")
    .reset_index(drop=True)
)

race_order_32f["race_ord"] = np.arange(
    len(race_order_32f),
    dtype=int,
)


# ============================================================
# 4. CURATED ORGANISATIONAL LINEAGE RULES
# ============================================================
#
# These are deliberately year-bounded.
#
# This prevents e.g.
#   1980s works Renault
# from being confused with
#   2000s Enstone Renault,
#
# or historical Alfa Romeo from being confused with
#   2019–23 Hinwil Alfa Romeo.
#
# ============================================================

LINEAGE_RULES_32F = [

    # ---------------- ENSTONE ----------------
    ("toleman",       1981, 1985, "ENSTONE"),
    ("benetton",      1986, 2001, "ENSTONE"),
    ("renault",       2002, 2011, "ENSTONE"),
    ("lotus_f1",      2012, 2015, "ENSTONE"),
    ("renault",       2016, 2020, "ENSTONE"),
    ("alpine",        2021, 9999, "ENSTONE"),

    # ---------------- BRACKLEY ----------------
    ("bar",           1999, 2005, "BRACKLEY"),
    ("honda",         2006, 2008, "BRACKLEY"),
    ("brawn",         2009, 2009, "BRACKLEY"),
    ("mercedes",      2010, 9999, "BRACKLEY"),

    # ---------------- SILVERSTONE ----------------
    ("jordan",        1991, 2005, "SILVERSTONE"),
    ("mf1",           2006, 2006, "SILVERSTONE"),
    ("midland",       2006, 2006, "SILVERSTONE"),
    ("spyker_mf1",    2006, 2006, "SILVERSTONE"),
    ("spyker",        2007, 2007, "SILVERSTONE"),
    ("force_india",   2008, 2018, "SILVERSTONE"),
    ("racing_point",  2019, 2020, "SILVERSTONE"),
    ("aston_martin",  2021, 9999, "SILVERSTONE"),

    # ---------------- FAENZA ----------------
    ("minardi",       1985, 2005, "FAENZA"),
    ("toro_rosso",    2006, 2019, "FAENZA"),
    ("alphatauri",    2020, 2023, "FAENZA"),
    ("rb",            2024, 9999, "FAENZA"),

    # ---------------- HINWIL ----------------
    ("sauber",        1993, 2005, "HINWIL"),
    ("bmw_sauber",    2006, 2009, "HINWIL"),
    ("sauber",        2010, 2018, "HINWIL"),
    ("alfa",          2019, 2023, "HINWIL"),
    ("sauber",        2024, 2025, "HINWIL"),
    ("audi",          2026, 9999, "HINWIL"),

    # ---------------- MILTON KEYNES ----------------
    ("stewart",       1997, 1999, "MILTON_KEYNES"),
    ("jaguar",        2000, 2004, "MILTON_KEYNES"),
    ("red_bull",      2005, 9999, "MILTON_KEYNES"),

    # ---------------- LIGIER / PROST ----------------
    ("ligier",        1976, 1996, "LIGIER_PROST"),
    ("prost",         1997, 2001, "LIGIER_PROST"),

    # ---------------- ARROWS / FOOTWORK ----------------
    ("arrows",        1978, 1990, "ARROWS_FOOTWORK"),
    ("footwork",      1991, 1996, "ARROWS_FOOTWORK"),
    ("arrows",        1997, 2002, "ARROWS_FOOTWORK"),

    # ---------------- MARCH / LEYTON HOUSE ----------------
    ("march",         1987, 1989, "MARCH_LEYTON"),
    ("leyton",        1990, 1991, "MARCH_LEYTON"),
    ("march",         1992, 1992, "MARCH_LEYTON"),

    # ---------------- VIRGIN / MARUSSIA / MANOR ----------------
    ("virgin",        2010, 2011, "VIRGIN_MARUSSIA_MANOR"),
    ("marussia",      2012, 2015, "VIRGIN_MARUSSIA_MANOR"),
    ("manor_marussia",2015, 2016, "VIRGIN_MARUSSIA_MANOR"),
    ("manor",         2016, 2016, "VIRGIN_MARUSSIA_MANOR"),

    # ---------------- TEAM LOTUS / CATERHAM ----------------
    ("lotus_racing",  2010, 2011, "TEAM_LOTUS_CATERHAM"),
    ("caterham",      2012, 2014, "TEAM_LOTUS_CATERHAM"),
]


lineage_rules_df_32f = pd.DataFrame(
    LINEAGE_RULES_32F,
    columns=[
        "constructorRef",
        "start_year",
        "end_year",
        "lineage",
    ],
)

lineage_rules_df_32f.to_csv(
    STAGE32F_OUT / "lineage_rules.csv",
    index=False,
)


# ============================================================
# 5. RESULT-APPEARANCE CONTEXT TABLE
#    + HISTORICAL DUPLICATE-WEEKEND HANDLING
# ============================================================
#
# raw.results is not globally unique in raceId x driverId.
#
# In some older races the same driver can appear in multiple result
# records within the same Grand Prix. For team-familiarity purposes,
# however, a Grand Prix is ONE exposure weekend.
#
# Therefore:
#
#   • first construct every raw result context;
#   • assign organisational lineage;
#   • audit duplicate driver/race records;
#   • collapse them to ONE driver/race exposure;
#   • refuse to continue if a model-era duplicate is structurally
#     ambiguous across multiple organisational lineages.
#
# This is preferable to blindly counting raw result rows, because
# familiarity is intended to represent experience with the team/car
# environment, not database row multiplicity.
#
# ============================================================

contexts_32f = (
    results_32f[
        ["raceId", "driverId", "constructorId"]
    ]
    .merge(
        race_order_32f,
        on="raceId",
        how="left",
        validate="many_to_one",
    )
    .merge(
        drivers_32f[
            ["driverId", driver_ref_col]
        ],
        on="driverId",
        how="left",
        validate="many_to_one",
    )
    .merge(
        constructors_32f[
            [
                "constructorId",
                constructor_ref_col,
                constructor_name_col,
            ]
        ].drop_duplicates("constructorId"),
        on="constructorId",
        how="left",
        validate="many_to_one",
    )
)

contexts_32f = contexts_32f.rename(
    columns={
        driver_ref_col: "driver",
        constructor_ref_col: "constructorRef",
        constructor_name_col: "constructor",
    }
)

contexts_32f["driver"] = (
    contexts_32f["driver"]
    .astype(str)
)

contexts_32f["constructorRef"] = (
    contexts_32f["constructorRef"]
    .astype(str)
)

contexts_32f = contexts_32f.dropna(
    subset=[
        "raceId",
        "driverId",
        "constructorId",
        "year",
        "round",
        "race_ord",
    ]
).copy()

contexts_32f["year"] = (
    contexts_32f["year"]
    .astype(int)
)

contexts_32f["race_ord"] = (
    contexts_32f["race_ord"]
    .astype(int)
)


# ============================================================
# 6. ASSIGN ORGANISATIONAL LINEAGE BEFORE DEDUPLICATION
# ============================================================

contexts_32f["lineage"] = (
    "LITERAL::"
    + contexts_32f["constructorRef"]
)

for (
    ref,
    start_year,
    end_year,
    lineage,
) in LINEAGE_RULES_32F:

    mask = (
        contexts_32f["constructorRef"].eq(ref)
        & contexts_32f["year"].between(
            start_year,
            end_year,
        )
    )

    contexts_32f.loc[
        mask,
        "lineage",
    ] = lineage


# ============================================================
# 6A. AUDIT DUPLICATE DRIVER/RACE RESULT RECORDS
# ============================================================
#
# First attach the race name explicitly. Do not assume that the
# earlier merge happened to preserve a column called "name".
#
# ============================================================

# ------------------------------------------------------------
# Explicit race-name lookup
# ------------------------------------------------------------

race_name_lookup_32f = (
    raw.races[
        ["raceId", "name"]
    ]
    .drop_duplicates("raceId")
    .rename(
        columns={
            "name": "race_name"
        }
    )
)

# Remove any stale race_name column in case this section is rerun.
if "race_name" in contexts_32f.columns:
    contexts_32f = contexts_32f.drop(
        columns=["race_name"]
    )

contexts_32f = contexts_32f.merge(
    race_name_lookup_32f,
    on="raceId",
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# Locate duplicate driver/race result records
# ------------------------------------------------------------

duplicate_mask_32f = (
    contexts_32f.duplicated(
        ["raceId", "driverId"],
        keep=False,
    )
)

duplicate_rows_32f = (
    contexts_32f.loc[
        duplicate_mask_32f
    ]
    .copy()
    .sort_values(
        [
            "year",
            "round",
            "driver",
            "constructorRef",
        ]
    )
)


# ------------------------------------------------------------
# Summarise duplicates
# ------------------------------------------------------------

if len(duplicate_rows_32f):

    duplicate_summary_32f = (
        duplicate_rows_32f
        .groupby(
            [
                "raceId",
                "driverId",
            ],
            as_index=False,
        )
        .agg(
            year=(
                "year",
                "first",
            ),

            round=(
                "round",
                "first",
            ),

            race=(
                "race_name",
                "first",
            ),

            driver=(
                "driver",
                "first",
            ),

            result_rows=(
                "constructorId",
                "size",
            ),

            n_constructor_ids=(
                "constructorId",
                "nunique",
            ),

            n_constructor_refs=(
                "constructorRef",
                "nunique",
            ),

            n_lineages=(
                "lineage",
                "nunique",
            ),

            constructor_refs=(
                "constructorRef",
                lambda s:
                    ", ".join(
                        sorted(
                            set(
                                map(str, s)
                            )
                        )
                    ),
            ),

            lineages=(
                "lineage",
                lambda s:
                    ", ".join(
                        sorted(
                            set(
                                map(str, s)
                            )
                        )
                    ),
            ),
        )
        .sort_values(
            [
                "year",
                "round",
                "driver",
            ]
        )
        .reset_index(drop=True)
    )

else:

    duplicate_summary_32f = pd.DataFrame(
        columns=[
            "raceId",
            "driverId",
            "year",
            "round",
            "race",
            "driver",
            "result_rows",
            "n_constructor_ids",
            "n_constructor_refs",
            "n_lineages",
            "constructor_refs",
            "lineages",
        ]
    )


# ------------------------------------------------------------
# Print audit
# ------------------------------------------------------------

print()
print("=" * 76)
print("RAW.RESULTS DUPLICATE DRIVER/RACE AUDIT")
print("=" * 76)

print(
    f"Duplicate result rows: "
    f"{len(duplicate_rows_32f):,}"
)

print(
    f"Duplicate driver/race combinations: "
    f"{len(duplicate_summary_32f):,}"
)

if len(duplicate_summary_32f):

    print(
        f"Year range: "
        f"{int(duplicate_summary_32f['year'].min())}–"
        f"{int(duplicate_summary_32f['year'].max())}"
    )

    print(
        "Duplicate driver/races from 1982 onward:",
        int(
            (
                duplicate_summary_32f[
                    "year"
                ] >= 1982
            ).sum()
        ),
    )

    print(
        "Duplicate driver/races spanning >1 constructor ID:",
        int(
            (
                duplicate_summary_32f[
                    "n_constructor_ids"
                ] > 1
            ).sum()
        ),
    )

    print(
        "Duplicate driver/races spanning >1 organisational lineage:",
        int(
            (
                duplicate_summary_32f[
                    "n_lineages"
                ] > 1
            ).sum()
        ),
    )

    display(
        duplicate_summary_32f
    )

else:

    print(
        "No duplicate driver/race result records found."
    )


# ============================================================
# 6B. STRUCTURAL SAFETY CHECK
# ============================================================
#
# A duplicate OLD historical result is fine: we collapse it to one
# weekend of familiarity.
#
# What would be concerning is a 1982+ driver/race represented under
# more than one organisational lineage, because then "which team was
# he familiar with that weekend?" would genuinely be ambiguous for
# the modelled period.
#
# ============================================================

ambiguous_model_era_duplicates_32f = (
    duplicate_summary_32f.loc[
        (
            duplicate_summary_32f[
                "year"
            ] >= 1982
        )
        & (
            duplicate_summary_32f[
                "n_lineages"
            ] > 1
        )
    ]
    .copy()
)


if len(
    ambiguous_model_era_duplicates_32f
):

    raise RuntimeError(
        "\nStage 32F found a 1982+ duplicate "
        "driver/race spanning multiple organisational "
        "lineages.\n\n"
        "This needs manual inspection before constructing "
        "familiarity.\n\n"
        + ambiguous_model_era_duplicates_32f.to_string(
            index=False
        )
    )


# ============================================================
# 6C. COLLAPSE TO ONE FAMILIARITY EXPOSURE PER DRIVER / GP
# ============================================================
#
# A GP weekend contributes at most ONE unit of familiarity.
#
# If duplicate rows belong to the same organisational lineage, the
# particular constructor record chosen does not affect the lineage
# familiarity calculation.
#
# We retain the first row only as a representative context.
#
# ============================================================

contexts_before_collapse_32f = len(
    contexts_32f
)

contexts_32f = (
    contexts_32f
    .sort_values(
        [
            "driverId",
            "race_ord",
            "constructorId",
        ],
        kind="mergesort",
    )
    .drop_duplicates(
        [
            "raceId",
            "driverId",
        ],
        keep="first",
    )
    .sort_values(
        [
            "driverId",
            "race_ord",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

contexts_removed_by_collapse_32f = (
    contexts_before_collapse_32f
    - len(contexts_32f)
)


# Final uniqueness assertion.
remaining_duplicate_count_32f = int(
    contexts_32f.duplicated(
        [
            "raceId",
            "driverId",
        ]
    ).sum()
)

if remaining_duplicate_count_32f:
    raise RuntimeError(
        "Duplicate driver/race rows remain after collapse."
    )


print()
print("=" * 76)
print("FAMILIARITY-WEEKEND COLLAPSE")
print("=" * 76)

print(
    f"Raw result contexts before collapse: "
    f"{contexts_before_collapse_32f:,}"
)

print(
    f"Duplicate rows removed: "
    f"{contexts_removed_by_collapse_32f:,}"
)

print(
    f"Unique driver/GP familiarity contexts: "
    f"{len(contexts_32f):,}"
)

print(
    f"Remaining duplicate driver/race contexts: "
    f"{remaining_duplicate_count_32f}"
)


# Save the duplicate audit now.
duplicate_summary_32f.to_csv(
    STAGE32F_OUT
    / "raw_results_duplicate_driver_race_audit.csv",
    index=False,
)

duplicate_rows_32f.to_csv(
    STAGE32F_OUT
    / "raw_results_duplicate_driver_race_rows.csv",
    index=False,
)

print(
    "\nHistorical duplicate audit saved."
)


# ============================================================
# 7. FAMILIARITY MEMORY DEFINITIONS
# ============================================================
#
# Half-life is expressed in MISSED GRAND PRIX WEEKENDS.
#
# In the modern calendar:
#
#     10 GP ~ half a season
#     20 GP ~ one season
#     40 GP ~ two seasons
#     80 GP ~ four seasons
#
# Importantly, normal consecutive weekends at the same team have:
#
#     missed_gps = 0
#
# so familiarity accumulates without decay.
#
# ============================================================

MEMORY_HALF_LIVES_32F = {
    "memory_10gp": 10.0,
    "memory_20gp": 20.0,
    "memory_40gp": 40.0,
    "memory_80gp": 80.0,
}

FAMILIARITY_SCALE_32F = 18.0


def _sat(stock):
    return np.tanh(
        np.asarray(stock, dtype=float)
        / FAMILIARITY_SCALE_32F
    )


# Initialise output columns.
contexts_32f["literal_appearances_before"] = 0
contexts_32f["lineage_appearances_before"] = 0

contexts_32f["fam_literal_production"] = np.nan
contexts_32f["fam_strict_stint"] = np.nan
contexts_32f["fam_no_decay"] = np.nan

for cand in MEMORY_HALF_LIVES_32F:
    contexts_32f[f"stock_{cand}"] = np.nan
    contexts_32f[f"fam_{cand}"] = np.nan

contexts_32f["previous_lineage"] = None
contexts_32f["last_same_lineage_race_ord"] = np.nan
contexts_32f["missed_gps_since_same_lineage"] = np.nan
contexts_32f["return_to_known_lineage"] = False


# ------------------------------------------------------------
# Driver-by-driver recursive construction
# ------------------------------------------------------------

for driver_id, idx in contexts_32f.groupby(
    "driverId",
    sort=False,
).groups.items():

    driver_idx = list(idx)

    # Frozen production-style literal-constructor counts.
    literal_counts = defaultdict(int)

    # No-decay organisational counts.
    lineage_counts = defaultdict(int)

    # Lazy-decay states:
    # candidate -> lineage -> (stock_after_last_exposure,
    #                          race_ord_of_last_exposure)
    decay_state = {
        cand: {}
        for cand in MEMORY_HALF_LIVES_32F
    }

    # Last exposure information.
    last_lineage_ord = {}
    previous_lineage = None
    previous_race_ord = None

    # Strict appearance-streak familiarity.
    strict_stint_count_after_previous = 0

    for row_i in driver_idx:

        row = contexts_32f.loc[row_i]

        ref = row["constructorRef"]
        lineage = row["lineage"]
        race_ord = int(row["race_ord"])

        # ----------------------------------------------------
        # Frozen literal-constructor production definition
        # ----------------------------------------------------

        n_literal_before = literal_counts[ref]

        contexts_32f.at[
            row_i,
            "literal_appearances_before",
        ] = n_literal_before

        contexts_32f.at[
            row_i,
            "fam_literal_production",
        ] = float(_sat(n_literal_before))


        # ----------------------------------------------------
        # No-decay organisational memory
        # ----------------------------------------------------

        n_lineage_before = lineage_counts[lineage]

        contexts_32f.at[
            row_i,
            "lineage_appearances_before",
        ] = n_lineage_before

        contexts_32f.at[
            row_i,
            "fam_no_decay",
        ] = float(_sat(n_lineage_before))


        # ----------------------------------------------------
        # Gap / return information
        # ----------------------------------------------------

        if lineage in last_lineage_ord:

            last_same = int(
                last_lineage_ord[lineage]
            )

            missed_same = max(
                0,
                race_ord - last_same - 1,
            )

            contexts_32f.at[
                row_i,
                "last_same_lineage_race_ord",
            ] = last_same

            contexts_32f.at[
                row_i,
                "missed_gps_since_same_lineage",
            ] = missed_same

            # A "return" here means there has been at least
            # one F1 race weekend since the driver's last
            # appearance for this organisation.
            if missed_same > 0:
                contexts_32f.at[
                    row_i,
                    "return_to_known_lineage",
                ] = True

        contexts_32f.at[
            row_i,
            "previous_lineage",
        ] = previous_lineage


        # ----------------------------------------------------
        # Strict continuous-appearance-streak baseline
        # ----------------------------------------------------
        #
        # This deliberately harsh baseline resets if:
        #
        #   - the driver was at another organisation; OR
        #   - the driver missed an intervening GP.
        #
        # It is NOT expected to be our preferred definition.
        # ----------------------------------------------------

        consecutive_same = (
            previous_lineage == lineage
            and previous_race_ord is not None
            and race_ord == previous_race_ord + 1
        )

        if consecutive_same:
            strict_before = (
                strict_stint_count_after_previous
            )
        else:
            strict_before = 0

        contexts_32f.at[
            row_i,
            "fam_strict_stint",
        ] = float(_sat(strict_before))

        strict_stint_count_after_previous = (
            strict_before + 1
        )


        # ----------------------------------------------------
        # Decaying organisational memory
        # ----------------------------------------------------

        for cand, half_life in (
            MEMORY_HALF_LIVES_32F.items()
        ):

            state = decay_state[cand]

            if lineage not in state:
                stock_before = 0.0

            else:
                stock_after_last, last_ord = (
                    state[lineage]
                )

                missed = max(
                    0,
                    race_ord - last_ord - 1,
                )

                stock_before = (
                    stock_after_last
                    * 0.5 ** (
                        missed / half_life
                    )
                )

            contexts_32f.at[
                row_i,
                f"stock_{cand}",
            ] = stock_before

            contexts_32f.at[
                row_i,
                f"fam_{cand}",
            ] = float(
                _sat(stock_before)
            )

            # Current weekend contributes one appearance
            # AFTER the pre-race familiarity has been recorded.
            state[lineage] = (
                stock_before + 1.0,
                race_ord,
            )


        # ----------------------------------------------------
        # Update non-decaying counters
        # ----------------------------------------------------

        literal_counts[ref] += 1
        lineage_counts[lineage] += 1

        last_lineage_ord[lineage] = race_ord

        previous_lineage = lineage
        previous_race_ord = race_ord


# ============================================================
# 8. CANDIDATE COLUMN DEFINITIONS
# ============================================================

FAM_CANDIDATES_32F = [
    "literal_production",
    "strict_stint",
    "memory_10gp",
    "memory_20gp",
    "memory_40gp",
    "memory_80gp",
    "no_decay",
]

FAM_COLS_32F = {
    cand: f"fam_{cand}"
    for cand in FAM_CANDIDATES_32F
}


# ============================================================
# 9. RETURN-TO-OLD-ORGANISATION EPISODES
# ============================================================

# ============================================================
# 9A. RESTORE HUMAN-READABLE RACE / CONSTRUCTOR LABELS
# ============================================================
#
# The familiarity calculations only require IDs / refs, so some
# earlier transformations do not preserve the human-readable
# columns "name" and "constructor".
#
# Reattach them explicitly here for the diagnostic tables.
# This is DISPLAY METADATA ONLY and does not alter any tenure
# calculation.
# ============================================================


# ------------------------------------------------------------
# Race-name lookup
# ------------------------------------------------------------

race_label_lookup_32f = (
    raw.races[
        [
            "raceId",
            "name",
        ]
    ]
    .drop_duplicates("raceId")
    .rename(
        columns={
            "name": "_race_name_32f",
        }
    )
)


# ------------------------------------------------------------
# Constructor-name lookup
# ------------------------------------------------------------

# Ergast/Jolpica-style constructor tables normally use "name".
# Make this robust in case the loaded table has a different
# display-name column.
constructor_name_candidates_32f = [
    "name",
    "constructor",
    "constructorName",
]

constructor_name_col_32f = next(
    (
        c
        for c in constructor_name_candidates_32f
        if c in raw.constructors.columns
    ),
    None,
)

if constructor_name_col_32f is None:

    # constructorRef is always sufficient as a readable fallback.
    constructor_label_lookup_32f = (
        raw.constructors[
            [
                "constructorId",
                "constructorRef",
            ]
        ]
        .drop_duplicates("constructorId")
        .rename(
            columns={
                "constructorRef": "_constructor_name_32f",
            }
        )
    )

else:

    constructor_label_lookup_32f = (
        raw.constructors[
            [
                "constructorId",
                constructor_name_col_32f,
            ]
        ]
        .drop_duplicates("constructorId")
        .rename(
            columns={
                constructor_name_col_32f:
                    "_constructor_name_32f",
            }
        )
    )


# ------------------------------------------------------------
# Remove stale helper columns if this section is rerun
# ------------------------------------------------------------

for col_32f in [
    "_race_name_32f",
    "_constructor_name_32f",
]:

    if col_32f in contexts_32f.columns:

        contexts_32f = contexts_32f.drop(
            columns=[col_32f]
        )


# ------------------------------------------------------------
# Merge labels back onto contexts
# ------------------------------------------------------------

contexts_32f = contexts_32f.merge(
    race_label_lookup_32f,
    on="raceId",
    how="left",
    validate="many_to_one",
)

contexts_32f = contexts_32f.merge(
    constructor_label_lookup_32f,
    on="constructorId",
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# Restore the names expected by the remainder of Stage 32F
# ------------------------------------------------------------

contexts_32f["name"] = (
    contexts_32f[
        "_race_name_32f"
    ]
)

contexts_32f["constructor"] = (
    contexts_32f[
        "_constructor_name_32f"
    ]
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

missing_race_labels_32f = int(
    contexts_32f[
        "name"
    ].isna().sum()
)

missing_constructor_labels_32f = int(
    contexts_32f[
        "constructor"
    ].isna().sum()
)

print()
print("=" * 76)
print("DISPLAY-LABEL RESTORATION")
print("=" * 76)

print(
    "Missing race labels:",
    missing_race_labels_32f,
)

print(
    "Missing constructor labels:",
    missing_constructor_labels_32f,
)

if missing_race_labels_32f:
    raise RuntimeError(
        "Some race IDs could not be mapped back to race names."
    )

if missing_constructor_labels_32f:
    raise RuntimeError(
        "Some constructor IDs could not be mapped back to "
        "constructor names."
    )

return_cases_32f = contexts_32f.loc[
    contexts_32f["return_to_known_lineage"]
].copy()

return_keep_cols = [
    "raceId",
    "year",
    "round",
    "name",
    "driver",
    "constructorRef",
    "constructor",
    "lineage",
    "previous_lineage",
    "lineage_appearances_before",
    "missed_gps_since_same_lineage",
] + list(FAM_COLS_32F.values())

return_cases_32f = return_cases_32f[
    return_keep_cols
].sort_values(
    [
        "missed_gps_since_same_lineage",
        "year",
        "round",
    ]
)


# ============================================================
# 10. IMPORTANT RETURN CASES
# ============================================================
#
# These are the cases we especially care about conceptually.
#
# Gasly / Kvyat:
#   short-ish returns to Faenza should retain familiarity.
#
# Ricciardo / Alonso / Räikkönen / etc:
#   long returns should have much less retained familiarity.
#
# Hülkenberg:
#   old 2013 Hinwil experience should have largely decayed by
#   2025, but 2025 Sauber -> 2026 Audi should carry strongly.
#
# ============================================================

FOCUS_RETURN_SPECS_32F = [

    # driver, target year, target lineage, label

    (
        "gasly",
        2019,
        "FAENZA",
        "Gasly returns Red Bull -> Toro Rosso",
    ),

    (
        "kvyat",
        2019,
        "FAENZA",
        "Kvyat returns to Toro Rosso",
    ),

    (
        "ricciardo",
        2023,
        "FAENZA",
        "Ricciardo returns to Faenza after long absence",
    ),

    (
        "alonso",
        2021,
        "ENSTONE",
        "Alonso returns to Enstone",
    ),

    (
        "raikkonen",
        2019,
        "HINWIL",
        "Räikkönen returns to Hinwil",
    ),

    (
        "hulkenberg",
        2025,
        "HINWIL",
        "Hülkenberg returns to Hinwil",
    ),

    (
        "fisichella",
        2005,
        "ENSTONE",
        "Fisichella returns to Enstone",
    ),

    (
        "webber",
        2007,
        "MILTON_KEYNES",
        "Webber returns to Milton Keynes lineage",
    ),

    (
        "kubica",
        2021,
        "HINWIL",
        "Kubica returns to Hinwil",
    ),
]


focus_return_rows_32f = []

for (
    driver,
    target_year,
    lineage,
    label,
) in FOCUS_RETURN_SPECS_32F:

    sub = return_cases_32f.loc[
        return_cases_32f["driver"].eq(driver)
        & return_cases_32f["year"].eq(target_year)
        & return_cases_32f["lineage"].eq(lineage)
    ].copy()

    if sub.empty:
        continue

    # First appearance in that return episode.
    row = (
        sub
        .sort_values(
            ["round", "raceId"]
        )
        .iloc[0]
        .copy()
    )

    row["case"] = label
    focus_return_rows_32f.append(row)


if focus_return_rows_32f:
    focus_returns_32f = pd.DataFrame(
        focus_return_rows_32f
    )
else:
    focus_returns_32f = pd.DataFrame()


# ============================================================
# 11. LOOKUP HELPERS FOR MODERN TABLES
# ============================================================

lookup_cols_32f = (
    ["raceId", "driver"]
    + list(FAM_COLS_32F.values())
)

fam_lookup_32f = (
    contexts_32f[
        lookup_cols_32f
    ]
    .drop_duplicates(
        ["raceId", "driver"]
    )
    .set_index(
        ["raceId", "driver"]
    )
)


def attach_driver_familiarities_32f(df):
    """
    Attach candidate familiarity values for driver_a and driver_b.
    """

    out = df.copy()

    keys_a = pd.MultiIndex.from_arrays(
        [
            out["raceId"].to_numpy(),
            out["driver_a"].astype(str).to_numpy(),
        ],
        names=["raceId", "driver"],
    )

    keys_b = pd.MultiIndex.from_arrays(
        [
            out["raceId"].to_numpy(),
            out["driver_b"].astype(str).to_numpy(),
        ],
        names=["raceId", "driver"],
    )

    for cand, fam_col in FAM_COLS_32F.items():

        source = fam_lookup_32f[fam_col]

        out[f"{cand}_a"] = (
            source
            .reindex(keys_a)
            .to_numpy()
        )

        out[f"{cand}_b"] = (
            source
            .reindex(keys_b)
            .to_numpy()
        )

        out[f"{cand}_diff_raw"] = (
            out[f"{cand}_a"]
            - out[f"{cand}_b"]
        )

        out[f"{cand}_mean_raw"] = (
            0.5
            * (
                out[f"{cand}_a"]
                + out[f"{cand}_b"]
            )
        )

    return out


pace_contrast_fam_32f = (
    attach_driver_familiarities_32f(
        pace_contrast_32f
    )
)

pace_level_fam_32f = (
    attach_driver_familiarities_32f(
        pace_level_32f
    )
)

q_contrast_fam_32f = (
    attach_driver_familiarities_32f(
        q_contrast_32f
    )
)

q1_level_fam_32f = (
    attach_driver_familiarities_32f(
        q1_level_32f
    )
)


# ============================================================
# 12. INFER THE PRODUCTION LEVEL-CENTRING OPERATOR
# ============================================================
#
# The contrast covariates are just:
#
#     tenure_a - tenure_b
#
# The level channels were centred within race.
#
# Stage 32E showed this via the constant implied centre.
#
# Here we explicitly determine whether the production centre is
# best reconstructed by:
#
#     unweighted race mean
#
# or
#
#     weight-weighted race mean.
#
# We then apply the SAME operator to every new candidate.
#
# ============================================================


def _weighted_group_mean(
    df,
    value_col,
    weight_col,
):
    """
    Return one weighted mean per row, grouped by raceId.
    """

    temp = df[
        ["raceId", value_col, weight_col]
    ].copy()

    temp["_vw"] = (
        temp[value_col]
        * temp[weight_col]
    )

    numerator = (
        temp
        .groupby("raceId")["_vw"]
        .transform("sum")
    )

    denominator = (
        temp
        .groupby("raceId")[weight_col]
        .transform("sum")
    )

    return numerator / denominator


def infer_level_centre_method_32f(
    df,
    stored_col="tenure_mean",
):
    """
    Compare candidate centring rules using literal-production familiarity.
    """

    if stored_col not in df.columns:
        raise RuntimeError(
            f"`{stored_col}` not found in level table."
        )

    raw_col = "literal_production_mean_raw"

    # Unweighted mean.
    centre_unweighted = (
        df
        .groupby("raceId")[raw_col]
        .transform("mean")
    )

    pred_unweighted = (
        df[raw_col]
        - centre_unweighted
    )

    err_unweighted = (
        pred_unweighted
        - df[stored_col]
    )

    scores = {
        "unweighted": float(
            np.nanmean(
                np.abs(err_unweighted)
            )
        )
    }

    predictions = {
        "unweighted": pred_unweighted
    }

    # Weighted alternative if available.
    if (
        "weight" in df.columns
        and np.isfinite(
            pd.to_numeric(
                df["weight"],
                errors="coerce",
            )
        ).any()
    ):

        work = df.copy()

        work["weight"] = pd.to_numeric(
            work["weight"],
            errors="coerce",
        )

        centre_weighted = (
            _weighted_group_mean(
                work,
                raw_col,
                "weight",
            )
        )

        pred_weighted = (
            work[raw_col]
            - centre_weighted
        )

        err_weighted = (
            pred_weighted
            - work[stored_col]
        )

        scores["weighted"] = float(
            np.nanmean(
                np.abs(err_weighted)
            )
        )

        predictions["weighted"] = (
            pred_weighted
        )

    best = min(
        scores,
        key=scores.get,
    )

    return best, scores, predictions[best]


pace_level_centre_method_32f, \
pace_level_centre_scores_32f, \
pace_level_literal_rebuild_32f = (
    infer_level_centre_method_32f(
        pace_level_fam_32f,
        "tenure_mean",
    )
)

q1_level_centre_method_32f, \
q1_level_centre_scores_32f, \
q1_level_literal_rebuild_32f = (
    infer_level_centre_method_32f(
        q1_level_fam_32f,
        "tenure_mean",
    )
)


def apply_level_centring_32f(
    df,
    method,
):
    """
    Apply inferred production-style within-race centring to every
    familiarity candidate.
    """

    out = df.copy()

    for cand in FAM_CANDIDATES_32F:

        raw_col = f"{cand}_mean_raw"
        out_col = f"{cand}_tenure_mean"

        if method == "weighted":

            centre = (
                _weighted_group_mean(
                    out,
                    raw_col,
                    "weight",
                )
            )

        elif method == "unweighted":

            centre = (
                out
                .groupby("raceId")[raw_col]
                .transform("mean")
            )

        else:
            raise ValueError(
                f"Unknown centring method: {method}"
            )

        out[out_col] = (
            out[raw_col]
            - centre
        )

    return out


pace_level_fam_32f = (
    apply_level_centring_32f(
        pace_level_fam_32f,
        pace_level_centre_method_32f,
    )
)

q1_level_fam_32f = (
    apply_level_centring_32f(
        q1_level_fam_32f,
        q1_level_centre_method_32f,
    )
)


# ============================================================
# 13. VERIFY FROZEN LITERAL-CONSTRUCTOR RECONSTRUCTION
# ============================================================

reconstruction_rows_32f = []


def contrast_reconstruction_summary_32f(
    label,
    df,
):
    stored = pd.to_numeric(
        df["tenure_diff"],
        errors="coerce",
    )

    pred = pd.to_numeric(
        df["literal_production_diff_raw"],
        errors="coerce",
    )

    err = pred - stored
    abs_err = np.abs(err)

    reconstruction_rows_32f.append(
        {
            "channel": label,
            "rows": len(df),
            "mean_abs_error": float(
                np.nanmean(abs_err)
            ),
            "median_abs_error": float(
                np.nanmedian(abs_err)
            ),
            "max_abs_error": float(
                np.nanmax(abs_err)
            ),
            "fraction_exact_1e-10": float(
                np.nanmean(abs_err < 1e-10)
            ),
        }
    )

    out = df[
        [
            "raceId",
            "driver_a",
            "driver_b",
            "tenure_diff",
        ]
    ].copy()

    if "year" in df.columns:
        out["year"] = df["year"]

    if "session" in df.columns:
        out["session"] = df["session"]

    out["reconstructed"] = pred
    out["error"] = err
    out["abs_error"] = abs_err

    return (
        out
        .sort_values(
            "abs_error",
            ascending=False,
        )
    )


def level_reconstruction_summary_32f(
    label,
    df,
):
    stored = pd.to_numeric(
        df["tenure_mean"],
        errors="coerce",
    )

    pred = pd.to_numeric(
        df[
            "literal_production_tenure_mean"
        ],
        errors="coerce",
    )

    err = pred - stored
    abs_err = np.abs(err)

    reconstruction_rows_32f.append(
        {
            "channel": label,
            "rows": len(df),
            "mean_abs_error": float(
                np.nanmean(abs_err)
            ),
            "median_abs_error": float(
                np.nanmedian(abs_err)
            ),
            "max_abs_error": float(
                np.nanmax(abs_err)
            ),
            "fraction_exact_1e-10": float(
                np.nanmean(abs_err < 1e-10)
            ),
        }
    )

    out = df[
        [
            "raceId",
            "driver_a",
            "driver_b",
            "tenure_mean",
        ]
    ].copy()

    if "year" in df.columns:
        out["year"] = df["year"]

    if "session" in df.columns:
        out["session"] = df["session"]

    out["reconstructed"] = pred
    out["error"] = err
    out["abs_error"] = abs_err

    return (
        out
        .sort_values(
            "abs_error",
            ascending=False,
        )
    )


pace_contrast_recon_32f = (
    contrast_reconstruction_summary_32f(
        "pace_contrast",
        pace_contrast_fam_32f,
    )
)

q_contrast_recon_32f = (
    contrast_reconstruction_summary_32f(
        "q_contrast",
        q_contrast_fam_32f,
    )
)

pace_level_recon_32f = (
    level_reconstruction_summary_32f(
        "pace_level",
        pace_level_fam_32f,
    )
)

q1_level_recon_32f = (
    level_reconstruction_summary_32f(
        "q1_level",
        q1_level_fam_32f,
    )
)

reconstruction_summary_32f = pd.DataFrame(
    reconstruction_rows_32f
)


# ============================================================
# 14. MODERN LIKELIHOOD IMPACT OF EACH MEMORY DEFINITION
# ============================================================

GAMMA_TENURE_REFERENCE_32F = 0.283208


impact_rows_32f = []


def _candidate_impact_32f(
    label,
    df,
    stored_col,
    candidate_suffix,
):

    stored = pd.to_numeric(
        df[stored_col],
        errors="coerce",
    )

    years = (
        pd.to_numeric(
            df["year"],
            errors="coerce",
        )
        if "year" in df.columns
        else pd.Series(
            np.nan,
            index=df.index,
        )
    )

    for cand in FAM_CANDIDATES_32F:

        if cand == "literal_production":
            continue

        new_col = (
            f"{cand}_{candidate_suffix}"
        )

        new = pd.to_numeric(
            df[new_col],
            errors="coerce",
        )

        delta = new - stored
        abs_delta = np.abs(delta)

        valid = (
            np.isfinite(stored)
            & np.isfinite(new)
        )

        valid_2026 = (
            valid
            & years.eq(2026)
        )

        if valid.sum() == 0:
            continue

        impact_rows_32f.append(
            {
                "channel": label,
                "candidate": cand,
                "rows": int(valid.sum()),
                "changed_rows_1e-10": int(
                    np.sum(
                        abs_delta[valid]
                        > 1e-10
                    )
                ),
                "changed_fraction": float(
                    np.mean(
                        abs_delta[valid]
                        > 1e-10
                    )
                ),
                "mean_abs_change": float(
                    np.mean(
                        abs_delta[valid]
                    )
                ),
                "median_abs_change": float(
                    np.median(
                        abs_delta[valid]
                    )
                ),
                "p95_abs_change": float(
                    np.quantile(
                        abs_delta[valid],
                        0.95,
                    )
                ),
                "max_abs_change": float(
                    np.max(
                        abs_delta[valid]
                    )
                ),
                "corr_with_frozen_covariate": float(
                    np.corrcoef(
                        stored[valid],
                        new[valid],
                    )[0, 1]
                ),
                "max_abs_mu_shift_gamma_ref": float(
                    GAMMA_TENURE_REFERENCE_32F
                    * np.max(
                        abs_delta[valid]
                    )
                ),
                "rows_2026": int(
                    valid_2026.sum()
                ),
                "mean_abs_change_2026": (
                    float(
                        np.mean(
                            abs_delta[
                                valid_2026
                            ]
                        )
                    )
                    if valid_2026.sum()
                    else np.nan
                ),
                "max_abs_change_2026": (
                    float(
                        np.max(
                            abs_delta[
                                valid_2026
                            ]
                        )
                    )
                    if valid_2026.sum()
                    else np.nan
                ),
            }
        )


_candidate_impact_32f(
    "pace_contrast",
    pace_contrast_fam_32f,
    "tenure_diff",
    "diff_raw",
)

_candidate_impact_32f(
    "pace_level",
    pace_level_fam_32f,
    "tenure_mean",
    "tenure_mean",
)

_candidate_impact_32f(
    "q_contrast",
    q_contrast_fam_32f,
    "tenure_diff",
    "diff_raw",
)

_candidate_impact_32f(
    "q1_level",
    q1_level_fam_32f,
    "tenure_mean",
    "tenure_mean",
)

candidate_impact_32f = pd.DataFrame(
    impact_rows_32f
)


# ============================================================
# 15. FIRST 2026 APPEARANCE FOR EACH DRIVER
# ============================================================
#
# This is particularly useful for:
#
#   Audi:
#       Bortoleto / Hülkenberg should inherit recent Hinwil
#       familiarity.
#
#   Cadillac:
#       Bottas / Pérez should start from zero.
#
#   Red Bull vs RB:
#       must remain separate organisations.
#
# ============================================================

first_2026_32f = (
    contexts_32f.loc[
        contexts_32f["year"].eq(2026)
    ]
    .sort_values(
        ["driver", "round"]
    )
    .groupby(
        "driver",
        as_index=False,
    )
    .first()
)

first_2026_cols_32f = [
    "driver",
    "constructor",
    "constructorRef",
    "lineage",
    "round",
    "literal_appearances_before",
    "lineage_appearances_before",
    "fam_literal_production",
    "fam_strict_stint",
    "fam_memory_10gp",
    "fam_memory_20gp",
    "fam_memory_40gp",
    "fam_memory_80gp",
    "fam_no_decay",
]

first_2026_32f = (
    first_2026_32f[
        first_2026_cols_32f
    ]
    .sort_values(
        ["constructor", "driver"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 16. RETURN-GAP SUMMARY
# ============================================================

if len(return_cases_32f):

    return_gap_summary_32f = pd.DataFrame(
        {
            "metric": [
                "return episodes",
                "median missed GPs",
                "mean missed GPs",
                "90th percentile missed GPs",
                "maximum missed GPs",
            ],
            "value": [
                len(return_cases_32f),
                float(
                    return_cases_32f[
                        "missed_gps_since_same_lineage"
                    ].median()
                ),
                float(
                    return_cases_32f[
                        "missed_gps_since_same_lineage"
                    ].mean()
                ),
                float(
                    return_cases_32f[
                        "missed_gps_since_same_lineage"
                    ].quantile(0.90)
                ),
                float(
                    return_cases_32f[
                        "missed_gps_since_same_lineage"
                    ].max()
                ),
            ],
        }
    )

else:
    return_gap_summary_32f = pd.DataFrame()


# ============================================================
# 17. THEORETICAL MEMORY RETENTION TABLE
# ============================================================
#
# This gives an intuitive view of what each half-life means.
#
# We use a driver with 40 accumulated team appearances as an
# illustrative reasonably experienced team member.
#
# ============================================================

example_stock_32f = 40.0

gap_grid_32f = np.array(
    [0, 5, 10, 15, 20, 30, 40, 60, 80, 120],
    dtype=float,
)

retention_rows_32f = []

for gap in gap_grid_32f:

    row = {
        "missed_gps": int(gap),
        "pre_absence_stock": example_stock_32f,
        "pre_absence_familiarity": float(
            _sat(example_stock_32f)
        ),
    }

    for cand, half_life in (
        MEMORY_HALF_LIVES_32F.items()
    ):

        stock_after_gap = (
            example_stock_32f
            * 0.5 ** (
                gap / half_life
            )
        )

        row[f"{cand}_stock"] = (
            stock_after_gap
        )

        row[f"{cand}_familiarity"] = float(
            _sat(stock_after_gap)
        )

    row["no_decay_familiarity"] = float(
        _sat(example_stock_32f)
    )

    retention_rows_32f.append(row)


memory_retention_32f = pd.DataFrame(
    retention_rows_32f
)


# ============================================================
# 18. PLOT — MEMORY CURVES
# ============================================================

plot_gaps = np.arange(
    0,
    161,
    dtype=float,
)

plt.figure(
    figsize=(10, 6)
)

for cand, half_life in (
    MEMORY_HALF_LIVES_32F.items()
):

    stock = (
        example_stock_32f
        * 0.5 ** (
            plot_gaps / half_life
        )
    )

    fam = _sat(stock)

    plt.plot(
        plot_gaps,
        fam,
        label=cand,
    )

plt.axhline(
    float(_sat(example_stock_32f)),
    linestyle="--",
    label="no_decay",
)

plt.axhline(
    0.0,
    linestyle=":",
)

plt.xlabel(
    "Missed F1 race weekends since last exposure"
)

plt.ylabel(
    "Retained organisational familiarity"
)

plt.title(
    "Candidate familiarity-memory decay curves\n"
    "(example starting stock = 40 prior appearances)"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    STAGE32F_OUT
    / "familiarity_memory_decay_curves.png",
    dpi=180,
)

plt.show()


# ============================================================
# 19. PLOT — FOCUS RETURN CASE HEATMAP
# ============================================================

if not focus_returns_32f.empty:

    heat_candidates = [
        "fam_strict_stint",
        "fam_memory_10gp",
        "fam_memory_20gp",
        "fam_memory_40gp",
        "fam_memory_80gp",
        "fam_no_decay",
    ]

    heat_labels = [
        "Strict stint",
        "10 GP",
        "20 GP",
        "40 GP",
        "80 GP",
        "No decay",
    ]

    heat = (
        focus_returns_32f[
            heat_candidates
        ]
        .astype(float)
        .to_numpy()
    )

    fig, ax = plt.subplots(
        figsize=(
            10,
            max(
                5,
                0.55 * len(
                    focus_returns_32f
                ),
            ),
        )
    )

    im = ax.imshow(
        heat,
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
    )

    ax.set_xticks(
        np.arange(
            len(heat_labels)
        )
    )

    ax.set_xticklabels(
        heat_labels,
        rotation=30,
        ha="right",
    )

    ylabels = []

    for _, row in (
        focus_returns_32f.iterrows()
    ):

        ylabels.append(
            f"{row['driver']} "
            f"{int(row['year'])} "
            f"({int(row['missed_gps_since_same_lineage'])} GP gap)"
        )

    ax.set_yticks(
        np.arange(
            len(ylabels)
        )
    )

    ax.set_yticklabels(
        ylabels
    )

    ax.set_title(
        "Organisational familiarity retained on important returns"
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
    )

    cbar.set_label(
        "Familiarity entering return weekend"
    )

    # Add values into cells.
    for i in range(
        heat.shape[0]
    ):
        for j in range(
            heat.shape[1]
        ):
            ax.text(
                j,
                i,
                f"{heat[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8,
            )

    plt.tight_layout()

    plt.savefig(
        STAGE32F_OUT
        / "focus_return_familiarity_heatmap.png",
        dpi=180,
    )

    plt.show()


# ============================================================
# 20. SAVE TABLES
# ============================================================

contexts_32f.to_csv(
    STAGE32F_OUT
    / "organisational_familiarity_contexts.csv",
    index=False,
)

return_cases_32f.to_csv(
    STAGE32F_OUT
    / "all_return_episodes.csv",
    index=False,
)

if not focus_returns_32f.empty:
    focus_returns_32f.to_csv(
        STAGE32F_OUT
        / "focus_return_episodes.csv",
        index=False,
    )

first_2026_32f.to_csv(
    STAGE32F_OUT
    / "first_2026_appearance_familiarity.csv",
    index=False,
)

candidate_impact_32f.to_csv(
    STAGE32F_OUT
    / "candidate_modern_likelihood_covariate_impact.csv",
    index=False,
)

reconstruction_summary_32f.to_csv(
    STAGE32F_OUT
    / "literal_production_reconstruction_summary.csv",
    index=False,
)

pace_contrast_recon_32f.to_csv(
    STAGE32F_OUT
    / "pace_contrast_literal_reconstruction_rows.csv",
    index=False,
)

q_contrast_recon_32f.to_csv(
    STAGE32F_OUT
    / "q_contrast_literal_reconstruction_rows.csv",
    index=False,
)

pace_level_recon_32f.to_csv(
    STAGE32F_OUT
    / "pace_level_literal_reconstruction_rows.csv",
    index=False,
)

q1_level_recon_32f.to_csv(
    STAGE32F_OUT
    / "q1_level_literal_reconstruction_rows.csv",
    index=False,
)

memory_retention_32f.to_csv(
    STAGE32F_OUT
    / "theoretical_memory_retention.csv",
    index=False,
)

if not return_gap_summary_32f.empty:
    return_gap_summary_32f.to_csv(
        STAGE32F_OUT
        / "return_gap_summary.csv",
        index=False,
    )


# ============================================================
# 21. DISPLAY RESULTS
# ============================================================

print()
print("=" * 76)
print("STAGE 32F — ORGANISATIONAL FAMILIARITY MEMORY AUDIT")
print("=" * 76)

print(
    f"Result-appearance contexts: "
    f"{len(contexts_32f):,}"
)

print(
    f"Return-to-known-organisation episodes: "
    f"{len(return_cases_32f):,}"
)

print()
print("=" * 76)
print("PRODUCTION TENURE RECONSTRUCTION")
print("=" * 76)

display(
    reconstruction_summary_32f
)

print()
print(
    "Pace level centring method:",
    pace_level_centre_method_32f,
    pace_level_centre_scores_32f,
)

print(
    "Q1 level centring method:",
    q1_level_centre_method_32f,
    q1_level_centre_scores_32f,
)


# ------------------------------------------------------------
# Show any genuine reconstruction anomalies.
# ------------------------------------------------------------

print()
print("=" * 76)
print("LARGEST LITERAL-PRODUCTION RECONSTRUCTION ERRORS")
print("=" * 76)

print("\nPace contrast:")
display(
    pace_contrast_recon_32f.head(10)
)

print("\nQualifying contrast:")
display(
    q_contrast_recon_32f.head(10)
)

print("\nPace level:")
display(
    pace_level_recon_32f.head(10)
)

print("\nQ1 level:")
display(
    q1_level_recon_32f.head(10)
)


print()
print("=" * 76)
print("FOCUS RETURN EPISODES")
print("=" * 76)

if focus_returns_32f.empty:
    print("No focus return episodes found.")
else:

    focus_display_cols = [
        "case",
        "driver",
        "year",
        "round",
        "constructorRef",
        "lineage",
        "previous_lineage",
        "lineage_appearances_before",
        "missed_gps_since_same_lineage",
        "fam_literal_production",
        "fam_strict_stint",
        "fam_memory_10gp",
        "fam_memory_20gp",
        "fam_memory_40gp",
        "fam_memory_80gp",
        "fam_no_decay",
    ]

    display(
        focus_returns_32f[
            focus_display_cols
        ]
        .sort_values(
            "missed_gps_since_same_lineage"
        )
    )


print()
print("=" * 76)
print("2026 FIRST-APPEARANCE FAMILIARITY")
print("=" * 76)

display(
    first_2026_32f
)


print()
print("=" * 76)
print("MODERN LIKELIHOOD COVARIATE IMPACT")
print("=" * 76)

display(
    candidate_impact_32f.sort_values(
        [
            "channel",
            "candidate",
        ]
    )
)


print()
print("=" * 76)
print("MEMORY RETENTION — EXAMPLE STOCK = 40 APPEARANCES")
print("=" * 76)

retention_display_cols = [
    "missed_gps",
    "pre_absence_familiarity",
    "memory_10gp_familiarity",
    "memory_20gp_familiarity",
    "memory_40gp_familiarity",
    "memory_80gp_familiarity",
    "no_decay_familiarity",
]

display(
    memory_retention_32f[
        retention_display_cols
    ]
)


# ============================================================
# 22. AUTOMATIC TRIAGE
# ============================================================

pace_contrast_maxerr = float(
    reconstruction_summary_32f.loc[
        reconstruction_summary_32f[
            "channel"
        ].eq("pace_contrast"),
        "max_abs_error",
    ].iloc[0]
)

pace_level_maxerr = float(
    reconstruction_summary_32f.loc[
        reconstruction_summary_32f[
            "channel"
        ].eq("pace_level"),
        "max_abs_error",
    ].iloc[0]
)

q_contrast_exact = float(
    reconstruction_summary_32f.loc[
        reconstruction_summary_32f[
            "channel"
        ].eq("q_contrast"),
        "fraction_exact_1e-10",
    ].iloc[0]
)

q1_level_exact = float(
    reconstruction_summary_32f.loc[
        reconstruction_summary_32f[
            "channel"
        ].eq("q1_level"),
        "fraction_exact_1e-10",
    ].iloc[0]
)


print()
print("=" * 76)
print("STAGE 32F TRIAGE SUMMARY")
print("=" * 76)

print(
    f"Pace contrast literal reconstruction max error: "
    f"{pace_contrast_maxerr:.3e}"
)

print(
    f"Pace level literal reconstruction max error: "
    f"{pace_level_maxerr:.3e}"
)

print(
    f"Qualifying contrast exact fraction: "
    f"{q_contrast_exact:.3%}"
)

print(
    f"Q1 level exact fraction: "
    f"{q1_level_exact:.3%}"
)

print(
    f"\nReturn episodes detected: "
    f"{len(return_cases_32f):,}"
)

print(
    "\nCandidate memory half-lives "
    "(missed race weekends):"
)

for cand, half_life in (
    MEMORY_HALF_LIVES_32F.items()
):
    print(
        f"  {cand}: {half_life:g} GP"
    )


print()
print("INTERPRETATION GUIDE:")
print(
    """
  1. SHORT RETURNS
     Gasly/Kvyat-style short returns should retain substantial
     familiarity under the preferred candidate.

     If a definition drives these close to zero, it is probably
     forgetting too quickly.

  2. LONG RETURNS
     Alonso 2021, Räikkönen 2019, etc. should NOT retain anything
     close to full old-team familiarity.

     If a definition leaves them near 1, it is probably remembering
     for too long.

  3. HÜLKENBERG / HINWIL
     His old 2013 Sauber experience should have largely faded before
     his 2025 return.

     But once he has spent 2025 at Sauber, the 2026 transition to
     Audi should retain that RECENT Hinwil familiarity.

  4. AUDI
     Bortoleto and Hülkenberg in 2026 should no longer be treated as
     completely new to the team merely because constructorId changed.

  5. CADILLAC
     Pérez and Bottas should still begin at zero because Cadillac is
     genuinely a new organisation.

  6. RB vs RED BULL
     Faenza and Milton Keynes remain distinct organisations. A move
     between them is a genuine team change.

  7. DO NOT SELECT A HALF-LIFE JUST FROM THESE TABLES.
     This stage establishes behavioural plausibility and the magnitude
     of the resulting likelihood changes.

     Once we see which candidates behave sensibly, Stage 32G should
     compare the plausible survivors using a proper predictive
     sensitivity test rather than choosing the half-life by eye.
"""
)

print(
    "\nSaved outputs ->\n"
    f"{STAGE32F_OUT.resolve()}"
)

print("\nStage 32F complete.")

In [ ]:
# ============================================================
# STAGE 32F.2A — FIX 2015 MARUSSIA / MANOR LINEAGE ALIAS
# ============================================================
#
# The raw data use two constructor identities for the same
# underlying 2015 operation:
#
#   qualifying: marussia
#   results:    manor
#
# For TEAM FAMILIARITY these must belong to the same lineage.
#
# Car / constructor-season identity remains untouched.
#
# This patch changes ONLY the organisational-familiarity lineage.
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 1. SHOW CURRENT RELEVANT RULES
# ============================================================

print("=" * 76)
print("CURRENT MARUSSIA / MANOR LINEAGE RULES")
print("=" * 76)

for rule in LINEAGE_RULES_32F:

    ref, start_year, end_year, lineage = rule

    if (
        ref in {
            "virgin",
            "marussia",
            "manor",
        }
        or lineage
        == "VIRGIN_MARUSSIA_MANOR"
    ):
        print(rule)


# ============================================================
# 2. PATCH MANOR TO INCLUDE 2015
# ============================================================
#
# Desired mapping:
#
#   Virgin   2010–2011
#   Marussia 2012–2015
#   Manor    2015–2016
#
# The 2015 overlap is deliberate because the source tables use
# different constructorRefs for the same underlying operation.
#
# ============================================================

patched_rules_32f2a = []

manor_rule_found_32f2a = False


for (
    ref,
    start_year,
    end_year,
    lineage,
) in LINEAGE_RULES_32F:

    if (
        ref == "manor"
        and lineage
        == "VIRGIN_MARUSSIA_MANOR"
    ):

        patched_rules_32f2a.append(
            (
                ref,
                min(
                    int(start_year),
                    2015,
                ),
                int(end_year),
                lineage,
            )
        )

        manor_rule_found_32f2a = True

    else:

        patched_rules_32f2a.append(
            (
                ref,
                int(start_year),
                int(end_year),
                lineage,
            )
        )


if not manor_rule_found_32f2a:

    patched_rules_32f2a.append(
        (
            "manor",
            2015,
            2016,
            "VIRGIN_MARUSSIA_MANOR",
        )
    )


LINEAGE_RULES_32F = (
    patched_rules_32f2a
)


# ============================================================
# 3. LINEAGE RESOLVER
# ============================================================

def lineage_from_ref_year_32f2a(
    constructor_ref,
    year,
):

    constructor_ref = str(
        constructor_ref
    )

    year = int(
        year
    )

    for (
        ref,
        start_year,
        end_year,
        lineage,
    ) in LINEAGE_RULES_32F:

        if (
            constructor_ref == ref
            and
            start_year
            <= year
            <= end_year
        ):

            return lineage

    return (
        "LITERAL::"
        + constructor_ref
    )


# ============================================================
# 4. REASSIGN LINEAGES IN THE RESULT-EXPOSURE CONTEXT TABLE
# ============================================================
#
# Stage 32F.2 builds its memory history from contexts_32f.
# Therefore patching the rule alone is not enough: the already
# constructed contexts table must also be re-labelled.
#
# ============================================================

required_context_cols_32f2a = {
    "constructorRef",
    "year",
    "lineage",
}

missing_context_cols_32f2a = (
    required_context_cols_32f2a
    - set(
        contexts_32f.columns
    )
)

if missing_context_cols_32f2a:

    raise RuntimeError(
        "contexts_32f is missing required columns: "
        + ", ".join(
            sorted(
                missing_context_cols_32f2a
            )
        )
    )


contexts_32f[
    "lineage"
] = [

    lineage_from_ref_year_32f2a(
        ref,
        year,
    )

    for ref, year
    in zip(
        contexts_32f[
            "constructorRef"
        ],
        contexts_32f[
            "year"
        ],
    )
]


# ============================================================
# 5. VERIFY THE TWO 2015 SOURCE IDENTITIES NOW COINCIDE
# ============================================================

marussia_manor_2015_32f2a = (
    contexts_32f.loc[
        (
            contexts_32f[
                "year"
            ].eq(2015)
        )
        &
        (
            contexts_32f[
                "constructorRef"
            ].isin(
                [
                    "marussia",
                    "manor",
                ]
            )
        )
    ]
    .copy()
)


print()
print("=" * 76)
print("2015 MARUSSIA / MANOR RESULT CONTEXTS AFTER PATCH")
print("=" * 76)


cols_32f2a = [
    c
    for c in [
        "raceId",
        "year",
        "round",
        "driver",
        "driverId",
        "constructorId",
        "constructorRef",
        "lineage",
        "race_ord",
    ]
    if c
    in marussia_manor_2015_32f2a.columns
]


display(
    marussia_manor_2015_32f2a[
        cols_32f2a
    ]
    .sort_values(
        [
            "race_ord",
            "driver",
        ]
    )
)


lineages_2015_32f2a = set(
    marussia_manor_2015_32f2a[
        "lineage"
    ]
    .dropna()
    .astype(str)
)


if (
    marussia_manor_2015_32f2a[
        "constructorRef"
    ]
    .nunique()
    >= 1
):

    assert (
        lineages_2015_32f2a
        ==
        {
            "VIRGIN_MARUSSIA_MANOR"
        }
    ), (
        "2015 Marussia/Manor identities are still not "
        "mapped to one organisational lineage."
    )


# ============================================================
# 6. SPECIFIC DRIVER EXPOSURE CHECK
# ============================================================

drivers_32f2a = [
    "stevens",
    "merhi",
    "rossi",
]


driver_exposure_32f2a = (
    contexts_32f.loc[
        (
            contexts_32f[
                "year"
            ].eq(2015)
        )
        &
        (
            contexts_32f[
                "driver"
            ].isin(
                drivers_32f2a
            )
        )
        &
        (
            contexts_32f[
                "lineage"
            ].eq(
                "VIRGIN_MARUSSIA_MANOR"
            )
        )
    ]
    .copy()
)


print()
print("=" * 76)
print("2015 MANOR DRIVER EXPOSURE HISTORY")
print("=" * 76)


display(
    driver_exposure_32f2a[
        [
            c
            for c in [
                "raceId",
                "round",
                "driver",
                "constructorRef",
                "lineage",
                "race_ord",
            ]
            if c
            in driver_exposure_32f2a.columns
        ]
    ]
    .sort_values(
        [
            "race_ord",
            "driver",
        ]
    )
)


# Stevens and Merhi should have repeated exposure records.
counts_32f2a = (
    driver_exposure_32f2a
    .groupby(
        "driver"
    )
    .size()
    .rename(
        "2015_lineage_exposures"
    )
    .reset_index()
)


print()
print(
    "Exposure counts:"
)

display(
    counts_32f2a
)


for driver in [
    "stevens",
    "merhi",
]:

    n = int(
        (
            driver_exposure_32f2a[
                "driver"
            ]
            == driver
        ).sum()
    )

    if n <= 1:

        raise RuntimeError(
            f"{driver} still has only {n} "
            "2015 familiarity exposure(s); "
            "the lineage patch has not worked."
        )


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

print()
print("=" * 76)
print("STAGE 32F.2A SUMMARY")
print("=" * 76)

print(
    "2015 Marussia/Manor familiarity lineage:",
    "VIRGIN_MARUSSIA_MANOR",
)

print(
    "Car constructor IDs remain unchanged."
)

print(
    "contexts_32f lineages have been rebuilt."
)

print()
print(
    "NOW RERUN THE FULL STAGE 32F.2 CELL."
)

print(
    "Do not run Stage 32G yet."
)

In [ ]:
# ============================================================
# STAGE 32F.2 — OBSERVATION-SAFE ORGANISATIONAL FAMILIARITY
# ============================================================
#
# PURPOSE
# -------
# Stage 32F/32F.1 constructed familiarity on raw.results contexts.
#
# That is sufficient for almost all modern observations, but a tiny
# number of qualifying rows have no matching same-race results context
# or use a different literal constructorId.
#
# Conceptually, that should NOT make familiarity missing:
#
#   familiarity entering race r
#
# depends only on the driver's PRIOR experience with the underlying
# organisation before race r.
#
# This cell therefore evaluates familiarity for ANY driver/race/team
# observation using the driver's stored pre-event organisational
# history.
#
# It:
#   • keeps result appearances as the exposure source;
#   • uses organisational lineage, not literal constructor ID;
#   • evaluates 10/20/40 GP memory at arbitrary modern observations;
#   • fixes the 2015 Manor constructor-ID mismatch naturally;
#   • centres at driver level exactly as production did;
#   • rebuilds the four modern likelihood tenure covariates;
#   • verifies zero missing values.
#
# NO MODEL IS FIT HERE.
#
# ============================================================

from pathlib import Path
import bisect

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 0. REQUIREMENTS
# ============================================================

required_32f2 = [
    "contexts_32f",
    "LINEAGE_RULES_32F",
    "MEMORY_HALF_LIVES_32F",
    "df_pace",
    "df_quali",
    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",
    "raw",
]

missing_32f2 = [
    x for x in required_32f2
    if x not in globals()
]

if missing_32f2:
    raise RuntimeError(
        "Stage 32F.2 missing required objects:\n  "
        + "\n  ".join(missing_32f2)
    )


OUTDIR_32F2 = (
    Path("model_outputs")
    / "joint_driver_car_v2_1_1982_2026_LIVE"
    / "validation_observation_safe_familiarity"
)

OUTDIR_32F2.mkdir(
    parents=True,
    exist_ok=True,
)


SHORTLIST_32F2 = {
    "memory_10gp": 10.0,
    "memory_20gp": 20.0,
    "memory_40gp": 40.0,
}

FAMILIARITY_SCALE_32F2 = 18.0


def sat_32f2(x):
    return np.tanh(
        np.asarray(x, dtype=float)
        / FAMILIARITY_SCALE_32F2
    )


# ============================================================
# 1. RACE ORDER LOOKUP
# ============================================================

race_order_lookup_32f2 = (
    raw.races[
        [
            "raceId",
            "year",
            "round",
        ]
    ]
    .copy()
)

for col in [
    "raceId",
    "year",
    "round",
]:
    race_order_lookup_32f2[col] = (
        pd.to_numeric(
            race_order_lookup_32f2[col],
            errors="coerce",
        )
    )

race_order_lookup_32f2 = (
    race_order_lookup_32f2
    .dropna(
        subset=[
            "raceId",
            "year",
            "round",
        ]
    )
    .sort_values(
        [
            "year",
            "round",
            "raceId",
        ],
        kind="mergesort",
    )
    .drop_duplicates("raceId")
    .reset_index(drop=True)
)

race_order_lookup_32f2["race_ord"] = (
    np.arange(
        len(race_order_lookup_32f2),
        dtype=int,
    )
)


# ============================================================
# 2. CONSTRUCTOR-ID -> REF LOOKUP
# ============================================================

constructor_lookup_32f2 = (
    raw.constructors[
        [
            "constructorId",
            "constructorRef",
        ]
    ]
    .copy()
)

constructor_lookup_32f2[
    "constructorId"
] = pd.to_numeric(
    constructor_lookup_32f2[
        "constructorId"
    ],
    errors="coerce",
)

constructor_lookup_32f2 = (
    constructor_lookup_32f2
    .dropna(
        subset=[
            "constructorId",
            "constructorRef",
        ]
    )
    .drop_duplicates("constructorId")
)

constructor_lookup_32f2[
    "constructorId"
] = constructor_lookup_32f2[
    "constructorId"
].astype(int)


# ============================================================
# 3. LINEAGE RESOLVER
# ============================================================

def lineage_from_ref_year_32f2(
    constructor_ref,
    year,
):
    """
    Resolve one constructorRef/year to the curated underlying
    organisational lineage.
    """

    constructor_ref = str(
        constructor_ref
    )

    year = int(year)

    for (
        ref,
        start_year,
        end_year,
        lineage,
    ) in LINEAGE_RULES_32F:

        if (
            constructor_ref == ref
            and start_year <= year <= end_year
        ):
            return lineage

    return (
        "LITERAL::"
        + constructor_ref
    )


# ============================================================
# 4. BUILD EXPOSURE HISTORY
# ============================================================
#
# contexts_32f is one exposure per driver per GP after the historical
# duplicate collapse.
#
# For each driver-lineage pair we need:
#
#   race ordinals of exposures
#
# and, for each candidate half-life:
#
#   familiarity stock immediately AFTER each exposure.
#
# This lets us query familiarity at an arbitrary later observation.
#
# ============================================================

history_32f2 = {}


for (
    driver_id,
    lineage,
), grp in contexts_32f.groupby(
    [
        "driverId",
        "lineage",
    ],
    sort=False,
):

    g = (
        grp
        .sort_values(
            "race_ord"
        )
        .copy()
    )

    ords = (
        g["race_ord"]
        .astype(int)
        .to_numpy()
    )

    entry = {
        "race_ord":
            ords,
    }

    for (
        candidate,
        half_life,
    ) in SHORTLIST_32F2.items():

        stock_after = np.zeros(
            len(g),
            dtype=float,
        )

        previous_stock_after = 0.0
        previous_ord = None

        for j, race_ord in enumerate(
            ords
        ):

            if previous_ord is None:

                stock_before = 0.0

            else:

                missed = max(
                    0,
                    int(race_ord)
                    - int(previous_ord)
                    - 1,
                )

                stock_before = (
                    previous_stock_after
                    * 0.5 ** (
                        missed
                        / half_life
                    )
                )

            current_stock_after = (
                stock_before
                + 1.0
            )

            stock_after[j] = (
                current_stock_after
            )

            previous_stock_after = (
                current_stock_after
            )

            previous_ord = int(
                race_ord
            )

        entry[
            f"stock_after_{candidate}"
        ] = stock_after

    history_32f2[
        (
            int(driver_id),
            str(lineage),
        )
    ] = entry


# ============================================================
# 5. QUERY FAMILIARITY BEFORE AN ARBITRARY OBSERVATION
# ============================================================

def familiarity_before_observation_32f2(
    driver_id,
    lineage,
    race_ord,
    candidate,
):
    """
    Familiarity entering an arbitrary race/weekend.

    Uses the most recent PRIOR exposure to this organisation.

    Current-race result existence is irrelevant.
    """

    key = (
        int(driver_id),
        str(lineage),
    )

    if key not in history_32f2:
        return 0.0

    hist = history_32f2[key]

    ords = hist[
        "race_ord"
    ]

    # Index of final exposure STRICTLY before current race.
    pos = (
        np.searchsorted(
            ords,
            int(race_ord),
            side="left",
        )
        - 1
    )

    if pos < 0:
        return 0.0

    last_ord = int(
        ords[pos]
    )

    stock_after = float(
        hist[
            f"stock_after_{candidate}"
        ][pos]
    )

    half_life = (
        SHORTLIST_32F2[
            candidate
        ]
    )

    missed = max(
        0,
        int(race_ord)
        - last_ord
        - 1,
    )

    stock_before = (
        stock_after
        * 0.5 ** (
            missed
            / half_life
        )
    )

    return float(
        sat_32f2(
            stock_before
        )
    )


# ============================================================
# 6. PREPARE ARBITRARY DRIVER-LEVEL OBSERVATIONS
# ============================================================

def prepare_observations_32f2(
    df,
    *,
    label,
):
    """
    Add year/race_ord/constructorRef/lineage and evaluate each
    familiarity candidate.
    """

    out = df.copy()

    # Remove stale helper columns if this cell is rerun.
    stale = [
        "race_ord",
        "_year_lookup",
        "constructorRef_32f2",
        "lineage_32f2",
    ]

    out = out.drop(
        columns=[
            c for c in stale
            if c in out.columns
        ],
        errors="ignore",
    )

    # Race order / fallback year.
    race_meta = (
        race_order_lookup_32f2[
            [
                "raceId",
                "year",
                "race_ord",
            ]
        ]
        .rename(
            columns={
                "year":
                    "_year_lookup",
            }
        )
    )

    out = out.merge(
        race_meta,
        on="raceId",
        how="left",
        validate="many_to_one",
    )

    if "year" not in out.columns:
        out["year"] = (
            out["_year_lookup"]
        )
    else:
        out["year"] = (
            pd.to_numeric(
                out["year"],
                errors="coerce",
            )
            .fillna(
                out["_year_lookup"]
            )
        )

    # Constructor ref.
    out = out.merge(
        constructor_lookup_32f2
        .rename(
            columns={
                "constructorRef":
                    "constructorRef_32f2",
            }
        ),
        on="constructorId",
        how="left",
        validate="many_to_one",
    )

    if out[
        "race_ord"
    ].isna().any():

        raise RuntimeError(
            f"{label}: missing race_ord "
            "after race lookup."
        )

    if out[
        "constructorRef_32f2"
    ].isna().any():

        bad = out.loc[
            out[
                "constructorRef_32f2"
            ].isna(),
            [
                "raceId",
                "driverId",
                "constructorId",
            ],
        ]

        raise RuntimeError(
            f"{label}: constructorRef lookup failed:\n"
            + bad.head(20).to_string(
                index=False
            )
        )

    # Organisational lineage at this observation.
    out["lineage_32f2"] = [
        lineage_from_ref_year_32f2(
            ref,
            year,
        )
        for ref, year in zip(
            out[
                "constructorRef_32f2"
            ],
            out["year"],
        )
    ]

    # Candidate raw familiarity.
    for candidate in (
        SHORTLIST_32F2
    ):

        vals = []

        for (
            driver_id,
            lineage,
            race_ord,
        ) in zip(
            out["driverId"],
            out["lineage_32f2"],
            out["race_ord"],
        ):

            vals.append(
                familiarity_before_observation_32f2(
                    driver_id,
                    lineage,
                    race_ord,
                    candidate,
                )
            )

        out[
            f"{candidate}_raw"
        ] = np.asarray(
            vals,
            dtype=float,
        )

    return out


pace_obs_32f2 = (
    prepare_observations_32f2(
        df_pace,
        label="pace",
    )
)

quali_obs_32f2 = (
    prepare_observations_32f2(
        df_quali,
        label="qualifying",
    )
)


# ============================================================
# 7. CENTRE EXACTLY AS THE MODEL DATA PIPELINE DOES
# ============================================================

for candidate in (
    SHORTLIST_32F2
):

    # Pace: whole race field.
    pace_obs_32f2[
        f"{candidate}_tenure_c"
    ] = (
        pace_obs_32f2[
            f"{candidate}_raw"
        ]
        -
        pace_obs_32f2.groupby(
            "raceId"
        )[
            f"{candidate}_raw"
        ]
        .transform("mean")
    )

    # Qualifying: each timed session separately.
    quali_obs_32f2[
        f"{candidate}_tenure_c"
    ] = (
        quali_obs_32f2[
            f"{candidate}_raw"
        ]
        -
        quali_obs_32f2.groupby(
            [
                "raceId",
                "session",
            ]
        )[
            f"{candidate}_raw"
        ]
        .transform("mean")
    )


# ============================================================
# 8. ATTACH CENTRED DRIVER FAMILIARITIES TO PAIR TABLES
# ============================================================

def rebuild_pair_table_32f2(
    pair_df,
    driver_df,
    *,
    include_session,
):
    """
    Rebuild candidate tenure_diff and tenure_mean using the exact
    driver-level centred candidate familiarities.
    """

    out = pair_df.copy()

    candidate_cols = [
        f"{c}_tenure_c"
        for c in SHORTLIST_32F2
    ]

    base_cols = [
        "raceId",
        "driverId",
    ]

    if include_session:
        base_cols.append(
            "session"
        )

    # Driver A.
    a = (
        driver_df[
            base_cols
            + candidate_cols
        ]
        .copy()
        .rename(
            columns={
                "driverId":
                    "driverId_a",

                **{
                    c:
                        f"{c}_a"
                    for c
                    in candidate_cols
                },
            }
        )
    )

    a_keys = [
        "raceId",
        "driverId_a",
    ]

    if include_session:
        a_keys.append(
            "session"
        )

    out = out.merge(
        a,
        on=a_keys,
        how="left",
        validate="many_to_one",
    )

    # Driver B.
    b = (
        driver_df[
            base_cols
            + candidate_cols
        ]
        .copy()
        .rename(
            columns={
                "driverId":
                    "driverId_b",

                **{
                    c:
                        f"{c}_b"
                    for c
                    in candidate_cols
                },
            }
        )
    )

    b_keys = [
        "raceId",
        "driverId_b",
    ]

    if include_session:
        b_keys.append(
            "session"
        )

    out = out.merge(
        b,
        on=b_keys,
        how="left",
        validate="many_to_one",
    )

    for candidate in (
        SHORTLIST_32F2
    ):

        c = (
            f"{candidate}_tenure_c"
        )

        out[
            f"{candidate}_tenure_diff"
        ] = (
            out[f"{c}_a"]
            -
            out[f"{c}_b"]
        )

        out[
            f"{candidate}_tenure_mean"
        ] = (
            0.5
            * (
                out[f"{c}_a"]
                +
                out[f"{c}_b"]
            )
        )

    return out


pace_contrast_32f2 = (
    rebuild_pair_table_32f2(
        pace_contrast_all,
        pace_obs_32f2,
        include_session=False,
    )
)

pace_level_32f2 = (
    rebuild_pair_table_32f2(
        pace_level_all,
        pace_obs_32f2,
        include_session=False,
    )
)

q_contrast_32f2 = (
    rebuild_pair_table_32f2(
        q_contrast_all,
        quali_obs_32f2,
        include_session=True,
    )
)

q1_level_32f2 = (
    rebuild_pair_table_32f2(
        q_level_all_model,
        quali_obs_32f2,
        include_session=True,
    )
)


# ============================================================
# 9. ZERO-MISSINGNESS AUDIT
# ============================================================

missing_rows_32f2 = []


for channel, df, suffix in [
    (
        "pace_contrast",
        pace_contrast_32f2,
        "tenure_diff",
    ),
    (
        "pace_level",
        pace_level_32f2,
        "tenure_mean",
    ),
    (
        "q_contrast",
        q_contrast_32f2,
        "tenure_diff",
    ),
    (
        "q1_level",
        q1_level_32f2,
        "tenure_mean",
    ),
]:

    for candidate in (
        SHORTLIST_32F2
    ):

        col = (
            f"{candidate}_{suffix}"
        )

        n_missing = int(
            df[col]
            .isna()
            .sum()
        )

        missing_rows_32f2.append(
            {
                "channel":
                    channel,

                "candidate":
                    candidate,

                "rows":
                    len(df),

                "missing":
                    n_missing,

                "finite":
                    int(
                        np.isfinite(
                            df[col]
                        ).sum()
                    ),
            }
        )


missingness_32f2 = (
    pd.DataFrame(
        missing_rows_32f2
    )
)


print()
print("=" * 76)
print("STAGE 32F.2 — CANDIDATE MISSINGNESS")
print("=" * 76)

display(
    missingness_32f2
)


if (
    missingness_32f2[
        "missing"
    ].sum()
    != 0
):

    raise RuntimeError(
        "Stage 32F.2 still has missing "
        "candidate familiarity values."
    )


# ============================================================
# 10. IDENTIFY THE OBSERVATIONS THAT STAGE 32F.1 MISSED
# ============================================================

# Candidate used only as a convenient probe; all three should now
# resolve identically in terms of missingness.
probe_candidate_32f2 = (
    "memory_20gp"
)

previous_q_missing_32f2 = (
    q_contrast_exact_32f1.loc[
        q_contrast_exact_32f1[
            f"{probe_candidate_32f2}_tenure_diff"
        ].isna()
    ]
    .copy()
    if (
        "q_contrast_exact_32f1"
        in globals()
    )
    else pd.DataFrame()
)


print()
print("=" * 76)
print("PREVIOUSLY MISSING QUALIFYING PAIRS")
print("=" * 76)

if previous_q_missing_32f2.empty:

    print(
        "No previous missing qualifying pairs "
        "found in memory."
    )

else:

    keys = [
        c for c in [
            "year",
            "raceId",
            "session",
            "driver_a",
            "driver_b",
            "constructorId",
        ]
        if c in previous_q_missing_32f2.columns
    ]

    display(
        previous_q_missing_32f2[
            keys
        ]
    )


# ============================================================
# 11. CONFIRM 2015 MANOR IS NOW HANDLED ORGANISATIONALLY
# ============================================================

manor_2015_32f2 = (
    quali_obs_32f2.loc[
        (
            quali_obs_32f2[
                "year"
            ].eq(2015)
        )
        &
        (
            quali_obs_32f2[
                "constructorId"
            ].eq(206)
        )
    ]
    .copy()
)


print()
print("=" * 76)
print("2015 MANOR QUALIFYING — CORRECTED ORGANISATIONAL FAMILIARITY")
print("=" * 76)

manor_cols_32f2 = [
    c for c in [
        "raceId",
        "year",
        "session",
        "driver",
        "driverId",
        "constructorId",
        "constructorRef_32f2",
        "lineage_32f2",
        "memory_10gp_raw",
        "memory_20gp_raw",
        "memory_40gp_raw",
        "memory_10gp_tenure_c",
        "memory_20gp_tenure_c",
        "memory_40gp_tenure_c",
    ]
    if c in manor_2015_32f2.columns
]

display(
    manor_2015_32f2[
        manor_cols_32f2
    ]
    .sort_values(
        [
            "raceId",
            "session",
            "driver",
        ]
    )
)


# ============================================================
# 12. CROSS-CANDIDATE SENSITIVITY
# ============================================================
#
# These are the rows that can actually discriminate between the
# 10/20/40-GP definitions in Stage 32G.
#
# ============================================================

sensitivity_rows_32f2 = []


for (
    channel,
    df,
    suffix,
) in [
    (
        "pace_contrast",
        pace_contrast_32f2,
        "tenure_diff",
    ),
    (
        "pace_level",
        pace_level_32f2,
        "tenure_mean",
    ),
    (
        "q_contrast",
        q_contrast_32f2,
        "tenure_diff",
    ),
    (
        "q1_level",
        q1_level_32f2,
        "tenure_mean",
    ),
]:

    vals = np.column_stack(
        [
            df[
                f"{candidate}_{suffix}"
            ].to_numpy(
                dtype=float
            )
            for candidate in SHORTLIST_32F2
        ]
    )

    spread = (
        np.max(
            vals,
            axis=1,
        )
        -
        np.min(
            vals,
            axis=1,
        )
    )

    temp = pd.DataFrame(
        {
            "channel":
                channel,

            "raceId":
                df[
                    "raceId"
                ].to_numpy(),

            "year":
                (
                    df["year"]
                    .to_numpy()
                    if "year" in df.columns
                    else np.nan
                ),

            "candidate_spread":
                spread,
        }
    )

    sensitivity_rows_32f2.append(
        temp
    )


sensitivity_32f2 = (
    pd.concat(
        sensitivity_rows_32f2,
        ignore_index=True,
    )
)


sensitivity_summary_32f2 = (
    sensitivity_32f2.groupby(
        "channel",
        as_index=False,
    )
    .agg(
        rows=(
            "candidate_spread",
            "size",
        ),
        rows_different_1e_6=(
            "candidate_spread",
            lambda x:
                int(
                    (
                        x > 1e-6
                    ).sum()
                ),
        ),
        rows_different_0_01=(
            "candidate_spread",
            lambda x:
                int(
                    (
                        x > 0.01
                    ).sum()
                ),
        ),
        median_spread=(
            "candidate_spread",
            "median",
        ),
        p90_spread=(
            "candidate_spread",
            lambda x:
                float(
                    np.quantile(
                        x,
                        0.90,
                    )
                ),
        ),
        p95_spread=(
            "candidate_spread",
            lambda x:
                float(
                    np.quantile(
                        x,
                        0.95,
                    )
                ),
        ),
        max_spread=(
            "candidate_spread",
            "max",
        ),
    )
)


print()
print("=" * 76)
print("10/20/40-GP DISCRIMINATING OBSERVATIONS")
print("=" * 76)

display(
    sensitivity_summary_32f2
)


# ============================================================
# 13. SAVE
# ============================================================

pace_obs_32f2.to_csv(
    OUTDIR_32F2
    / "pace_driver_candidate_familiarity.csv",
    index=False,
)

quali_obs_32f2.to_csv(
    OUTDIR_32F2
    / "quali_driver_candidate_familiarity.csv",
    index=False,
)

missingness_32f2.to_csv(
    OUTDIR_32F2
    / "candidate_missingness.csv",
    index=False,
)

sensitivity_32f2.to_csv(
    OUTDIR_32F2
    / "candidate_discrimination_rows.csv",
    index=False,
)

sensitivity_summary_32f2.to_csv(
    OUTDIR_32F2
    / "candidate_discrimination_summary.csv",
    index=False,
)


# ============================================================
# 14. FINAL TRIAGE
# ============================================================

print()
print("=" * 76)
print("STAGE 32F.2 TRIAGE SUMMARY")
print("=" * 76)

print(
    "Candidate familiarity missing values:",
    int(
        missingness_32f2[
            "missing"
        ].sum()
    ),
)

print(
    "Pace driver rows:",
    f"{len(pace_obs_32f2):,}",
)

print(
    "Qualifying driver/session rows:",
    f"{len(quali_obs_32f2):,}",
)

print(
    "\nStage-32G candidates:"
)

for candidate in (
    SHORTLIST_32F2
):
    print(
        f"  • {candidate}"
    )

print(
    "\nIf missing values = 0, the candidate covariates are now "
    "safe for predictive model comparison."
)

print(
    "\nSaved outputs ->"
)
print(
    OUTDIR_32F2.resolve()
)

print(
    "\nStage 32F.2 complete."
)

## Part V — Pantheon v2.2 LIVE production model

In [ ]:
# ============================================================
# STAGE 32H-A — LOCK 20-GP ORGANISATIONAL FAMILIARITY,
#               REBUILD FULL 1982–2026 LIVE MODEL INPUTS
# ============================================================
#
# PURPOSE
# -------
# We have now completed the familiarity-memory sensitivity study.
#
# Predictive CV could not distinguish 10 / 20 / 40 GP half-lives:
#
#   20 vs 40:  ΔELPD = +0.317 ± 2.221
#   20 vs 10:  ΔELPD = +0.753 ± 2.256
#
# We therefore lock:
#
#       familiarity memory half-life = 20 missed GP weekends
#
# as the central behaviourally-plausible specification.
#
# This cell:
#
#   1. takes the already-audited Stage-32F familiarity history;
#   2. uses memory_20gp as the production familiarity variable;
#   3. switches tenure lookup to driver/race rather than
#      driver/race/constructor ID, fixing rebrand-ID mismatches;
#   4. rebuilds:
#        - driver-event familiarity lookup
#        - modern pace data
#        - modern qualifying data
#        - pace contrasts / levels
#        - qualifying contrasts / Q1 levels
#        - historical qualifying/race PL tables
#        - combined driver/car state index
#        - gap-aware driver-drift operator
#        - historical ranking packs
#        - sparse matrices
#        - complete 1982–2026 LIVE PyMC model
#   5. performs structural audits before any expensive sampling.
#
# IMPORTANT
# ---------
# NO SAMPLING occurs here.
#
# The frozen 1982–2025 trace and model outputs are not touched.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import pytensor.sparse as ps

from IPython.display import display


# ============================================================
# 0. LOCKED LIVE SPECIFICATION
# ============================================================

LIVE_FAMILIARITY_SPEC = "memory_20gp"
LIVE_FAMILIARITY_HALF_LIFE_GP = 20

EXPECTED_LIVE_CEILING_YEAR = 2026

# These are the full 1996–2026 evidence counts already verified
# in Stage 32B.
EXPECTED_MODERN_COUNTS_32H = {
    "pace_contrast": 5197,
    "pace_level": 5197,
    "q_contrast": 9303,
    "q1_level": 5095,
}

# Historical likelihood remains 1982–1995.
EXPECTED_HIST_COUNTS_32H = {
    "hist_quali": 5726,
    "hist_race": 2490,
}


print("=" * 76)
print("STAGE 32H-A — LOCKED LIVE FAMILIARITY SPECIFICATION")
print("=" * 76)

print(
    "Selected familiarity memory:",
    LIVE_FAMILIARITY_SPEC,
)

print(
    "Half-life:",
    LIVE_FAMILIARITY_HALF_LIFE_GP,
    "missed GP weekends",
)

print(
    "CFG.ceiling_year:",
    CFG.ceiling_year,
)


if int(CFG.ceiling_year) != EXPECTED_LIVE_CEILING_YEAR:

    raise RuntimeError(
        "CFG.ceiling_year is not 2026. "
        "Do not continue with the LIVE rebuild."
    )


# ============================================================
# 1. REQUIRED EXISTING OBJECTS
# ============================================================

required_objects_32h = [
    "raw",
    "CFG",

    # Stage 32F memory-history object
    "contexts_32f",

    # Existing construction functions
    "build_lap_relative_pace",
    "build_timed_qualifying",
    "build_pair_and_level_data",
    "build_historical_rankings",
    "build_combined_index",
    "build_gap_aware_driver_drift_operator",
    "pack_rankings",
    "plackett_luce_logp_vectorized",

    # Frozen model builder architecture
    "build_joint_model_1982_2025",
]


missing_objects_32h = [
    name
    for name in required_objects_32h
    if name not in globals()
]


if missing_objects_32h:

    raise RuntimeError(
        "Stage 32H-A is missing required objects:\n  "
        +
        "\n  ".join(
            missing_objects_32h
        )
    )


# ============================================================
# 2. LOCATE THE EXACT 20-GP FAMILIARITY SOURCES
# ============================================================
#
# IMPORTANT
# ---------
# contexts_32f is based primarily on raw.results contexts.
#
# That is sufficient for almost all driver/race observations,
# but Stage 32F.2 demonstrated that a tiny number of valid
# qualifying observations do not have a matching raw.results
# context.
#
# Example:
#
#   Stroll
#   raceId = 1153
#   2025
#   Q1 / Q2
#
# Stage 32F.2 explicitly built observation-safe driver tables
# which solve this problem.
#
# Therefore the production familiarity lookup must be the UNION
# of:
#
#   1. full contexts_32f history;
#   2. pace_obs_32f2;
#   3. quali_obs_32f2.
#
# The lookup itself remains one raw familiarity value per:
#
#       raceId + driverId
#
# ============================================================

required_familiarity_objects_32h = [
    "contexts_32f",
    "pace_obs_32f2",
    "quali_obs_32f2",
]


missing_familiarity_objects_32h = [
    name
    for name in required_familiarity_objects_32h
    if name not in globals()
]


if missing_familiarity_objects_32h:

    raise RuntimeError(
        "Stage 32H-A requires the observation-safe Stage-32F.2 "
        "objects:\n  "
        +
        "\n  ".join(
            missing_familiarity_objects_32h
        )
        +
        "\n\nRun Stage 32F.2 first."
    )


# ------------------------------------------------------------
# Full-history Stage-32F source.
# ------------------------------------------------------------

context_memory_candidates_32h = [
    "fam_memory_20gp",
    "memory_20gp_raw",
    "memory_20gp",
]


context_memory_col_32h = next(
    (
        col
        for col in context_memory_candidates_32h
        if col in contexts_32f.columns
    ),
    None,
)


if context_memory_col_32h is None:

    raise RuntimeError(
        "Could not identify the 20-GP familiarity column "
        "in contexts_32f."
    )


# ------------------------------------------------------------
# Observation-safe Stage-32F.2 sources.
# ------------------------------------------------------------

obs_memory_col_32h = "memory_20gp_raw"


for name, df in [
    (
        "pace_obs_32f2",
        pace_obs_32f2,
    ),
    (
        "quali_obs_32f2",
        quali_obs_32f2,
    ),
]:

    required_cols = {
        "raceId",
        "driverId",
        obs_memory_col_32h,
    }


    missing_cols = (
        required_cols
        -
        set(
            df.columns
        )
    )


    if missing_cols:

        raise RuntimeError(
            f"{name} is missing required columns: "
            f"{sorted(missing_cols)}"
        )


print()
print("=" * 76)
print("20-GP FAMILIARITY SOURCES")
print("=" * 76)

print(
    "contexts_32f:",
    context_memory_col_32h,
)

print(
    "pace_obs_32f2:",
    obs_memory_col_32h,
)

print(
    "quali_obs_32f2:",
    obs_memory_col_32h,
)


# ============================================================
# 3. BUILD OBSERVATION-SAFE DRIVER / GP FAMILIARITY LOOKUP
# ============================================================

# ------------------------------------------------------------
# A. Full historical / results-based familiarity contexts
# ------------------------------------------------------------

familiarity_from_contexts_32h = (
    contexts_32f[
        [
            "raceId",
            "driverId",
            context_memory_col_32h,
        ]
    ]
    .copy()
    .rename(
        columns={
            context_memory_col_32h:
                "tenure_raw",
        }
    )
)


familiarity_from_contexts_32h[
    "source_32h"
] = "contexts_32f"


# ------------------------------------------------------------
# B. Observation-safe pace contexts
# ------------------------------------------------------------

familiarity_from_pace_32h = (
    pace_obs_32f2[
        [
            "raceId",
            "driverId",
            obs_memory_col_32h,
        ]
    ]
    .copy()
    .rename(
        columns={
            obs_memory_col_32h:
                "tenure_raw",
        }
    )
)


familiarity_from_pace_32h[
    "source_32h"
] = "pace_obs_32f2"


# ------------------------------------------------------------
# C. Observation-safe qualifying contexts
# ------------------------------------------------------------

familiarity_from_quali_32h = (
    quali_obs_32f2[
        [
            "raceId",
            "driverId",
            obs_memory_col_32h,
        ]
    ]
    .copy()
    .rename(
        columns={
            obs_memory_col_32h:
                "tenure_raw",
        }
    )
)


familiarity_from_quali_32h[
    "source_32h"
] = "quali_obs_32f2"


# ------------------------------------------------------------
# D. Combine all sources
# ------------------------------------------------------------

familiarity_union_32h = pd.concat(
    [
        familiarity_from_contexts_32h,
        familiarity_from_pace_32h,
        familiarity_from_quali_32h,
    ],
    ignore_index=True,
)


familiarity_union_32h[
    "tenure_raw"
] = pd.to_numeric(
    familiarity_union_32h[
        "tenure_raw"
    ],
    errors="coerce",
)


if familiarity_union_32h[
    "tenure_raw"
].isna().any():

    bad = familiarity_union_32h.loc[
        familiarity_union_32h[
            "tenure_raw"
        ].isna()
    ]


    display(
        bad.head(30)
    )


    raise RuntimeError(
        "Missing familiarity values found in the combined "
        "20-GP familiarity sources."
    )


if not np.isfinite(
    familiarity_union_32h[
        "tenure_raw"
    ].to_numpy(
        dtype=float
    )
).all():

    raise RuntimeError(
        "Non-finite familiarity values found in combined sources."
    )


# ============================================================
# 3A. VERIFY CROSS-SOURCE CONSISTENCY
# ============================================================
#
# If contexts_32f, pace_obs_32f2 and quali_obs_32f2 all contain
# the same driver/race, they MUST agree on raw familiarity.
#
# We never silently choose one conflicting source.
# ============================================================

familiarity_consistency_32h = (
    familiarity_union_32h
    .groupby(
        [
            "raceId",
            "driverId",
        ],
        as_index=False,
    )
    .agg(
        n_rows=(
            "tenure_raw",
            "size",
        ),

        n_sources=(
            "source_32h",
            "nunique",
        ),

        min_familiarity=(
            "tenure_raw",
            "min",
        ),

        max_familiarity=(
            "tenure_raw",
            "max",
        ),
    )
)


familiarity_consistency_32h[
    "spread"
] = (
    familiarity_consistency_32h[
        "max_familiarity"
    ]
    -
    familiarity_consistency_32h[
        "min_familiarity"
    ]
)


bad_consistency_32h = (
    familiarity_consistency_32h[
        familiarity_consistency_32h[
            "spread"
        ].abs()
        > 1.0e-10
    ]
)


print()
print("=" * 76)
print("CROSS-SOURCE FAMILIARITY CONSISTENCY")
print("=" * 76)

print(
    "Combined source rows:",
    f"{len(familiarity_union_32h):,}",
)

print(
    "Unique driver/GP keys:",
    f"{len(familiarity_consistency_32h):,}",
)

print(
    "Driver/GP keys represented in multiple sources:",
    f"{int((familiarity_consistency_32h['n_sources'] > 1).sum()):,}",
)

print(
    "Conflicting driver/GP familiarity values:",
    f"{len(bad_consistency_32h):,}",
)


if len(
    bad_consistency_32h
):

    display(
        bad_consistency_32h
        .sort_values(
            "spread",
            ascending=False,
        )
        .head(30)
    )


    raise RuntimeError(
        "The Stage-32F / Stage-32F.2 familiarity sources "
        "disagree for at least one driver/race."
    )


# ============================================================
# 3B. COLLAPSE TO ONE VALUE PER DRIVER / GP
# ============================================================

driver_event_cov = (
    familiarity_union_32h
    .groupby(
        [
            "raceId",
            "driverId",
        ],
        as_index=False,
    )
    .agg(
        tenure_raw=(
            "tenure_raw",
            "first",
        )
    )
)


if driver_event_cov.duplicated(
    [
        "raceId",
        "driverId",
    ]
).any():

    raise RuntimeError(
        "driver_event_cov is not unique by raceId + driverId."
    )


print()
print("=" * 76)
print("OBSERVATION-SAFE PRODUCTION FAMILIARITY LOOKUP")
print("=" * 76)

print(
    "Driver/GP contexts:",
    f"{len(driver_event_cov):,}",
)

print(
    "Missing familiarity:",
    int(
        driver_event_cov[
            "tenure_raw"
        ].isna().sum()
    ),
)


# ============================================================
# 3C. EXPLICITLY VERIFY THE PREVIOUSLY-MISSING STROLL CASE
# ============================================================

driver_id_to_ref_32h = {
    int(driver_id):
        driver_ref
    for driver_id, driver_ref
    in raw.driver_ref.items()
}


stroll_ids_32h = [
    driver_id
    for driver_id, driver_ref
    in driver_id_to_ref_32h.items()
    if driver_ref == "stroll"
]


if not stroll_ids_32h:

    raise RuntimeError(
        "Could not resolve Stroll's driverId."
    )


stroll_1153_32h = (
    driver_event_cov[
        (
            driver_event_cov[
                "raceId"
            ]
            == 1153
        )
        &
        (
            driver_event_cov[
                "driverId"
            ].isin(
                stroll_ids_32h
            )
        )
    ]
    .copy()
)


print()
print("=" * 76)
print("EXPLICIT STROLL 2025 RACE-1153 CHECK")
print("=" * 76)

display(
    stroll_1153_32h
)


if len(
    stroll_1153_32h
) != 1:

    raise RuntimeError(
        "Stroll raceId=1153 still does not have exactly one "
        "observation-safe familiarity context."
    )


if not np.isfinite(
    stroll_1153_32h[
        "tenure_raw"
    ].iloc[
        0
    ]
):

    raise RuntimeError(
        "Stroll raceId=1153 familiarity is not finite."
    )


print(
    "Stroll raceId=1153 familiarity successfully recovered."
)


# ============================================================
# 4. REDEFINE MODERN TENURE-CENTRING HELPER
# ============================================================
#
# Same mathematical construction as before:
#
#       tenure_c = raw familiarity
#                  - within-observation-group mean familiarity
#
# But lookup is now driver/race, not literal constructor ID.
# ============================================================

def add_centered_tenure(
    df,
    driver_event_cov,
    group_cols,
):

    out = df.merge(
        driver_event_cov[
            [
                "raceId",
                "driverId",
                "tenure_raw",
            ]
        ],
        on=[
            "raceId",
            "driverId",
        ],
        how="left",
        validate="many_to_one",
    )


    missing = int(
        out[
            "tenure_raw"
        ].isna().sum()
    )


    if missing:

        missing_rows = (
            out.loc[
                out[
                    "tenure_raw"
                ].isna(),
                [
                    col
                    for col in [
                        "raceId",
                        "year",
                        "session",
                        "driverId",
                        "driver",
                        "constructorId",
                    ]
                    if col in out.columns
                ],
            ]
            .head(30)
        )

        display(
            missing_rows
        )

        raise RuntimeError(
            f"Modern familiarity lookup failed for "
            f"{missing} observations."
        )


    out[
        "tenure_c"
    ] = (
        out[
            "tenure_raw"
        ]
        -
        out.groupby(
            group_cols
        )[
            "tenure_raw"
        ]
        .transform(
            "mean"
        )
    )


    return out


# ============================================================
# 5. REDEFINE HISTORICAL TENURE-CENTRING HELPER
# ============================================================

def add_historical_tenure(
    df,
    driver_event_cov,
):

    out = df.merge(
        driver_event_cov[
            [
                "raceId",
                "driverId",
                "tenure_raw",
            ]
        ],
        on=[
            "raceId",
            "driverId",
        ],
        how="left",
        validate="many_to_one",
    )


    missing = int(
        out[
            "tenure_raw"
        ].isna().sum()
    )


    if missing:

        missing_rows = (
            out.loc[
                out[
                    "tenure_raw"
                ].isna(),
                [
                    col
                    for col in [
                        "raceId",
                        "year",
                        "round",
                        "driverId",
                        "driver",
                        "constructorId",
                    ]
                    if col in out.columns
                ],
            ]
            .head(30)
        )

        display(
            missing_rows
        )

        raise RuntimeError(
            f"Historical familiarity lookup failed for "
            f"{missing} observations."
        )


    out[
        "tenure_c"
    ] = (
        out[
            "tenure_raw"
        ]
        -
        out.groupby(
            "raceId"
        )[
            "tenure_raw"
        ]
        .transform(
            "mean"
        )
    )


    return out


# ============================================================
# 6. REBUILD FULL 1996–2026 DRIVER-LEVEL MODERN DATA
# ============================================================

print()
print("=" * 76)
print("REBUILDING FULL 1996–2026 MODERN DRIVER OBSERVATIONS")
print("=" * 76)


df_pace = build_lap_relative_pace(
    raw,
    driver_event_cov,
    CFG,
)


df_quali = build_timed_qualifying(
    raw,
    driver_event_cov,
    CFG,
)


print(
    "Pace driver/race observations:",
    f"{len(df_pace):,}",
)

print(
    "Qualifying driver/session observations:",
    f"{len(df_quali):,}",
)


print(
    "Pace years:",
    int(df_pace["year"].min()),
    "–",
    int(df_pace["year"].max()),
)

print(
    "Qualifying years:",
    int(df_quali["year"].min()),
    "–",
    int(df_quali["year"].max()),
)


if int(
    df_pace[
        "year"
    ].max()
) != 2026:

    raise RuntimeError(
        "2026 is missing from rebuilt race-pace data."
    )


if int(
    df_quali[
        "year"
    ].max()
) != 2026:

    raise RuntimeError(
        "2026 is missing from rebuilt qualifying data."
    )


# ============================================================
# 7. REBUILD MODERN TEAM CONTRAST + LEVEL TABLES
# ============================================================

pace_contrast, pace_level = (
    build_pair_and_level_data(
        df_pace,
        y_col=
            "y_pace",
        individual_weight_col=
            "pace_weight",
    )
)


q_contrast, q_level_all = (
    build_pair_and_level_data(
        df_quali,
        y_col=
            "y_quali",
        individual_weight_col=
            "quali_weight",
        session_col=
            "session",
    )
)


# ------------------------------------------------------------
# Same frozen qualifying weighting:
#
# Q1/Q2/Q3 common sessions must not triple-count a team pair.
# ------------------------------------------------------------

n_common_sessions_32h = (
    q_contrast
    .groupby(
        [
            "raceId",
            "constructorId",
        ]
    )[
        "session"
    ]
    .transform(
        "count"
    )
    .astype(float)
)


q_contrast[
    "weight"
] = (
    1.0
    /
    n_common_sessions_32h
)


# ------------------------------------------------------------
# Same frozen team-level qualifying anchor:
# Q1 only.
# ------------------------------------------------------------

q_level = (
    q_level_all[
        q_level_all[
            "session"
        ]
        == "q1"
    ]
    .copy()
)


q_level[
    "weight"
] = 1.0


print()
print("=" * 76)
print("FULL MODERN LIKELIHOOD TABLES")
print("=" * 76)


modern_counts_32h = {
    "pace_contrast":
        len(
            pace_contrast
        ),

    "pace_level":
        len(
            pace_level
        ),

    "q_contrast":
        len(
            q_contrast
        ),

    "q1_level":
        len(
            q_level
        ),
}


for name, n in modern_counts_32h.items():

    print(
        f"{name:15s}:",
        f"{n:,}",
    )


for name, expected in EXPECTED_MODERN_COUNTS_32H.items():

    observed = modern_counts_32h[
        name
    ]

    if observed != expected:

        raise RuntimeError(
            f"{name}: expected {expected:,} rows from the "
            f"verified Stage-32B 2026 ingestion, but rebuilt "
            f"{observed:,}."
        )


# ============================================================
# 8. TENURE / FAMILIARITY MISSINGNESS + CENTRING AUDIT
# ============================================================

modern_familiarity_audit_32h = pd.DataFrame(
    [
        {
            "table":
                "df_pace",

            "rows":
                len(
                    df_pace
                ),

            "missing":
                int(
                    df_pace[
                        "tenure_c"
                    ].isna().sum()
                ),

            "max_abs_group_mean":
                float(
                    df_pace
                    .groupby(
                        "raceId"
                    )[
                        "tenure_c"
                    ]
                    .mean()
                    .abs()
                    .max()
                ),
        },

        {
            "table":
                "df_quali",

            "rows":
                len(
                    df_quali
                ),

            "missing":
                int(
                    df_quali[
                        "tenure_c"
                    ].isna().sum()
                ),

            "max_abs_group_mean":
                float(
                    df_quali
                    .groupby(
                        [
                            "raceId",
                            "session",
                        ]
                    )[
                        "tenure_c"
                    ]
                    .mean()
                    .abs()
                    .max()
                ),
        },

        {
            "table":
                "pace_contrast",

            "rows":
                len(
                    pace_contrast
                ),

            "missing":
                int(
                    pace_contrast[
                        "tenure_diff"
                    ].isna().sum()
                ),

            "max_abs_group_mean":
                np.nan,
        },

        {
            "table":
                "pace_level",

            "rows":
                len(
                    pace_level
                ),

            "missing":
                int(
                    pace_level[
                        "tenure_mean"
                    ].isna().sum()
                ),

            "max_abs_group_mean":
                np.nan,
        },

        {
            "table":
                "q_contrast",

            "rows":
                len(
                    q_contrast
                ),

            "missing":
                int(
                    q_contrast[
                        "tenure_diff"
                    ].isna().sum()
                ),

            "max_abs_group_mean":
                np.nan,
        },

        {
            "table":
                "q1_level",

            "rows":
                len(
                    q_level
                ),

            "missing":
                int(
                    q_level[
                        "tenure_mean"
                    ].isna().sum()
                ),

            "max_abs_group_mean":
                np.nan,
        },
    ]
)


display(
    modern_familiarity_audit_32h
)


if modern_familiarity_audit_32h[
    "missing"
].sum() != 0:

    raise RuntimeError(
        "Missing familiarity values remain in modern model data."
    )


# ============================================================
# 9. REBUILD HISTORICAL 1982–1995 TABLES WITH SAME MEMORY RULE
# ============================================================

print()
print("=" * 76)
print("REBUILDING HISTORICAL 1982–1995 TABLES")
print("=" * 76)


hist_quali, hist_race = (
    build_historical_rankings(
        raw,
        driver_event_cov,
        CFG,
    )
)


historical_counts_32h = {
    "hist_quali":
        len(
            hist_quali
        ),

    "hist_race":
        len(
            hist_race
        ),
}


print(
    "Historical qualifying:",
    f"{len(hist_quali):,}",
)

print(
    "Historical classified race:",
    f"{len(hist_race):,}",
)


for name, expected in EXPECTED_HIST_COUNTS_32H.items():

    observed = historical_counts_32h[
        name
    ]

    if observed != expected:

        raise RuntimeError(
            f"{name}: expected {expected:,} rows, "
            f"rebuilt {observed:,}."
        )


hist_centering_32h = pd.DataFrame(
    [
        {
            "table":
                "hist_quali",

            "rows":
                len(
                    hist_quali
                ),

            "missing":
                int(
                    hist_quali[
                        "tenure_c"
                    ].isna().sum()
                ),

            "max_abs_race_mean":
                float(
                    hist_quali
                    .groupby(
                        "raceId"
                    )[
                        "tenure_c"
                    ]
                    .mean()
                    .abs()
                    .max()
                ),
        },

        {
            "table":
                "hist_race",

            "rows":
                len(
                    hist_race
                ),

            "missing":
                int(
                    hist_race[
                        "tenure_c"
                    ].isna().sum()
                ),

            "max_abs_race_mean":
                float(
                    hist_race
                    .groupby(
                        "raceId"
                    )[
                        "tenure_c"
                    ]
                    .mean()
                    .abs()
                    .max()
                ),
        },
    ]
)


display(
    hist_centering_32h
)


if hist_centering_32h[
    "missing"
].sum() != 0:

    raise RuntimeError(
        "Missing historical familiarity values remain."
    )


# ============================================================
# 10. INSPECT IMPORTANT 2026 FAMILIARITY CASES
# ============================================================
#
# IMPORTANT:
# raw.results contains a small number of historical pre-1982
# driver/race duplicates caused by multiple constructor entries.
#
# This section is ONLY a 2026 audit, so restrict to 2026 before
# enforcing one-row-per-driver/race metadata.
# ============================================================

race_meta_32h = (
    raw.races[
        [
            "raceId",
            "year",
            "round",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Identify 2026 race IDs first
# ------------------------------------------------------------

race_ids_2026_32h = set(
    race_meta_32h.loc[
        race_meta_32h[
            "year"
        ]
        == 2026,
        "raceId",
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Driver / constructor metadata only for 2026
# ------------------------------------------------------------

driver_meta_32h = (
    raw.results.loc[
        raw.results[
            "raceId"
        ].isin(
            race_ids_2026_32h
        ),
        [
            "raceId",
            "driverId",
            "constructorId",
        ],
    ]
    .copy()
)


# ------------------------------------------------------------
# Check that 2026 really is unique by driver/race.
#
# Unlike the old Indianapolis-era data, it should be.
# ------------------------------------------------------------

duplicate_driver_races_2026_32h = (
    driver_meta_32h
    .duplicated(
        [
            "raceId",
            "driverId",
        ],
        keep=False,
    )
)


if duplicate_driver_races_2026_32h.any():

    bad_2026_meta_32h = (
        driver_meta_32h.loc[
            duplicate_driver_races_2026_32h
        ]
        .sort_values(
            [
                "raceId",
                "driverId",
            ]
        )
    )

    display(
        bad_2026_meta_32h
    )

    raise RuntimeError(
        "Unexpected duplicate driver/race metadata exists in 2026."
    )


# ------------------------------------------------------------
# Driver labels
# ------------------------------------------------------------

driver_meta_32h[
    "driver"
] = (
    driver_meta_32h[
        "driverId"
    ]
    .map(
        raw.driver_ref
    )
)


# ------------------------------------------------------------
# Constructor display names
# ------------------------------------------------------------

if (
    "constructorId"
    in raw.constructors.columns
    and
    "name"
    in raw.constructors.columns
):

    constructor_name_map_32h = {
        int(constructor_id):
            name
        for constructor_id, name
        in zip(
            raw.constructors[
                "constructorId"
            ],
            raw.constructors[
                "name"
            ],
        )
    }

else:

    constructor_name_map_32h = {}


constructor_ref_map_32h = {
    int(k):
        v
    for k, v
    in raw.constructor_ref.items()
}


driver_meta_32h[
    "constructor"
] = (
    driver_meta_32h[
        "constructorId"
    ]
    .map(
        constructor_name_map_32h
    )
)


driver_meta_32h[
    "constructor"
] = (
    driver_meta_32h[
        "constructor"
    ]
    .fillna(
        driver_meta_32h[
            "constructorId"
        ]
        .map(
            constructor_ref_map_32h
        )
    )
)


driver_meta_32h[
    "constructor"
] = (
    driver_meta_32h[
        "constructor"
    ]
    .fillna(
        driver_meta_32h[
            "constructorId"
        ]
        .astype(str)
    )
)


print()
print("=" * 76)
print("2026 DRIVER / CONSTRUCTOR METADATA CHECK")
print("=" * 76)

print(
    "2026 metadata rows:",
    f"{len(driver_meta_32h):,}",
)

print(
    "Duplicate 2026 driver/race keys:",
    int(
        driver_meta_32h
        .duplicated(
            [
                "raceId",
                "driverId",
            ]
        )
        .sum()
    ),
)

print(
    "Missing driver labels:",
    int(
        driver_meta_32h[
            "driver"
        ].isna().sum()
    ),
)

print(
    "Missing constructor labels:",
    int(
        driver_meta_32h[
            "constructor"
        ].isna().sum()
    ),
)


# ------------------------------------------------------------
# Restrict familiarity lookup to 2026 BEFORE metadata merge
# ------------------------------------------------------------

live_familiarity_audit_32h = (
    driver_event_cov
    .merge(
        race_meta_32h,
        on="raceId",
        how="left",
        validate="many_to_one",
    )
)


live_familiarity_audit_32h = (
    live_familiarity_audit_32h[
        live_familiarity_audit_32h[
            "year"
        ]
        == 2026
    ]
    .copy()
)


live_familiarity_audit_32h = (
    live_familiarity_audit_32h
    .merge(
        driver_meta_32h,
        on=[
            "raceId",
            "driverId",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "driver",
            "round",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Final metadata coverage check
# ------------------------------------------------------------

if live_familiarity_audit_32h[
    "driver"
].isna().any():

    display(
        live_familiarity_audit_32h.loc[
            live_familiarity_audit_32h[
                "driver"
            ].isna()
        ]
        .head(30)
    )

    raise RuntimeError(
        "At least one 2026 familiarity context lacks driver metadata."
    )


if live_familiarity_audit_32h[
    "constructor"
].isna().any():

    display(
        live_familiarity_audit_32h.loc[
            live_familiarity_audit_32h[
                "constructor"
            ].isna()
        ]
        .head(30)
    )

    raise RuntimeError(
        "At least one 2026 familiarity context lacks constructor metadata."
    )


# ------------------------------------------------------------
# First 2026 familiarity observation by driver
# ------------------------------------------------------------

first_2026_familiarity_32h = (
    live_familiarity_audit_32h
    .sort_values(
        [
            "driver",
            "round",
        ]
    )
    .groupby(
        "driver",
        as_index=False,
    )
    .first()
    [
        [
            "driver",
            "constructor",
            "round",
            "tenure_raw",
        ]
    ]
    .sort_values(
        [
            "constructor",
            "driver",
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 76)
print("FIRST 2026 FAMILIARITY CONTEXT BY DRIVER")
print("=" * 76)


display(
    first_2026_familiarity_32h
)


# ------------------------------------------------------------
# Explicit cases we care about
# ------------------------------------------------------------

important_drivers_32h = [
    "bortoleto",
    "hulkenberg",
    "bottas",
    "perez",
    "lawson",
    "tsunoda",
    "hadjar",
]


print()
print("=" * 76)
print("IMPORTANT 2026 FAMILIARITY CASES")
print("=" * 76)


display(
    first_2026_familiarity_32h[
        first_2026_familiarity_32h[
            "driver"
        ].isin(
            important_drivers_32h
        )
    ]
)


# ============================================================
# 11. FREEZE THE FULL LIVE MODERN TABLE COPIES
# ============================================================
#
# These are now the actual inputs for the LIVE model.
# ============================================================

pace_contrast_all = (
    pace_contrast.copy()
)

pace_level_all = (
    pace_level.copy()
)

q_contrast_all = (
    q_contrast.copy()
)

q_level_all_model = (
    q_level.copy()
)


# ============================================================
# 12. REBUILD COMBINED 1982–2026 STATE INDEX
# ============================================================

print()
print("=" * 76)
print("REBUILDING 1982–2026 COMBINED STATE INDEX")
print("=" * 76)


idx_all = build_combined_index(
    df_pace,
    df_quali,
    [
        pace_contrast_all,
        pace_level_all,
        q_contrast_all,
        q_level_all_model,
    ],
    hist_quali,
    hist_race,
)


L_driver_all = (
    build_gap_aware_driver_drift_operator(
        idx_all,
        CFG.rho_driver,
    )
)


state_years_32h = np.array(
    [
        year
        for _, year
        in idx_all.active_driver_years
    ],
    dtype=int,
)


constructor_state_years_32h = np.array(
    [
        year
        for _, year
        in idx_all.active_constructor_years
    ],
    dtype=int,
)


print(
    "Drivers:",
    len(
        idx_all.unique_drivers
    ),
)

print(
    "Driver-seasons:",
    len(
        idx_all.active_driver_years
    ),
)

print(
    "Constructor-seasons:",
    len(
        idx_all.active_constructor_years
    ),
)

print(
    "Driver transitions:",
    L_driver_all.shape[1],
)

print(
    "Driver-state year range:",
    state_years_32h.min(),
    "–",
    state_years_32h.max(),
)


if state_years_32h.max() != 2026:

    raise RuntimeError(
        "Combined driver state space does not reach 2026."
    )


if constructor_state_years_32h.max() != 2026:

    raise RuntimeError(
        "Combined constructor state space does not reach 2026."
    )


# ============================================================
# 13. 2026 DRIVER / CONSTRUCTOR STATE AUDIT
# ============================================================

driver_states_2026_32h = sorted(
    [
        driver
        for driver, year
        in idx_all.active_driver_years
        if year == 2026
    ]
)


constructor_states_2026_32h = sorted(
    [
        constructor_id
        for constructor_id, year
        in idx_all.active_constructor_years
        if year == 2026
    ]
)


print()
print(
    "2026 driver states:",
    len(
        driver_states_2026_32h
    ),
)

print(
    driver_states_2026_32h
)


print()
print(
    "2026 constructor states:",
    len(
        constructor_states_2026_32h
    ),
)


constructor_state_names_2026_32h = pd.DataFrame(
    {
        "constructorId":
            constructor_states_2026_32h,

        "constructor":
            [
                constructor_name_map_32h.get(
                    int(constructor_id),
                    constructor_ref_map_32h.get(
                        int(constructor_id),
                        str(
                            constructor_id
                        ),
                    ),
                )
                for constructor_id
                in constructor_states_2026_32h
            ],
    }
)


display(
    constructor_state_names_2026_32h
)


if len(
    driver_states_2026_32h
) != 23:

    raise RuntimeError(
        "Expected 23 unique 2026 driver-season states."
    )


if len(
    constructor_states_2026_32h
) != 11:

    raise RuntimeError(
        "Expected 11 2026 constructor-season states."
    )


# ============================================================
# 14. REPACK HISTORICAL PL LIKELIHOODS
# ============================================================

hist_q_pack = (
    pack_rankings(
        hist_quali
    )
)


hist_race_pack = (
    pack_rankings(
        hist_race
    )
)


print()
print(
    "Historical qualifying pack:",
    hist_q_pack[
        "n_races"
    ],
    "races ×",
    hist_q_pack[
        "max_field"
    ],
    "max drivers",
)


print(
    "Historical race pack:",
    hist_race_pack[
        "n_races"
    ],
    "races ×",
    hist_race_pack[
        "max_field"
    ],
    "max drivers",
)


# ============================================================
# 15. REBUILD SPARSE MATRICES FOR THE NEW STATE SPACE
# ============================================================
#
# build_joint_model_1982_2025() uses these as globals.
#
# We deliberately replace the in-memory sparse matrices only.
# No frozen files are modified.
# ============================================================

L_driver_sp = sp.csr_matrix(
    L_driver_all,
    dtype=np.float64,
)


P_driver_sp = sp.csr_matrix(
    idx_all.P_driver,
    dtype=np.float64,
)


P_car_sp = sp.csr_matrix(
    idx_all.P_car,
    dtype=np.float64,
)


L_driver_pt = (
    ps.as_sparse_variable(
        L_driver_sp
    )
)


P_driver_pt = (
    ps.as_sparse_variable(
        P_driver_sp
    )
)


P_car_pt = (
    ps.as_sparse_variable(
        P_car_sp
    )
)


def sparse_matvec(
    A,
    x,
):

    return ps.structured_dot(
        A,
        x[
            :,
            None,
        ],
    )[
        :,
        0,
    ]


print()
print("=" * 76)
print("LIVE SPARSE MATRICES")
print("=" * 76)


for name, matrix in [
    (
        "L_driver",
        L_driver_sp,
    ),
    (
        "P_driver",
        P_driver_sp,
    ),
    (
        "P_car",
        P_car_sp,
    ),
]:

    density = (
        matrix.nnz
        /
        (
            matrix.shape[0]
            *
            matrix.shape[1]
        )
    )

    print(
        f"{name:10s}",
        f"shape={matrix.shape}",
        f"nnz={matrix.nnz:,}",
        f"density={density:.6f}",
    )


# ============================================================
# 16. BUILD THE FULL 1982–2026 LIVE MODEL
# ============================================================
#
# The function name still says 1982_2025 because that is the
# frozen architecture's historical name.
#
# There is no year hard-coding in the statistical likelihood:
# it operates on the supplied tables and combined state index.
#
# We do NOT overwrite model_1982_2025.
# ============================================================

print()
print("=" * 76)
print("BUILDING 1982–2026 LIVE MODEL")
print("=" * 76)


model_1982_2026_live = (
    build_joint_model_1982_2025(
        pace_contrast_all,
        pace_level_all,
        q_contrast_all,
        q_level_all_model,
        hist_q_pack,
        hist_race_pack,
        idx_all,
        cfg=CFG,
    )
)


print(
    "Free RVs:",
    len(
        model_1982_2026_live.free_RVs
    ),
)

print(
    "Observed RVs:",
    len(
        model_1982_2026_live.observed_RVs
    ),
)

print(
    "Potentials:",
    len(
        model_1982_2026_live.potentials
    ),
)


# ============================================================
# 17. FINAL LIVE INPUT AUDIT
# ============================================================

audit_rows_32h = []


for label, df, tenure_col in [
    (
        "pace_contrast",
        pace_contrast_all,
        "tenure_diff",
    ),
    (
        "pace_level",
        pace_level_all,
        "tenure_mean",
    ),
    (
        "q_contrast",
        q_contrast_all,
        "tenure_diff",
    ),
    (
        "q1_level",
        q_level_all_model,
        "tenure_mean",
    ),
]:

    vals = pd.to_numeric(
        df[
            tenure_col
        ],
        errors="coerce",
    )


    audit_rows_32h.append(
        {
            "table":
                label,

            "rows":
                len(
                    df
                ),

            "first_year":
                int(
                    df[
                        "year"
                    ].min()
                ),

            "last_year":
                int(
                    df[
                        "year"
                    ].max()
                ),

            "rows_2026":
                int(
                    (
                        df[
                            "year"
                        ]
                        == 2026
                    ).sum()
                ),

            "missing_familiarity":
                int(
                    vals.isna().sum()
                ),

            "finite_familiarity":
                int(
                    np.isfinite(
                        vals
                    ).sum()
                ),

            "min_familiarity_covariate":
                float(
                    vals.min()
                ),

            "max_familiarity_covariate":
                float(
                    vals.max()
                ),
        }
    )


final_input_audit_32h = (
    pd.DataFrame(
        audit_rows_32h
    )
)


display(
    final_input_audit_32h
)


if (
    final_input_audit_32h[
        "last_year"
    ]
    != 2026
).any():

    raise RuntimeError(
        "At least one modern likelihood table does not reach 2026."
    )


if (
    final_input_audit_32h[
        "rows_2026"
    ]
    <= 0
).any():

    raise RuntimeError(
        "At least one modern likelihood channel contains no 2026 rows."
    )


if (
    final_input_audit_32h[
        "missing_familiarity"
    ]
    != 0
).any():

    raise RuntimeError(
        "Missing familiarity values remain in final LIVE inputs."
    )


# ============================================================
# 18. SAVE A SMALL REPRODUCIBILITY MANIFEST
# ============================================================

stage32h_dir = (
    Path(
        CFG.output_dir
    )
    /
    "validation_live_2026_model_inputs"
)


stage32h_dir.mkdir(
    parents=True,
    exist_ok=True,
)


familiarity_manifest_32h = pd.DataFrame(
    [
        {
            "specification":
                LIVE_FAMILIARITY_SPEC,

            "half_life_missed_gp":
                LIVE_FAMILIARITY_HALF_LIFE_GP,

            "ceiling_year":
                int(
                    CFG.ceiling_year
                ),

            "pace_contrast_rows":
                len(
                    pace_contrast_all
                ),

            "pace_level_rows":
                len(
                    pace_level_all
                ),

            "q_contrast_rows":
                len(
                    q_contrast_all
                ),

            "q1_level_rows":
                len(
                    q_level_all_model
                ),

            "historical_quali_rows":
                len(
                    hist_quali
                ),

            "historical_race_rows":
                len(
                    hist_race
                ),

            "driver_states":
                len(
                    idx_all.active_driver_years
                ),

            "constructor_states":
                len(
                    idx_all.active_constructor_years
                ),

            "driver_states_2026":
                len(
                    driver_states_2026_32h
                ),

            "constructor_states_2026":
                len(
                    constructor_states_2026_32h
                ),
        }
    ]
)


familiarity_manifest_32h.to_csv(
    stage32h_dir
    /
    "stage32h_live_model_input_manifest.csv",
    index=False,
)


final_input_audit_32h.to_csv(
    stage32h_dir
    /
    "stage32h_modern_likelihood_audit.csv",
    index=False,
)


first_2026_familiarity_32h.to_csv(
    stage32h_dir
    /
    "stage32h_first_2026_familiarity.csv",
    index=False,
)


print()
print("=" * 76)
print("STAGE 32H-A TRIAGE SUMMARY")
print("=" * 76)

print(
    "Locked familiarity specification:",
    LIVE_FAMILIARITY_SPEC,
)

print(
    "Modern horizon:",
    int(
        final_input_audit_32h[
            "first_year"
        ].min()
    ),
    "–",
    int(
        final_input_audit_32h[
            "last_year"
        ].max()
    ),
)

print(
    "Modern likelihood rows:",
    int(
        final_input_audit_32h[
            "rows"
        ].sum()
    ),
)

print(
    "2026 likelihood rows:",
    int(
        final_input_audit_32h[
            "rows_2026"
        ].sum()
    ),
)

print(
    "Historical qualifying rows:",
    len(
        hist_quali
    ),
)

print(
    "Historical race rows:",
    len(
        hist_race
    ),
)

print(
    "Driver-season states:",
    len(
        idx_all.active_driver_years
    ),
)

print(
    "2026 driver-season states:",
    len(
        driver_states_2026_32h
    ),
)

print(
    "Constructor-season states:",
    len(
        idx_all.active_constructor_years
    ),
)

print(
    "2026 constructor-season states:",
    len(
        constructor_states_2026_32h
    ),
)

print(
    "Missing familiarity values:",
    int(
        final_input_audit_32h[
            "missing_familiarity"
        ].sum()
    ),
)

print()
print(
    "LIVE model object built:",
    "model_1982_2026_live",
)

print()
print("HARD FLAGS: none")

print()
print(
    "Stage 32H-A complete."
)

print(
    "Do NOT sample yet; inspect this output first."
)

In [ ]:
# ============================================================
# STAGE 32H-B — FULL 1982–2026 LIVE PRODUCTION FIT
# ============================================================
#
# LOCKED LIVE MODEL SPECIFICATION
#
#   Historical horizon:
#       1982–1995
#
#   Modern horizon:
#       1996–2026
#
#   Familiarity:
#       organisational familiarity
#       20-GP missed-weekend half-life
#
#   Sampling:
#       4 chains
#       1000 tuning iterations / chain
#       1000 posterior draws / chain
#       target_accept = 0.92
#       seed = 42
#       nutpie + JAX
#
# IMPORTANT
# ---------
# This is a LIVE 2026 posterior.
#
# It does NOT replace or overwrite the frozen official
# 1982–2025 Pantheon posterior.
#
# The checkpoint name contains a deterministic snapshot
# identifier so that future 2026 updates cannot silently load
# an older live-season fit.
# ============================================================

from pathlib import Path

import hashlib
import json
import time

import numpy as np
import pandas as pd
import pymc as pm
import xarray as xr


# ============================================================
# 0. REQUIRED OBJECTS
# ============================================================

required_objects_32hb = [

    "model_1982_2026_live",

    "pace_contrast_all",
    "pace_level_all",
    "q_contrast_all",
    "q_level_all_model",

    "hist_quali",
    "hist_race",

    "idx_all",
    "CFG",
]


missing_objects_32hb = [

    name

    for name
    in required_objects_32hb

    if name not in globals()
]


if missing_objects_32hb:

    raise RuntimeError(
        "Stage 32H-B is missing required objects:\n  "
        +
        "\n  ".join(
            missing_objects_32hb
        )
        +
        "\n\nRun Stage 32H-A first."
    )


# ============================================================
# 1. LOCK SAMPLING CONFIGURATION
# ============================================================

LIVE_DRAWS_32HB = 1000
LIVE_TUNE_32HB = 1000
LIVE_CHAINS_32HB = 4

LIVE_TARGET_ACCEPT_32HB = 0.92
LIVE_SEED_32HB = 42

LIVE_CHECKPOINT_ENGINE_32HB = "h5netcdf"


# ============================================================
# 2. VERIFY THIS REALLY IS THE LIVE 2026 MODEL INPUT
# ============================================================

modern_tables_32hb = {

    "pace_contrast":
        pace_contrast_all,

    "pace_level":
        pace_level_all,

    "q_contrast":
        q_contrast_all,

    "q1_level":
        q_level_all_model,
}


expected_2026_rows_32hb = {

    "pace_contrast": 143,
    "pace_level": 143,
    "q_contrast": 324,
    "q1_level": 159,
}


preflight_rows_32hb = []


for table_name, df in modern_tables_32hb.items():

    if "year" not in df.columns:

        raise RuntimeError(
            f"{table_name} has no year column."
        )


    n_2026 = int(
        (
            pd.to_numeric(
                df["year"],
                errors="coerce",
            )
            == 2026
        ).sum()
    )


    preflight_rows_32hb.append(
        {
            "table":
                table_name,

            "rows":
                len(df),

            "first_year":
                int(
                    df[
                        "year"
                    ].min()
                ),

            "last_year":
                int(
                    df[
                        "year"
                    ].max()
                ),

            "rows_2026":
                n_2026,

            "expected_2026_rows":
                expected_2026_rows_32hb[
                    table_name
                ],

            "matches_stage_32h_a":
                (
                    n_2026
                    ==
                    expected_2026_rows_32hb[
                        table_name
                    ]
                ),
        }
    )


preflight_table_32hb = pd.DataFrame(
    preflight_rows_32hb
)


print(
    "=" * 76
)

print(
    "STAGE 32H-B — LIVE FIT PREFLIGHT"
)

print(
    "=" * 76
)


display(
    preflight_table_32hb
)


if not preflight_table_32hb[
    "matches_stage_32h_a"
].all():

    raise RuntimeError(
        "The modern likelihood tables no longer match the "
        "Stage 32H-A snapshot. Re-run Stage 32H-A before sampling."
    )


if int(
    preflight_table_32hb[
        "rows"
    ].sum()
) != 24792:

    raise RuntimeError(
        "Modern likelihood row total is not the locked "
        "Stage 32H-A value of 24,792."
    )


if len(
    hist_quali
) != 5726:

    raise RuntimeError(
        "Historical qualifying row count changed from 5,726."
    )


if len(
    hist_race
) != 2490:

    raise RuntimeError(
        "Historical race row count changed from 2,490."
    )


if len(
    idx_all.active_driver_years
) != 1242:

    raise RuntimeError(
        "Driver-season state count changed from 1,242."
    )


if len(
    idx_all.active_constructor_years
) != 555:

    raise RuntimeError(
        "Constructor-season state count changed from 555."
    )


# ============================================================
# 3. DETERMINE CURRENT COMPLETED 2026 ROUND
# ============================================================
#
# Use raw.results rather than the published schedule:
# future scheduled races should NOT affect the live snapshot tag.
# ============================================================

if "raw" not in globals():

    raise RuntimeError(
        "raw is required to determine the completed 2026 "
        "race snapshot."
    )


race_year_round_32hb = (
    raw.races[
        [
            "raceId",
            "year",
            "round",
        ]
    ]
    .copy()
)


completed_2026_results_32hb = (
    raw.results[
        [
            "raceId",
        ]
    ]
    .drop_duplicates()
    .merge(
        race_year_round_32hb,
        on="raceId",
        how="left",
        validate="one_to_one",
    )
)


completed_2026_results_32hb = (
    completed_2026_results_32hb[
        completed_2026_results_32hb[
            "year"
        ]
        == 2026
    ]
    .copy()
)


if len(
    completed_2026_results_32hb
) == 0:

    raise RuntimeError(
        "No completed 2026 races found in raw.results."
    )


LIVE_COMPLETED_RACES_32HB = int(
    completed_2026_results_32hb[
        "raceId"
    ].nunique()
)


LIVE_LAST_ROUND_32HB = int(
    completed_2026_results_32hb[
        "round"
    ].max()
)


print()
print(
    "Completed 2026 races:",
    LIVE_COMPLETED_RACES_32HB,
)

print(
    "Latest completed round:",
    LIVE_LAST_ROUND_32HB,
)


# ============================================================
# 4. BUILD DETERMINISTIC MODEL-INPUT SNAPSHOT HASH
# ============================================================
#
# This protects us against silently loading an old LIVE
# checkpoint if the API data are revised while the nominal
# completed round remains the same.
# ============================================================

def update_dataframe_hash_32hb(
    hasher,
    label,
    df,
    preferred_columns,
):

    hasher.update(
        label.encode(
            "utf-8"
        )
    )


    columns = [
        col
        for col
        in preferred_columns
        if col in df.columns
    ]


    if not columns:

        raise RuntimeError(
            f"No hashable model-input columns found for {label}."
        )


    x = (
        df[
            columns
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    # Convert object/string values consistently.
    for col in x.columns:

        if (
            x[col].dtype
            == object
        ):

            x[col] = (
                x[col]
                .astype(
                    "string"
                )
                .fillna(
                    "<NA>"
                )
            )


    row_hash = pd.util.hash_pandas_object(
        x,
        index=False,
    ).to_numpy(
        dtype=np.uint64
    )


    hasher.update(
        row_hash.tobytes()
    )


snapshot_hasher_32hb = hashlib.sha256()


modern_hash_columns_32hb = {

    "pace_contrast": [
        "year",
        "raceId",
        "dy_a",
        "dy_b",
        "y",
        "weight",
        "tenure_diff",
    ],

    "pace_level": [
        "year",
        "raceId",
        "dy_a",
        "dy_b",
        "csy",
        "y",
        "weight",
        "tenure_mean",
    ],

    "q_contrast": [
        "year",
        "raceId",
        "session",
        "dy_a",
        "dy_b",
        "y",
        "weight",
        "tenure_diff",
    ],

    "q1_level": [
        "year",
        "raceId",
        "session",
        "dy_a",
        "dy_b",
        "csy",
        "y",
        "weight",
        "tenure_mean",
    ],
}


for table_name, df in modern_tables_32hb.items():

    update_dataframe_hash_32hb(
        snapshot_hasher_32hb,
        table_name,
        df,
        modern_hash_columns_32hb[
            table_name
        ],
    )


# Include historical data too, even though it should be fixed.

update_dataframe_hash_32hb(
    snapshot_hasher_32hb,
    "hist_quali",
    hist_quali,
    [
        "year",
        "raceId",
        "driverId",
        "constructorId",
        "position",
        "tenure_c",
    ],
)


update_dataframe_hash_32hb(
    snapshot_hasher_32hb,
    "hist_race",
    hist_race,
    [
        "year",
        "raceId",
        "driverId",
        "constructorId",
        "position",
        "tenure_c",
    ],
)


snapshot_hasher_32hb.update(
    (
        f"driver_states={len(idx_all.active_driver_years)};"
        f"constructor_states={len(idx_all.active_constructor_years)};"
        f"familiarity=memory_20gp;"
        f"half_life=20"
    )
    .encode(
        "utf-8"
    )
)


LIVE_SNAPSHOT_HASH_32HB = (
    snapshot_hasher_32hb
    .hexdigest()[
        :12
    ]
)


LIVE_SNAPSHOT_TAG_32HB = (
    f"through_r{LIVE_LAST_ROUND_32HB:02d}_"
    f"{LIVE_COMPLETED_RACES_32HB}gp_"
    f"{LIVE_SNAPSHOT_HASH_32HB}"
)


print(
    "Snapshot tag:",
    LIVE_SNAPSHOT_TAG_32HB,
)


# ============================================================
# 5. LIVE OUTPUT PATHS
# ============================================================

LIVE_OUTPUT_DIR_32HB = Path(
    CFG.output_dir
)


LIVE_OUTPUT_DIR_32HB.mkdir(
    parents=True,
    exist_ok=True,
)


LIVE_TRACE_PATH_32HB = (
    LIVE_OUTPUT_DIR_32HB
    /
    (
        "joint_driver_car_v2_1_"
        "1982_2026_LIVE_"
        f"{LIVE_SNAPSHOT_TAG_32HB}_"
        "nutpie_jax_"
        "1000x1000_4chain.nc"
    )
)


LIVE_MANIFEST_PATH_32HB = (
    LIVE_OUTPUT_DIR_32HB
    /
    (
        "joint_driver_car_v2_1_"
        "1982_2026_LIVE_"
        f"{LIVE_SNAPSHOT_TAG_32HB}_"
        "manifest.json"
    )
)


print()
print(
    "LIVE checkpoint:"
)

print(
    LIVE_TRACE_PATH_32HB
)


print()
print(
    "LIVE manifest:"
)

print(
    LIVE_MANIFEST_PATH_32HB
)


# ============================================================
# 6. CHECKPOINT BACKEND
# ============================================================

def ensure_live_checkpoint_backend_32hb():

    try:

        import h5netcdf  # noqa: F401
        import h5py      # noqa: F401

    except ImportError as exc:

        raise RuntimeError(
            "The LIVE checkpoint requires h5netcdf + h5py.\n\n"
            "Install with:\n"
            "    %pip install h5netcdf h5py"
        ) from exc


ensure_live_checkpoint_backend_32hb()


# ============================================================
# 7. MANIFEST
# ============================================================

live_manifest_32hb = {

    "model":
        "joint_driver_car_v2_1_1982_2026_LIVE",

    "status":
        "live_partial_2026",

    "historical_start_year":
        1982,

    "modern_start_year":
        1996,

    "ceiling_year":
        2026,

    "completed_2026_races":
        LIVE_COMPLETED_RACES_32HB,

    "latest_completed_round":
        LIVE_LAST_ROUND_32HB,

    "snapshot_hash":
        LIVE_SNAPSHOT_HASH_32HB,

    "familiarity_definition":
        "organisational_memory",

    "familiarity_half_life_missed_gp":
        20,

    "sampling": {

        "draws":
            LIVE_DRAWS_32HB,

        "tune":
            LIVE_TUNE_32HB,

        "chains":
            LIVE_CHAINS_32HB,

        "target_accept":
            LIVE_TARGET_ACCEPT_32HB,

        "seed":
            LIVE_SEED_32HB,

        "sampler":
            "nutpie",

        "backend":
            "jax",
    },

    "likelihood_rows": {

        "pace_contrast":
            len(
                pace_contrast_all
            ),

        "pace_level":
            len(
                pace_level_all
            ),

        "q_contrast":
            len(
                q_contrast_all
            ),

        "q1_level":
            len(
                q_level_all_model
            ),

        "historical_qualifying":
            len(
                hist_quali
            ),

        "historical_race":
            len(
                hist_race
            ),
    },

    "state_counts": {

        "drivers":
            len(
                idx_all.unique_drivers
            ),

        "driver_seasons":
            len(
                idx_all.active_driver_years
            ),

        "constructor_seasons":
            len(
                idx_all.active_constructor_years
            ),
    },
}


# ============================================================
# 8. LOAD EXACT CHECKPOINT OR SAMPLE
# ============================================================

if LIVE_TRACE_PATH_32HB.exists():

    print()
    print(
        "=" * 76
    )

    print(
        "EXISTING EXACT LIVE SNAPSHOT FOUND"
    )

    print(
        "=" * 76
    )

    print(
        LIVE_TRACE_PATH_32HB
    )


    if not LIVE_MANIFEST_PATH_32HB.exists():

        raise RuntimeError(
            "The LIVE trace exists but its snapshot manifest "
            "is missing.\n\n"
            "Do not automatically trust or overwrite it.\n"
            f"Trace:\n{LIVE_TRACE_PATH_32HB}"
        )


    with open(
        LIVE_MANIFEST_PATH_32HB,
        "r",
        encoding="utf-8",
    ) as f:

        existing_manifest_32hb = (
            json.load(
                f
            )
        )


    if (
        existing_manifest_32hb.get(
            "snapshot_hash"
        )
        != LIVE_SNAPSHOT_HASH_32HB
    ):

        raise RuntimeError(
            "Existing LIVE checkpoint manifest does not match "
            "the current model-input snapshot."
        )


    try:

        live_trace_2026 = (
            xr.open_datatree(
                LIVE_TRACE_PATH_32HB,
                engine=
                    LIVE_CHECKPOINT_ENGINE_32HB,
            )
        )

    except Exception as exc:

        raise RuntimeError(
            "An exact-snapshot LIVE checkpoint exists but cannot "
            "be read as a valid inference checkpoint.\n\n"
            f"{LIVE_TRACE_PATH_32HB}\n\n"
            "Do not overwrite it automatically. Inspect, rename, "
            "or delete it deliberately."
        ) from exc


    LIVE_SAMPLING_MINUTES_32HB = np.nan


    print()
    print(
        "Loaded existing exact-snapshot LIVE trace."
    )

    print(
        "NO RESAMPLING performed."
    )


else:

    print()
    print(
        "=" * 76
    )

    print(
        "FULL 1982–2026 LIVE PRODUCTION FIT"
    )

    print(
        "=" * 76
    )


    print(
        f"{LIVE_CHAINS_32HB} chains × "
        f"({LIVE_TUNE_32HB} tune + "
        f"{LIVE_DRAWS_32HB} posterior)"
    )


    print(
        "Sampler: nutpie / JAX"
    )


    print(
        "target_accept:",
        LIVE_TARGET_ACCEPT_32HB,
    )


    print(
        "seed:",
        LIVE_SEED_32HB,
    )


    print(
        "Familiarity:",
        "memory_20gp",
    )


    print(
        "Latest completed 2026 round:",
        LIVE_LAST_ROUND_32HB,
    )


    print(
        "Snapshot hash:",
        LIVE_SNAPSHOT_HASH_32HB,
    )


    print()
    print(
        "Starting sampling..."
    )


    t0_32hb = (
        time.perf_counter()
    )


    with model_1982_2026_live:

        live_trace_2026 = pm.sample(

            draws=
                LIVE_DRAWS_32HB,

            tune=
                LIVE_TUNE_32HB,

            chains=
                LIVE_CHAINS_32HB,

            random_seed=
                LIVE_SEED_32HB,

            target_accept=
                LIVE_TARGET_ACCEPT_32HB,

            nuts_sampler=
                "nutpie",

            nuts_sampler_kwargs={
                "backend":
                    "jax",
            },

            return_inferencedata=
                True,

            progressbar=
                True,
        )


    LIVE_SAMPLING_MINUTES_32HB = (
        time.perf_counter()
        -
        t0_32hb
    ) / 60.0


    print()
    print(
        "Sampling wall time:",
        f"{LIVE_SAMPLING_MINUTES_32HB:.2f} minutes",
    )


    # --------------------------------------------------------
    # Save the posterior FIRST.
    # --------------------------------------------------------

    live_trace_2026.to_netcdf(

        LIVE_TRACE_PATH_32HB,

        engine=
            LIVE_CHECKPOINT_ENGINE_32HB,

        mode=
            "w",
    )


    print()
    print(
        "Saved LIVE posterior ->"
    )

    print(
        LIVE_TRACE_PATH_32HB
    )


    # --------------------------------------------------------
    # Only write the manifest after the posterior has
    # successfully been written.
    # --------------------------------------------------------

    live_manifest_32hb[
        "sampling_wall_minutes"
    ] = float(
        LIVE_SAMPLING_MINUTES_32HB
    )


    with open(
        LIVE_MANIFEST_PATH_32HB,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            live_manifest_32hb,
            f,
            indent=2,
        )


    print()
    print(
        "Saved LIVE manifest ->"
    )

    print(
        LIVE_MANIFEST_PATH_32HB
    )


# ============================================================
# 9. CHECK LOADED POSTERIOR STRUCTURE
# ============================================================

if "posterior" not in live_trace_2026:

    raise RuntimeError(
        "LIVE trace contains no posterior group."
    )


posterior_live_32hb = (
    live_trace_2026[
        "posterior"
    ]
    .to_dataset()
)


sample_stats_live_32hb = (
    live_trace_2026[
        "sample_stats"
    ]
    .to_dataset()
)


expected_posterior_vars_32hb = {

    "sigma_driver_base",
    "theta_z",

    "sigma_driver_drift",
    "drift_z",

    "alpha_dy",

    "sigma_car",
    "car_z",
    "car_csy",

    "gamma_tenure",

    "sigma_pace_contrast",
    "sigma_pace_level",

    "sigma_q_contrast",
    "sigma_q_level",

    "tau_hist_quali",
    "tau_hist_race",
}


missing_posterior_vars_32hb = (

    expected_posterior_vars_32hb
    -
    set(
        posterior_live_32hb.data_vars
    )
)


if missing_posterior_vars_32hb:

    raise RuntimeError(
        "LIVE posterior is missing expected variables:\n  "
        +
        "\n  ".join(
            sorted(
                missing_posterior_vars_32hb
            )
        )
    )


n_chain_32hb = int(
    posterior_live_32hb.sizes[
        "chain"
    ]
)


n_draw_32hb = int(
    posterior_live_32hb.sizes[
        "draw"
    ]
)


if n_chain_32hb != LIVE_CHAINS_32HB:

    raise RuntimeError(
        f"Expected {LIVE_CHAINS_32HB} chains, "
        f"found {n_chain_32hb}."
    )


if n_draw_32hb != LIVE_DRAWS_32HB:

    raise RuntimeError(
        f"Expected {LIVE_DRAWS_32HB} posterior draws per chain, "
        f"found {n_draw_32hb}."
    )


# ============================================================
# 10. VERY BASIC SAMPLER SANITY CHECK
# ============================================================
#
# Full R-hat / ESS analysis belongs in Stage 32H-C.
# Here we only make sure the saved trace has the correct
# dimensions and report divergences.
# ============================================================

if (
    "diverging"
    in sample_stats_live_32hb
):

    live_divergences_32hb = int(
        np.asarray(
            sample_stats_live_32hb[
                "diverging"
            ].values
        ).sum()
    )

else:

    live_divergences_32hb = 0


print()
print(
    "=" * 76
)

print(
    "STAGE 32H-B FIT SUMMARY"
)

print(
    "=" * 76
)


print(
    "Snapshot:",
    LIVE_SNAPSHOT_TAG_32HB,
)


print(
    "Posterior chains:",
    n_chain_32hb,
)


print(
    "Posterior draws / chain:",
    n_draw_32hb,
)


print(
    "Total posterior draws:",
    (
        n_chain_32hb
        *
        n_draw_32hb
    ),
)


print(
    "Divergences:",
    live_divergences_32hb,
)


if np.isfinite(
    LIVE_SAMPLING_MINUTES_32HB
):

    print(
        "Sampling wall time:",
        f"{LIVE_SAMPLING_MINUTES_32HB:.2f} min",
    )

else:

    print(
        "Sampling wall time:",
        "checkpoint loaded",
    )


print()
print(
    "Checkpoint:"
)

print(
    LIVE_TRACE_PATH_32HB
)


print()
print(
    "Manifest:"
)

print(
    LIVE_MANIFEST_PATH_32HB
)


print()
print(
    "Stage 32H-B complete."
)

print(
    "Do NOT calculate LIVE JAWS yet."
)

print(
    "Next: Stage 32H-C — full convergence / posterior-health "
    "gate and inspection of the 2026 latent states."
)

In [ ]:
# ============================================================
# STAGE 32H-C — FULL POSTERIOR HEALTH GATE
#                 + 2026 LATENT-STATE INSPECTION
# ============================================================
#
# PURPOSE
# -------
# Diagnose the completed 1982–2026 LIVE posterior before doing
# ANY downstream LIVE scoring.
#
# This stage:
#
#   1. loads the completed snapshot checkpoint;
#   2. verifies dimensions and finite values;
#   3. checks divergences / BFMI / NUTS behaviour;
#   4. computes rank-normalised R-hat;
#   5. computes bulk and tail ESS;
#   6. compares global parameters with frozen 1982–2025;
#   7. extracts the 23 driver-season alpha states for 2026;
#   8. extracts the 11 constructor-season beta states for 2026;
#   9. computes posterior ranks within the 2026 field;
#  10. gives a convergence PASS / WATCH / FAIL summary.
#
# IMPORTANT
# ---------
# NO JAWS is calculated here.
#
# This stage is diagnostic only.
# ============================================================

from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd
import xarray as xr

from IPython.display import display


# ============================================================
# 0. LOAD THE COMPLETED LIVE CHECKPOINT
# ============================================================

trace_path_32hc = Path(
    LIVE_TRACE_PATH_32HB
)


if not trace_path_32hc.exists():

    raise RuntimeError(
        "Completed LIVE checkpoint does not exist:\n"
        f"{trace_path_32hc}"
    )


print(
    "=" * 80
)

print(
    "STAGE 32H-C — FULL LIVE POSTERIOR HEALTH GATE"
)

print(
    "=" * 80
)

print(
    "Checkpoint:"
)

print(
    trace_path_32hc
)


live_idata_32hc = az.from_netcdf(
    trace_path_32hc
)


if not hasattr(
    live_idata_32hc,
    "posterior",
):

    raise RuntimeError(
        "Loaded object contains no posterior group."
    )


posterior_32hc = (
    live_idata_32hc.posterior
)


sample_stats_32hc = (
    live_idata_32hc.sample_stats
)


# ============================================================
# 1. BASIC DIMENSION CHECK
# ============================================================

n_chains_32hc = int(
    posterior_32hc.sizes[
        "chain"
    ]
)


n_draws_32hc = int(
    posterior_32hc.sizes[
        "draw"
    ]
)


print()
print(
    "=" * 80
)

print(
    "POSTERIOR DIMENSIONS"
)

print(
    "=" * 80
)


print(
    "Chains:",
    n_chains_32hc,
)


print(
    "Draws / chain:",
    n_draws_32hc,
)


print(
    "Total posterior draws:",
    n_chains_32hc
    *
    n_draws_32hc,
)


if n_chains_32hc != 4:

    raise RuntimeError(
        f"Expected 4 chains, found {n_chains_32hc}."
    )


if n_draws_32hc != 1000:

    raise RuntimeError(
        f"Expected 1000 draws / chain, found {n_draws_32hc}."
    )


# ============================================================
# 2. FINITE-VALUE AUDIT
# ============================================================

finite_rows_32hc = []


for variable in posterior_32hc.data_vars:

    values = np.asarray(
        posterior_32hc[
            variable
        ].values
    )


    try:

        finite = np.isfinite(
            values
        )

        n_nonfinite = int(
            (
                ~finite
            ).sum()
        )

    except TypeError:

        n_nonfinite = np.nan


    finite_rows_32hc.append(
        {
            "variable":
                variable,

            "shape":
                str(
                    values.shape
                ),

            "n_values":
                values.size,

            "n_nonfinite":
                n_nonfinite,
        }
    )


finite_audit_32hc = pd.DataFrame(
    finite_rows_32hc
)


print()
print(
    "=" * 80
)

print(
    "FINITE-VALUE AUDIT"
)

print(
    "=" * 80
)


display(
    finite_audit_32hc
)


numeric_nonfinite_32hc = (
    finite_audit_32hc[
        "n_nonfinite"
    ]
    .dropna()
    .sum()
)


print(
    "Total non-finite numeric posterior values:",
    int(
        numeric_nonfinite_32hc
    ),
)


# ============================================================
# 3. DIVERGENCES
# ============================================================

print()
print(
    "=" * 80
)

print(
    "DIVERGENCES"
)

print(
    "=" * 80
)


if "diverging" in sample_stats_32hc:

    diverging_32hc = np.asarray(
        sample_stats_32hc[
            "diverging"
        ].values,
        dtype=bool,
    )


    divergences_by_chain_32hc = (
        diverging_32hc
        .sum(
            axis=1
        )
        .astype(int)
    )


    divergence_table_32hc = pd.DataFrame(
        {
            "chain":
                np.arange(
                    n_chains_32hc
                ),

            "divergences":
                divergences_by_chain_32hc,
        }
    )


    display(
        divergence_table_32hc
    )


    n_divergences_32hc = int(
        divergences_by_chain_32hc.sum()
    )


else:

    n_divergences_32hc = np.nan

    print(
        "'diverging' not present in sample_stats."
    )


print(
    "Total divergences:",
    n_divergences_32hc,
)


# ============================================================
# 4. BFMI
# ============================================================
#
# Calculate directly from energy to avoid the xarray/DataTree
# incompatibility encountered during the interrupted-run audit.
#
# BFMI = mean[(E_t - E_{t-1})^2] / Var(E)
# ============================================================

print()
print(
    "=" * 80
)

print(
    "BFMI"
)

print(
    "=" * 80
)


if "energy" in sample_stats_32hc:

    energy_32hc = np.asarray(
        sample_stats_32hc[
            "energy"
        ].values,
        dtype=float,
    )


    bfmi_32hc = (
        np.mean(
            np.diff(
                energy_32hc,
                axis=1,
            )
            ** 2,
            axis=1,
        )
        /
        np.var(
            energy_32hc,
            axis=1,
            ddof=1,
        )
    )


    bfmi_table_32hc = pd.DataFrame(
        {
            "chain":
                np.arange(
                    n_chains_32hc
                ),

            "BFMI":
                bfmi_32hc,
        }
    )


    display(
        bfmi_table_32hc
    )


    min_bfmi_32hc = float(
        np.min(
            bfmi_32hc
        )
    )


    print(
        "Minimum BFMI:",
        f"{min_bfmi_32hc:.4f}",
    )


else:

    bfmi_32hc = np.full(
        n_chains_32hc,
        np.nan,
    )


    min_bfmi_32hc = np.nan


    print(
        "'energy' not present in sample_stats."
    )


# ============================================================
# 5. NUTS TRAJECTORY INFORMATION
# ============================================================

print()
print(
    "=" * 80
)

print(
    "NUTS TRAJECTORY INFORMATION"
)

print(
    "=" * 80
)


print(
    "sample_stats variables:"
)

print(
    sorted(
        list(
            sample_stats_32hc.data_vars
        )
    )
)


if "n_steps" in sample_stats_32hc:

    n_steps_32hc = np.asarray(
        sample_stats_32hc[
            "n_steps"
        ].values,
        dtype=float,
    )


    max_n_steps_32hc = int(
        np.max(
            n_steps_32hc
        )
    )


    print()
    print(
        "Minimum n_steps:",
        int(
            np.min(
                n_steps_32hc
            )
        ),
    )


    print(
        "Median n_steps:",
        float(
            np.median(
                n_steps_32hc
            )
        ),
    )


    print(
        "90th percentile n_steps:",
        float(
            np.quantile(
                n_steps_32hc,
                0.90,
            )
        ),
    )


    print(
        "99th percentile n_steps:",
        float(
            np.quantile(
                n_steps_32hc,
                0.99,
            )
        ),
    )


    print(
        "Maximum n_steps:",
        max_n_steps_32hc,
    )


    print(
        "Fraction at observed maximum:",
        float(
            np.mean(
                n_steps_32hc
                ==
                max_n_steps_32hc
            )
        ),
    )


maxdepth_fields_32hc = [

    "reached_max_treedepth",
    "reached_max_tree_depth",
    "maxdepth_reached",
]


maxdepth_hits_32hc = None


for field in maxdepth_fields_32hc:

    if field in sample_stats_32hc:

        vals = np.asarray(
            sample_stats_32hc[
                field
            ].values,
            dtype=bool,
        )


        maxdepth_hits_32hc = int(
            vals.sum()
        )


        print()
        print(
            f"{field}:",
            maxdepth_hits_32hc,
        )


# ============================================================
# 6. COMPUTE R-HAT / ESS
# ============================================================

print()
print(
    "=" * 80
)

print(
    "COMPUTING R-HAT / ESS"
)

print(
    "=" * 80
)


rhat_raw_32hc = az.rhat(
    posterior_32hc,
    method="rank",
)


ess_bulk_raw_32hc = az.ess(
    posterior_32hc,
    method="bulk",
)


ess_tail_raw_32hc = az.ess(
    posterior_32hc,
    method="tail",
)


# ------------------------------------------------------------
# Normalise Dataset/DataTree return types
# ------------------------------------------------------------

def normalise_diag_dataset_32hc(
    obj,
):

    if isinstance(
        obj,
        xr.Dataset,
    ):

        return obj


    if isinstance(
        obj,
        xr.DataTree,
    ):

        # Most versions return the diagnostic variables at the
        # root. If not, use the first child containing data vars.

        root_ds = obj.to_dataset()


        if len(
            root_ds.data_vars
        ):

            return root_ds


        for child_name in obj.children:

            child_ds = (
                obj[
                    child_name
                ]
                .to_dataset()
            )


            if len(
                child_ds.data_vars
            ):

                return child_ds


    raise TypeError(
        f"Unsupported diagnostic object type: {type(obj)}"
    )


rhat_ds_32hc = normalise_diag_dataset_32hc(
    rhat_raw_32hc
)


ess_bulk_ds_32hc = normalise_diag_dataset_32hc(
    ess_bulk_raw_32hc
)


ess_tail_ds_32hc = normalise_diag_dataset_32hc(
    ess_tail_raw_32hc
)


# ============================================================
# 7. DIAGNOSTIC HELPERS
# ============================================================

def finite_diag_values_32hc(
    dataset,
    variable,
):

    if variable not in dataset:

        return np.array(
            [],
            dtype=float,
        )


    values = np.asarray(
        dataset[
            variable
        ].values,
        dtype=float,
    ).ravel()


    return values[
        np.isfinite(
            values
        )
    ]


def diagnostic_row_32hc(
    variable,
):

    rh = finite_diag_values_32hc(
        rhat_ds_32hc,
        variable,
    )


    eb = finite_diag_values_32hc(
        ess_bulk_ds_32hc,
        variable,
    )


    et = finite_diag_values_32hc(
        ess_tail_ds_32hc,
        variable,
    )


    return {
        "variable":
            variable,

        "n_elements":
            len(
                rh
            ),

        "max_rhat":
            (
                float(
                    np.max(
                        rh
                    )
                )
                if len(
                    rh
                )
                else np.nan
            ),

        "p95_rhat":
            (
                float(
                    np.quantile(
                        rh,
                        0.95,
                    )
                )
                if len(
                    rh
                )
                else np.nan
            ),

        "n_rhat_gt_1.01":
            int(
                (
                    rh
                    >
                    1.01
                ).sum()
            ),

        "n_rhat_gt_1.03":
            int(
                (
                    rh
                    >
                    1.03
                ).sum()
            ),

        "min_ess_bulk":
            (
                float(
                    np.min(
                        eb
                    )
                )
                if len(
                    eb
                )
                else np.nan
            ),

        "p05_ess_bulk":
            (
                float(
                    np.quantile(
                        eb,
                        0.05,
                    )
                )
                if len(
                    eb
                )
                else np.nan
            ),

        "min_ess_tail":
            (
                float(
                    np.min(
                        et
                    )
                )
                if len(
                    et
                )
                else np.nan
            ),

        "p05_ess_tail":
            (
                float(
                    np.quantile(
                        et,
                        0.05,
                    )
                )
                if len(
                    et
                )
                else np.nan
            ),
    }


# ============================================================
# 8. GLOBAL PARAMETER HEALTH
# ============================================================

global_vars_32hc = [

    "sigma_driver_base",
    "sigma_driver_drift",

    "sigma_car",

    "gamma_tenure",

    "sigma_pace_contrast",
    "sigma_pace_level",

    "sigma_q_contrast",
    "sigma_q_level",

    "tau_hist_quali",
    "tau_hist_race",
]


missing_globals_32hc = [
    var
    for var
    in global_vars_32hc
    if var not in posterior_32hc
]


if missing_globals_32hc:

    raise RuntimeError(
        "Missing expected global posterior variables:\n"
        +
        "\n".join(
            missing_globals_32hc
        )
    )


global_diag_32hc = pd.DataFrame(
    [
        diagnostic_row_32hc(
            variable
        )
        for variable
        in global_vars_32hc
    ]
)


print()
print(
    "=" * 80
)

print(
    "GLOBAL PARAMETER DIAGNOSTICS"
)

print(
    "=" * 80
)


display(
    global_diag_32hc
    .sort_values(
        "max_rhat",
        ascending=False,
    )
)


# ============================================================
# 9. LATENT-STATE HEALTH
# ============================================================

latent_vars_32hc = [

    "alpha_dy",
    "car_csy",

    "theta_z",
    "drift_z",
    "car_z",
]


latent_vars_32hc = [
    var
    for var
    in latent_vars_32hc
    if var
    in posterior_32hc
]


latent_diag_32hc = pd.DataFrame(
    [
        diagnostic_row_32hc(
            variable
        )
        for variable
        in latent_vars_32hc
    ]
)


print()
print(
    "=" * 80
)

print(
    "LATENT-STATE DIAGNOSTICS"
)

print(
    "=" * 80
)


display(
    latent_diag_32hc
)


# ============================================================
# 10. OVERALL POSTERIOR HEALTH
# ============================================================

all_rhat_chunks_32hc = []


for variable in rhat_ds_32hc.data_vars:

    vals = finite_diag_values_32hc(
        rhat_ds_32hc,
        variable,
    )


    if len(
        vals
    ):

        all_rhat_chunks_32hc.append(
            vals
        )


all_bulk_chunks_32hc = []


for variable in ess_bulk_ds_32hc.data_vars:

    vals = finite_diag_values_32hc(
        ess_bulk_ds_32hc,
        variable,
    )


    if len(
        vals
    ):

        all_bulk_chunks_32hc.append(
            vals
        )


all_tail_chunks_32hc = []


for variable in ess_tail_ds_32hc.data_vars:

    vals = finite_diag_values_32hc(
        ess_tail_ds_32hc,
        variable,
    )


    if len(
        vals
    ):

        all_tail_chunks_32hc.append(
            vals
        )


all_rhat_32hc = np.concatenate(
    all_rhat_chunks_32hc
)


all_bulk_ess_32hc = np.concatenate(
    all_bulk_chunks_32hc
)


all_tail_ess_32hc = np.concatenate(
    all_tail_chunks_32hc
)


max_rhat_32hc = float(
    np.max(
        all_rhat_32hc
    )
)


min_bulk_ess_32hc = float(
    np.min(
        all_bulk_ess_32hc
    )
)


min_tail_ess_32hc = float(
    np.min(
        all_tail_ess_32hc
    )
)


print()
print(
    "=" * 80
)

print(
    "OVERALL POSTERIOR HEALTH"
)

print(
    "=" * 80
)


print(
    "R-hat elements:",
    f"{len(all_rhat_32hc):,}",
)


print(
    "Max R-hat:",
    f"{max_rhat_32hc:.6f}",
)


print(
    "95th percentile R-hat:",
    f"{np.quantile(all_rhat_32hc, 0.95):.6f}",
)


print(
    "R-hat > 1.01:",
    int(
        (
            all_rhat_32hc
            >
            1.01
        ).sum()
    ),
)


print(
    "R-hat > 1.03:",
    int(
        (
            all_rhat_32hc
            >
            1.03
        ).sum()
    ),
)


print(
    "Minimum bulk ESS:",
    f"{min_bulk_ess_32hc:.1f}",
)


print(
    "5th percentile bulk ESS:",
    f"{np.quantile(all_bulk_ess_32hc, 0.05):.1f}",
)


print(
    "Median bulk ESS:",
    f"{np.median(all_bulk_ess_32hc):.1f}",
)


print(
    "Minimum tail ESS:",
    f"{min_tail_ess_32hc:.1f}",
)


print(
    "5th percentile tail ESS:",
    f"{np.quantile(all_tail_ess_32hc, 0.05):.1f}",
)


# ============================================================
# 11. GLOBAL POSTERIOR SUMMARY
# ============================================================

global_summary_rows_32hc = []


for variable in global_vars_32hc:

    values = np.asarray(
        posterior_32hc[
            variable
        ].values,
        dtype=float,
    ).reshape(
        -1
    )


    global_summary_rows_32hc.append(
        {
            "variable":
                variable,

            "mean":
                float(
                    np.mean(
                        values
                    )
                ),

            "sd":
                float(
                    np.std(
                        values,
                        ddof=1,
                    )
                ),

            "q055":
                float(
                    np.quantile(
                        values,
                        0.055,
                    )
                ),

            "q945":
                float(
                    np.quantile(
                        values,
                        0.945,
                    )
                ),
        }
    )


global_summary_32hc = pd.DataFrame(
    global_summary_rows_32hc
)


# ------------------------------------------------------------
# Frozen 1982–2025 reference values
# ------------------------------------------------------------

frozen_globals_32hc = {

    "sigma_driver_base":
        0.3830,

    "sigma_driver_drift":
        0.1740,

    "sigma_car":
        0.9210,

    "gamma_tenure":
        0.2832,

    "sigma_pace_contrast":
        0.6054,

    "sigma_pace_level":
        0.5112,

    "sigma_q_contrast":
        0.4452,

    "sigma_q_level":
        0.5121,

    "tau_hist_quali":
        0.5090,

    "tau_hist_race":
        0.7330,
}


global_summary_32hc[
    "frozen_2025_mean"
] = global_summary_32hc[
    "variable"
].map(
    frozen_globals_32hc
)


global_summary_32hc[
    "live_minus_frozen"
] = (
    global_summary_32hc[
        "mean"
    ]
    -
    global_summary_32hc[
        "frozen_2025_mean"
    ]
)


global_summary_32hc[
    "shift_in_live_sd"
] = (
    global_summary_32hc[
        "live_minus_frozen"
    ]
    /
    global_summary_32hc[
        "sd"
    ]
)


print()
print(
    "=" * 80
)

print(
    "GLOBAL POSTERIOR — LIVE VS FROZEN 1982–2025"
)

print(
    "=" * 80
)


display(
    global_summary_32hc
)


# ============================================================
# 12. HELPER — EXTRACT VECTOR POSTERIOR
# ============================================================

def posterior_matrix_32hc(
    variable,
):

    da = posterior_32hc[
        variable
    ]


    values = np.asarray(
        da.values,
        dtype=float,
    )


    if values.ndim != 3:

        raise RuntimeError(
            f"{variable} expected shape "
            "(chain, draw, state), "
            f"found {values.shape}."
        )


    return values.reshape(
        (
            -1,
            values.shape[
                -1
            ],
        )
    )


# ============================================================
# 13. 2026 DRIVER-SEASON STATES
# ============================================================

alpha_draws_32hc = posterior_matrix_32hc(
    "alpha_dy"
)


driver_states_32hc = list(
    idx_all.active_driver_years
)


if alpha_draws_32hc.shape[
    1
] != len(
    driver_states_32hc
):

    raise RuntimeError(
        "alpha_dy state count does not match "
        "idx_all.active_driver_years."
    )


driver_2026_indices_32hc = []


for i, state in enumerate(
    driver_states_32hc
):

    driver_id, year = state


    if int(
        year
    ) == 2026:

        driver_2026_indices_32hc.append(
            (
                i,
                driver_id,
            )
        )


if len(
    driver_2026_indices_32hc
) != 23:

    raise RuntimeError(
        "Expected 23 2026 driver-season states, "
        f"found {len(driver_2026_indices_32hc)}."
    )


driver_2026_idx_32hc = np.array(
    [
        x[
            0
        ]
        for x
        in driver_2026_indices_32hc
    ],
    dtype=int,
)


driver_2026_ids_32hc = [
    x[
        1
    ]
    for x
    in driver_2026_indices_32hc
]


alpha_2026_draws_32hc = (
    alpha_draws_32hc[
        :,
        driver_2026_idx_32hc,
    ]
)


# ------------------------------------------------------------
# Per-draw ranks among 2026 driver states
#
# 1 = highest alpha
# ------------------------------------------------------------

alpha_order_32hc = np.argsort(
    -alpha_2026_draws_32hc,
    axis=1,
)


alpha_rank_draws_32hc = np.empty_like(
    alpha_order_32hc,
    dtype=int,
)


row_indices_32hc = np.arange(
    alpha_rank_draws_32hc.shape[
        0
    ]
)[
    :,
    None,
]


alpha_rank_draws_32hc[
    row_indices_32hc,
    alpha_order_32hc,
] = (
    np.arange(
        1,
        alpha_rank_draws_32hc.shape[
            1
        ]
        +
        1,
    )[
        None,
        :
    ]
)


driver_rows_32hc = []


for j, driver_id in enumerate(
    driver_2026_ids_32hc
):

    values = alpha_2026_draws_32hc[
        :,
        j,
    ]


    ranks = alpha_rank_draws_32hc[
        :,
        j,
    ]


    driver_rows_32hc.append(
        {
            "driver":
                driver_id,

            "alpha_mean":
                float(
                    np.mean(
                        values
                    )
                ),

            "alpha_sd":
                float(
                    np.std(
                        values,
                        ddof=1,
                    )
                ),

            "alpha_89_lb":
                float(
                    np.quantile(
                        values,
                        0.055,
                    )
                ),

            "alpha_89_ub":
                float(
                    np.quantile(
                        values,
                        0.945,
                    )
                ),

            "p_alpha_gt_0":
                float(
                    np.mean(
                        values
                        >
                        0
                    )
                ),

            "mean_rank_2026":
                float(
                    np.mean(
                        ranks
                    )
                ),

            "median_rank_2026":
                float(
                    np.median(
                        ranks
                    )
                ),

            "rank_89_lb":
                int(
                    np.quantile(
                        ranks,
                        0.055,
                        method="nearest",
                    )
                ),

            "rank_89_ub":
                int(
                    np.quantile(
                        ranks,
                        0.945,
                        method="nearest",
                    )
                ),
        }
    )


driver_2026_summary_32hc = (
    pd.DataFrame(
        driver_rows_32hc
    )
    .sort_values(
        "alpha_mean",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


driver_2026_summary_32hc.insert(
    0,
    "mean_order",
    np.arange(
        1,
        len(
            driver_2026_summary_32hc
        )
        +
        1,
    ),
)


print()
print(
    "=" * 80
)

print(
    "2026 DRIVER-SEASON LATENT ABILITY"
)

print(
    "=" * 80
)


display(
    driver_2026_summary_32hc
)


# ============================================================
# 14. 2026 CONSTRUCTOR-SEASON STATES
# ============================================================

car_draws_32hc = posterior_matrix_32hc(
    "car_csy"
)


constructor_states_32hc = list(
    idx_all.active_constructor_years
)


if car_draws_32hc.shape[
    1
] != len(
    constructor_states_32hc
):

    raise RuntimeError(
        "car_csy state count does not match "
        "idx_all.active_constructor_years."
    )


constructor_2026_indices_32hc = []


for i, state in enumerate(
    constructor_states_32hc
):

    constructor_id, year = state


    if int(
        year
    ) == 2026:

        constructor_2026_indices_32hc.append(
            (
                i,
                constructor_id,
            )
        )


if len(
    constructor_2026_indices_32hc
) != 11:

    raise RuntimeError(
        "Expected 11 2026 constructor-season states, "
        f"found {len(constructor_2026_indices_32hc)}."
    )


constructor_2026_idx_32hc = np.array(
    [
        x[
            0
        ]
        for x
        in constructor_2026_indices_32hc
    ],
    dtype=int,
)


constructor_2026_ids_32hc = [
    x[
        1
    ]
    for x
    in constructor_2026_indices_32hc
]


car_2026_draws_32hc = (
    car_draws_32hc[
        :,
        constructor_2026_idx_32hc,
    ]
)


car_order_32hc = np.argsort(
    -car_2026_draws_32hc,
    axis=1,
)


car_rank_draws_32hc = np.empty_like(
    car_order_32hc,
    dtype=int,
)


car_rank_draws_32hc[
    np.arange(
        car_rank_draws_32hc.shape[
            0
        ]
    )[
        :,
        None,
    ],
    car_order_32hc,
] = (
    np.arange(
        1,
        car_rank_draws_32hc.shape[
            1
        ]
        +
        1,
    )[
        None,
        :
    ]
)


# ------------------------------------------------------------
# Constructor labels
# ------------------------------------------------------------

constructor_name_lookup_32hc = {}


if hasattr(
    raw,
    "constructors",
):

    constructors_meta_32hc = (
        raw.constructors.copy()
    )


    if (
        "constructorId"
        in constructors_meta_32hc.columns
        and
        "name"
        in constructors_meta_32hc.columns
    ):

        constructor_name_lookup_32hc = dict(
            zip(
                constructors_meta_32hc[
                    "constructorId"
                ],
                constructors_meta_32hc[
                    "name"
                ],
            )
        )


constructor_rows_32hc = []


for j, constructor_id in enumerate(
    constructor_2026_ids_32hc
):

    values = car_2026_draws_32hc[
        :,
        j,
    ]


    ranks = car_rank_draws_32hc[
        :,
        j,
    ]


    constructor_rows_32hc.append(
        {
            "constructor_id":
                constructor_id,

            "constructor":
                constructor_name_lookup_32hc.get(
                    constructor_id,
                    str(
                        constructor_id
                    ),
                ),

            "beta_mean":
                float(
                    np.mean(
                        values
                    )
                ),

            "beta_sd":
                float(
                    np.std(
                        values,
                        ddof=1,
                    )
                ),

            "beta_89_lb":
                float(
                    np.quantile(
                        values,
                        0.055,
                    )
                ),

            "beta_89_ub":
                float(
                    np.quantile(
                        values,
                        0.945,
                    )
                ),

            "p_beta_gt_0":
                float(
                    np.mean(
                        values
                        >
                        0
                    )
                ),

            "mean_rank_2026":
                float(
                    np.mean(
                        ranks
                    )
                ),

            "median_rank_2026":
                float(
                    np.median(
                        ranks
                    )
                ),

            "rank_89_lb":
                int(
                    np.quantile(
                        ranks,
                        0.055,
                        method="nearest",
                    )
                ),

            "rank_89_ub":
                int(
                    np.quantile(
                        ranks,
                        0.945,
                        method="nearest",
                    )
                ),
        }
    )


constructor_2026_summary_32hc = (
    pd.DataFrame(
        constructor_rows_32hc
    )
    .sort_values(
        "beta_mean",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


constructor_2026_summary_32hc.insert(
    0,
    "mean_order",
    np.arange(
        1,
        len(
            constructor_2026_summary_32hc
        )
        +
        1,
    ),
)


print()
print(
    "=" * 80
)

print(
    "2026 CONSTRUCTOR-SEASON LATENT EFFECTS"
)

print(
    "=" * 80
)


display(
    constructor_2026_summary_32hc
)


# ============================================================
# 15. GLOBAL HEALTH GATE
# ============================================================
#
# HARD:
#   - any non-finite posterior value
#   - any divergence
#   - max R-hat > 1.01
#
# WATCH:
#   - minimum bulk ESS < 400
#   - minimum tail ESS < 400
#   - BFMI < 0.30
#   - explicit maximum-tree-depth hits
#
# The frozen 1982–2025 reference fit had:
#
#   max R-hat       = 1.007758
#   R-hat > 1.01    = 0
#   min bulk ESS    = 972.3
#   divergences     = 0
#
# ============================================================

hard_flags_32hc = []
watch_flags_32hc = []


if numeric_nonfinite_32hc > 0:

    hard_flags_32hc.append(
        f"{int(numeric_nonfinite_32hc)} "
        "non-finite posterior values"
    )


if (
    np.isfinite(
        n_divergences_32hc
    )
    and
    n_divergences_32hc
    >
    0
):

    hard_flags_32hc.append(
        f"{n_divergences_32hc} divergences"
    )


if max_rhat_32hc > 1.01:

    hard_flags_32hc.append(
        f"max R-hat {max_rhat_32hc:.6f} > 1.01"
    )


if min_bulk_ess_32hc < 400:

    watch_flags_32hc.append(
        f"minimum bulk ESS {min_bulk_ess_32hc:.1f} < 400"
    )


if min_tail_ess_32hc < 400:

    watch_flags_32hc.append(
        f"minimum tail ESS {min_tail_ess_32hc:.1f} < 400"
    )


if (
    np.isfinite(
        min_bfmi_32hc
    )
    and
    min_bfmi_32hc
    <
    0.30
):

    watch_flags_32hc.append(
        f"minimum BFMI {min_bfmi_32hc:.4f} < 0.30"
    )


if (
    maxdepth_hits_32hc
    is not None
    and
    maxdepth_hits_32hc
    >
    0
):

    watch_flags_32hc.append(
        f"{maxdepth_hits_32hc} explicit "
        "maximum-tree-depth hits"
    )


if hard_flags_32hc:

    health_status_32hc = (
        "FAIL"
    )

elif watch_flags_32hc:

    health_status_32hc = (
        "WATCH"
    )

else:

    health_status_32hc = (
        "PASS"
    )


print()
print(
    "=" * 80
)

print(
    "STAGE 32H-C HEALTH GATE"
)

print(
    "=" * 80
)


print(
    "Status:",
    health_status_32hc,
)


print()
print(
    "Completed posterior:"
)

print(
    f"  {n_chains_32hc} chains × "
    f"{n_draws_32hc} draws"
)


print(
    "  divergences:",
    n_divergences_32hc,
)


print(
    "  max R-hat:",
    f"{max_rhat_32hc:.6f}",
)


print(
    "  R-hat > 1.01:",
    int(
        (
            all_rhat_32hc
            >
            1.01
        ).sum()
    ),
)


print(
    "  min bulk ESS:",
    f"{min_bulk_ess_32hc:.1f}",
)


print(
    "  min tail ESS:",
    f"{min_tail_ess_32hc:.1f}",
)


print(
    "  min BFMI:",
    (
        f"{min_bfmi_32hc:.4f}"
        if np.isfinite(
            min_bfmi_32hc
        )
        else "NA"
    ),
)


print()
print(
    "Frozen 1982–2025 reference:"
)

print(
    "  max R-hat:    1.007758"
)

print(
    "  R-hat >1.01:  0"
)

print(
    "  min bulk ESS: 972.3"
)

print(
    "  divergences:  0"
)


print()
print(
    "Hard flags:"
)


if hard_flags_32hc:

    for flag in hard_flags_32hc:

        print(
            "  -",
            flag,
        )

else:

    print(
        "  none"
    )


print()
print(
    "Watch flags:"
)


if watch_flags_32hc:

    for flag in watch_flags_32hc:

        print(
            "  -",
            flag,
        )

else:

    print(
        "  none"
    )


print()
print(
    "NO LIVE JAWS HAS BEEN CALCULATED."
)

print(
    "Next decision depends on this health gate "
    "and inspection of the 2026 states above."
)

## Part VI — Partial-season scoring and release export

In [ ]:
# ============================================================
# STAGE 32H-F — PARTIAL-SEASON JAWS POLICY AUDIT
# ============================================================
#
# PURPOSE
# -------
# Compare three ways of handling the incomplete 2026 season:
#
#   A. HISTORICAL ONLY
#      2026 receives zero peak/longevity credit.
#
#   B. PRO-RATA EQUIVALENT-SEASON  [PROPOSED]
#      Let f = completed GPs / scheduled GPs.
#
#      longevity:
#          historical surplus
#        + f * max(alpha_2026, 0)
#
#      peak:
#          best five EQUIVALENT season-units.
#
#      Historical seasons have weight 1.
#      2026 has weight f.
#
#      Therefore if 2026 is sufficiently strong to enter the
#      peak window, it occupies only f of one season and the
#      next-best historical season fills the remainder.
#
#   C. FULL 2026
#      Naive upper-bound policy treating the partial season
#      exactly like a completed season.
#
# The proposed policy satisfies:
#
#      f = 0  -> exact through-2025 metric
#      f = 1  -> exact ordinary completed-season metric
#
# NO production LIVE leaderboard is saved here.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 0. PREREQUISITES
# ============================================================

required_32hf = [
    "posterior_32hc",
    "idx_all",
    "raw",
]


missing_32hf = [
    name
    for name in required_32hf
    if name not in globals()
]


if missing_32hf:

    raise RuntimeError(
        "Stage 32H-F requires:\n  "
        +
        "\n  ".join(
            missing_32hf
        )
    )


print("=" * 80)
print("STAGE 32H-F — PARTIAL-SEASON JAWS POLICY AUDIT")
print("=" * 80)


# ============================================================
# 1. LOCKED JAWS CONSTANTS
# ============================================================

PANTHEON_START_YEAR_32HF = 1982

PANTHEON_MIN_STARTS_32HF = 35
PANTHEON_MIN_TEAMMATES_32HF = 3
PANTHEON_MIN_EQUIV_SEASONS_32HF = 5.0

PANTHEON_PEAK_WEIGHT_32HF = 0.60
PANTHEON_LONGEVITY_WEIGHT_32HF = 0.40


# ============================================================
# 2. DETERMINE 2026 SEASON PROGRESS — CORRECTED
# ============================================================
#
# IMPORTANT:
#
# raw.races contains COMPLETED 2026 rounds only by design.
# It therefore CANNOT be used as the denominator for the
# scheduled season length.
#
# Stage 32A keeps the full live schedule in race_by_round,
# constructed from the Jolpica season schedule endpoint.
#
# completed numerator:
#     actual completed GP result rounds
#
# scheduled denominator:
#     full current 2026 schedule
# ============================================================

races_32hf = raw.races.copy()
results_32hf = raw.results.copy()


# ------------------------------------------------------------
# Full CURRENT schedule from Stage 32A
# ------------------------------------------------------------

if "race_by_round" not in globals():

    raise RuntimeError(
        "race_by_round is missing.\n\n"
        "Stage 32H-F needs the full 2026 schedule assembled "
        "by Stage 32A. Run Stage 32A in this kernel before "
        "running the policy audit."
    )


schedule_rounds_2026_32hf = sorted(
    {
        int(
            rr
        )

        for rr
        in race_by_round.keys()
    }
)


scheduled_2026_races_32hf = len(
    schedule_rounds_2026_32hf
)


if scheduled_2026_races_32hf <= 0:

    raise RuntimeError(
        "No 2026 scheduled rounds found in race_by_round."
    )


# ------------------------------------------------------------
# Completed rounds from the actual fitted raw.results data
# ------------------------------------------------------------

results_with_year_32hf = results_32hf.merge(

    races_32hf[
        [
            "raceId",
            "year",
            "round",
        ]
    ],

    on="raceId",
    how="left",
    validate="many_to_one",
)


completed_rounds_2026_32hf = sorted(
    pd.to_numeric(
        results_with_year_32hf.loc[
            results_with_year_32hf[
                "year"
            ].eq(
                2026
            ),
            "round",
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


completed_2026_races_32hf = len(
    completed_rounds_2026_32hf
)


# ------------------------------------------------------------
# Cross-check completed rounds against full schedule
# ------------------------------------------------------------

missing_from_schedule_32hf = sorted(
    set(
        completed_rounds_2026_32hf
    )
    -
    set(
        schedule_rounds_2026_32hf
    )
)


if missing_from_schedule_32hf:

    raise RuntimeError(
        "Completed result rounds are missing from the "
        "full 2026 schedule:\n"
        f"{missing_from_schedule_32hf}"
    )


if (
    completed_2026_races_32hf
    >
    scheduled_2026_races_32hf
):

    raise RuntimeError(
        "Completed 2026 race count exceeds scheduled count."
    )


SEASON_FRACTION_32HF = (
    completed_2026_races_32hf
    /
    scheduled_2026_races_32hf
)


print()
print(
    "Completed 2026 rounds:",
    completed_rounds_2026_32hf,
)


print(
    "Full scheduled 2026 rounds:",
    schedule_rounds_2026_32hf,
)


print(
    "2026 completed GPs:",
    completed_2026_races_32hf,
)


print(
    "2026 scheduled GPs:",
    scheduled_2026_races_32hf,
)


print(
    "2026 season fraction:",
    f"{SEASON_FRACTION_32HF:.6f}",
)


# ------------------------------------------------------------
# Current-snapshot sanity checks
# ------------------------------------------------------------

if completed_2026_races_32hf != 15:

    print(
        "WARNING: this snapshot no longer contains exactly "
        "15 completed GPs."
    )


if scheduled_2026_races_32hf != 22:

    print(
        "WARNING: the current schedule does not contain "
        "22 rounds. This may be legitimate if the official "
        "calendar has changed again; inspect race_by_round."
    )


# ============================================================
# 3. LIVE ALPHA POSTERIOR
# ============================================================

alpha_post_32hf = np.asarray(
    posterior_32hc[
        "alpha_dy"
    ].values,
    dtype=float,
)


alpha_post_32hf = alpha_post_32hf.reshape(
    -1,
    alpha_post_32hf.shape[
        -1
    ],
)


n_draws_32hf = int(
    alpha_post_32hf.shape[
        0
    ]
)


driver_states_32hf = [
    (
        driver,
        int(
            year
        ),
    )

    for driver, year
    in idx_all.active_driver_years
]


if alpha_post_32hf.shape[
    1
] != len(
    driver_states_32hf
):

    raise RuntimeError(
        "alpha_dy state count mismatch."
    )


state_to_idx_32hf = {
    state: i
    for i, state
    in enumerate(
        driver_states_32hf
    )
}


all_drivers_32hf = sorted(
    {
        driver
        for driver, year
        in driver_states_32hf
    }
)


# ============================================================
# 4. DRIVER -> MODELLED YEARS
# ============================================================

driver_years_32hf = {}


for driver, year in driver_states_32hf:

    if year < PANTHEON_START_YEAR_32HF:

        continue


    driver_years_32hf.setdefault(
        driver,
        [],
    ).append(
        year
    )


for driver in driver_years_32hf:

    driver_years_32hf[
        driver
    ] = sorted(
        set(
            driver_years_32hf[
                driver
            ]
        )
    )


# ============================================================
# 5. CURRENT ACTUAL-START / TEAMMATE ELIGIBILITY
# ============================================================
#
# Unlike peak/longevity exposure, starts and teammates are
# factual events that have already occurred.
#
# So the LIVE eligibility audit uses actual starts through the
# currently completed 2026 rounds.
# ============================================================

career_results_32hf = (
    results_with_year_32hf.loc[
        results_with_year_32hf[
            "year"
        ].between(
            PANTHEON_START_YEAR_32HF,
            2026,
        )
    ]
    .copy()
)


career_results_32hf[
    "driver"
] = (
    career_results_32hf[
        "driverId"
    ]
    .map(
        raw.driver_ref
    )
)


career_results_32hf[
    "status_str"
] = (
    career_results_32hf[
        "statusId"
    ]
    .map(
        raw.status_map
    )
    .fillna("")
    .astype(str)
    .str.lower()
)


NON_START_PATTERN_32HF = (
    r"did not start"
    r"|did not qualify"
    r"|did not prequalify"
    r"|withdrawn"
    r"|withdrew"
    r"|injured"
    r"|illness"
    r"|unwell"
    r"|not permitted to start"
)


career_results_32hf[
    "is_actual_start"
] = ~(
    career_results_32hf[
        "status_str"
    ]
    .str.contains(
        NON_START_PATTERN_32HF,
        regex=True,
        na=False,
    )
)


actual_starts_32hf = (
    career_results_32hf.loc[
        career_results_32hf[
            "is_actual_start"
        ],
        [
            "raceId",
            "year",
            "driverId",
            "driver",
            "constructorId",
        ],
    ]
    .drop_duplicates(
        subset=[
            "raceId",
            "driverId",
            "constructorId",
        ]
    )
    .copy()
)


starts_lookup_32hf = (
    actual_starts_32hf
    .groupby(
        "driver"
    )[
        "raceId"
    ]
    .nunique()
    .to_dict()
)


teammate_sets_32hf = {
    driver: set()
    for driver
    in all_drivers_32hf
}


for (
    race_id,
    constructor_id,
), grp in actual_starts_32hf.groupby(
    [
        "raceId",
        "constructorId",
    ]
):

    drivers_here = (
        grp[
            "driver"
        ]
        .dropna()
        .unique()
        .tolist()
    )


    for driver in drivers_here:

        teammate_sets_32hf.setdefault(
            driver,
            set(),
        )


        teammate_sets_32hf[
            driver
        ].update(
            other

            for other
            in drivers_here

            if other
            != driver
        )


teammates_lookup_32hf = {
    driver:
        len(
            teammate_sets_32hf.get(
                driver,
                set(),
            )
        )

    for driver
    in all_drivers_32hf
}


# ============================================================
# 6. WEIGHTED BEST-FIVE-EQUIVALENT-SEASONS HELPER
# ============================================================
#
# states:
#     shape = draws x seasons
#
# weights:
#     shape = seasons
#
# We sort seasons by alpha independently in every posterior
# draw, then take the highest-alpha season exposure until
# exactly five equivalent season-units have been accumulated.
#
# Example with f = 0.625:
#
#   2026 alpha        weight 0.625
#   best historical   weight 1
#   next historical   weight 1
#   next historical   weight 1
#   next historical   weight 1
#   next historical   only 0.375 needed
#
# Total exposure = exactly 5.
# ============================================================

def weighted_top5_equivalent_32hf(
    states,
    weights,
):

    states = np.asarray(
        states,
        dtype=float,
    )


    weights = np.asarray(
        weights,
        dtype=float,
    )


    if states.ndim != 2:

        raise ValueError(
            "states must be draws x seasons."
        )


    if weights.ndim != 1:

        raise ValueError(
            "weights must be one-dimensional."
        )


    if states.shape[
        1
    ] != len(
        weights
    ):

        raise ValueError(
            "states/weights length mismatch."
        )


    total_weight = float(
        np.sum(
            weights
        )
    )


    if (
        total_weight
        <
        PANTHEON_MIN_EQUIV_SEASONS_32HF
        -
        1e-12
    ):

        raise ValueError(
            "Driver has fewer than five equivalent seasons."
        )


    order = np.argsort(
        -states,
        axis=1,
    )


    sorted_states = np.take_along_axis(
        states,
        order,
        axis=1,
    )


    sorted_weights = weights[
        order
    ]


    cumulative_before = (
        np.cumsum(
            sorted_weights,
            axis=1,
        )
        -
        sorted_weights
    )


    remaining = (
        PANTHEON_MIN_EQUIV_SEASONS_32HF
        -
        cumulative_before
    )


    used_weight = np.minimum(
        sorted_weights,
        np.maximum(
            remaining,
            0.0,
        ),
    )


    peak = (
        (
            sorted_states
            *
            used_weight
        )
        .sum(
            axis=1
        )
        /
        PANTHEON_MIN_EQUIV_SEASONS_32HF
    )


    used_total = used_weight.sum(
        axis=1
    )


    if not np.allclose(
        used_total,
        PANTHEON_MIN_EQUIV_SEASONS_32HF,
        atol=1e-10,
        rtol=0.0,
    ):

        raise RuntimeError(
            "Weighted peak failed to accumulate exactly "
            "five equivalent seasons."
        )


    return peak


# ============================================================
# 7. POLICY SCORING FUNCTION
# ============================================================

def build_policy_32hf(
    policy_name,
    current_season_weight,
):

    current_season_weight = float(
        current_season_weight
    )


    if not (
        0.0
        <=
        current_season_weight
        <=
        1.0
    ):

        raise ValueError(
            "Current-season weight must lie in [0, 1]."
        )


    # --------------------------------------------------------
    # Equivalent-season exposure for normalisation eligibility
    # --------------------------------------------------------

    equivalent_seasons = {}


    for driver in all_drivers_32hf:

        years = driver_years_32hf.get(
            driver,
            [],
        )


        n_historical = sum(
            year <= 2025
            for year
            in years
        )


        has_2026 = (
            2026
            in years
        )


        equivalent_seasons[
            driver
        ] = (
            float(
                n_historical
            )
            +
            (
                current_season_weight
                if has_2026
                else 0.0
            )
        )


    scored_drivers = [
        driver

        for driver
        in all_drivers_32hf

        if (
            equivalent_seasons.get(
                driver,
                0.0,
            )
            >=
            PANTHEON_MIN_EQUIV_SEASONS_32HF
            -
            1e-12
        )
    ]


    n_scored = len(
        scored_drivers
    )


    surplus_mat = np.zeros(
        (
            n_draws_32hf,
            n_scored,
        ),
        dtype=float,
    )


    peak_mat = np.zeros_like(
        surplus_mat
    )


    for j, driver in enumerate(
        scored_drivers
    ):

        years = [
            year

            for year
            in driver_years_32hf[
                driver
            ]

            if year <= 2026
        ]


        states_idx = np.array(
            [
                state_to_idx_32hf[
                    (
                        driver,
                        year,
                    )
                ]

                for year
                in years
            ],
            dtype=int,
        )


        states = alpha_post_32hf[
            :,
            states_idx,
        ]


        weights = np.array(
            [
                (
                    1.0

                    if year <= 2025

                    else current_season_weight
                )

                for year
                in years
            ],
            dtype=float,
        )


        # Remove zero-weight 2026 column in the historical-only
        # policy so it cannot affect sorting numerically.

        keep = weights > 0.0


        states = states[
            :,
            keep,
        ]


        weights = weights[
            keep
        ]


        surplus_mat[
            :,
            j,
        ] = (
            np.maximum(
                states,
                0.0,
            )
            *
            weights[
                None,
                :
            ]
        ).sum(
            axis=1
        )


        peak_mat[
            :,
            j,
        ] = weighted_top5_equivalent_32hf(
            states,
            weights,
        )


    # --------------------------------------------------------
    # Per-draw normalisation
    # --------------------------------------------------------

    longevity_raw = np.sqrt(
        surplus_mat
    )


    peak_denominator = peak_mat.max(
        axis=1,
        keepdims=True,
    )


    longevity_denominator = (
        longevity_raw.max(
            axis=1,
            keepdims=True,
        )
    )


    if np.any(
        peak_denominator
        <= 0
    ):

        raise RuntimeError(
            f"{policy_name}: non-positive peak denominator."
        )


    if np.any(
        longevity_denominator
        <= 0
    ):

        raise RuntimeError(
            f"{policy_name}: non-positive longevity denominator."
        )


    peak_score = (
        peak_mat
        /
        peak_denominator
        *
        100.0
    )


    longevity_score = (
        longevity_raw
        /
        longevity_denominator
        *
        100.0
    )


    jaws = (
        PANTHEON_PEAK_WEIGHT_32HF
        *
        peak_score

        +

        PANTHEON_LONGEVITY_WEIGHT_32HF
        *
        longevity_score
    )


    # --------------------------------------------------------
    # LIVE eligibility
    # --------------------------------------------------------

    eligible_drivers = [
        driver

        for driver
        in scored_drivers

        if (
            starts_lookup_32hf.get(
                driver,
                0,
            )
            >=
            PANTHEON_MIN_STARTS_32HF
        )

        and

        (
            teammates_lookup_32hf.get(
                driver,
                0,
            )
            >=
            PANTHEON_MIN_TEAMMATES_32HF
        )
    ]


    eligible_indices = np.array(
        [
            scored_drivers.index(
                driver
            )

            for driver
            in eligible_drivers
        ],
        dtype=int,
    )


    eligible_jaws = jaws[
        :,
        eligible_indices,
    ]


    # --------------------------------------------------------
    # Ranks
    # --------------------------------------------------------

    order = np.argsort(
        -eligible_jaws,
        axis=1,
    )


    ranks = np.empty_like(
        order,
        dtype=int,
    )


    ranks[
        np.arange(
            n_draws_32hf
        )[
            :,
            None,
        ],
        order,
    ] = np.arange(
        1,
        len(
            eligible_drivers
        )
        +
        1,
    )[
        None,
        :
    ]


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    rows = []


    for eligible_j, driver in enumerate(
        eligible_drivers
    ):

        scored_j = eligible_indices[
            eligible_j
        ]


        jd = eligible_jaws[
            :,
            eligible_j,
        ]


        rd = ranks[
            :,
            eligible_j,
        ]


        rows.append(
            {
                "driver":
                    driver,

                "policy":
                    policy_name,

                "season_weight_2026":
                    current_season_weight,

                "equivalent_seasons":
                    equivalent_seasons[
                        driver
                    ],

                "jaws_mean":
                    float(
                        np.mean(
                            jd
                        )
                    ),

                "jaws_median":
                    float(
                        np.median(
                            jd
                        )
                    ),

                "jaws_89_lb":
                    float(
                        np.quantile(
                            jd,
                            0.055,
                        )
                    ),

                "jaws_89_ub":
                    float(
                        np.quantile(
                            jd,
                            0.945,
                        )
                    ),

                "peak_raw_mean":
                    float(
                        peak_mat[
                            :,
                            scored_j,
                        ].mean()
                    ),

                "surplus_raw_mean":
                    float(
                        surplus_mat[
                            :,
                            scored_j,
                        ].mean()
                    ),

                "peak_score_mean":
                    float(
                        peak_score[
                            :,
                            scored_j,
                        ].mean()
                    ),

                "longevity_score_mean":
                    float(
                        longevity_score[
                            :,
                            scored_j,
                        ].mean()
                    ),

                "rank_mean":
                    float(
                        rd.mean()
                    ),

                "rank_median":
                    float(
                        np.median(
                            rd
                        )
                    ),

                "rank_89_lb":
                    float(
                        np.quantile(
                            rd,
                            0.055,
                        )
                    ),

                "rank_89_ub":
                    float(
                        np.quantile(
                            rd,
                            0.945,
                        )
                    ),

                "starts":
                    int(
                        starts_lookup_32hf.get(
                            driver,
                            0,
                        )
                    ),

                "unique_teammates":
                    int(
                        teammates_lookup_32hf.get(
                            driver,
                            0,
                        )
                    ),
            }
        )


    board = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "jaws_mean",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )


    board[
        "point_rank"
    ] = np.arange(
        1,
        len(
            board
        )
        +
        1,
    )


    return {
        "name":
            policy_name,

        "weight":
            current_season_weight,

        "scored_drivers":
            scored_drivers,

        "eligible_drivers":
            eligible_drivers,

        "equivalent_seasons":
            equivalent_seasons,

        "peak_raw":
            peak_mat,

        "surplus_raw":
            surplus_mat,

        "peak_score":
            peak_score,

        "longevity_score":
            longevity_score,

        "jaws":
            jaws,

        "eligible_jaws":
            eligible_jaws,

        "ranks":
            ranks,

        "board":
            board,
    }


# ============================================================
# 8. BUILD THREE POLICIES
# ============================================================

policy_2025only_32hf = build_policy_32hf(
    "2025_ONLY",
    0.0,
)


policy_prorata_32hf = build_policy_32hf(
    "PRORATA_2026",
    SEASON_FRACTION_32HF,
)


policy_full_32hf = build_policy_32hf(
    "FULL_2026",
    1.0,
)


print()
print("=" * 80)
print("POLICY POOLS / ELIGIBILITY")
print("=" * 80)


policy_pool_table_32hf = pd.DataFrame(
    [
        {
            "policy":
                policy[
                    "name"
                ],

            "2026_weight":
                policy[
                    "weight"
                ],

            "normalisation_pool":
                len(
                    policy[
                        "scored_drivers"
                    ]
                ),

            "eligible":
                len(
                    policy[
                        "eligible_drivers"
                    ]
                ),
        }

        for policy
        in [
            policy_2025only_32hf,
            policy_prorata_32hf,
            policy_full_32hf,
        ]
    ]
)


display(
    policy_pool_table_32hf
)


# ============================================================
# 9. ELIGIBILITY CHANGES FROM FROZEN 2025
# ============================================================

frozen_csv_32hf = (
    Path(
        "model_outputs"
    )
    /
    "joint_driver_car_v2_1_1982_2025"
    /
    "pantheon_jaws_1982_2025_FINAL.csv"
)


if not frozen_csv_32hf.exists():

    raise RuntimeError(
        "Frozen Pantheon CSV not found."
    )


frozen_board_32hf = pd.read_csv(
    frozen_csv_32hf
)


frozen_eligible_32hf = set(
    frozen_board_32hf[
        "driver"
    ].astype(str)
)


prorata_eligible_32hf = set(
    policy_prorata_32hf[
        "eligible_drivers"
    ]
)


print()
print("=" * 80)
print("LIVE ELIGIBILITY CHANGES")
print("=" * 80)


print(
    "Newly eligible in LIVE pro-rata policy:",
    sorted(
        prorata_eligible_32hf
        -
        frozen_eligible_32hf
    ),
)


print(
    "No longer eligible:",
    sorted(
        frozen_eligible_32hf
        -
        prorata_eligible_32hf
    ),
)


# ============================================================
# 10. MERGE POLICY BOARDS
# ============================================================

def compact_policy_board_32hf(
    policy,
    suffix,
):

    board = policy[
        "board"
    ].copy()


    return (
        board[
            [
                "driver",
                "point_rank",
                "jaws_mean",
                "peak_score_mean",
                "longevity_score_mean",
                "rank_mean",
            ]
        ]
        .rename(
            columns={
                "point_rank":
                    f"rank_{suffix}",

                "jaws_mean":
                    f"jaws_{suffix}",

                "peak_score_mean":
                    f"peak_{suffix}",

                "longevity_score_mean":
                    f"longevity_{suffix}",

                "rank_mean":
                    f"expected_rank_{suffix}",
            }
        )
    )


comparison_32hf = (
    compact_policy_board_32hf(
        policy_2025only_32hf,
        "2025",
    )
    .merge(
        compact_policy_board_32hf(
            policy_prorata_32hf,
            "prorata",
        ),
        on="driver",
        how="outer",
        validate="one_to_one",
    )
    .merge(
        compact_policy_board_32hf(
            policy_full_32hf,
            "full",
        ),
        on="driver",
        how="outer",
        validate="one_to_one",
    )
)


comparison_32hf[
    "prorata_minus_2025"
] = (
    comparison_32hf[
        "jaws_prorata"
    ]
    -
    comparison_32hf[
        "jaws_2025"
    ]
)


comparison_32hf[
    "full_minus_prorata"
] = (
    comparison_32hf[
        "jaws_full"
    ]
    -
    comparison_32hf[
        "jaws_prorata"
    ]
)


comparison_32hf[
    "rank_move_prorata_vs_2025"
] = (
    comparison_32hf[
        "rank_2025"
    ]
    -
    comparison_32hf[
        "rank_prorata"
    ]
)


comparison_32hf[
    "rank_move_full_vs_prorata"
] = (
    comparison_32hf[
        "rank_prorata"
    ]
    -
    comparison_32hf[
        "rank_full"
    ]
)


# ============================================================
# 11. PRO-RATA LIVE LEADERBOARD — AUDIT VIEW
# ============================================================

print()
print("=" * 80)
print("PRO-RATA LIVE POLICY — TOP 30")
print("=" * 80)


display(
    policy_prorata_32hf[
        "board"
    ]
    .head(
        30
    )
)


# ============================================================
# 12. BIGGEST 2026 EFFECTS UNDER PRO-RATA POLICY
# ============================================================

print()
print("=" * 80)
print("LARGEST PRO-RATA 2026 SCORE EFFECTS")
print("=" * 80)


display(
    comparison_32hf
    .dropna(
        subset=[
            "prorata_minus_2025",
        ]
    )
    .assign(
        abs_effect=lambda df:
            np.abs(
                df[
                    "prorata_minus_2025"
                ]
            )
    )
    .sort_values(
        "abs_effect",
        ascending=False,
    )
    .head(
        30
    )
    [
        [
            "driver",

            "rank_2025",
            "rank_prorata",
            "rank_move_prorata_vs_2025",

            "jaws_2025",
            "jaws_prorata",
            "prorata_minus_2025",

            "peak_2025",
            "peak_prorata",

            "longevity_2025",
            "longevity_prorata",
        ]
    ]
)


# ============================================================
# 13. HOW MUCH DOES NAIVE FULL-SEASON TREATMENT OVERSTATE?
# ============================================================

print()
print("=" * 80)
print("FULL-2026 VS PRO-RATA DIFFERENCE")
print("=" * 80)


display(
    comparison_32hf
    .dropna(
        subset=[
            "full_minus_prorata",
        ]
    )
    .assign(
        abs_difference=lambda df:
            np.abs(
                df[
                    "full_minus_prorata"
                ]
            )
    )
    .sort_values(
        "abs_difference",
        ascending=False,
    )
    .head(
        30
    )
    [
        [
            "driver",

            "rank_prorata",
            "rank_full",
            "rank_move_full_vs_prorata",

            "jaws_prorata",
            "jaws_full",
            "full_minus_prorata",

            "peak_prorata",
            "peak_full",

            "longevity_prorata",
            "longevity_full",
        ]
    ]
)


# ============================================================
# 14. GLOBAL POLICY-STABILITY METRICS
# ============================================================

common_2025_prorata_32hf = (
    comparison_32hf
    .dropna(
        subset=[
            "jaws_2025",
            "jaws_prorata",
        ]
    )
)


common_prorata_full_32hf = (
    comparison_32hf
    .dropna(
        subset=[
            "jaws_prorata",
            "jaws_full",
        ]
    )
)


spearman_2025_prorata_32hf = float(
    common_2025_prorata_32hf[
        [
            "jaws_2025",
            "jaws_prorata",
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1,
    ]
)


spearman_prorata_full_32hf = float(
    common_prorata_full_32hf[
        [
            "jaws_prorata",
            "jaws_full",
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1,
    ]
)


max_rank_move_2025_prorata_32hf = int(
    common_2025_prorata_32hf[
        "rank_move_prorata_vs_2025"
    ]
    .abs()
    .max()
)


max_rank_move_prorata_full_32hf = int(
    common_prorata_full_32hf[
        "rank_move_full_vs_prorata"
    ]
    .abs()
    .max()
)


print()
print("=" * 80)
print("POLICY SENSITIVITY SUMMARY")
print("=" * 80)


print(
    "Season fraction:",
    f"{SEASON_FRACTION_32HF:.6f}",
)


print()
print(
    "2025-only vs pro-rata Spearman:",
    f"{spearman_2025_prorata_32hf:.6f}",
)

print(
    "Max rank move, 2025-only -> pro-rata:",
    max_rank_move_2025_prorata_32hf,
)


print()
print(
    "Pro-rata vs full-2026 Spearman:",
    f"{spearman_prorata_full_32hf:.6f}",
)

print(
    "Max rank move, pro-rata -> full-2026:",
    max_rank_move_prorata_full_32hf,
)


# ============================================================
# 15. SAVE AUDIT TABLES ONLY
# ============================================================

audit_dir_32hf = Path(
    CFG.output_dir
)


audit_dir_32hf.mkdir(
    parents=True,
    exist_ok=True,
)


policy_comparison_path_32hf = (
    audit_dir_32hf
    /
    "partial_2026_jaws_policy_comparison.csv"
)


prorata_board_path_32hf = (
    audit_dir_32hf
    /
    "partial_2026_prorata_jaws_AUDIT.csv"
)


comparison_32hf.to_csv(
    policy_comparison_path_32hf,
    index=False,
)


policy_prorata_32hf[
    "board"
].to_csv(
    prorata_board_path_32hf,
    index=False,
)


print()
print(
    "Saved policy comparison ->"
)

print(
    policy_comparison_path_32hf
)


print()
print(
    "Saved pro-rata AUDIT leaderboard ->"
)

print(
    prorata_board_path_32hf
)


print()
print("=" * 80)
print("STAGE 32H-F COMPLETE")
print("=" * 80)

print()
print(
    "No production LIVE Pantheon has been locked."
)

print(
    "This stage compares partial-season scoring policies only."
)

In [ ]:
# ============================================================
# STAGE 32H-G — LOCK + EXPORT LIVE PANTHEON SNAPSHOT
# ============================================================
#
# LIVE PRODUCT VERSION:
#     Pantheon v2.2 LIVE
#
# WHY v2.2?
# ---------
# The frozen official 1982–2025 Pantheon remains v2.1.
#
# The LIVE model changes the familiarity specification to
# organisational memory with a 20-GP missed-weekend half-life,
# so the LIVE product is methodologically distinct.
#
# IMPORTANT:
# ---------
# We DO NOT rename, move, alter, or overwrite the fitted
# posterior checkpoint. Its existing v2_1 filename is retained
# as immutable provenance.
#
# LOCKED PARTIAL-SEASON POLICY
# ----------------------------
# 2026 season exposure:
#
#     f = completed GPs / currently scheduled GPs
#
# Current snapshot:
#
#     15 / 23 = 0.652173913...
#
# Longevity:
#
#     historical positive surplus
#     + f * max(alpha_2026, 0)
#
# Peak:
#
#     best five equivalent season-units
#
# Completed seasons have weight 1.
# 2026 has weight f.
#
# JAWS:
#
#     0.60 * peak score
#   + 0.40 * longevity score
#
# This cell:
#
#   1. validates the locked policy
#   2. constructs the production LIVE leaderboard
#   3. saves posterior score/rank draws
#   4. writes a reproducibility manifest
#
# NO SAMPLING OCCURS.
# ============================================================

from pathlib import Path

import json
import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 0. PRODUCT LOCK
# ============================================================

LIVE_PRODUCT_VERSION_32HG = "v2_2"

LIVE_PRODUCT_NAME_32HG = (
    "F1_Pantheon_1982_2026_LIVE"
)

LIVE_SCORING_POLICY_32HG = (
    "prorata_equivalent_season"
)

LIVE_FAMILIARITY_32HG = (
    "organisational_memory_20gp_half_life"
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_32hg = [

    "policy_prorata_32hf",
    "policy_2025only_32hf",
    "policy_full_32hf",

    "SEASON_FRACTION_32HF",
    "completed_2026_races_32hf",
    "scheduled_2026_races_32hf",

    "LIVE_SNAPSHOT_TAG_32HB",
    "LIVE_SNAPSHOT_HASH_32HB",
    "LIVE_TRACE_PATH_32HB",

    "posterior_32hc",
]


missing_32hg = [

    name

    for name
    in required_32hg

    if name not in globals()
]


if missing_32hg:

    raise RuntimeError(
        "Stage 32H-G is missing required objects:\n  "
        +
        "\n  ".join(
            missing_32hg
        )
        +
        "\n\nRun Stages 32H-B through 32H-F first."
    )


print("=" * 80)
print("STAGE 32H-G — LOCK + EXPORT LIVE PANTHEON")
print("=" * 80)


# ============================================================
# 2. VALIDATE CURRENT SNAPSHOT
# ============================================================

expected_completed_32hg = 15
expected_schedule_32hg = 23

expected_fraction_32hg = (
    expected_completed_32hg
    /
    expected_schedule_32hg
)


if (
    completed_2026_races_32hf
    !=
    expected_completed_32hg
):

    raise RuntimeError(
        "This release cell is locked to the current "
        "15-GP snapshot.\n"
        f"Found {completed_2026_races_32hf} completed GPs."
    )


if (
    scheduled_2026_races_32hf
    !=
    expected_schedule_32hg
):

    raise RuntimeError(
        "Current scheduled-race denominator changed.\n"
        f"Expected 23, found {scheduled_2026_races_32hf}.\n\n"
        "Inspect the current official calendar before "
        "releasing this snapshot."
    )


if not np.isclose(
    SEASON_FRACTION_32HF,
    expected_fraction_32hg,
    atol=1e-12,
    rtol=0.0,
):

    raise RuntimeError(
        "Season fraction does not equal 15/23."
    )


if not np.isclose(
    policy_prorata_32hf[
        "weight"
    ],
    SEASON_FRACTION_32HF,
    atol=1e-12,
    rtol=0.0,
):

    raise RuntimeError(
        "Pro-rata policy weight does not match "
        "the locked season fraction."
    )


if not np.isclose(
    policy_2025only_32hf[
        "weight"
    ],
    0.0,
):

    raise RuntimeError(
        "2025-only policy is not weight 0."
    )


if not np.isclose(
    policy_full_32hf[
        "weight"
    ],
    1.0,
):

    raise RuntimeError(
        "Full-2026 policy is not weight 1."
    )


print()
print(
    "Completed GPs:",
    completed_2026_races_32hf,
)

print(
    "Scheduled GPs:",
    scheduled_2026_races_32hf,
)

print(
    "Locked 2026 exposure:",
    f"{SEASON_FRACTION_32HF:.9f}",
)

print(
    "Snapshot:",
    LIVE_SNAPSHOT_TAG_32HB,
)


# ============================================================
# 3. VALIDATE PRODUCTION POOLS
# ============================================================

scored_drivers_32hg = list(
    policy_prorata_32hf[
        "scored_drivers"
    ]
)


eligible_drivers_32hg = list(
    policy_prorata_32hf[
        "eligible_drivers"
    ]
)


if len(
    scored_drivers_32hg
) != 99:

    raise RuntimeError(
        "Expected 99-driver normalisation pool, found "
        f"{len(scored_drivers_32hg)}."
    )


if len(
    eligible_drivers_32hg
) != 98:

    raise RuntimeError(
        "Expected 98 Pantheon-eligible drivers, found "
        f"{len(eligible_drivers_32hg)}."
    )


print()
print(
    "Normalisation pool:",
    len(
        scored_drivers_32hg
    ),
)

print(
    "Pantheon eligible:",
    len(
        eligible_drivers_32hg
    ),
)


# ============================================================
# 4. PRODUCTION BOARD
# ============================================================

live_board_32hg = (
    policy_prorata_32hf[
        "board"
    ]
    .copy()
)


if set(
    live_board_32hg[
        "driver"
    ]
) != set(
    eligible_drivers_32hg
):

    raise RuntimeError(
        "Leaderboard driver set does not match "
        "the eligible-driver set."
    )


# ------------------------------------------------------------
# Add immutable snapshot metadata
# ------------------------------------------------------------

live_board_32hg[
    "product_version"
] = LIVE_PRODUCT_VERSION_32HG


live_board_32hg[
    "snapshot_tag"
] = LIVE_SNAPSHOT_TAG_32HB


live_board_32hg[
    "snapshot_hash"
] = LIVE_SNAPSHOT_HASH_32HB


live_board_32hg[
    "completed_2026_gps"
] = completed_2026_races_32hf


live_board_32hg[
    "scheduled_2026_gps"
] = scheduled_2026_races_32hf


live_board_32hg[
    "season_fraction_2026"
] = SEASON_FRACTION_32HF


live_board_32hg[
    "scoring_policy"
] = LIVE_SCORING_POLICY_32HG


live_board_32hg[
    "familiarity_model"
] = LIVE_FAMILIARITY_32HG


# ------------------------------------------------------------
# Put important presentation columns first
# ------------------------------------------------------------

preferred_cols_32hg = [

    "point_rank",
    "driver",

    "jaws_mean",
    "jaws_median",
    "jaws_89_lb",
    "jaws_89_ub",

    "rank_mean",
    "rank_median",
    "rank_89_lb",
    "rank_89_ub",

    "peak_raw_mean",
    "surplus_raw_mean",

    "peak_score_mean",
    "longevity_score_mean",

    "equivalent_seasons",

    "starts",
    "unique_teammates",

    "season_weight_2026",

    "product_version",
    "snapshot_tag",
    "snapshot_hash",

    "completed_2026_gps",
    "scheduled_2026_gps",
    "season_fraction_2026",

    "scoring_policy",
    "familiarity_model",
]


missing_board_cols_32hg = [

    col

    for col
    in preferred_cols_32hg

    if col not in live_board_32hg.columns
]


if missing_board_cols_32hg:

    raise RuntimeError(
        "Production board missing columns:\n  "
        +
        "\n  ".join(
            missing_board_cols_32hg
        )
    )


remaining_cols_32hg = [

    col

    for col
    in live_board_32hg.columns

    if col not in preferred_cols_32hg
]


live_board_32hg = live_board_32hg[
    preferred_cols_32hg
    +
    remaining_cols_32hg
]


# ============================================================
# 5. EXTRACT DRAW-LEVEL PRODUCTION MATRICES
# ============================================================

eligible_jaws_draws_32hg = np.asarray(

    policy_prorata_32hf[
        "eligible_jaws"
    ],

    dtype=float,
)


rank_draws_32hg = np.asarray(

    policy_prorata_32hf[
        "ranks"
    ],

    dtype=np.int16,
)


if (
    eligible_jaws_draws_32hg.shape
    !=
    rank_draws_32hg.shape
):

    raise RuntimeError(
        "JAWS and rank draw matrices have different shapes."
    )


if (
    eligible_jaws_draws_32hg.shape[
        1
    ]
    !=
    len(
        eligible_drivers_32hg
    )
):

    raise RuntimeError(
        "Draw matrix driver dimension does not match "
        "eligible-driver count."
    )


# ------------------------------------------------------------
# Peak / longevity score matrices are stored over the complete
# normalisation pool, so subset them to eligible drivers.
# ------------------------------------------------------------

scored_index_32hg = {

    driver: i

    for i, driver
    in enumerate(
        scored_drivers_32hg
    )
}


eligible_scored_idx_32hg = np.array(

    [

        scored_index_32hg[
            driver
        ]

        for driver
        in eligible_drivers_32hg
    ],

    dtype=int,
)


peak_score_draws_32hg = np.asarray(

    policy_prorata_32hf[
        "peak_score"
    ][
        :,
        eligible_scored_idx_32hg,
    ],

    dtype=float,
)


longevity_score_draws_32hg = np.asarray(

    policy_prorata_32hf[
        "longevity_score"
    ][
        :,
        eligible_scored_idx_32hg,
    ],

    dtype=float,
)


peak_raw_draws_32hg = np.asarray(

    policy_prorata_32hf[
        "peak_raw"
    ][
        :,
        eligible_scored_idx_32hg,
    ],

    dtype=float,
)


surplus_raw_draws_32hg = np.asarray(

    policy_prorata_32hf[
        "surplus_raw"
    ][
        :,
        eligible_scored_idx_32hg,
    ],

    dtype=float,
)


n_draws_32hg = int(
    eligible_jaws_draws_32hg.shape[
        0
    ]
)


if n_draws_32hg != 4000:

    raise RuntimeError(
        f"Expected 4000 posterior draws, found {n_draws_32hg}."
    )


# ============================================================
# 6. DRAW-LEVEL INTERNAL CONSISTENCY CHECK
# ============================================================

reconstructed_jaws_32hg = (

    0.60
    *
    peak_score_draws_32hg

    +

    0.40
    *
    longevity_score_draws_32hg
)


max_reconstruction_error_32hg = float(

    np.max(
        np.abs(
            reconstructed_jaws_32hg
            -
            eligible_jaws_draws_32hg
        )
    )
)


print()
print(
    "Max draw-level JAWS reconstruction error:",
    f"{max_reconstruction_error_32hg:.3e}",
)


if max_reconstruction_error_32hg > 1e-10:

    raise RuntimeError(
        "JAWS draw reconstruction failed."
    )


# ============================================================
# 7. CHECK BOARD MEANS AGAINST DRAW MATRICES
# ============================================================

board_mean_lookup_32hg = dict(
    zip(
        live_board_32hg[
            "driver"
        ],
        live_board_32hg[
            "jaws_mean"
        ],
    )
)


max_board_mean_error_32hg = 0.0


for j, driver in enumerate(
    eligible_drivers_32hg
):

    draw_mean = float(
        eligible_jaws_draws_32hg[
            :,
            j,
        ].mean()
    )


    error = abs(
        draw_mean
        -
        board_mean_lookup_32hg[
            driver
        ]
    )


    max_board_mean_error_32hg = max(
        max_board_mean_error_32hg,
        error,
    )


print(
    "Max board-vs-draw mean error:",
    f"{max_board_mean_error_32hg:.3e}",
)


if max_board_mean_error_32hg > 1e-10:

    raise RuntimeError(
        "Leaderboard means do not reproduce draw means."
    )


# ============================================================
# 8. OUTPUT FILENAMES
# ============================================================

live_output_dir_32hg = Path(
    CFG.output_dir
)


live_output_dir_32hg.mkdir(
    parents=True,
    exist_ok=True,
)


release_tag_32hg = (

    f"{LIVE_PRODUCT_VERSION_32HG}_"
    f"{LIVE_SNAPSHOT_TAG_32HB}_"
    f"{completed_2026_races_32hf}of"
    f"{scheduled_2026_races_32hf}gp"
)


LIVE_PANTHEON_CSV_32HG = (

    live_output_dir_32hg
    /
    (
        "pantheon_jaws_1982_2026_LIVE_"
        f"{release_tag_32hg}.csv"
    )
)


LIVE_PANTHEON_DRAWS_32HG = (

    live_output_dir_32hg
    /
    (
        "pantheon_jaws_1982_2026_LIVE_"
        f"{release_tag_32hg}_draws.npz"
    )
)


LIVE_PANTHEON_MANIFEST_32HG = (

    live_output_dir_32hg
    /
    (
        "pantheon_jaws_1982_2026_LIVE_"
        f"{release_tag_32hg}_manifest.json"
    )
)


# ============================================================
# 9. REFUSE SILENT OVERWRITE
# ============================================================

for path_32hg in [

    LIVE_PANTHEON_CSV_32HG,
    LIVE_PANTHEON_DRAWS_32HG,
    LIVE_PANTHEON_MANIFEST_32HG,
]:

    if path_32hg.exists():

        raise RuntimeError(
            "LIVE release file already exists:\n"
            f"{path_32hg}\n\n"
            "Nothing has been overwritten."
        )


# ============================================================
# 10. SAVE LEADERBOARD
# ============================================================

live_board_32hg.to_csv(

    LIVE_PANTHEON_CSV_32HG,

    index=False,
)


# ============================================================
# 11. SAVE DRAW ARCHIVE
# ============================================================

np.savez_compressed(

    LIVE_PANTHEON_DRAWS_32HG,

    drivers=
        np.asarray(
            eligible_drivers_32hg,
            dtype=str,
        ),

    jaws=
        eligible_jaws_draws_32hg,

    rank=
        rank_draws_32hg,

    peak_score=
        peak_score_draws_32hg,

    longevity_score=
        longevity_score_draws_32hg,

    peak_raw=
        peak_raw_draws_32hg,

    career_surplus=
        surplus_raw_draws_32hg,

    season_fraction_2026=
        np.asarray(
            SEASON_FRACTION_32HF,
            dtype=float,
        ),

    completed_2026_gps=
        np.asarray(
            completed_2026_races_32hf,
            dtype=int,
        ),

    scheduled_2026_gps=
        np.asarray(
            scheduled_2026_races_32hf,
            dtype=int,
        ),

    snapshot_hash=
        np.asarray(
            LIVE_SNAPSHOT_HASH_32HB,
            dtype=str,
        ),
)


# ============================================================
# 12. RELEASE MANIFEST
# ============================================================

live_release_manifest_32hg = {

    "product":
        LIVE_PRODUCT_NAME_32HG,

    "product_version":
        LIVE_PRODUCT_VERSION_32HG,

    "status":
        "LIVE_partial_season_snapshot",

    "snapshot_tag":
        LIVE_SNAPSHOT_TAG_32HB,

    "snapshot_hash":
        LIVE_SNAPSHOT_HASH_32HB,

    "source_trace":
        str(
            LIVE_TRACE_PATH_32HB
        ),

    "source_trace_note":
        (
            "Source checkpoint retains its historical v2_1 "
            "filename. The LIVE product is labelled v2.2 "
            "because organisational familiarity memory changes "
            "the model specification relative to the frozen "
            "1982–2025 v2.1 model."
        ),

    "historical_window": {
        "start_year":
            1982,

        "last_complete_year":
            2025,

        "live_year":
            2026,
    },

    "live_2026": {

        "completed_grands_prix":
            int(
                completed_2026_races_32hf
            ),

        "scheduled_grands_prix":
            int(
                scheduled_2026_races_32hf
            ),

        "season_fraction":
            float(
                SEASON_FRACTION_32HF
            ),

        "latest_completed_round":
            int(
                LIVE_LAST_ROUND_32HB
            ),
    },

    "model": {

        "familiarity_definition":
            "organisational familiarity",

        "familiarity_decay":
            (
                "exponential decay while away from "
                "organisational lineage"
            ),

        "familiarity_half_life_missed_gp":
            20,

        "posterior_draws":
            4000,

        "chains":
            4,
    },

    "jaws": {

        "peak_weight":
            0.60,

        "longevity_weight":
            0.40,

        "peak_definition":
            (
                "posterior-draw mean ability over the best "
                "five equivalent season-units"
            ),

        "complete_season_weight":
            1.0,

        "live_2026_season_weight":
            float(
                SEASON_FRACTION_32HF
            ),

        "career_surplus_definition":
            (
                "sum of positive completed-season alpha plus "
                "season_fraction * max(alpha_2026, 0)"
            ),

        "longevity_transform":
            "sqrt(career_surplus)",

        "normalisation":
            (
                "peak and longevity independently normalised "
                "to 100 within every posterior draw"
            ),
    },

    "eligibility": {

        "minimum_actual_starts":
            35,

        "minimum_unique_actual_start_teammates":
            3,

        "minimum_equivalent_modelled_seasons":
            5.0,

        "normalisation_pool_size":
            int(
                len(
                    scored_drivers_32hg
                )
            ),

        "eligible_driver_count":
            int(
                len(
                    eligible_drivers_32hg
                )
            ),
    },

    "validation": {

        "stage_32h_c_health_gate":
            "PASS",

        "divergences":
            0,

        "max_rhat":
            1.006234,

        "min_bulk_ess":
            1032.8,

        "min_tail_ess":
            1258.1,

        "min_bfmi":
            0.6563,

        "stage_32h_e_historical_score_stability": {

            "jaws_pearson":
                0.998856,

            "jaws_spearman":
                0.998432,

            "top10_set_identical":
                True,

            "top10_order_identical":
                True,

            "maximum_point_rank_movement":
                5,
        },

        "stage_32h_f_partial_season_policy": {

            "through_2025_vs_prorata_spearman":
                0.999923,

            "through_2025_vs_prorata_max_rank_move":
                1,

            "prorata_vs_full_2026_spearman":
                0.999962,

            "prorata_vs_full_2026_max_rank_move":
                1,
        },
    },

    "files": {

        "leaderboard_csv":
            str(
                LIVE_PANTHEON_CSV_32HG
            ),

        "posterior_draw_archive":
            str(
                LIVE_PANTHEON_DRAWS_32HG
            ),
    },
}


with open(

    LIVE_PANTHEON_MANIFEST_32HG,

    "w",

    encoding="utf-8",

) as f:

    json.dump(

        live_release_manifest_32hg,

        f,

        indent=2,
    )


# ============================================================
# 13. FINAL DISPLAY
# ============================================================

print()
print("=" * 80)
print("PANTHEON v2.2 LIVE — CURRENT TOP 30")
print("=" * 80)


display_cols_32hg = [

    "point_rank",
    "driver",

    "jaws_mean",
    "jaws_89_lb",
    "jaws_89_ub",

    "rank_mean",
    "rank_89_lb",
    "rank_89_ub",

    "peak_score_mean",
    "longevity_score_mean",

    "equivalent_seasons",
    "starts",
]


display(
    live_board_32hg[
        display_cols_32hg
    ]
    .head(
        30
    )
)


print()
print("=" * 80)
print("LIVE RELEASE FILES")
print("=" * 80)


print()
print("Leaderboard:")
print(
    LIVE_PANTHEON_CSV_32HG
)


print()
print("Posterior score/rank draws:")
print(
    LIVE_PANTHEON_DRAWS_32HG
)


print()
print("Release manifest:")
print(
    LIVE_PANTHEON_MANIFEST_32HG
)


print()
print("=" * 80)
print("STAGE 32H-G COMPLETE")
print("=" * 80)


print()
print(
    "Frozen official 1982–2025 Pantheon: UNCHANGED."
)

print(
    "Pantheon v2.2 LIVE snapshot: RELEASED."
)

print(
    "2026 exposure:",
    f"{completed_2026_races_32hf}/"
    f"{scheduled_2026_races_32hf} = "
    f"{SEASON_FRACTION_32HF:.6f}",
)

## Part VII — Final dashboard

The validated Stage 20 dashboard engine has been extracted to `dashboard_base_stage20.py`. The cells below apply the v2.2 LIVE data adapter and the final presentation changes. These remain separated here for provenance; they can later be collapsed into a standalone dashboard builder once the production pipeline is packaged into modules.


In [ ]:
# ============================================================
# STAGE 32H-H — PANTHEON v2.2 LIVE DASHBOARD
# ============================================================
#
# PURPOSE
# -------
# Build the full interactive Pantheon dashboard for the locked
# v2.2 LIVE Round-15 snapshot whilst preserving the frozen
# 1982–2025 dashboard unchanged.
#
# We deliberately REUSE the validated Stage 20 dashboard engine:
#
#   - leaderboard
#   - driver explorer
#   - clickable season context
#   - posterior rank distribution
#   - peak vs longevity
#   - pairwise posterior comparison
#   - 3D teammate network
#   - shortest teammate path
#   - Teammate Chain game
#
# Rather than maintaining a second independent 13k-line HTML/JS
# implementation, this stage:
#
#   1. translates the v2.2 LIVE objects into the legacy Stage 20
#      dashboard data contract;
#   2. locates the existing Stage 20 source locally;
#   3. patches ONLY the product/version/scoring text and a few
#      LIVE-specific display details;
#   4. runs the dashboard builder in an isolated namespace;
#   5. validates the generated HTML.
#
# OUTPUT
# ------
#
#   f1_pantheon_1982_2026_v2_2_LIVE_
#       through_r15_15gp_cf1f82aaab32_15of23gp.html
#
# Frozen v2.1 dashboard: UNCHANGED.
#
# ============================================================

from pathlib import Path

import json
import os
import webbrowser

import numpy as np
import pandas as pd


# ============================================================
# 0. CURRENT LIVE RELEASE LOCK
# ============================================================

LIVE_PRODUCT_VERSION_32HH = "v2.2"

LIVE_COMPLETED_GPS_32HH = 15

LIVE_SCHEDULED_GPS_32HH = 23

LIVE_SEASON_FRACTION_32HH = (
    LIVE_COMPLETED_GPS_32HH
    /
    LIVE_SCHEDULED_GPS_32HH
)


LIVE_DASHBOARD_FILENAME_32HH = (
    "f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp.html"
)


LIVE_DASHBOARD_BUILDER_FILENAME_32HH = (
    "build_f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp.py"
)


# ------------------------------------------------------------
# Normally leave this as None.
#
# If automatic discovery cannot locate the old Stage 20 source,
# set this manually to either:
#
#   Stage20_FINAL_With_Teammate_Chain_Game.py
#
# or the old dashboard notebook containing that cell.
# ------------------------------------------------------------

STAGE20_SOURCE_PATH_32HH = Path("dashboard_base_stage20.py")


# ------------------------------------------------------------
# Browser behaviour
# ------------------------------------------------------------

OPEN_LIVE_DASHBOARD_AFTER_BUILD_32HH = True


# ------------------------------------------------------------
# Refuse silent replacement of a released dashboard.
# ------------------------------------------------------------

ALLOW_LIVE_DASHBOARD_OVERWRITE_32HH = False


# ============================================================
# 1. REQUIRED LIVE OBJECTS
# ============================================================

required_objects_32hh = [

    "live_board_32hg",

    "eligible_drivers_32hg",

    "policy_prorata_32hf",

    "posterior_32hc",

    "idx_all",

    "actual_starts_32hf",

    "raw",

    "CFG",

    "LIVE_SNAPSHOT_TAG_32HB",

    "LIVE_SNAPSHOT_HASH_32HB",
]


missing_objects_32hh = [

    name

    for name
    in required_objects_32hh

    if name not in globals()
]


if missing_objects_32hh:

    raise RuntimeError(
        "Stage 32H-H requires:\n  "
        +
        "\n  ".join(
            missing_objects_32hh
        )
        +
        "\n\nRun Stages 32H-B through 32H-G first."
    )


print("=" * 80)
print("STAGE 32H-H — PANTHEON v2.2 LIVE DASHBOARD")
print("=" * 80)


# ============================================================
# 2. VERIFY RELEASE SNAPSHOT
# ============================================================

if LIVE_SNAPSHOT_TAG_32HB != "through_r15_15gp_cf1f82aaab32":

    raise RuntimeError(
        "Unexpected LIVE snapshot:\n"
        f"{LIVE_SNAPSHOT_TAG_32HB}\n\n"
        "This dashboard release cell is locked to the "
        "Round-15 snapshot."
    )


if not np.isclose(
    policy_prorata_32hf[
        "weight"
    ],
    LIVE_SEASON_FRACTION_32HH,
    atol=1e-12,
    rtol=0.0,
):

    raise RuntimeError(
        "Pro-rata JAWS weight does not equal 15/23."
    )


if len(
    live_board_32hg
) != 98:

    raise RuntimeError(
        "Expected 98 LIVE Pantheon drivers, found "
        f"{len(live_board_32hg)}."
    )


print()
print(
    "Snapshot:",
    LIVE_SNAPSHOT_TAG_32HB,
)

print(
    "LIVE exposure:",
    f"{LIVE_COMPLETED_GPS_32HH}/"
    f"{LIVE_SCHEDULED_GPS_32HH} = "
    f"{LIVE_SEASON_FRACTION_32HH:.6f}",
)

print(
    "Pantheon drivers:",
    len(
        live_board_32hg
    ),
)


# ============================================================
# 3. LIVE POSTERIOR MATRICES
# ============================================================

alpha_post_live_32hh = np.asarray(

    posterior_32hc[
        "alpha_dy"
    ].values,

    dtype=float,
)


alpha_post_live_32hh = (
    alpha_post_live_32hh.reshape(
        -1,
        alpha_post_live_32hh.shape[
            -1
        ],
    )
)


eligible_jaws_live_32hh = np.asarray(

    policy_prorata_32hf[
        "eligible_jaws"
    ],

    dtype=float,
)


rank_live_32hh = np.asarray(

    policy_prorata_32hf[
        "ranks"
    ],

    dtype=int,
)


eligible_drivers_live_32hh = list(
    eligible_drivers_32hg
)


if alpha_post_live_32hh.shape[
    0
] != 4000:

    raise RuntimeError(
        "Expected 4000 alpha posterior draws."
    )


if eligible_jaws_live_32hh.shape != (
    4000,
    98,
):

    raise RuntimeError(
        "Unexpected LIVE JAWS draw dimensions:\n"
        f"{eligible_jaws_live_32hh.shape}"
    )


if rank_live_32hh.shape != (
    4000,
    98,
):

    raise RuntimeError(
        "Unexpected LIVE rank draw dimensions:\n"
        f"{rank_live_32hh.shape}"
    )


# ============================================================
# 4. REBUILD DRIVER-YEAR LOOKUP FOR DASHBOARD
# ============================================================

driver_years_live_32hh = {}


for driver, year in idx_all.active_driver_years:

    driver_years_live_32hh.setdefault(
        driver,
        [],
    ).append(
        int(
            year
        )
    )


for driver in driver_years_live_32hh:

    driver_years_live_32hh[
        driver
    ] = sorted(
        set(
            driver_years_live_32hh[
                driver
            ]
        )
    )


# ============================================================
# 5. TRANSLATE v2.2 BOARD -> STAGE 20 DATA CONTRACT
# ============================================================
#
# Stage 20 expects:
#
#   rank
#   driver
#   jaws_mean
#   jaws_median
#   jaws_lb
#   jaws_ub
#   top5_peak_alpha_mean
#   career_surplus_mean
#   peak_score_mean
#   longevity_score_mean
#   rank_mean
#   rank_median
#   rank_lb
#   rank_ub
#   p_rank_1
#   seasons
#   starts
#   unique_teammates
#
# The meaning of "seasons" in LIVE is now EQUIVALENT seasons,
# because 2026 contributes 15/23 rather than one complete year.
# ============================================================

live_board_stage20_32hh = (
    live_board_32hg
    .copy()
    .sort_values(
        "point_rank"
    )
    .reset_index(
        drop=True
    )
)


eligible_driver_col_32hh = {

    driver: j

    for j, driver
    in enumerate(
        eligible_drivers_live_32hh
    )
}


p_rank_1_32hh = {}


for driver in eligible_drivers_live_32hh:

    j = eligible_driver_col_32hh[
        driver
    ]


    p_rank_1_32hh[
        driver
    ] = float(
        np.mean(
            rank_live_32hh[
                :,
                j,
            ]
            ==
            1
        )
    )


pantheon_live_legacy_32hh = pd.DataFrame(
    {
        "rank":
            live_board_stage20_32hh[
                "point_rank"
            ].astype(int),

        "driver":
            live_board_stage20_32hh[
                "driver"
            ].astype(str),

        "jaws_mean":
            live_board_stage20_32hh[
                "jaws_mean"
            ].astype(float),

        "jaws_median":
            live_board_stage20_32hh[
                "jaws_median"
            ].astype(float),

        "jaws_lb":
            live_board_stage20_32hh[
                "jaws_89_lb"
            ].astype(float),

        "jaws_ub":
            live_board_stage20_32hh[
                "jaws_89_ub"
            ].astype(float),

        "top5_peak_alpha_mean":
            live_board_stage20_32hh[
                "peak_raw_mean"
            ].astype(float),

        "career_surplus_mean":
            live_board_stage20_32hh[
                "surplus_raw_mean"
            ].astype(float),

        "peak_score_mean":
            live_board_stage20_32hh[
                "peak_score_mean"
            ].astype(float),

        "longevity_score_mean":
            live_board_stage20_32hh[
                "longevity_score_mean"
            ].astype(float),

        "rank_mean":
            live_board_stage20_32hh[
                "rank_mean"
            ].astype(float),

        "rank_median":
            live_board_stage20_32hh[
                "rank_median"
            ].astype(float),

        "rank_lb":
            live_board_stage20_32hh[
                "rank_89_lb"
            ].astype(float),

        "rank_ub":
            live_board_stage20_32hh[
                "rank_89_ub"
            ].astype(float),

        "p_rank_1":
            live_board_stage20_32hh[
                "driver"
            ]
            .map(
                p_rank_1_32hh
            )
            .astype(float),

        # LIVE equivalent-season exposure.
        "seasons":
            live_board_stage20_32hh[
                "equivalent_seasons"
            ].astype(float),

        "starts":
            live_board_stage20_32hh[
                "starts"
            ].astype(int),

        "unique_teammates":
            live_board_stage20_32hh[
                "unique_teammates"
            ].astype(int),
    }
)


if pantheon_live_legacy_32hh[
    "p_rank_1"
].isna().any():

    raise RuntimeError(
        "Missing P(rank 1) while translating LIVE board."
    )


if set(
    pantheon_live_legacy_32hh[
        "driver"
    ]
) != set(
    eligible_drivers_live_32hh
):

    raise RuntimeError(
        "Translated leaderboard driver set mismatch."
    )


print()
print(
    "Stage-20-compatible LIVE leaderboard:",
    pantheon_live_legacy_32hh.shape,
)


# ============================================================
# 6. SOURCE DISCOVERY
# ============================================================
#
# We reuse the exact final Stage 20 source.
#
# Automatic discovery searches the working project for:
#
#   .py
#   .txt
#   .ipynb
#
# and extracts the code containing the Stage 20 dashboard
# marker.
# ============================================================

STAGE20_MARKER_32HH = (
    "STAGE 20 — FINAL INTERACTIVE 1982–2025 "
    "PANTHEON DASHBOARD"
)


def extract_stage20_source_32hh(
    path,
):

    path = Path(
        path
    )


    suffix = (
        path.suffix
        .lower()
    )


    if suffix == ".ipynb":

        try:

            notebook = json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:

            return None


        for cell in notebook.get(
            "cells",
            [],
        ):

            if cell.get(
                "cell_type"
            ) != "code":

                continue


            cell_source = cell.get(
                "source",
                "",
            )


            if isinstance(
                cell_source,
                list,
            ):

                cell_source = "".join(
                    cell_source
                )


            if (
                STAGE20_MARKER_32HH
                in cell_source
            ):

                return (
                    cell_source
                    .replace(
                        "\r\n",
                        "\n",
                    )
                )


        return None


    if suffix in {
        ".py",
        ".txt",
    }:

        try:

            source = path.read_text(
                encoding="utf-8"
            )

        except Exception:

            return None


        if (
            STAGE20_MARKER_32HH
            in source
        ):

            return (
                source
                .replace(
                    "\r\n",
                    "\n",
                )
            )


    return None


def discover_stage20_source_32hh():

    # --------------------------------------------------------
    # Explicit override
    # --------------------------------------------------------

    if STAGE20_SOURCE_PATH_32HH is not None:

        explicit = Path(
            STAGE20_SOURCE_PATH_32HH
        )


        if not explicit.exists():

            raise RuntimeError(
                "STAGE20_SOURCE_PATH_32HH does not exist:\n"
                f"{explicit}"
            )


        source = extract_stage20_source_32hh(
            explicit
        )


        if source is None:

            raise RuntimeError(
                "The explicitly supplied source does not "
                "contain the final Stage 20 dashboard cell:\n"
                f"{explicit}"
            )


        return (
            explicit,
            source,
        )


    cwd = Path.cwd()


    # --------------------------------------------------------
    # Preferred exact filenames first
    # --------------------------------------------------------

    preferred_names = [

        "Stage20_FINAL_With_Teammate_Chain_Game.py",

        "Stage20_FINAL_With_Teammate_Chain_Game_copy_paste.txt",

        "F1_Pantheon_1982_2025_FINAL_Dashboard_"
        "With_Teammate_Chain.ipynb",

        "F1_Pantheon_1982_2025_FINAL_Dashboard_"
        "ReadableShortestPath.ipynb",

        "F1_Pantheon_1982_2025_FINAL_Dashboard_"
        "ShortestPath_CleanLabels.ipynb",

        "F1_Pantheon_1982_2025_FINAL_Dashboard_"
        "ShortestPath.ipynb",
    ]


    for preferred_name in preferred_names:

        for candidate in cwd.rglob(
            preferred_name
        ):

            candidate_str = str(
                candidate
            )


            if (
                "model_outputs"
                in candidate_str
            ):

                continue


            source = extract_stage20_source_32hh(
                candidate
            )


            if source is not None:

                return (
                    candidate,
                    source,
                )


    # --------------------------------------------------------
    # Fallback scan.
    #
    # Prune directories which can be very large and cannot
    # contain the desired notebook source.
    # --------------------------------------------------------

    ignored_dirs = {

        ".git",
        ".venv",
        "venv",
        "__pycache__",
        ".ipynb_checkpoints",
        "node_modules",
        "model_outputs",
    }


    for root, dirs, files in os.walk(
        cwd
    ):

        dirs[
            :
        ] = [
            d

            for d
            in dirs

            if d not in ignored_dirs
        ]


        root_path = Path(
            root
        )


        for filename in files:

            suffix = Path(
                filename
            ).suffix.lower()


            if suffix not in {
                ".py",
                ".txt",
                ".ipynb",
            }:

                continue


            candidate = (
                root_path
                /
                filename
            )


            source = extract_stage20_source_32hh(
                candidate
            )


            if source is not None:

                return (
                    candidate,
                    source,
                )


    raise RuntimeError(
        "Could not automatically find the final Stage 20 "
        "dashboard source.\n\n"
        "Set STAGE20_SOURCE_PATH_32HH near the top of this "
        "cell to the old Stage 20 .py/.txt file or dashboard "
        ".ipynb, then rerun Stage 32H-H."
    )


stage20_source_path_32hh, dashboard_source_32hh = (
    discover_stage20_source_32hh()
)


print()
print(
    "Using Stage 20 source:"
)

print(
    stage20_source_path_32hh.resolve()
)


# ============================================================
# 7. SAFE PATCH HELPER
# ============================================================

def replace_once_32hh(
    source,
    old,
    new,
    label,
):

    n = source.count(
        old
    )


    if n != 1:

        raise RuntimeError(
            f"Dashboard source patch '{label}' expected "
            f"exactly one match, found {n}."
        )


    return source.replace(
        old,
        new,
        1,
    )


# ============================================================
# 8. PATCH PRODUCT SETTINGS
# ============================================================

patched_source_32hh = dashboard_source_32hh


# ------------------------------------------------------------
# Change original historical text FIRST.
#
# New LIVE-vs-frozen wording is inserted only afterwards so
# references to the frozen 1982–2025 release remain correct.
# ------------------------------------------------------------

if "1982–2025" not in patched_source_32hh:

    raise RuntimeError(
        "Stage 20 source does not contain expected "
        "1982–2025 text."
    )


patched_source_32hh = (
    patched_source_32hh.replace(
        "1982–2025",
        "1982–2026 LIVE",
    )
)


# ------------------------------------------------------------
# 2026 settings / output path
# ------------------------------------------------------------

patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    "assert END_YEAR == 2025",

    "assert END_YEAR == 2026",

    "END_YEAR assertion",
)


old_dashboard_path_block_32hh = """PANTHEON_DASHBOARD_PATH = OUTPUT_DIR / (
    f"f1_pantheon_" f"{START_YEAR}_{END_YEAR}_" f"FINAL.html"
)"""


new_dashboard_path_block_32hh = """PANTHEON_DASHBOARD_PATH = (
    OUTPUT_DIR
    /
    LIVE_DASHBOARD_FILENAME_32HH
)"""


patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    old_dashboard_path_block_32hh,

    new_dashboard_path_block_32hh,

    "dashboard output path",
)


patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    "OPEN_AFTER_BUILD = True",

    (
        "OPEN_AFTER_BUILD = "
        "OPEN_LIVE_DASHBOARD_AFTER_BUILD_32HH"
    ),

    "browser-open setting",
)


# ============================================================
# 9. PATCH LIVE LEADERBOARD CONTRACT
# ============================================================

patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    """int(
                    row.seasons
                )""",

    """float(
                    row.seasons
                )""",

    "equivalent-season float payload",
)


# ============================================================
# 10. ADD LIVE METADATA TO PAYLOAD
# ============================================================

old_metadata_tail_32hh = """        "modern_floor_year":
            int(
                CFG.modern_floor_year
            ),
"""


new_metadata_tail_32hh = """        "modern_floor_year":
            int(
                CFG.modern_floor_year
            ),

        "product_version":
            LIVE_PRODUCT_VERSION_32HH,

        "live":
            True,

        "snapshot_tag":
            LIVE_SNAPSHOT_TAG_32HB,

        "snapshot_hash":
            LIVE_SNAPSHOT_HASH_32HB,

        "completed_2026_gps":
            LIVE_COMPLETED_GPS_32HH,

        "scheduled_2026_gps":
            LIVE_SCHEDULED_GPS_32HH,

        "season_fraction_2026":
            LIVE_SEASON_FRACTION_32HH,

        "familiarity_model":
            "organisational_memory_20gp_half_life",
"""


patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    old_metadata_tail_32hh,

    new_metadata_tail_32hh,

    "LIVE payload metadata",
)


# ============================================================
# 11. PATCH PAGE TITLE / HEADER
# ============================================================

patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    """<title>
F1 Pantheon — __START_YEAR__–__END_YEAR__
</title>""",

    """<title>
F1 Pantheon v2.2 LIVE — through Round 15
</title>""",

    "HTML title",
)


patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    """<h1>
F1 Pantheon
</h1>""",

    """<h1>
F1 Pantheon v2.2 LIVE
</h1>""",

    "page heading",
)


old_subtitle_32hh = """Bayesian driver–constructor ranking,
__START_YEAR__–__END_YEAR__

&nbsp;•&nbsp;

60% five-season peak

&nbsp;•&nbsp;

40% square-root-damped longevity"""


new_subtitle_32hh = """Bayesian driver–constructor ranking,
__START_YEAR__–__END_YEAR__

&nbsp;•&nbsp;

through Round 15

&nbsp;•&nbsp;

15/23 GPs

&nbsp;•&nbsp;

65.2% season exposure

&nbsp;•&nbsp;

60% five-equivalent-season peak

&nbsp;•&nbsp;

40% square-root-damped longevity"""


patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    old_subtitle_32hh,

    new_subtitle_32hh,

    "LIVE subtitle",
)


# ============================================================
# 12. ADD LIVE SNAPSHOT BANNER
# ============================================================

old_nav_start_32hh = """</div>


<nav class="sticky-nav" aria-label="Dashboard sections">"""


new_nav_start_32hh = """</div>


<div class="warning">

<strong>
LIVE snapshot:
</strong>

2026 driver and constructor states use evidence through Round 15.

JAWS gives the incomplete 2026 season

<strong>
15/23 = 65.2%
</strong>

of one season of peak/longevity exposure.

The frozen Pantheon v2.1 1982–2025 release remains the
immutable historical benchmark.

</div>


<nav class="sticky-nav" aria-label="Dashboard sections">"""


patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    old_nav_start_32hh,

    new_nav_start_32hh,

    "LIVE snapshot banner",
)


# ============================================================
# 13. PATCH SUMMARY CARDS
# ============================================================

old_eligibility_metric_32hh = """        <div class="metric">

            <div class="metric-label">
                Eligibility
            </div>"""


new_eligibility_metric_32hh = """        <div class="metric">

            <div class="metric-label">
                LIVE exposure
            </div>

            <div class="metric-value">
                15 / 23 GPs
            </div>

            <div class="note">
                65.2% of 2026
            </div>

        </div>


        <div class="metric">

            <div class="metric-label">
                Eligibility
            </div>"""


patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    old_eligibility_metric_32hh,

    new_eligibility_metric_32hh,

    "LIVE summary metric",
)


patched_source_32hh = patched_source_32hh.replace(

    "starts / teammates / seasons",

    "starts / teammates / equiv. seasons",
)


# ============================================================
# 14. PATCH DRIVER SUMMARY LANGUAGE
# ============================================================

patched_source_32hh = patched_source_32hh.replace(

    "Five-season peak",

    "Five-equivalent-season peak",
)


patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    "${row.seasons} modelled seasons",

    "${num(row.seasons, 2)} equivalent seasons",

    "career equivalent-season display",
)


# ============================================================
# 15. PATCH NETWORK DESCRIPTION
# ============================================================

patched_source_32hh = patched_source_32hh.replace(

    (
        "Every node is a modelled F1 driver from "
        "1982–2026 LIVE and every"
    ),

    (
        "Every node is a modelled F1 driver from 1982 "
        "through the current 2026 LIVE snapshot and every"
    ),
)


# ============================================================
# 16. PATCH MODEL DESCRIPTION
# ============================================================

patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    (
        "constructor performance and "
        "constructor-tenure context."
    ),

    (
        "constructor performance and organisational-familiarity "
        "context. Organisational familiarity uses exponentially "
        "decaying memory while a driver is away from that "
        "lineage, with a 20-GP missed-weekend half-life."
    ),

    "familiarity description",
)


old_scoring_method_32hh = """The final Pantheon score combines

<strong>60% peak</strong>

— the mean of the driver's best five latent seasons —

with

<strong>40% longevity</strong>

— the square root of cumulative positive career α surplus.

Peak and longevity are independently normalised to 100
within every posterior draw before combination."""


new_scoring_method_32hh = """The LIVE Pantheon score combines

<strong>60% peak</strong>

— mean ability across the best five equivalent season-units —

with

<strong>40% longevity</strong>

— the square root of cumulative positive career α surplus.

Completed seasons have weight 1.

The incomplete 2026 season has weight

<strong>
15/23 = 0.6522
</strong>.

If 2026 enters a driver's peak window, it therefore occupies
only 0.6522 of one of the five season-units, with the next-best
completed season filling the remaining exposure.

Positive 2026 career surplus is weighted by the same 15/23
factor.

Peak and longevity are independently normalised to 100
within every posterior draw before combination."""


patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    old_scoring_method_32hh,

    new_scoring_method_32hh,

    "LIVE JAWS methodology",
)


patched_source_32hh = patched_source_32hh.replace(

    "<strong>5 modelled seasons</strong>.",

    "<strong>5 equivalent modelled seasons</strong>.",
)


# ============================================================
# 17. PATCH HISTORICAL/LIVE CAVEAT
# ============================================================

patched_source_32hh = patched_source_32hh.replace(

    """<strong>
1982–2026 LIVE F1 Pantheon
</strong>,""",

    """<strong>
Pantheon v2.2 LIVE, beginning in 1982
</strong>,""",
)


old_production_note_32hh = """<p class="note">

The final production posterior contains 4,000 draws."""


new_production_note_32hh = """<p class="note">

This LIVE snapshot does not replace the frozen v2.1 1982–2025
release.

New 2026 evidence can revise earlier latent driver states through
the temporal model, and v2.2 also uses the updated organisational-
familiarity specification.

The 2026 latent state itself is inferred from the 15 completed
Grands Prix. Only its JAWS career exposure is pro-rated to 15/23.

</p>


<p class="note">

The final production posterior contains 4,000 draws."""


patched_source_32hh = replace_once_32hh(

    patched_source_32hh,

    old_production_note_32hh,

    new_production_note_32hh,

    "LIVE provenance note",
)


# ============================================================
# 18. PATCH FOOTER / BUILD LABELS
# ============================================================

patched_source_32hh = patched_source_32hh.replace(

    "Final validated joint driver–constructor model.",

    "Pantheon v2.2 LIVE joint driver–constructor model.",
)


patched_source_32hh = patched_source_32hh.replace(

    'print("FINAL PANTHEON DASHBOARD BUILT")',

    'print("PANTHEON v2.2 LIVE DASHBOARD BUILT")',
)


patched_source_32hh = patched_source_32hh.replace(

    'print("\\nStage 20 complete.")',

    'print("\\nStage 32H-H complete.")',
)


# ============================================================
# 19. PATCH-SOURCE SANITY
# ============================================================

required_live_phrases_32hh = [

    "F1 Pantheon v2.2 LIVE",

    "15/23 GPs",

    "65.2% season exposure",

    "five-equivalent-season peak",

    "20-GP missed-weekend half-life",

    "equivalent modelled seasons",

    "Stage 32H-H complete.",
]


missing_live_phrases_32hh = [

    phrase

    for phrase
    in required_live_phrases_32hh

    if phrase not in patched_source_32hh
]


if missing_live_phrases_32hh:

    raise RuntimeError(
        "Patched dashboard source is missing:\n  "
        +
        "\n  ".join(
            missing_live_phrases_32hh
        )
    )


if "assert END_YEAR == 2025" in patched_source_32hh:

    raise RuntimeError(
        "Old 2025 END_YEAR assertion survived patching."
    )


# ============================================================
# 20. OUTPUT PATHS / OVERWRITE GUARD
# ============================================================

dashboard_output_dir_32hh = Path(
    CFG.output_dir
)


dashboard_output_dir_32hh.mkdir(
    parents=True,
    exist_ok=True,
)


expected_dashboard_path_32hh = (
    dashboard_output_dir_32hh
    /
    LIVE_DASHBOARD_FILENAME_32HH
)


dashboard_builder_path_32hh = (
    dashboard_output_dir_32hh
    /
    LIVE_DASHBOARD_BUILDER_FILENAME_32HH
)


if not ALLOW_LIVE_DASHBOARD_OVERWRITE_32HH:

    existing_32hh = [

        path

        for path
        in [
            expected_dashboard_path_32hh,
            dashboard_builder_path_32hh,
        ]

        if path.exists()
    ]


    if existing_32hh:

        raise RuntimeError(
            "Refusing to overwrite existing LIVE dashboard "
            "release files:\n  "
            +
            "\n  ".join(
                str(
                    path
                )
                for path
                in existing_32hh
            )
            +
            "\n\nSet "
            "ALLOW_LIVE_DASHBOARD_OVERWRITE_32HH = True "
            "only if you intentionally want to rebuild this "
            "same snapshot."
        )


# ============================================================
# 21. SAVE THE PATCHED BUILDER FOR PROVENANCE
# ============================================================

dashboard_builder_path_32hh.write_text(

    patched_source_32hh,

    encoding="utf-8",
)


print()
print(
    "Saved patched dashboard builder ->"
)

print(
    dashboard_builder_path_32hh
)


# ============================================================
# 22. BUILD ISOLATED DASHBOARD NAMESPACE
# ============================================================
#
# The old dashboard source expects the original Stage 20 names.
#
# We provide those names ONLY inside this isolated execution
# namespace. The current LIVE notebook objects are untouched.
# ============================================================

dashboard_namespace_32hh = dict(
    globals()
)


dashboard_namespace_32hh.update(
    {
        # ----------------------------------------------------
        # Original Stage 20 contract
        # ----------------------------------------------------

        "pantheon_2025":
            pantheon_live_legacy_32hh,

        "eligible_drivers":
            eligible_drivers_live_32hh,

        "eligible_jaws_mat":
            eligible_jaws_live_32hh,

        "rank_mat":
            rank_live_32hh,

        "alpha_post":
            alpha_post_live_32hh,

        "driver_years":
            driver_years_live_32hh,

        "actual_starts":
            actual_starts_32hf,


        # ----------------------------------------------------
        # Force Stage 20 to use the LIVE constructor posterior
        # rather than looking for the frozen trace.
        # ----------------------------------------------------

        "posterior_final":
            posterior_32hc,


        # ----------------------------------------------------
        # LIVE dashboard configuration consumed by our patch.
        # ----------------------------------------------------

        "LIVE_DASHBOARD_FILENAME_32HH":
            LIVE_DASHBOARD_FILENAME_32HH,

        "LIVE_PRODUCT_VERSION_32HH":
            LIVE_PRODUCT_VERSION_32HH,

        "LIVE_COMPLETED_GPS_32HH":
            LIVE_COMPLETED_GPS_32HH,

        "LIVE_SCHEDULED_GPS_32HH":
            LIVE_SCHEDULED_GPS_32HH,

        "LIVE_SEASON_FRACTION_32HH":
            LIVE_SEASON_FRACTION_32HH,

        "LIVE_SNAPSHOT_TAG_32HB":
            LIVE_SNAPSHOT_TAG_32HB,

        "LIVE_SNAPSHOT_HASH_32HB":
            LIVE_SNAPSHOT_HASH_32HB,

        "OPEN_LIVE_DASHBOARD_AFTER_BUILD_32HH":
            OPEN_LIVE_DASHBOARD_AFTER_BUILD_32HH,
    }
)


# ============================================================
# 23. EXECUTE THE VALIDATED DASHBOARD ENGINE
# ============================================================

print()
print("=" * 80)
print("BUILDING PANTHEON v2.2 LIVE DASHBOARD")
print("=" * 80)


compiled_dashboard_32hh = compile(

    patched_source_32hh,

    str(
        dashboard_builder_path_32hh
    ),

    "exec",
)


exec(

    compiled_dashboard_32hh,

    dashboard_namespace_32hh,

    dashboard_namespace_32hh,
)


# ============================================================
# 24. RETRIEVE GENERATED PATH
# ============================================================

if (
    "PANTHEON_DASHBOARD_PATH"
    not in dashboard_namespace_32hh
):

    raise RuntimeError(
        "Dashboard builder did not expose "
        "PANTHEON_DASHBOARD_PATH."
    )


LIVE_DASHBOARD_PATH_32HH = Path(

    dashboard_namespace_32hh[
        "PANTHEON_DASHBOARD_PATH"
    ]
)


if not LIVE_DASHBOARD_PATH_32HH.exists():

    raise RuntimeError(
        "LIVE dashboard file was not created:\n"
        f"{LIVE_DASHBOARD_PATH_32HH}"
    )


# ============================================================
# 25. GENERATED HTML VALIDATION
# ============================================================

live_html_32hh = (
    LIVE_DASHBOARD_PATH_32HH
    .read_text(
        encoding="utf-8"
    )
)


html_checks_32hh = {

    "v2.2 LIVE title":
        (
            "F1 Pantheon v2.2 LIVE"
            in live_html_32hh
        ),

    "Round 15":
        (
            "through Round 15"
            in live_html_32hh
        ),

    "15/23 exposure":
        (
            "15/23"
            in live_html_32hh
        ),

    "65.2% exposure":
        (
            "65.2%"
            in live_html_32hh
        ),

    "20-GP familiarity":
        (
            "20-GP missed-weekend half-life"
            in live_html_32hh
        ),

    "equivalent-season peak":
        (
            "five equivalent season-units"
            in live_html_32hh
        ),

    "teammate network":
        (
            'id="teammate-network"'
            in live_html_32hh
        ),

    "Teammate Chain":
        (
            "Teammate Chain"
            in live_html_32hh
        ),

    "payload inserted":
        (
            "__PANTHEON_DATA__"
            not in live_html_32hh
        ),

    "start-year placeholder replaced":
        (
            "__START_YEAR__"
            not in live_html_32hh
        ),

    "end-year placeholder replaced":
        (
            "__END_YEAR__"
            not in live_html_32hh
        ),
}


html_check_table_32hh = pd.DataFrame(
    [
        {
            "check":
                key,

            "passed":
                value,
        }

        for key, value
        in html_checks_32hh.items()
    ]
)


print()
print("=" * 80)
print("LIVE DASHBOARD VALIDATION")
print("=" * 80)


display(
    html_check_table_32hh
)


if not all(
    html_checks_32hh.values()
):

    failed_checks_32hh = [

        key

        for key, value
        in html_checks_32hh.items()

        if not value
    ]


    raise RuntimeError(
        "LIVE dashboard HTML failed validation:\n  "
        +
        "\n  ".join(
            failed_checks_32hh
        )
    )


# ============================================================
# 26. DATA PAYLOAD SANITY
# ============================================================

dashboard_payload_32hh = (
    dashboard_namespace_32hh[
        "payload"
    ]
)


if dashboard_payload_32hh[
    "metadata"
][
    "n_drivers"
] != 98:

    raise RuntimeError(
        "Dashboard payload does not contain 98 drivers."
    )


if dashboard_payload_32hh[
    "metadata"
][
    "n_draws"
] != 4000:

    raise RuntimeError(
        "Dashboard payload does not contain 4000 draws."
    )


if dashboard_payload_32hh[
    "metadata"
][
    "end_year"
] != 2026:

    raise RuntimeError(
        "Dashboard payload end year is not 2026."
    )


if not np.isclose(

    dashboard_payload_32hh[
        "metadata"
    ][
        "season_fraction_2026"
    ],

    LIVE_SEASON_FRACTION_32HH,

):

    raise RuntimeError(
        "Dashboard payload season fraction mismatch."
    )


# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("STAGE 32H-H COMPLETE")
print("=" * 80)


print()
print(
    "Pantheon v2.2 LIVE dashboard:"
)

print(
    LIVE_DASHBOARD_PATH_32HH.resolve()
)


print()
print(
    "Reusable patched builder:"
)

print(
    dashboard_builder_path_32hh.resolve()
)


print()
print(
    "Source dashboard:"
)

print(
    stage20_source_path_32hh.resolve()
)


print()
print(
    "Drivers:",
    dashboard_payload_32hh[
        "metadata"
    ][
        "n_drivers"
    ],
)


print(
    "Posterior draws:",
    dashboard_payload_32hh[
        "metadata"
    ][
        "n_draws"
    ],
)


print(
    "Coverage:",
    (
        f"{dashboard_payload_32hh['metadata']['start_year']}"
        "–"
        f"{dashboard_payload_32hh['metadata']['end_year']}"
    ),
)


print(
    "2026 exposure:",
    (
        f"{LIVE_COMPLETED_GPS_32HH}/"
        f"{LIVE_SCHEDULED_GPS_32HH} = "
        f"{100 * LIVE_SEASON_FRACTION_32HH:.1f}%"
    ),
)


print()
print(
    "Frozen v2.1 dashboard: UNCHANGED."
)

print(
    "v2.2 LIVE dashboard: BUILT AND VALIDATED."
)

In [ ]:
# ============================================================
# STAGE 32H-I — ADD CURRENT 2026 DRIVER RANKINGS
# ============================================================
#
# PURPOSE
# -------
# Add a dedicated "Current 2026 driver rankings" section to
# the Pantheon v2.2 LIVE dashboard.
#
# This ranking is:
#
#   - based on posterior alpha_{driver, 2026}
#   - restricted to the 23 modelled 2026 driver-season states
#   - ranked separately in every posterior draw
#   - NOT championship standings
#   - NOT JAWS
#
# Display:
#
#   rank by posterior mean alpha
#   driver
#   current team
#   posterior mean alpha
#   89% alpha interval
#   expected 2026 rank
#   89% rank interval
#   P(top 5)
#   P(alpha > 0)
#
# Clicking a row opens that driver's 2026 season in the
# existing Driver Explorer.
#
# The original v2.2 LIVE dashboard remains untouched.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 0. REQUIRE STAGE 32H-H
# ============================================================

required_32hi = [

    "dashboard_builder_path_32hh",
    "dashboard_namespace_32hh",

    "posterior_32hc",
    "idx_all",

    "raw",
    "CFG",
]


missing_32hi = [

    name

    for name
    in required_32hi

    if name not in globals()
]


if missing_32hi:

    raise RuntimeError(
        "Stage 32H-I requires:\n  "
        +
        "\n  ".join(
            missing_32hi
        )
        +
        "\n\nRun Stage 32H-H first."
    )


print("=" * 80)
print("STAGE 32H-I — CURRENT 2026 DRIVER RANKINGS")
print("=" * 80)


# ============================================================
# 1. READ THE VALIDATED v2.2 LIVE BUILDER
# ============================================================

base_builder_path_32hi = Path(
    dashboard_builder_path_32hh
)


if not base_builder_path_32hi.exists():

    raise RuntimeError(
        "The reusable Stage 32H-H dashboard builder "
        "does not exist:\n"
        f"{base_builder_path_32hi}"
    )


dashboard_source_32hi = (
    base_builder_path_32hi
    .read_text(
        encoding="utf-8"
    )
)


print()
print(
    "Base LIVE builder:"
)

print(
    base_builder_path_32hi.resolve()
)


# ============================================================
# 2. EXTRACT 2026 DRIVER POSTERIOR
# ============================================================

alpha_post_32hi = np.asarray(

    posterior_32hc[
        "alpha_dy"
    ].values,

    dtype=float,
)


alpha_post_32hi = alpha_post_32hi.reshape(
    -1,
    alpha_post_32hi.shape[-1],
)


driver_states_32hi = [

    (
        str(driver),
        int(year),
    )

    for driver, year
    in idx_all.active_driver_years
]


if alpha_post_32hi.shape[1] != len(
    driver_states_32hi
):

    raise RuntimeError(
        "alpha_dy state count does not match "
        "idx_all.active_driver_years."
    )


driver_2026_states_32hi = [

    (
        i,
        driver,
    )

    for i, (
        driver,
        year,
    )
    in enumerate(
        driver_states_32hi
    )

    if year == 2026
]


if len(
    driver_2026_states_32hi
) != 23:

    raise RuntimeError(
        "Expected 23 driver-season states in 2026, "
        f"found {len(driver_2026_states_32hi)}."
    )


driver_2026_indices_32hi = np.array(

    [
        i
        for i, driver
        in driver_2026_states_32hi
    ],

    dtype=int,
)


driver_2026_ids_32hi = [

    driver

    for i, driver
    in driver_2026_states_32hi
]


alpha_2026_32hi = alpha_post_32hi[
    :,
    driver_2026_indices_32hi,
]


print()
print(
    "2026 drivers:",
    len(
        driver_2026_ids_32hi
    ),
)

print(
    "Posterior draws:",
    alpha_2026_32hi.shape[0],
)


# ============================================================
# 3. POSTERIOR 2026 RANKS
# ============================================================
#
# Rank 1 = highest alpha within the 2026 field.
# ============================================================

order_2026_32hi = np.argsort(

    -alpha_2026_32hi,

    axis=1,
)


rank_2026_32hi = np.empty_like(

    order_2026_32hi,

    dtype=int,
)


rank_2026_32hi[

    np.arange(
        alpha_2026_32hi.shape[0]
    )[
        :,
        None,
    ],

    order_2026_32hi,

] = np.arange(

    1,
    alpha_2026_32hi.shape[1] + 1,

)[
    None,
    :
]


# ============================================================
# 4. DISPLAY-NAME LOOKUP
# ============================================================

drivers_meta_32hi = (
    raw.drivers.copy()
)


if (
    "driverRef"
    not in drivers_meta_32hi.columns
):

    raise RuntimeError(
        "raw.drivers has no driverRef column."
    )


if (
    "forename"
    in drivers_meta_32hi.columns
    and
    "surname"
    in drivers_meta_32hi.columns
):

    display_name_lookup_32hi = {

        str(row.driverRef):
            (
                f"{row.forename} "
                f"{row.surname}"
            ).strip()

        for row
        in drivers_meta_32hi.itertuples(
            index=False
        )
    }


else:

    display_name_lookup_32hi = {}


def display_name_32hi(
    driver,
):

    return display_name_lookup_32hi.get(

        str(
            driver
        ),

        str(
            driver
        )
        .replace(
            "_",
            " ",
        )
        .title(),
    )


# ============================================================
# 5. CURRENT TEAM LOOKUP
# ============================================================
#
# "Current team" = constructor in the driver's latest
# completed 2026 GP appearance.
#
# This handles drivers who changed constructor during 2026.
# ============================================================

results_32hi = (
    raw.results.copy()
)


races_32hi = (
    raw.races.copy()
)


race_cols_32hi = [

    "raceId",
    "year",
    "round",
]


missing_race_cols_32hi = [

    col

    for col
    in race_cols_32hi

    if col not in races_32hi.columns
]


if missing_race_cols_32hi:

    raise RuntimeError(
        "raw.races missing columns: "
        +
        str(
            missing_race_cols_32hi
        )
    )


result_context_32hi = results_32hi.merge(

    races_32hi[
        race_cols_32hi
    ],

    on="raceId",
    how="left",
    validate="many_to_one",
)


driver_ref_by_id_32hi = dict(
    zip(
        raw.drivers[
            "driverId"
        ],
        raw.drivers[
            "driverRef"
        ].astype(str),
    )
)


result_context_32hi[
    "driver"
] = (
    result_context_32hi[
        "driverId"
    ]
    .map(
        driver_ref_by_id_32hi
    )
)


constructor_name_lookup_32hi = {}


if (
    hasattr(
        raw,
        "constructors",
    )
    and
    "constructorId"
    in raw.constructors.columns
):

    if (
        "name"
        in raw.constructors.columns
    ):

        constructor_name_lookup_32hi = dict(
            zip(
                raw.constructors[
                    "constructorId"
                ],
                raw.constructors[
                    "name"
                ],
            )
        )


if not constructor_name_lookup_32hi:

    if hasattr(
        raw,
        "constructor_ref",
    ):

        constructor_name_lookup_32hi = dict(
            raw.constructor_ref
        )


result_2026_32hi = (
    result_context_32hi.loc[
        result_context_32hi[
            "year"
        ].eq(
            2026
        )
    ]
    .copy()
)


result_2026_32hi[
    "round"
] = pd.to_numeric(

    result_2026_32hi[
        "round"
    ],

    errors="coerce",
)


latest_team_rows_32hi = (

    result_2026_32hi
    .sort_values(
        [
            "driver",
            "round",
            "raceId",
        ]
    )
    .groupby(
        "driver",
        as_index=False,
    )
    .tail(
        1
    )
)


current_constructor_lookup_32hi = dict(
    zip(
        latest_team_rows_32hi[
            "driver"
        ],
        latest_team_rows_32hi[
            "constructorId"
        ],
    )
)


current_team_lookup_32hi = {

    driver:
        constructor_name_lookup_32hi.get(
            constructor_id,
            str(
                constructor_id
            ),
        )

    for driver, constructor_id
    in current_constructor_lookup_32hi.items()
}


# ============================================================
# 6. BUILD CURRENT-RANKING TABLE
# ============================================================

current_ranking_rows_32hi = []


for j, driver in enumerate(
    driver_2026_ids_32hi
):

    alpha_draws = alpha_2026_32hi[
        :,
        j,
    ]


    rank_draws = rank_2026_32hi[
        :,
        j,
    ]


    current_ranking_rows_32hi.append(
        {
            "driver":
                driver,

            "display_name":
                display_name_32hi(
                    driver
                ),

            "team":
                current_team_lookup_32hi.get(
                    driver,
                    "—",
                ),

            "alpha_mean":
                float(
                    np.mean(
                        alpha_draws
                    )
                ),

            "alpha_median":
                float(
                    np.median(
                        alpha_draws
                    )
                ),

            "alpha_lb":
                float(
                    np.quantile(
                        alpha_draws,
                        0.055,
                    )
                ),

            "alpha_ub":
                float(
                    np.quantile(
                        alpha_draws,
                        0.945,
                    )
                ),

            "rank_mean":
                float(
                    np.mean(
                        rank_draws
                    )
                ),

            "rank_median":
                float(
                    np.median(
                        rank_draws
                    )
                ),

            "rank_lb":
                int(
                    np.quantile(
                        rank_draws,
                        0.055,
                        method="nearest",
                    )
                ),

            "rank_ub":
                int(
                    np.quantile(
                        rank_draws,
                        0.945,
                        method="nearest",
                    )
                ),

            "p_top5":
                float(
                    np.mean(
                        rank_draws
                        <=
                        5
                    )
                ),

            "p_alpha_gt_zero":
                float(
                    np.mean(
                        alpha_draws
                        >
                        0.0
                    )
                ),
        }
    )


current_rankings_32hi = (

    pd.DataFrame(
        current_ranking_rows_32hi
    )
    .sort_values(
        "alpha_mean",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


current_rankings_32hi.insert(

    0,

    "rank",

    np.arange(
        1,
        len(
            current_rankings_32hi
        )
        +
        1,
    ),
)


print()
print("=" * 80)
print("CURRENT 2026 DRIVER RANKINGS")
print("=" * 80)


display(
    current_rankings_32hi
)


# ============================================================
# 7. SERIALISABLE PAYLOAD
# ============================================================

current_rankings_payload_32hi = [

    {
        "rank":
            int(
                row.rank
            ),

        "driver":
            str(
                row.driver
            ),

        "display_name":
            str(
                row.display_name
            ),

        "team":
            str(
                row.team
            ),

        "alpha_mean":
            float(
                row.alpha_mean
            ),

        "alpha_median":
            float(
                row.alpha_median
            ),

        "alpha_lb":
            float(
                row.alpha_lb
            ),

        "alpha_ub":
            float(
                row.alpha_ub
            ),

        "rank_mean":
            float(
                row.rank_mean
            ),

        "rank_median":
            float(
                row.rank_median
            ),

        "rank_lb":
            int(
                row.rank_lb
            ),

        "rank_ub":
            int(
                row.rank_ub
            ),

        "p_top5":
            float(
                row.p_top5
            ),

        "p_alpha_gt_zero":
            float(
                row.p_alpha_gt_zero
            ),
    }

    for row
    in current_rankings_32hi.itertuples(
        index=False
    )
]


if len(
    current_rankings_payload_32hi
) != 23:

    raise RuntimeError(
        "Current-ranking payload does not contain 23 drivers."
    )


# ============================================================
# 8. SAFE PATCH HELPER
# ============================================================

def replace_once_32hi(
    source,
    old,
    new,
    label,
):

    count = source.count(
        old
    )


    if count != 1:

        raise RuntimeError(
            f"Patch '{label}' expected exactly one match, "
            f"found {count}."
        )


    return source.replace(
        old,
        new,
        1,
    )


patched_dashboard_32hi = dashboard_source_32hi


# ============================================================
# 9. ADD PAYLOAD FIELD
# ============================================================

old_payload_32hi = """    "leaderboard":
        leaderboard_payload,"""


new_payload_32hi = """    "leaderboard":
        leaderboard_payload,

    "current_rankings":
        current_rankings_payload_32hi,"""


patched_dashboard_32hi = replace_once_32hi(

    patched_dashboard_32hi,

    old_payload_32hi,

    new_payload_32hi,

    "current ranking payload",
)


# ============================================================
# 10. ADD NAVIGATION LINK
# ============================================================

patched_dashboard_32hi = replace_once_32hi(

    patched_dashboard_32hi,

    """<a href="#leaderboard-section">Leaderboard</a>""",

    """<a href="#current-rankings-section">2026 rankings</a>
<a href="#leaderboard-section">Pantheon</a>""",

    "current rankings navigation",
)


# ============================================================
# 11. INSERT CURRENT-RANKING HTML SECTION
# ============================================================

leaderboard_marker_32hi = """<!-- ======================================================
     LEADERBOARD
======================================================= -->"""


current_section_html_32hi = r"""
<!-- ======================================================
     CURRENT 2026 DRIVER RANKINGS
======================================================= -->

<div id="current-rankings-section" class="section">

<h2>
Current 2026 driver rankings
</h2>


<p class="note">

Ranking of the 23 current driver-season states by posterior
mean latent ability

<strong>α<sub>2026</sub></strong>

using evidence through Round 15.

This is a ranking of modelled 2026 realised driver ability,
not championship points and not the career Pantheon score.

The expected-rank and rank-interval columns are calculated
within the 23-driver 2026 field independently in every
posterior draw.

Because the model separates driver and constructor effects,
absolute driver levels carry appreciable driver–car allocation
uncertainty. Direct teammate differences are generally more
tightly identified.

Click any driver to open their 2026 season in the
Driver Explorer.

</p>


<div class="table-wrap">

<table>

<thead>

<tr>

<th>
Rank
</th>

<th>
Driver
</th>

<th>
Current team
</th>

<th>
2026 α
</th>

<th>
89% α interval
</th>

<th>
Expected rank
</th>

<th>
89% rank interval
</th>

<th>
P(top 5)
</th>

<th>
P(α &gt; 0)
</th>

</tr>

</thead>


<tbody id="current-rankings-body">
</tbody>

</table>

</div>

</div>


"""


patched_dashboard_32hi = replace_once_32hi(

    patched_dashboard_32hi,

    leaderboard_marker_32hi,

    (
        current_section_html_32hi
        +
        leaderboard_marker_32hi
    ),

    "current rankings HTML section",
)


# ============================================================
# 12. ADD JS DATA HANDLE
# ============================================================

old_table_data_32hi = """const tableData =
    DATA.leaderboard;"""


new_table_data_32hi = """const tableData =
    DATA.leaderboard;


const currentRankings =
    DATA.current_rankings || [];"""


patched_dashboard_32hi = replace_once_32hi(

    patched_dashboard_32hi,

    old_table_data_32hi,

    new_table_data_32hi,

    "current rankings JS data",
)


# ============================================================
# 13. ADD CURRENT-RANKING RENDER FUNCTION
# ============================================================

render_current_rankings_js_32hi = r"""
function renderCurrentRankings() {

    const tbody =
        document.getElementById(
            "current-rankings-body"
        );


    if (!tbody) {

        return;

    }


    const fmt = (
        value,
        digits = 3
    ) => {

        if (
            value === null
            ||
            value === undefined
            ||
            Number.isNaN(
                Number(
                    value
                )
            )
        ) {

            return "—";

        }


        return Number(
            value
        ).toFixed(
            digits
        );

    };


    const pct = value => {

        if (
            value === null
            ||
            value === undefined
        ) {

            return "—";

        }


        return (
            100
            *
            Number(
                value
            )
        ).toFixed(
            1
        )
        +
        "%";

    };


    tbody.innerHTML =
        currentRankings.map(

            row => `

            <tr
                class="current-ranking-row"
                data-driver="${row.driver}"
                title="Open ${row.display_name}'s 2026 season"
                style="cursor: pointer;"
            >

                <td>
                    #${row.rank}
                </td>

                <td class="driver-name">
                    ${row.display_name}
                </td>

                <td>
                    ${row.team}
                </td>

                <td>
                    ${fmt(row.alpha_mean, 3)}
                </td>

                <td>
                    ${fmt(row.alpha_lb, 3)}
                    –
                    ${fmt(row.alpha_ub, 3)}
                </td>

                <td>
                    #${fmt(row.rank_mean, 1)}
                </td>

                <td>
                    #${row.rank_lb}
                    –
                    #${row.rank_ub}
                </td>

                <td>
                    ${pct(row.p_top5)}
                </td>

                <td>
                    ${pct(row.p_alpha_gt_zero)}
                </td>

            </tr>

            `

        ).join(
            ""
        );


    tbody.querySelectorAll(
        ".current-ranking-row"
    )
    .forEach(

        tr => {

            tr.addEventListener(

                "click",

                () => {

                    const driver =
                        tr.dataset.driver;


                    const driverSelect =
                        document.getElementById(
                            "driver-select"
                        );


                    if (driverSelect) {

                        driverSelect.value =
                            driver;

                    }


                    renderDriver(
                        driver,
                        2026
                    );


                    const explorer =
                        document.getElementById(
                            "driver-explorer-section"
                        );


                    if (explorer) {

                        explorer.scrollIntoView(
                            {
                                behavior:
                                    "smooth",

                                block:
                                    "start",
                            }
                        );

                    }

                }

            );

        }

    );

}


"""


patched_dashboard_32hi = replace_once_32hi(

    patched_dashboard_32hi,

    """function renderLeaderboard() {""",

    (
        render_current_rankings_js_32hi
        +
        "\nfunction renderLeaderboard() {"
    ),

    "current rankings render function",
)


# ============================================================
# 14. INITIALISE CURRENT-RANKING TABLE
# ============================================================

patched_dashboard_32hi = replace_once_32hi(

    patched_dashboard_32hi,

    """    renderLeaderboard();""",

    """    renderCurrentRankings();


    renderLeaderboard();""",

    "current rankings initial render",
)


# ============================================================
# 15. NEW OUTPUT FILENAMES
# ============================================================

LIVE_DASHBOARD_FILENAME_32HI = (
    "f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "with_current_driver_rankings.html"
)


LIVE_DASHBOARD_BUILDER_FILENAME_32HI = (
    "build_f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "with_current_driver_rankings.py"
)


output_dir_32hi = Path(
    CFG.output_dir
)


output_dir_32hi.mkdir(
    parents=True,
    exist_ok=True,
)


LIVE_DASHBOARD_PATH_32HI = (
    output_dir_32hi
    /
    LIVE_DASHBOARD_FILENAME_32HI
)


LIVE_DASHBOARD_BUILDER_PATH_32HI = (
    output_dir_32hi
    /
    LIVE_DASHBOARD_BUILDER_FILENAME_32HI
)


# ============================================================
# 16. REFUSE SILENT OVERWRITE
# ============================================================

existing_32hi = [

    path

    for path
    in [
        LIVE_DASHBOARD_PATH_32HI,
        LIVE_DASHBOARD_BUILDER_PATH_32HI,
    ]

    if path.exists()
]


if existing_32hi:

    raise RuntimeError(
        "Stage 32H-I output already exists:\n  "
        +
        "\n  ".join(
            str(
                path
            )
            for path
            in existing_32hi
        )
        +
        "\n\nDelete/rename it explicitly if you want to "
        "rebuild this exact snapshot."
    )


# ============================================================
# 17. SAVE PATCHED BUILDER
# ============================================================

LIVE_DASHBOARD_BUILDER_PATH_32HI.write_text(

    patched_dashboard_32hi,

    encoding="utf-8",
)


print()
print(
    "Saved extended LIVE builder ->"
)

print(
    LIVE_DASHBOARD_BUILDER_PATH_32HI
)


# ============================================================
# 18. BUILD EXECUTION NAMESPACE
# ============================================================
#
# Start from the working Stage 32H-H namespace so all of the
# original Stage 20 compatibility aliases remain available.
# ============================================================

dashboard_namespace_32hi = dict(
    dashboard_namespace_32hh
)


dashboard_namespace_32hi.update(
    {
        # New payload
        "current_rankings_payload_32hi":
            current_rankings_payload_32hi,

        # New output filename
        "LIVE_DASHBOARD_FILENAME_32HH":
            LIVE_DASHBOARD_FILENAME_32HI,

        # Do not let the builder choose a different name
        "LIVE_DASHBOARD_FILENAME_32HI":
            LIVE_DASHBOARD_FILENAME_32HI,

        # Open after successful build
        "OPEN_LIVE_DASHBOARD_AFTER_BUILD_32HH":
            True,
    }
)


# ============================================================
# 19. EXECUTE EXTENDED DASHBOARD BUILDER
# ============================================================

print()
print("=" * 80)
print("BUILDING LIVE DASHBOARD WITH CURRENT RANKINGS")
print("=" * 80)


compiled_32hi = compile(

    patched_dashboard_32hi,

    str(
        LIVE_DASHBOARD_BUILDER_PATH_32HI
    ),

    "exec",
)


exec(

    compiled_32hi,

    dashboard_namespace_32hi,

    dashboard_namespace_32hi,
)


# ============================================================
# 20. FIND GENERATED FILE
# ============================================================

generated_path_32hi = Path(

    dashboard_namespace_32hi[
        "PANTHEON_DASHBOARD_PATH"
    ]
)


if not generated_path_32hi.exists():

    raise RuntimeError(
        "Extended dashboard was not generated:\n"
        f"{generated_path_32hi}"
    )


if (
    generated_path_32hi.resolve()
    !=
    LIVE_DASHBOARD_PATH_32HI.resolve()
):

    raise RuntimeError(
        "Dashboard builder wrote to an unexpected path:\n"
        f"{generated_path_32hi}"
    )


# ============================================================
# 21. VALIDATE GENERATED HTML
# ============================================================

html_32hi = (
    generated_path_32hi
    .read_text(
        encoding="utf-8"
    )
)


html_checks_32hi = {

    "current rankings section":
        (
            'id="current-rankings-section"'
            in html_32hi
        ),

    "current rankings table":
        (
            'id="current-rankings-body"'
            in html_32hi
        ),

    "navigation link":
        (
            '#current-rankings-section'
            in html_32hi
        ),

    "JS ranking payload":
        (
            "DATA.current_rankings"
            in html_32hi
        ),

    "render function":
        (
            "function renderCurrentRankings()"
            in html_32hi
        ),

    "driver explorer click-through":
        (
            "renderDriver("
            in html_32hi
        ),

    "LIVE title":
        (
            "F1 Pantheon v2.2 LIVE"
            in html_32hi
        ),

    "15/23 exposure":
        (
            "15/23"
            in html_32hi
        ),

    "Teammate Chain preserved":
        (
            "Teammate Chain"
            in html_32hi
        ),
}


validation_32hi = pd.DataFrame(
    [
        {
            "check":
                key,

            "passed":
                value,
        }

        for key, value
        in html_checks_32hi.items()
    ]
)


print()
print("=" * 80)
print("STAGE 32H-I HTML VALIDATION")
print("=" * 80)


display(
    validation_32hi
)


if not all(
    html_checks_32hi.values()
):

    failed_32hi = [

        key

        for key, value
        in html_checks_32hi.items()

        if not value
    ]


    raise RuntimeError(
        "Current-ranking dashboard failed validation:\n  "
        +
        "\n  ".join(
            failed_32hi
        )
    )


# ============================================================
# 22. PAYLOAD VALIDATION
# ============================================================

payload_32hi = (
    dashboard_namespace_32hi[
        "payload"
    ]
)


if (
    "current_rankings"
    not in payload_32hi
):

    raise RuntimeError(
        "Generated payload has no current_rankings field."
    )


if len(
    payload_32hi[
        "current_rankings"
    ]
) != 23:

    raise RuntimeError(
        "Generated payload does not contain 23 current "
        "driver rankings."
    )


payload_ranks_32hi = [

    row[
        "rank"
    ]

    for row
    in payload_32hi[
        "current_rankings"
    ]
]


if payload_ranks_32hi != list(
    range(
        1,
        24,
    )
):

    raise RuntimeError(
        "Current-driver point ranks are not exactly 1–23."
    )


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("STAGE 32H-I COMPLETE")
print("=" * 80)


print()
print(
    "LIVE dashboard with current 2026 rankings:"
)

print(
    generated_path_32hi.resolve()
)


print()
print(
    "Reusable extended builder:"
)

print(
    LIVE_DASHBOARD_BUILDER_PATH_32HI.resolve()
)


print()
print(
    "Current drivers:",
    len(
        current_rankings_payload_32hi
    ),
)


print(
    "Posterior draws:",
    alpha_2026_32hi.shape[0],
)


print()
print(
    "Current-driver ranking definition:"
)

print(
    "  posterior mean 2026 latent alpha"
)

print(
    "  rank uncertainty propagated draw-by-draw"
)

print(
    "  89% alpha and rank intervals"
)

print(
    "  current team = latest constructor through Round 15"
)

print()
print(
    "Pantheon leaderboard: PRESERVED."
)

print(
    "Driver Explorer: PRESERVED."
)

print(
    "3D teammate network: PRESERVED."
)

print(
    "Teammate Chain: PRESERVED."
)

In [ ]:
# ============================================================
# STAGE 32H-J — ADD 2026 DRIVER / TEAM RANKING TABS
# ============================================================
#
# PURPOSE
# -------
# Extend the existing "Current 2026 driver rankings" section
# into a two-tab current-strength section:
#
#       DRIVERS | TEAMS
#
# DRIVERS
# -------
# Posterior alpha_{driver, 2026}
#
# This is the inferred 2026 season driver ability using ALL
# model evidence available through Round 15.
#
# It is not:
#
#   - championship points
#   - raw average over the 15 races
#   - JAWS
#   - race-by-race current form
#
#
# TEAMS
# -----
# Posterior beta_{constructor, 2026}
#
# This is the constructor/car latent effect after accounting
# for driver ability and familiarity.
#
# Because the joint model has a known driver/car allocation
# ridge, absolute beta uncertainty should be shown explicitly.
#
#
# OUTPUT
# ------
# A NEW HTML release.
#
# The Stage 32H-I dashboard remains untouched.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 0. REQUIRE STAGE 32H-I
# ============================================================

required_32hj = [

    "LIVE_DASHBOARD_BUILDER_PATH_32HI",

    "dashboard_namespace_32hi",

    "current_rankings_payload_32hi",

    "posterior_32hc",

    "idx_all",

    "raw",

    "CFG",
]


missing_32hj = [

    name

    for name
    in required_32hj

    if name not in globals()
]


if missing_32hj:

    raise RuntimeError(
        "Stage 32H-J requires:\n  "
        +
        "\n  ".join(
            missing_32hj
        )
        +
        "\n\nRun Stage 32H-I first."
    )


print("=" * 80)
print("STAGE 32H-J — 2026 DRIVER / TEAM RANKING TABS")
print("=" * 80)


# ============================================================
# 1. READ CURRENT EXTENDED BUILDER
# ============================================================

base_builder_32hj = Path(
    LIVE_DASHBOARD_BUILDER_PATH_32HI
)


if not base_builder_32hj.exists():

    raise RuntimeError(
        "Stage 32H-I builder does not exist:\n"
        f"{base_builder_32hj}"
    )


dashboard_source_32hj = (
    base_builder_32hj
    .read_text(
        encoding="utf-8"
    )
)


print()
print("Base builder:")
print(
    base_builder_32hj.resolve()
)


# ============================================================
# 2. EXTRACT 2026 CONSTRUCTOR POSTERIOR
# ============================================================

car_post_32hj = np.asarray(

    posterior_32hc[
        "car_csy"
    ].values,

    dtype=float,
)


car_post_32hj = car_post_32hj.reshape(
    -1,
    car_post_32hj.shape[-1],
)


constructor_states_32hj = [

    (
        constructor_id,
        int(year),
    )

    for constructor_id, year
    in idx_all.active_constructor_years
]


if car_post_32hj.shape[1] != len(
    constructor_states_32hj
):

    raise RuntimeError(
        "car_csy state count does not match "
        "idx_all.active_constructor_years."
    )


constructor_2026_states_32hj = [

    (
        i,
        constructor_id,
    )

    for i, (
        constructor_id,
        year,
    )
    in enumerate(
        constructor_states_32hj
    )

    if year == 2026
]


if len(
    constructor_2026_states_32hj
) != 11:

    raise RuntimeError(
        "Expected 11 constructor-season states in 2026, "
        f"found {len(constructor_2026_states_32hj)}."
    )


constructor_2026_indices_32hj = np.array(

    [
        i
        for i, constructor_id
        in constructor_2026_states_32hj
    ],

    dtype=int,
)


constructor_2026_ids_32hj = [

    constructor_id

    for i, constructor_id
    in constructor_2026_states_32hj
]


beta_2026_32hj = car_post_32hj[
    :,
    constructor_2026_indices_32hj,
]


print()
print(
    "2026 constructor states:",
    len(
        constructor_2026_ids_32hj
    ),
)

print(
    "Posterior draws:",
    beta_2026_32hj.shape[0],
)


# ============================================================
# 3. POSTERIOR TEAM RANKS
# ============================================================
#
# Rank 1 = highest constructor/car beta in each posterior draw.
# ============================================================

team_order_32hj = np.argsort(

    -beta_2026_32hj,

    axis=1,
)


team_rank_32hj = np.empty_like(

    team_order_32hj,

    dtype=int,
)


team_rank_32hj[

    np.arange(
        beta_2026_32hj.shape[0]
    )[
        :,
        None,
    ],

    team_order_32hj,

] = np.arange(

    1,
    beta_2026_32hj.shape[1] + 1,

)[
    None,
    :
]


# ============================================================
# 4. CONSTRUCTOR NAME LOOKUP
# ============================================================

constructor_name_lookup_32hj = {}


if (
    hasattr(
        raw,
        "constructors",
    )
    and
    "constructorId"
    in raw.constructors.columns
):

    if (
        "name"
        in raw.constructors.columns
    ):

        constructor_name_lookup_32hj = dict(
            zip(
                raw.constructors[
                    "constructorId"
                ],
                raw.constructors[
                    "name"
                ],
            )
        )


if not constructor_name_lookup_32hj:

    if hasattr(
        raw,
        "constructor_ref",
    ):

        constructor_name_lookup_32hj = dict(
            raw.constructor_ref
        )


# ============================================================
# 5. BUILD TEAM-RANKING TABLE
# ============================================================

team_ranking_rows_32hj = []


for j, constructor_id in enumerate(
    constructor_2026_ids_32hj
):

    beta_draws = beta_2026_32hj[
        :,
        j,
    ]


    rank_draws = team_rank_32hj[
        :,
        j,
    ]


    team_ranking_rows_32hj.append(
        {
            "constructor_id":
                constructor_id,

            "team":
                constructor_name_lookup_32hj.get(
                    constructor_id,
                    str(
                        constructor_id
                    ),
                ),

            "beta_mean":
                float(
                    np.mean(
                        beta_draws
                    )
                ),

            "beta_median":
                float(
                    np.median(
                        beta_draws
                    )
                ),

            "beta_lb":
                float(
                    np.quantile(
                        beta_draws,
                        0.055,
                    )
                ),

            "beta_ub":
                float(
                    np.quantile(
                        beta_draws,
                        0.945,
                    )
                ),

            "rank_mean":
                float(
                    np.mean(
                        rank_draws
                    )
                ),

            "rank_median":
                float(
                    np.median(
                        rank_draws
                    )
                ),

            "rank_lb":
                int(
                    np.quantile(
                        rank_draws,
                        0.055,
                        method="nearest",
                    )
                ),

            "rank_ub":
                int(
                    np.quantile(
                        rank_draws,
                        0.945,
                        method="nearest",
                    )
                ),

            "p_top3":
                float(
                    np.mean(
                        rank_draws
                        <=
                        3
                    )
                ),

            "p_beta_gt_zero":
                float(
                    np.mean(
                        beta_draws
                        >
                        0
                    )
                ),
        }
    )


current_team_rankings_32hj = (

    pd.DataFrame(
        team_ranking_rows_32hj
    )
    .sort_values(
        "beta_mean",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


current_team_rankings_32hj.insert(

    0,

    "rank",

    np.arange(
        1,
        len(
            current_team_rankings_32hj
        )
        +
        1,
    ),
)


print()
print("=" * 80)
print("CURRENT 2026 TEAM / CAR RANKINGS")
print("=" * 80)


display(
    current_team_rankings_32hj
)


# ============================================================
# 6. SERIALISABLE TEAM PAYLOAD
# ============================================================

current_team_rankings_payload_32hj = [

    {
        "rank":
            int(
                row.rank
            ),

        "constructor_id":
            int(
                row.constructor_id
            ),

        "team":
            str(
                row.team
            ),

        "beta_mean":
            float(
                row.beta_mean
            ),

        "beta_median":
            float(
                row.beta_median
            ),

        "beta_lb":
            float(
                row.beta_lb
            ),

        "beta_ub":
            float(
                row.beta_ub
            ),

        "rank_mean":
            float(
                row.rank_mean
            ),

        "rank_median":
            float(
                row.rank_median
            ),

        "rank_lb":
            int(
                row.rank_lb
            ),

        "rank_ub":
            int(
                row.rank_ub
            ),

        "p_top3":
            float(
                row.p_top3
            ),

        "p_beta_gt_zero":
            float(
                row.p_beta_gt_zero
            ),
    }

    for row
    in current_team_rankings_32hj.itertuples(
        index=False
    )
]


if len(
    current_team_rankings_payload_32hj
) != 11:

    raise RuntimeError(
        "Team ranking payload does not contain 11 teams."
    )


# ============================================================
# 7. SAFE PATCH HELPER
# ============================================================

def replace_once_32hj(
    source,
    old,
    new,
    label,
):

    count = source.count(
        old
    )


    if count != 1:

        raise RuntimeError(
            f"Patch '{label}' expected exactly one match, "
            f"found {count}."
        )


    return source.replace(
        old,
        new,
        1,
    )


patched_dashboard_32hj = dashboard_source_32hj


# ============================================================
# 8. ADD TEAM PAYLOAD
# ============================================================

old_payload_32hj = """    "current_rankings":
        current_rankings_payload_32hi,"""


new_payload_32hj = """    "current_rankings":
        current_rankings_payload_32hi,

    "current_team_rankings":
        current_team_rankings_payload_32hj,"""


patched_dashboard_32hj = replace_once_32hj(

    patched_dashboard_32hj,

    old_payload_32hj,

    new_payload_32hj,

    "team ranking payload",
)


# ============================================================
# 9. UPDATE CURRENT-SECTION TITLE / EXPLANATION
# ============================================================

patched_dashboard_32hj = replace_once_32hj(

    patched_dashboard_32hj,

    """<h2>
Current 2026 driver rankings
</h2>""",

    """<h2>
Current 2026 rankings
</h2>""",

    "current section heading",
)


old_driver_note_32hj = """Ranking of the 23 current driver-season states by posterior
mean latent ability

<strong>α<sub>2026</sub></strong>

using evidence through Round 15.

This is a ranking of modelled 2026 realised driver ability,
not championship points and not the career Pantheon score.

The expected-rank and rank-interval columns are calculated
within the 23-driver 2026 field independently in every
posterior draw.

Because the model separates driver and constructor effects,
absolute driver levels carry appreciable driver–car allocation
uncertainty. Direct teammate differences are generally more
tightly identified.

Click any driver to open their 2026 season in the
Driver Explorer."""


new_driver_note_32hj = """The current rankings use the

<strong>
2026 latent states from the full joint posterior
</strong>

given all model evidence available through Round 15.

The 15 completed 2026 Grands Prix provide the direct 2026
likelihood, but the estimates are not fitted to those races in
isolation: the temporal driver model, earlier seasons, teammate
network and constructor effects are inferred jointly.

The Driver tab ranks

<strong>α<sub>driver,2026</sub></strong>,

the driver's realised 2026 season ability.

The Teams tab ranks

<strong>β<sub>constructor,2026</sub></strong>,

the constructor/car effect after accounting for driver ability
and organisational familiarity.

Neither table is a championship-points ranking or a career
Pantheon ranking.

Absolute driver and constructor levels carry the known
driver–car allocation uncertainty, so posterior rank intervals
are shown explicitly."""


patched_dashboard_32hj = replace_once_32hj(

    patched_dashboard_32hj,

    old_driver_note_32hj,

    new_driver_note_32hj,

    "current rankings explanation",
)


# ============================================================
# 10. ADD TAB BUTTONS ABOVE DRIVER TABLE
# ============================================================

old_driver_table_start_32hj = """</p>


<div class="table-wrap">

<table>

<thead>

<tr>

<th>
Rank
</th>

<th>
Driver
</th>"""


new_driver_table_start_32hj = """</p>


<div class="current-ranking-tabs">

<button
    id="current-driver-tab"
    class="current-ranking-tab active"
    type="button"
>
Drivers
</button>

<button
    id="current-team-tab"
    class="current-ranking-tab"
    type="button"
>
Teams
</button>

</div>


<div
    id="current-driver-panel"
    class="current-ranking-panel"
>

<div class="table-wrap">

<table>

<thead>

<tr>

<th>
Rank
</th>

<th>
Driver
</th>"""


patched_dashboard_32hj = replace_once_32hj(

    patched_dashboard_32hj,

    old_driver_table_start_32hj,

    new_driver_table_start_32hj,

    "driver ranking tab panel",
)


# ============================================================
# 11. CLOSE DRIVER PANEL + ADD TEAM PANEL
# ============================================================

old_current_section_end_32hj = """<tbody id="current-rankings-body">
</tbody>

</table>

</div>

</div>


<!-- ======================================================
     LEADERBOARD"""


new_current_section_end_32hj = """<tbody id="current-rankings-body">
</tbody>

</table>

</div>

</div>


<div
    id="current-team-panel"
    class="current-ranking-panel"
    style="display: none;"
>

<p class="note">

<strong>Team ability</strong>

is the 2026 constructor/car latent effect after adjusting for
the drivers in the car and their organisational familiarity.

A higher β means the model attributes more performance to the
constructor/car component.

Because driver α and constructor β can trade credit against one
another, the model identifies their combined signal more tightly
than either absolute component in isolation. The intervals below
therefore matter as much as the point ordering.

</p>


<div class="table-wrap">

<table>

<thead>

<tr>

<th>
Rank
</th>

<th>
Team
</th>

<th>
2026 β
</th>

<th>
89% β interval
</th>

<th>
Expected rank
</th>

<th>
89% rank interval
</th>

<th>
P(top 3)
</th>

<th>
P(β &gt; 0)
</th>

</tr>

</thead>


<tbody id="current-team-rankings-body">
</tbody>

</table>

</div>

</div>

</div>


<!-- ======================================================
     LEADERBOARD"""


patched_dashboard_32hj = replace_once_32hj(

    patched_dashboard_32hj,

    old_current_section_end_32hj,

    new_current_section_end_32hj,

    "team ranking panel",
)


# ============================================================
# 12. ADD TAB CSS
# ============================================================

tab_css_32hj = r"""
.current-ranking-tabs {

    display: flex;
    gap: 8px;
    margin: 18px 0 16px 0;

}


.current-ranking-tab {

    border: 1px solid #3a4652;
    background: #171d23;
    color: #b8c2cc;

    padding: 9px 18px;

    border-radius: 7px;

    font: inherit;
    font-weight: 700;

    cursor: pointer;

    transition:
        background 0.15s ease,
        border-color 0.15s ease,
        color 0.15s ease;

}


.current-ranking-tab:hover {

    background: #202832;
    color: #ffffff;

}


.current-ranking-tab.active {

    background: #27313b;
    border-color: #7f9db8;
    color: #ffffff;

}


.current-ranking-panel {

    width: 100%;

}
"""


patched_dashboard_32hj = replace_once_32hj(

    patched_dashboard_32hj,

    "</style>",

    (
        tab_css_32hj
        +
        "\n</style>"
    ),

    "ranking tab CSS",
)


# ============================================================
# 13. ADD TEAM DATA HANDLE
# ============================================================

old_current_data_32hj = """const currentRankings =
    DATA.current_rankings || [];"""


new_current_data_32hj = """const currentRankings =
    DATA.current_rankings || [];


const currentTeamRankings =
    DATA.current_team_rankings || [];"""


patched_dashboard_32hj = replace_once_32hj(

    patched_dashboard_32hj,

    old_current_data_32hj,

    new_current_data_32hj,

    "team ranking JS data",
)


# ============================================================
# 14. ADD TEAM RENDER + TAB LOGIC
# ============================================================

team_js_32hj = r"""
function renderCurrentTeamRankings() {

    const tbody =
        document.getElementById(
            "current-team-rankings-body"
        );


    if (!tbody) {

        return;

    }


    const fmt = (
        value,
        digits = 3
    ) => {

        if (
            value === null
            ||
            value === undefined
            ||
            Number.isNaN(
                Number(
                    value
                )
            )
        ) {

            return "—";

        }


        return Number(
            value
        ).toFixed(
            digits
        );

    };


    const pct = value => {

        if (
            value === null
            ||
            value === undefined
        ) {

            return "—";

        }


        return (
            100
            *
            Number(
                value
            )
        ).toFixed(
            1
        )
        +
        "%";

    };


    tbody.innerHTML =
        currentTeamRankings.map(

            row => `

            <tr>

                <td>
                    #${row.rank}
                </td>

                <td class="driver-name">
                    ${row.team}
                </td>

                <td>
                    ${fmt(row.beta_mean, 3)}
                </td>

                <td>
                    ${fmt(row.beta_lb, 3)}
                    –
                    ${fmt(row.beta_ub, 3)}
                </td>

                <td>
                    #${fmt(row.rank_mean, 1)}
                </td>

                <td>
                    #${row.rank_lb}
                    –
                    #${row.rank_ub}
                </td>

                <td>
                    ${pct(row.p_top3)}
                </td>

                <td>
                    ${pct(row.p_beta_gt_zero)}
                </td>

            </tr>

            `

        ).join(
            ""
        );

}


function setCurrentRankingTab(
    tab
) {

    const driverPanel =
        document.getElementById(
            "current-driver-panel"
        );


    const teamPanel =
        document.getElementById(
            "current-team-panel"
        );


    const driverButton =
        document.getElementById(
            "current-driver-tab"
        );


    const teamButton =
        document.getElementById(
            "current-team-tab"
        );


    if (
        !driverPanel
        ||
        !teamPanel
        ||
        !driverButton
        ||
        !teamButton
    ) {

        return;

    }


    const showTeams =
        tab === "teams";


    driverPanel.style.display =
        showTeams
        ? "none"
        : "block";


    teamPanel.style.display =
        showTeams
        ? "block"
        : "none";


    driverButton.classList.toggle(
        "active",
        !showTeams
    );


    teamButton.classList.toggle(
        "active",
        showTeams
    );

}


function initialiseCurrentRankingTabs() {

    const driverButton =
        document.getElementById(
            "current-driver-tab"
        );


    const teamButton =
        document.getElementById(
            "current-team-tab"
        );


    if (driverButton) {

        driverButton.addEventListener(

            "click",

            () => {

                setCurrentRankingTab(
                    "drivers"
                );

            }

        );

    }


    if (teamButton) {

        teamButton.addEventListener(

            "click",

            () => {

                setCurrentRankingTab(
                    "teams"
                );

            }

        );

    }


    setCurrentRankingTab(
        "drivers"
    );

}


"""


patched_dashboard_32hj = replace_once_32hj(

    patched_dashboard_32hj,

    """function renderCurrentRankings() {""",

    (
        team_js_32hj
        +
        "\nfunction renderCurrentRankings() {"
    ),

    "team ranking JS functions",
)


# ============================================================
# 15. INITIALISE TEAM TABLE + TABS
# ============================================================

patched_dashboard_32hj = replace_once_32hj(

    patched_dashboard_32hj,

    """    renderCurrentRankings();


    renderLeaderboard();""",

    """    renderCurrentRankings();

    renderCurrentTeamRankings();

    initialiseCurrentRankingTabs();


    renderLeaderboard();""",

    "team ranking initialisation",
)


# ============================================================
# 16. NEW OUTPUT FILENAMES
# ============================================================

LIVE_DASHBOARD_FILENAME_32HJ = (
    "f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "driver_team_current_rankings.html"
)


LIVE_DASHBOARD_BUILDER_FILENAME_32HJ = (
    "build_f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "driver_team_current_rankings.py"
)


output_dir_32hj = Path(
    CFG.output_dir
)


LIVE_DASHBOARD_PATH_32HJ = (
    output_dir_32hj
    /
    LIVE_DASHBOARD_FILENAME_32HJ
)


LIVE_DASHBOARD_BUILDER_PATH_32HJ = (
    output_dir_32hj
    /
    LIVE_DASHBOARD_BUILDER_FILENAME_32HJ
)


# ============================================================
# 17. REFUSE SILENT OVERWRITE
# ============================================================

existing_32hj = [

    path

    for path
    in [
        LIVE_DASHBOARD_PATH_32HJ,
        LIVE_DASHBOARD_BUILDER_PATH_32HJ,
    ]

    if path.exists()
]


if existing_32hj:

    raise RuntimeError(
        "Stage 32H-J output already exists:\n  "
        +
        "\n  ".join(
            str(
                path
            )
            for path
            in existing_32hj
        )
        +
        "\n\nDelete or rename it explicitly if you want "
        "to rebuild this exact release."
    )


# ============================================================
# 18. SAVE EXTENDED BUILDER
# ============================================================

LIVE_DASHBOARD_BUILDER_PATH_32HJ.write_text(

    patched_dashboard_32hj,

    encoding="utf-8",
)


print()
print(
    "Saved driver/team dashboard builder ->"
)

print(
    LIVE_DASHBOARD_BUILDER_PATH_32HJ
)


# ============================================================
# 19. EXECUTION NAMESPACE
# ============================================================

dashboard_namespace_32hj = dict(
    dashboard_namespace_32hi
)


dashboard_namespace_32hj.update(
    {
        "current_team_rankings_payload_32hj":
            current_team_rankings_payload_32hj,

        "LIVE_DASHBOARD_FILENAME_32HH":
            LIVE_DASHBOARD_FILENAME_32HJ,

        "OPEN_LIVE_DASHBOARD_AFTER_BUILD_32HH":
            True,
    }
)


# ============================================================
# 20. BUILD DASHBOARD
# ============================================================

print()
print("=" * 80)
print("BUILDING LIVE DRIVER / TEAM RANKING DASHBOARD")
print("=" * 80)


compiled_32hj = compile(

    patched_dashboard_32hj,

    str(
        LIVE_DASHBOARD_BUILDER_PATH_32HJ
    ),

    "exec",
)


exec(

    compiled_32hj,

    dashboard_namespace_32hj,

    dashboard_namespace_32hj,
)


# ============================================================
# 21. RETRIEVE GENERATED PATH
# ============================================================

generated_path_32hj = Path(

    dashboard_namespace_32hj[
        "PANTHEON_DASHBOARD_PATH"
    ]
)


if not generated_path_32hj.exists():

    raise RuntimeError(
        "Dashboard was not generated:\n"
        f"{generated_path_32hj}"
    )


if (
    generated_path_32hj.resolve()
    !=
    LIVE_DASHBOARD_PATH_32HJ.resolve()
):

    raise RuntimeError(
        "Dashboard builder wrote to an unexpected path:\n"
        f"{generated_path_32hj}"
    )


# ============================================================
# 22. VALIDATE HTML
# ============================================================

html_32hj = (
    generated_path_32hj
    .read_text(
        encoding="utf-8"
    )
)


html_checks_32hj = {

    "Current rankings heading":
        (
            "Current 2026 rankings"
            in html_32hj
        ),

    "Drivers tab":
        (
            'id="current-driver-tab"'
            in html_32hj
        ),

    "Teams tab":
        (
            'id="current-team-tab"'
            in html_32hj
        ),

    "Driver panel":
        (
            'id="current-driver-panel"'
            in html_32hj
        ),

    "Team panel":
        (
            'id="current-team-panel"'
            in html_32hj
        ),

    "Team table":
        (
            'id="current-team-rankings-body"'
            in html_32hj
        ),

    "Team payload":
        (
            "DATA.current_team_rankings"
            in html_32hj
        ),

    "Team render function":
        (
            "function renderCurrentTeamRankings()"
            in html_32hj
        ),

    "Tab function":
        (
            "function setCurrentRankingTab("
            in html_32hj
        ),

    "all-evidence explanation":
        (
            "given all model evidence available through Round 15"
            in html_32hj
        ),

    "driver-car warning":
        (
            "driver–car allocation uncertainty"
            in html_32hj
        ),

    "Pantheon preserved":
        (
            "Pantheon leaderboard"
            in html_32hj
        ),

    "Teammate Chain preserved":
        (
            "Teammate Chain"
            in html_32hj
        ),
}


validation_32hj = pd.DataFrame(
    [
        {
            "check":
                key,

            "passed":
                passed,
        }

        for key, passed
        in html_checks_32hj.items()
    ]
)


print()
print("=" * 80)
print("STAGE 32H-J HTML VALIDATION")
print("=" * 80)


display(
    validation_32hj
)


if not all(
    html_checks_32hj.values()
):

    failed_32hj = [

        key

        for key, passed
        in html_checks_32hj.items()

        if not passed
    ]


    raise RuntimeError(
        "Driver/team dashboard failed validation:\n  "
        +
        "\n  ".join(
            failed_32hj
        )
    )


# ============================================================
# 23. VALIDATE PAYLOAD
# ============================================================

payload_32hj = (
    dashboard_namespace_32hj[
        "payload"
    ]
)


if len(
    payload_32hj[
        "current_rankings"
    ]
) != 23:

    raise RuntimeError(
        "Driver payload does not contain 23 drivers."
    )


if len(
    payload_32hj[
        "current_team_rankings"
    ]
) != 11:

    raise RuntimeError(
        "Team payload does not contain 11 teams."
    )


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("STAGE 32H-J COMPLETE")
print("=" * 80)


print()
print(
    "LIVE dashboard with Driver / Team tabs:"
)

print(
    generated_path_32hj.resolve()
)


print()
print(
    "Driver ranking:"
)

print(
    "  alpha_{driver,2026}"
)

print(
    "  all model evidence through Round 15"
)

print(
    "  23-driver posterior ranking"
)


print()
print(
    "Team ranking:"
)

print(
    "  beta_{constructor,2026}"
)

print(
    "  driver/familiarity-adjusted constructor effect"
)

print(
    "  11-team posterior ranking"
)


print()
print(
    "Current 2026 Driver / Team tabs: BUILT."
)

In [ ]:
# ============================================================
# STAGE 32H-K — 95% CREDIBLE INTERVALS
#               + RENAME LONGEVITY -> CAREER VALUE
# ============================================================
#
# PURPOSE
# -------
# Finalise public-facing uncertainty / scoring terminology:
#
#   1. replace every central 89% posterior interval with a
#      central 95% credible interval:
#
#          2.5th percentile -> 97.5th percentile
#
#   2. rename the public-facing "Longevity" component to:
#
#          Career value
#
#      The underlying formula remains IDENTICAL:
#
#          sqrt(cumulative positive career alpha surplus)
#
#   3. keep internal variable names such as:
#
#          longevity_score
#
#      unchanged for backwards compatibility and provenance.
#
#
# IMPORTANT
# ---------
# NO MODEL REFIT.
# NO JAWS SCORE CHANGE.
# NO POSTERIOR CHANGE.
# NO RANKING CHANGE.
#
# Only:
#
#   - interval width
#   - displayed terminology
#
# change.
#
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 0. REQUIRE STAGE 32H-J
# ============================================================

required_32hk = [

    "LIVE_DASHBOARD_BUILDER_PATH_32HJ",

    "dashboard_namespace_32hj",

    "posterior_32hc",

    "eligible_jaws_live_32hh",
    "rank_live_32hh",

    "eligible_drivers_live_32hh",

    "pantheon_live_legacy_32hh",

    "alpha_2026_32hi",
    "rank_2026_32hi",
    "driver_2026_ids_32hi",

    "beta_2026_32hj",
    "team_rank_32hj",
    "constructor_2026_ids_32hj",

    "current_team_rankings_payload_32hj",

    "CFG",
]


missing_32hk = [

    name

    for name
    in required_32hk

    if name not in globals()
]


if missing_32hk:

    raise RuntimeError(
        "Stage 32H-K requires:\n  "
        +
        "\n  ".join(
            missing_32hk
        )
        +
        "\n\nRun Stage 32H-J first."
    )


print("=" * 80)
print("STAGE 32H-K — 95% INTERVALS + CAREER VALUE")
print("=" * 80)


# ============================================================
# 1. PUBLIC INTERVAL DEFINITION
# ============================================================

PUBLIC_CREDIBLE_INTERVAL_32HK = 0.95

PUBLIC_LOWER_Q_32HK = 0.025

PUBLIC_UPPER_Q_32HK = 0.975


print()
print(
    "Public interval:"
)

print(
    "  central 95% credible interval"
)

print(
    "  lower quantile:",
    PUBLIC_LOWER_Q_32HK,
)

print(
    "  upper quantile:",
    PUBLIC_UPPER_Q_32HK,
)


# ============================================================
# 2. READ THE CURRENT DRIVER / TEAM DASHBOARD BUILDER
# ============================================================

base_builder_32hk = Path(
    LIVE_DASHBOARD_BUILDER_PATH_32HJ
)


if not base_builder_32hk.exists():

    raise RuntimeError(
        "Stage 32H-J dashboard builder does not exist:\n"
        f"{base_builder_32hk}"
    )


dashboard_source_32hk = (
    base_builder_32hk
    .read_text(
        encoding="utf-8"
    )
)


print()
print(
    "Base builder:"
)

print(
    base_builder_32hk.resolve()
)


# ============================================================
# 3. REBUILD PANTHEON LEADERBOARD INTERVALS AT 95%
# ============================================================
#
# Stage 20 consumes a backwards-compatible dataframe called
# pantheon_2025.
#
# Keep all means/scores identical, but replace:
#
#   jaws_lb / jaws_ub
#   rank_lb / rank_ub
#
# using the actual 4,000 posterior draws.
# ============================================================

pantheon_live_legacy_32hk = (
    pantheon_live_legacy_32hh
    .copy()
)


eligible_driver_column_32hk = {

    driver: j

    for j, driver
    in enumerate(
        eligible_drivers_live_32hh
    )
}


for row_index, row in (
    pantheon_live_legacy_32hk
    .iterrows()
):

    driver = str(
        row[
            "driver"
        ]
    )


    if driver not in eligible_driver_column_32hk:

        raise RuntimeError(
            "Leaderboard driver missing from posterior matrix: "
            f"{driver}"
        )


    j = eligible_driver_column_32hk[
        driver
    ]


    jaws_draws = np.asarray(

        eligible_jaws_live_32hh[
            :,
            j,
        ],

        dtype=float,
    )


    rank_draws = np.asarray(

        rank_live_32hh[
            :,
            j,
        ],

        dtype=float,
    )


    pantheon_live_legacy_32hk.loc[
        row_index,
        "jaws_lb",
    ] = float(
        np.quantile(
            jaws_draws,
            PUBLIC_LOWER_Q_32HK,
        )
    )


    pantheon_live_legacy_32hk.loc[
        row_index,
        "jaws_ub",
    ] = float(
        np.quantile(
            jaws_draws,
            PUBLIC_UPPER_Q_32HK,
        )
    )


    pantheon_live_legacy_32hk.loc[
        row_index,
        "rank_lb",
    ] = float(
        np.quantile(
            rank_draws,
            PUBLIC_LOWER_Q_32HK,
        )
    )


    pantheon_live_legacy_32hk.loc[
        row_index,
        "rank_ub",
    ] = float(
        np.quantile(
            rank_draws,
            PUBLIC_UPPER_Q_32HK,
        )
    )


print()
print(
    "Pantheon leaderboard 95% intervals rebuilt:",
    len(
        pantheon_live_legacy_32hk
    ),
)


# ============================================================
# 4. REBUILD CURRENT 2026 DRIVER PAYLOAD WITH 95% INTERVALS
# ============================================================

current_rankings_payload_32hk = []


# ------------------------------------------------------------
# Reuse display/team metadata from Stage 32H-I payload.
# ------------------------------------------------------------

old_driver_payload_lookup_32hk = {

    row[
        "driver"
    ]:
        row

    for row
    in current_rankings_payload_32hi
}


# ------------------------------------------------------------
# Mean-order rank remains rank by posterior mean alpha.
# ------------------------------------------------------------

alpha_mean_2026_32hk = (
    np.mean(
        alpha_2026_32hi,
        axis=0,
    )
)


mean_order_2026_32hk = np.argsort(
    -alpha_mean_2026_32hk
)


point_rank_by_driver_32hk = {}


for position, j in enumerate(
    mean_order_2026_32hk,
    start=1,
):

    point_rank_by_driver_32hk[
        driver_2026_ids_32hi[
            j
        ]
    ] = int(
        position
    )


for j, driver in enumerate(
    driver_2026_ids_32hi
):

    old = old_driver_payload_lookup_32hk[
        driver
    ]


    alpha_draws = np.asarray(

        alpha_2026_32hi[
            :,
            j,
        ],

        dtype=float,
    )


    rank_draws = np.asarray(

        rank_2026_32hi[
            :,
            j,
        ],

        dtype=int,
    )


    current_rankings_payload_32hk.append(
        {
            "rank":
                point_rank_by_driver_32hk[
                    driver
                ],

            "driver":
                str(
                    driver
                ),

            "display_name":
                str(
                    old[
                        "display_name"
                    ]
                ),

            "team":
                str(
                    old[
                        "team"
                    ]
                ),

            "alpha_mean":
                float(
                    np.mean(
                        alpha_draws
                    )
                ),

            "alpha_median":
                float(
                    np.median(
                        alpha_draws
                    )
                ),

            "alpha_lb":
                float(
                    np.quantile(
                        alpha_draws,
                        PUBLIC_LOWER_Q_32HK,
                    )
                ),

            "alpha_ub":
                float(
                    np.quantile(
                        alpha_draws,
                        PUBLIC_UPPER_Q_32HK,
                    )
                ),

            "rank_mean":
                float(
                    np.mean(
                        rank_draws
                    )
                ),

            "rank_median":
                float(
                    np.median(
                        rank_draws
                    )
                ),

            "rank_lb":
                int(
                    np.quantile(
                        rank_draws,
                        PUBLIC_LOWER_Q_32HK,
                        method="nearest",
                    )
                ),

            "rank_ub":
                int(
                    np.quantile(
                        rank_draws,
                        PUBLIC_UPPER_Q_32HK,
                        method="nearest",
                    )
                ),

            "p_top5":
                float(
                    np.mean(
                        rank_draws
                        <=
                        5
                    )
                ),

            "p_alpha_gt_zero":
                float(
                    np.mean(
                        alpha_draws
                        >
                        0
                    )
                ),
        }
    )


current_rankings_payload_32hk = sorted(

    current_rankings_payload_32hk,

    key=lambda row:
        row[
            "rank"
        ],
)


if len(
    current_rankings_payload_32hk
) != 23:

    raise RuntimeError(
        "Expected 23 current-driver payload rows."
    )


# ============================================================
# 5. REBUILD CURRENT 2026 TEAM PAYLOAD WITH 95% INTERVALS
# ============================================================

old_team_payload_lookup_32hk = {

    int(
        row[
            "constructor_id"
        ]
    ):
        row

    for row
    in current_team_rankings_payload_32hj
}


beta_mean_2026_32hk = (
    np.mean(
        beta_2026_32hj,
        axis=0,
    )
)


mean_team_order_2026_32hk = np.argsort(
    -beta_mean_2026_32hk
)


point_rank_by_constructor_32hk = {}


for position, j in enumerate(
    mean_team_order_2026_32hk,
    start=1,
):

    point_rank_by_constructor_32hk[
        int(
            constructor_2026_ids_32hj[
                j
            ]
        )
    ] = int(
        position
    )


current_team_rankings_payload_32hk = []


for j, constructor_id in enumerate(
    constructor_2026_ids_32hj
):

    constructor_id = int(
        constructor_id
    )


    old = old_team_payload_lookup_32hk[
        constructor_id
    ]


    beta_draws = np.asarray(

        beta_2026_32hj[
            :,
            j,
        ],

        dtype=float,
    )


    rank_draws = np.asarray(

        team_rank_32hj[
            :,
            j,
        ],

        dtype=int,
    )


    current_team_rankings_payload_32hk.append(
        {
            "rank":
                point_rank_by_constructor_32hk[
                    constructor_id
                ],

            "constructor_id":
                constructor_id,

            "team":
                str(
                    old[
                        "team"
                    ]
                ),

            "beta_mean":
                float(
                    np.mean(
                        beta_draws
                    )
                ),

            "beta_median":
                float(
                    np.median(
                        beta_draws
                    )
                ),

            "beta_lb":
                float(
                    np.quantile(
                        beta_draws,
                        PUBLIC_LOWER_Q_32HK,
                    )
                ),

            "beta_ub":
                float(
                    np.quantile(
                        beta_draws,
                        PUBLIC_UPPER_Q_32HK,
                    )
                ),

            "rank_mean":
                float(
                    np.mean(
                        rank_draws
                    )
                ),

            "rank_median":
                float(
                    np.median(
                        rank_draws
                    )
                ),

            "rank_lb":
                int(
                    np.quantile(
                        rank_draws,
                        PUBLIC_LOWER_Q_32HK,
                        method="nearest",
                    )
                ),

            "rank_ub":
                int(
                    np.quantile(
                        rank_draws,
                        PUBLIC_UPPER_Q_32HK,
                        method="nearest",
                    )
                ),

            "p_top3":
                float(
                    np.mean(
                        rank_draws
                        <=
                        3
                    )
                ),

            "p_beta_gt_zero":
                float(
                    np.mean(
                        beta_draws
                        >
                        0
                    )
                ),
        }
    )


current_team_rankings_payload_32hk = sorted(

    current_team_rankings_payload_32hk,

    key=lambda row:
        row[
            "rank"
        ],
)


if len(
    current_team_rankings_payload_32hk
) != 11:

    raise RuntimeError(
        "Expected 11 current-team payload rows."
    )


# ============================================================
# 6. SANITY CHECK — 95% MUST NOT BE NARROWER THAN 89%
# ============================================================

for new_row in current_rankings_payload_32hk:

    old_row = old_driver_payload_lookup_32hk[
        new_row[
            "driver"
        ]
    ]


    new_width = (
        new_row[
            "alpha_ub"
        ]
        -
        new_row[
            "alpha_lb"
        ]
    )


    old_width = (
        old_row[
            "alpha_ub"
        ]
        -
        old_row[
            "alpha_lb"
        ]
    )


    if (
        new_width
        +
        1e-12
        <
        old_width
    ):

        raise RuntimeError(
            "95% driver interval became narrower than "
            "the old 89% interval for "
            f"{new_row['driver']}."
        )


for new_row in current_team_rankings_payload_32hk:

    old_row = old_team_payload_lookup_32hk[
        new_row[
            "constructor_id"
        ]
    ]


    new_width = (
        new_row[
            "beta_ub"
        ]
        -
        new_row[
            "beta_lb"
        ]
    )


    old_width = (
        old_row[
            "beta_ub"
        ]
        -
        old_row[
            "beta_lb"
        ]
    )


    if (
        new_width
        +
        1e-12
        <
        old_width
    ):

        raise RuntimeError(
            "95% team interval became narrower than "
            "the old 89% interval for "
            f"{new_row['team']}."
        )


print()
print(
    "Current-driver 95% interval check: PASS"
)

print(
    "Current-team 95% interval check: PASS"
)


# ============================================================
# 7. PATCH ALL ORIGINAL STAGE-20 POSTERIOR QUANTILES
# ============================================================
#
# These cover:
#
#   - driver-season alpha
#   - constructor beta
#   - season ranks
#   - teammate alpha
#   - teammate deltas
#   - pairwise comparisons
#   - network career alpha summaries
#
# They are all central posterior intervals.
# ============================================================

patched_dashboard_32hk = dashboard_source_32hk


lower_count_32hk = (
    patched_dashboard_32hk.count(
        "0.055,"
    )
)


upper_count_32hk = (
    patched_dashboard_32hk.count(
        "0.945,"
    )
)


print()
print(
    "Original 5.5% quantile occurrences:",
    lower_count_32hk,
)

print(
    "Original 94.5% quantile occurrences:",
    upper_count_32hk,
)


if lower_count_32hk != 9:

    raise RuntimeError(
        "Expected exactly 9 occurrences of 0.055 in the "
        "validated dashboard builder, found "
        f"{lower_count_32hk}."
    )


if upper_count_32hk != 9:

    raise RuntimeError(
        "Expected exactly 9 occurrences of 0.945 in the "
        "validated dashboard builder, found "
        f"{upper_count_32hk}."
    )


patched_dashboard_32hk = (
    patched_dashboard_32hk
    .replace(
        "0.055,",
        "0.025,",
    )
    .replace(
        "0.945,",
        "0.975,",
    )
)


# ============================================================
# 8. PATCH ALL PUBLIC 89% LABELS -> 95%
# ============================================================

n_89_labels_32hk = (
    patched_dashboard_32hk.count(
        "89%"
    )
)


print()
print(
    "Public 89% labels found:",
    n_89_labels_32hk,
)


if n_89_labels_32hk == 0:

    raise RuntimeError(
        "No 89% labels found to replace."
    )


patched_dashboard_32hk = (
    patched_dashboard_32hk
    .replace(
        "89%",
        "95%",
    )
    .replace(
        "5.5th",
        "2.5th",
    )
    .replace(
        "94.5th",
        "97.5th",
    )
)


# ============================================================
# 9. RENAME PUBLIC "LONGEVITY" -> "CAREER VALUE"
# ============================================================
#
# IMPORTANT:
#
# Internal identifiers remain:
#
#       longevity_score
#       longevity_weight
#
# We change ONLY human-facing wording.
# ============================================================


# ------------------------------------------------------------
# Header subtitle
# ------------------------------------------------------------

patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        "40% square-root-damped longevity",

        "40% square-root-damped career value",
    )
)


# ------------------------------------------------------------
# Leaderboard sort option
# ------------------------------------------------------------

patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        """<option value="longevity_score">
Longevity score
</option>""",

        """<option value="longevity_score">
Career value score
</option>""",
    )
)


# ------------------------------------------------------------
# Leaderboard table
# ------------------------------------------------------------

patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        """<th>
Longevity
</th>""",

        """<th>
Career value
</th>""",
    )
)


# ------------------------------------------------------------
# Peak / Career value section title
# ------------------------------------------------------------

patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        """<h2>
Peak vs longevity
</h2>""",

        """<h2>
Peak vs career value
</h2>""",
    )
)


# ------------------------------------------------------------
# Methodology wording
# ------------------------------------------------------------

patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        "<strong>40% longevity</strong>",

        "<strong>40% career value</strong>",
    )
)


patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        "Peak and longevity are independently normalised",

        "Peak and career value are independently normalised",
    )
)


patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        "peak/longevity exposure",

        "peak/career-value exposure",
    )
)


# ------------------------------------------------------------
# Driver summary card
# ------------------------------------------------------------

patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        """<div class="small-label">
                Longevity
            </div>""",

        """<div class="small-label">
                Career value
            </div>""",
    )
)


# ------------------------------------------------------------
# Plot hover + y axis
# ------------------------------------------------------------

patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        '+ "Longevity %{y:.1f}"',

        '+ "Career value %{y:.1f}"',
    )
)


patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        '"Longevity score"',

        '"Career value score"',
    )
)


# ------------------------------------------------------------
# Relevant HTML comment only
# ------------------------------------------------------------

patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        "PEAK / LONGEVITY + HEAD TO HEAD",

        "PEAK / CAREER VALUE + HEAD TO HEAD",
    )
)


# ============================================================
# 10. IMPROVE CAREER-VALUE EXPLANATION
# ============================================================

old_career_value_explanation_32hk = (
    "— the square root of cumulative positive career α surplus."
)


new_career_value_explanation_32hk = (
    "— square-root-damped cumulative positive career α surplus. "
    "It rewards sustained above-average seasons rather than "
    "career length alone."
)


if (
    old_career_value_explanation_32hk
    in patched_dashboard_32hk
):

    patched_dashboard_32hk = (
        patched_dashboard_32hk.replace(

            old_career_value_explanation_32hk,

            new_career_value_explanation_32hk,
        )
    )


# ============================================================
# 11. MAKE LEADERBOARD RANK INTERVAL HEADERS EXPLICIT
# ============================================================

patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        """<th>
Rank LB
</th>

<th>
Rank UB
</th>""",

        """<th>
95% Rank LB
</th>

<th>
95% Rank UB
</th>""",
    )
)


# ============================================================
# 12. PATCH METHODOLOGY INTERVAL EXPLANATION
# ============================================================

old_interval_method_32hk = """Rank LB and Rank UB are the 2.5th and 97.5th percentiles of
the posterior rank distribution, forming an 95% posterior
rank interval."""


new_interval_method_32hk = """Rank LB and Rank UB are the 2.5th and 97.5th percentiles of
the posterior rank distribution, forming a central 95%
Bayesian credible interval for rank."""


if (
    old_interval_method_32hk
    in patched_dashboard_32hk
):

    patched_dashboard_32hk = (
        patched_dashboard_32hk.replace(

            old_interval_method_32hk,

            new_interval_method_32hk,
        )
    )


# ------------------------------------------------------------
# In case line wrapping in the current builder differs slightly
# from the exact block above.
# ------------------------------------------------------------

patched_dashboard_32hk = (
    patched_dashboard_32hk.replace(

        "forming an 95% posterior\nrank interval.",

        "forming a central 95% Bayesian credible\n"
        "interval for rank.",
    )
)


# ============================================================
# 13. NEW OUTPUT FILES
# ============================================================

LIVE_DASHBOARD_FILENAME_32HK = (
    "f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "FINAL_95CI_career_value.html"
)


LIVE_DASHBOARD_BUILDER_FILENAME_32HK = (
    "build_f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "FINAL_95CI_career_value.py"
)


output_dir_32hk = Path(
    CFG.output_dir
)


LIVE_DASHBOARD_PATH_32HK = (
    output_dir_32hk
    /
    LIVE_DASHBOARD_FILENAME_32HK
)


LIVE_DASHBOARD_BUILDER_PATH_32HK = (
    output_dir_32hk
    /
    LIVE_DASHBOARD_BUILDER_FILENAME_32HK
)


# ============================================================
# 14. REFUSE SILENT OVERWRITE
# ============================================================

existing_32hk = [

    path

    for path
    in [
        LIVE_DASHBOARD_PATH_32HK,
        LIVE_DASHBOARD_BUILDER_PATH_32HK,
    ]

    if path.exists()
]


if existing_32hk:

    raise RuntimeError(
        "Stage 32H-K output already exists:\n  "
        +
        "\n  ".join(
            str(
                path
            )
            for path
            in existing_32hk
        )
        +
        "\n\nDelete or rename it explicitly if you want "
        "to rebuild this exact release."
    )


# ============================================================
# 15. SAVE FINAL PATCHED BUILDER
# ============================================================

LIVE_DASHBOARD_BUILDER_PATH_32HK.write_text(

    patched_dashboard_32hk,

    encoding="utf-8",
)


print()
print(
    "Saved 95%-CI / Career-value builder ->"
)

print(
    LIVE_DASHBOARD_BUILDER_PATH_32HK
)


# ============================================================
# 16. EXECUTION NAMESPACE
# ============================================================
#
# Start from Stage 32H-J so the Driver / Team tab machinery
# remains intact.
#
# Override only:
#
#   - legacy leaderboard interval dataframe
#   - current driver payload
#   - current team payload
#   - output filename
# ============================================================

dashboard_namespace_32hk = dict(
    dashboard_namespace_32hj
)


dashboard_namespace_32hk.update(
    {
        # ----------------------------------------------------
        # 95% Pantheon leaderboard intervals
        # ----------------------------------------------------

        "pantheon_2025":
            pantheon_live_legacy_32hk,


        # ----------------------------------------------------
        # 95% current-driver intervals
        # ----------------------------------------------------

        "current_rankings_payload_32hi":
            current_rankings_payload_32hk,


        # ----------------------------------------------------
        # 95% current-team intervals
        # ----------------------------------------------------

        "current_team_rankings_payload_32hj":
            current_team_rankings_payload_32hk,


        # ----------------------------------------------------
        # New output file
        # ----------------------------------------------------

        "LIVE_DASHBOARD_FILENAME_32HH":
            LIVE_DASHBOARD_FILENAME_32HK,


        # ----------------------------------------------------
        # Open browser after successful build
        # ----------------------------------------------------

        "OPEN_LIVE_DASHBOARD_AFTER_BUILD_32HH":
            True,
    }
)


# ============================================================
# 17. BUILD FINAL DASHBOARD
# ============================================================

print()
print("=" * 80)
print("BUILDING 95%-CI / CAREER-VALUE DASHBOARD")
print("=" * 80)


compiled_32hk = compile(

    patched_dashboard_32hk,

    str(
        LIVE_DASHBOARD_BUILDER_PATH_32HK
    ),

    "exec",
)


exec(

    compiled_32hk,

    dashboard_namespace_32hk,

    dashboard_namespace_32hk,
)


# ============================================================
# 18. RETRIEVE GENERATED DASHBOARD
# ============================================================

generated_path_32hk = Path(

    dashboard_namespace_32hk[
        "PANTHEON_DASHBOARD_PATH"
    ]
)


if not generated_path_32hk.exists():

    raise RuntimeError(
        "Final dashboard was not generated:\n"
        f"{generated_path_32hk}"
    )


if (
    generated_path_32hk.resolve()
    !=
    LIVE_DASHBOARD_PATH_32HK.resolve()
):

    raise RuntimeError(
        "Dashboard was written to an unexpected path:\n"
        f"{generated_path_32hk}"
    )


# ============================================================
# 19. VALIDATE GENERATED HTML
# ============================================================

html_32hk = (
    generated_path_32hk
    .read_text(
        encoding="utf-8"
    )
)


html_checks_32hk = {

    # --------------------------------------------------------
    # 95% intervals
    # --------------------------------------------------------

    "95% JAWS interval":
        (
            "95% JAWS interval"
            in html_32hk
        ),

    "95% current driver alpha":
        (
            "95% α interval"
            in html_32hk
        ),

    "95% current team beta":
        (
            "95% β interval"
            in html_32hk
        ),

    "95% posterior labels":
        (
            "95%"
            in html_32hk
        ),

    "no visible 89% labels":
        (
            "89%"
            not in html_32hk
        ),

    "2.5th percentile":
        (
            "2.5th percentile"
            in html_32hk
        ),

    "97.5th percentile":
        (
            "97.5th percentile"
            in html_32hk
        ),


    # --------------------------------------------------------
    # Career value terminology
    # --------------------------------------------------------

    "Career value leaderboard":
        (
            "Career value"
            in html_32hk
        ),

    "Career value score option":
        (
            "Career value score"
            in html_32hk
        ),

    "Peak vs career value":
        (
            "Peak vs career value"
            in html_32hk
        ),

    "40% career value":
        (
            "40% career value"
            in html_32hk
        ),

    "career value plot axis":
        (
            "Career value score"
            in html_32hk
        ),


    # --------------------------------------------------------
    # Existing features preserved
    # --------------------------------------------------------

    "Driver current tab":
        (
            'id="current-driver-tab"'
            in html_32hk
        ),

    "Team current tab":
        (
            'id="current-team-tab"'
            in html_32hk
        ),

    "Driver explorer":
        (
            "driver explorer"
            in html_32hk.lower()
        ),

    "Teammate network":
        (
            'id="teammate-network"'
            in html_32hk
        ),

    "Teammate Chain":
        (
            "Teammate Chain"
            in html_32hk
        ),
}


validation_32hk = pd.DataFrame(
    [
        {
            "check":
                check,

            "passed":
                passed,
        }

        for check, passed
        in html_checks_32hk.items()
    ]
)


print()
print("=" * 80)
print("STAGE 32H-K HTML VALIDATION")
print("=" * 80)


display(
    validation_32hk
)


if not all(
    html_checks_32hk.values()
):

    failed_32hk = [

        check

        for check, passed
        in html_checks_32hk.items()

        if not passed
    ]


    raise RuntimeError(
        "95%-CI / Career-value dashboard failed "
        "validation:\n  "
        +
        "\n  ".join(
            failed_32hk
        )
    )


# ============================================================
# 20. VERIFY INTERVALS REALLY CHANGED
# ============================================================

payload_32hk = (
    dashboard_namespace_32hk[
        "payload"
    ]
)


# ------------------------------------------------------------
# Example: Max current-driver interval
# ------------------------------------------------------------

max_row_32hk = next(

    row

    for row
    in payload_32hk[
        "current_rankings"
    ]

    if row[
        "driver"
    ]
    ==
    "max_verstappen"
)


print()
print("=" * 80)
print("95% INTERVAL SPOT CHECK")
print("=" * 80)


print()
print(
    "Max Verstappen 2026 alpha:"
)

print(
    f"  mean = {max_row_32hk['alpha_mean']:.3f}"
)

print(
    "  95% CrI = "
    f"[{max_row_32hk['alpha_lb']:.3f}, "
    f"{max_row_32hk['alpha_ub']:.3f}]"
)

print(
    "  95% rank interval = "
    f"#{max_row_32hk['rank_lb']}"
    "–"
    f"#{max_row_32hk['rank_ub']}"
)


# ============================================================
# 21. FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("STAGE 32H-K COMPLETE")
print("=" * 80)


print()
print(
    "Final dashboard:"
)

print(
    generated_path_32hk.resolve()
)


print()
print(
    "Public uncertainty:"
)

print(
    "  central 95% Bayesian credible intervals"
)

print(
    "  posterior quantiles 2.5% – 97.5%"
)


print()
print(
    "Public score terminology:"
)

print(
    "  Peak        = 60%"
)

print(
    "  Career value = 40%"
)

print(
    "  Career value formula is unchanged:"
)

print(
    "  sqrt(cumulative positive career alpha surplus)"
)


print()
print(
    "Internal variable names remain unchanged "
    "(e.g. longevity_score)."
)

print()
print(
    "No model, posterior, JAWS score or ranking was altered."
)

In [ ]:
# ============================================================
# STAGE 32H-L — FINAL DASHBOARD WORDING + UI POLISH
# ============================================================
#
# PRESENTATION ONLY.
#
# NO MODEL CHANGE.
# NO POSTERIOR CHANGE.
# NO PANTHEON SCORE CHANGE.
# NO RANKING CHANGE.
#
#
# Main changes
# ------------
#
# - clearer statement of what Pantheon measures
# - "Pantheon score" used publicly instead of unexplained JAWS
# - concise definitions of alpha, beta and credible intervals
# - clearer current-2026 explanation
# - simplify main leaderboard
# - combine Rank LB / Rank UB into one 95% rank interval column
# - improve leaderboard column tooltips
# - move Compare earlier in navigation
# - rename Head-to-head posterior -> Compare drivers
# - improve methodology description of alpha
# - make clear this ranks driving performance, not
#   championship success
#
# Internal variable names and JAWS machinery remain untouched.
# ============================================================

from pathlib import Path
import re
import pandas as pd


# ============================================================
# 0. REQUIRE STAGE 32H-K
# ============================================================

required_32hl = [

    "LIVE_DASHBOARD_BUILDER_PATH_32HK",
    "dashboard_namespace_32hk",
    "CFG",
]


missing_32hl = [

    name

    for name
    in required_32hl

    if name not in globals()
]


if missing_32hl:

    raise RuntimeError(
        "Stage 32H-L requires:\n  "
        +
        "\n  ".join(
            missing_32hl
        )
        +
        "\n\nRun Stage 32H-K first."
    )


print("=" * 80)
print("STAGE 32H-L — FINAL WORDING + UI POLISH")
print("=" * 80)


# ============================================================
# 1. LOAD CURRENT FINAL BUILDER
# ============================================================

base_builder_32hl = Path(
    LIVE_DASHBOARD_BUILDER_PATH_32HK
)


if not base_builder_32hl.exists():

    raise RuntimeError(
        "Stage 32H-K builder does not exist:\n"
        f"{base_builder_32hl}"
    )


source_32hl = (
    base_builder_32hl
    .read_text(
        encoding="utf-8"
    )
)


print()
print("Base builder:")
print(
    base_builder_32hl.resolve()
)


# ============================================================
# 2. HELPERS
# ============================================================

def replace_once_32hl(
    source,
    old,
    new,
    label,
):

    count = source.count(
        old
    )


    if count != 1:

        raise RuntimeError(
            f"Patch '{label}' expected exactly one match, "
            f"found {count}."
        )


    return source.replace(
        old,
        new,
        1,
    )


def regex_once_32hl(
    source,
    pattern,
    replacement,
    label,
    flags=re.S,
):

    new_source, count = re.subn(
        pattern,
        replacement,
        source,
        count=1,
        flags=flags,
    )


    if count != 1:

        raise RuntimeError(
            f"Regex patch '{label}' expected exactly one "
            f"match, found {count}."
        )


    return new_source


patched_32hl = source_32hl


# ============================================================
# 3. FINAL POLISH CSS
# ============================================================

polish_css_32hl = r"""

/* =========================================================
   Stage 32H-L — final presentation polish
========================================================= */

.pantheon-intro {

    margin:
        -8px 0 18px 0;

    max-width:
        980px;

    color:
        var(--muted);

    font-size:
        14px;

    line-height:
        1.65;

}


.pantheon-intro strong {

    color:
        var(--text);

}


.metric-key-grid {

    display:
        grid;

    grid-template-columns:
        repeat(
            4,
            minmax(
                0,
                1fr
            )
        );

    gap:
        10px;

    margin-top:
        12px;

}


.metric-key-card {

    padding:
        13px 14px;

    background:
        var(--panel2);

    border:
        1px solid
        var(--border);

    border-radius:
        10px;

}


.metric-key-name {

    margin-bottom:
        5px;

    color:
        var(--text);

    font-size:
        12px;

    font-weight:
        750;

}


.metric-key-description {

    color:
        var(--muted);

    font-size:
        11px;

    line-height:
        1.45;

}


.section-lead {

    max-width:
        1000px;

    margin:
        -5px 0 18px 0;

    color:
        var(--muted);

    font-size:
        13px;

    line-height:
        1.6;

}


.section-lead strong {

    color:
        var(--text);

}


table th[title] {

    cursor:
        help;

}


.measurement-note {

    padding:
        14px 16px;

    margin:
        12px 0 18px 0;

    background:
        var(--panel2);

    border-left:
        3px solid
        var(--accent);

    border-radius:
        8px;

    color:
        var(--muted);

    font-size:
        13px;

    line-height:
        1.6;

}


.measurement-note strong {

    color:
        var(--text);

}


@media (
    max-width:
        900px
) {

    .metric-key-grid {

        grid-template-columns:
            repeat(
                2,
                minmax(
                    0,
                    1fr
                )
            );

    }

}


@media (
    max-width:
        520px
) {

    .metric-key-grid {

        grid-template-columns:
            1fr;

    }

}

"""


patched_32hl = replace_once_32hl(

    patched_32hl,

    "</style>",

    (
        polish_css_32hl
        +
        "\n</style>"
    ),

    "final polish CSS",
)


# ============================================================
# 4. TOP-OF-PAGE PURPOSE STATEMENT
# ============================================================

top_intro_32hl = r"""
<div class="pantheon-intro">

Pantheon is a

<strong>
model-based ranking of Formula One driver performance
</strong>,

rather than championship success.

It estimates the driver's underlying season-level performance
after separating constructor performance and organisational
familiarity, then combines

<strong>
60% Peak
</strong>

with

<strong>
40% Career value
</strong>

to produce the overall Pantheon score.

</div>
"""


patched_32hl = regex_once_32hl(

    patched_32hl,

    r'(<div class="subtitle">.*?</div>)\s*(<nav class="sticky-nav")',

    (
        r'\1'
        +
        "\n\n"
        +
        top_intro_32hl
        +
        "\n"
        +
        r'\2'
    ),

    "top purpose statement",
)


# ============================================================
# 5. CLEAN NAVIGATION ORDER
# ============================================================
#
# Rankings and comparison first.
# Network/game remain available but are secondary.
# ============================================================

nav_32hl = r"""<nav class="sticky-nav" aria-label="Dashboard sections">

<a href="#current-rankings-section">2026 rankings</a>
<a href="#leaderboard-section">Pantheon</a>
<a href="#driver-explorer-section">Driver explorer</a>
<a href="#compare-section">Compare</a>
<a href="#network-section">Teammate network</a>
<a href="#game-section">Teammate Chain</a>
<a href="#method-section">Method</a>

</nav>"""


patched_32hl = regex_once_32hl(

    patched_32hl,

    r'<nav class="sticky-nav" aria-label="Dashboard sections">.*?</nav>',

    nav_32hl,

    "navigation order",
)


# ============================================================
# 6. ADD COMPACT METRIC KEY
# ============================================================

metric_key_32hl = r"""
<div class="metric-key-grid">

<div class="metric-key-card">

<div class="metric-key-name">
Pantheon score
</div>

<div class="metric-key-description">
Overall career score: 60% Peak + 40% Career value.
Higher is better; the scale is relative to the modelled field.
</div>

</div>


<div class="metric-key-card">

<div class="metric-key-name">
α — driver
</div>

<div class="metric-key-description">
Estimated realised driver performance in that season relative
to the average occupied F1 seat.
</div>

</div>


<div class="metric-key-card">

<div class="metric-key-name">
β — constructor
</div>

<div class="metric-key-description">
Constructor-season performance context after separating
driver ability and organisational familiarity.
</div>

</div>


<div class="metric-key-card">

<div class="metric-key-name">
95% credible interval
</div>

<div class="metric-key-description">
Central posterior interval from the 2.5th to 97.5th
percentiles. Wider intervals mean greater uncertainty.
</div>

</div>

</div>
"""


patched_32hl = regex_once_32hl(

    patched_32hl,

    r'(<div\s+id="summary"\s+class="summary-grid">\s*</div>)',

    (
        r'\1'
        +
        "\n\n"
        +
        metric_key_32hl
    ),

    "metric key",
)


# ============================================================
# 7. TIGHTEN CURRENT-2026 EXPLANATION
# ============================================================

current_start_32hl = patched_32hl.find(
    '<div id="current-rankings-section"'
)


current_tabs_32hl = patched_32hl.find(
    '<div class="current-ranking-tabs"',
    current_start_32hl,
)


if (
    current_start_32hl < 0
    or
    current_tabs_32hl < 0
):

    raise RuntimeError(
        "Could not locate current 2026 ranking section."
    )


current_prefix_32hl = patched_32hl[
    current_start_32hl:
    current_tabs_32hl
]


new_current_note_32hl = r"""
<p class="note">

These are the model's

<strong>
2026 season-level estimates
</strong>

using all evidence available through Round 15.

The 15 completed 2026 Grands Prix provide the direct
current-season observations, while previous seasons inform the
2026 states through the temporal model.

The

<strong>Drivers</strong>

tab ranks driver α; the

<strong>Teams</strong>

tab ranks constructor β.

These are modelled performance rankings, not championship
standings.

Click a driver row to open that driver's 2026 season in the
Driver Explorer.

</p>
"""


current_prefix_new_32hl, current_note_count_32hl = re.subn(

    r'<p class="note">.*?</p>',

    new_current_note_32hl,

    current_prefix_32hl,

    count=1,

    flags=re.S,
)


if current_note_count_32hl != 1:

    raise RuntimeError(
        "Could not replace the current-ranking explanation."
    )


patched_32hl = (
    patched_32hl[
        :current_start_32hl
    ]
    +
    current_prefix_new_32hl
    +
    patched_32hl[
        current_tabs_32hl:
    ]
)


# ============================================================
# 8. POLISH CURRENT-RANKING TABLE HEADERS
# ============================================================

patched_32hl = patched_32hl.replace(

    """<th>
2026 α
</th>""",

    """<th title="Estimated 2026 driver performance relative to the average occupied F1 seat">
2026 α
</th>""",
)


patched_32hl = patched_32hl.replace(

    """<th>
95% α interval
</th>""",

    """<th title="Central 95% Bayesian credible interval for 2026 driver α">
95% α interval
</th>""",
)


patched_32hl = patched_32hl.replace(

    """<th>
Expected rank
</th>""",

    """<th title="Mean rank across posterior draws">
Expected rank
</th>""",
)


# ============================================================
# 9. PANTHEON SECTION — CLEAR PURPOSE
# ============================================================

leader_start_32hl = patched_32hl.find(
    '<div id="leaderboard-section"'
)


driver_explorer_marker_32hl = patched_32hl.find(
    '<div id="driver-explorer-section"',
    leader_start_32hl,
)


if (
    leader_start_32hl < 0
    or
    driver_explorer_marker_32hl < 0
):

    raise RuntimeError(
        "Could not locate Pantheon leaderboard section."
    )


leader_segment_32hl = patched_32hl[
    leader_start_32hl:
    driver_explorer_marker_32hl
]


leader_segment_32hl, leader_heading_count_32hl = re.subn(

    r'<h2>\s*Pantheon leaderboard\s*</h2>',

    r"""<h2>
Pantheon — overall ranking
</h2>

<p class="section-lead">

The overall ranking balances

<strong>
Peak
</strong>

— performance across a driver's best five equivalent
season-units — with

<strong>
Career value
</strong>

— square-root-damped cumulative positive career α.

The Pantheon score is a relative model score, not a
percentage or probability.

</p>""",

    leader_segment_32hl,

    count=1,

    flags=re.S,
)


if leader_heading_count_32hl != 1:

    raise RuntimeError(
        "Could not replace Pantheon leaderboard heading."
    )


# ============================================================
# 10. MAKE SORT OPTIONS PUBLIC-FRIENDLY
# ============================================================

leader_segment_32hl = (
    leader_segment_32hl
    .replace(
        "Posterior mean JAWS",
        "Pantheon score",
    )
    .replace(
        "Rank lower bound",
        "95% rank lower bound",
    )
    .replace(
        "Rank upper bound",
        "95% rank upper bound",
    )
    .replace(
        "Rank interval width",
        "95% rank interval width",
    )
)


# ============================================================
# 11. SIMPLIFY MAIN LEADERBOARD HEADERS
# ============================================================

leader_segment_32hl = leader_segment_32hl.replace(

    """<th>
JAWS
</th>""",

    """<th title="Overall Pantheon score: 60% Peak + 40% Career value">
Pantheon score
</th>""",
)


leader_segment_32hl = leader_segment_32hl.replace(

    """<th>
95% JAWS interval
</th>""",

    """<th title="Central 95% credible interval for the Pantheon score">
95% score interval
</th>""",
)


leader_segment_32hl = leader_segment_32hl.replace(

    """<th>
Peak
</th>""",

    """<th title="Normalised score from the driver's best five equivalent season-units">
Peak
</th>""",
)


leader_segment_32hl = leader_segment_32hl.replace(

    """<th>
Career value
</th>""",

    """<th title="Square-root-damped cumulative positive career alpha">
Career value
</th>""",
)


leader_segment_32hl = leader_segment_32hl.replace(

    """<th>
Starts
</th>""",

    """<th title="Actual Formula One race starts">
Starts
</th>""",
)


# ------------------------------------------------------------
# Combine the two posterior-rank bound columns.
#
# Be deliberately tolerant of the exact public wording used
# by the previous dashboard stage, e.g.
#
#   Rank LB
#   95% Rank LB
#   95% rank interval — LB
#
# and the corresponding UB version.
# ------------------------------------------------------------

rank_header_pattern_32hl = r"""
<th[^>]*>
\s*
(?:
    (?:95\%\s*)?
    Rank\s*LB
    |
    (?:95\%\s*)?
    Rank\s*Lower\s*Bound
    |
    95\%\s*Rank\s*Interval\s*(?:—|-)\s*LB
)
\s*
</th>

\s*

<th[^>]*>
\s*
(?:
    (?:95\%\s*)?
    Rank\s*UB
    |
    (?:95\%\s*)?
    Rank\s*Upper\s*Bound
    |
    95\%\s*Rank\s*Interval\s*(?:—|-)\s*UB
)
\s*
</th>
"""


leader_segment_32hl, rank_header_count_32hl = re.subn(

    rank_header_pattern_32hl,

    """<th title="Central 95% posterior credible interval for rank">
95% rank interval
</th>""",

    leader_segment_32hl,

    count=1,

    flags=(
        re.S
        |
        re.X
        |
        re.I
    ),
)


if rank_header_count_32hl != 1:

    # Print the actual leaderboard headers so any future
    # presentation change is immediately obvious.
    actual_headers_32hl = [

        re.sub(
            r"\s+",
            " ",
            re.sub(
                r"<[^>]+>",
                "",
                cell,
            )
        ).strip()

        for cell
        in re.findall(
            r"<th[^>]*>.*?</th>",
            leader_segment_32hl,
            flags=re.S | re.I,
        )
    ]


    raise RuntimeError(
        "Could not combine Pantheon rank interval headers.\n\n"
        "Actual leaderboard headers were:\n  "
        +
        "\n  ".join(
            actual_headers_32hl
        )
    )


patched_32hl = (
    patched_32hl[
        :leader_start_32hl
    ]
    +
    leader_segment_32hl
    +
    patched_32hl[
        driver_explorer_marker_32hl:
    ]
)


# ============================================================
# 12. COMBINE LEADERBOARD RANK LB / UB CELLS
# ============================================================

render_lb_start_32hl = patched_32hl.find(
    "function renderLeaderboard()"
)


render_lb_end_32hl = patched_32hl.find(
    "let currentDriver",
    render_lb_start_32hl,
)


if (
    render_lb_start_32hl < 0
    or
    render_lb_end_32hl < 0
):

    raise RuntimeError(
        "Could not locate renderLeaderboard()."
    )


render_lb_segment_32hl = patched_32hl[
    render_lb_start_32hl:
    render_lb_end_32hl
]


rank_cells_pattern_32hl = r"""
<td>
\s*
\#\$\{num\(row\.rank_lb,\s*0\)\}
\s*
</td>
\s*
<td>
\s*
\#\$\{num\(row\.rank_ub,\s*0\)\}
\s*
</td>
"""


render_lb_segment_32hl, rank_cells_count_32hl = re.subn(

    rank_cells_pattern_32hl,

    """<td>
                    #${num(row.rank_lb, 0)}
                    –
                    #${num(row.rank_ub, 0)}
                </td>""",

    render_lb_segment_32hl,

    count=1,

    flags=re.S | re.X,
)


if rank_cells_count_32hl != 1:

    raise RuntimeError(
        "Could not combine leaderboard rank-bound cells."
    )


patched_32hl = (
    patched_32hl[
        :render_lb_start_32hl
    ]
    +
    render_lb_segment_32hl
    +
    patched_32hl[
        render_lb_end_32hl:
    ]
)


# ============================================================
# 13. DRIVER SUMMARY — PUBLIC TERMINOLOGY
# ============================================================

patched_32hl = patched_32hl.replace(

    "Posterior mean JAWS",

    "Pantheon score",
)


# ============================================================
# 14. COMPARISON SECTION
# ============================================================

patched_32hl = patched_32hl.replace(

    """<h2>
Head-to-head posterior
</h2>""",

    """<h2>
Compare drivers
</h2>""",
)


patched_32hl = patched_32hl.replace(

    """<strong>
P(JAWS A &gt; JAWS B)
</strong>""",

    """<strong>
P(Pantheon score A &gt; Pantheon score B)
</strong>""",
)


patched_32hl = patched_32hl.replace(

    '"JAWS"',

    '"Pantheon score"',
)


patched_32hl = patched_32hl.replace(

    '+ "JAWS %{customdata[2]:.1f}<br>"',

    '+ "Pantheon score %{customdata[2]:.1f}<br>"',
)


# ============================================================
# 15. REWRITE THE CORE "WHAT DOES ALPHA MEAN?" PARAGRAPH
# ============================================================
#
# This is deliberately more precise than simply saying
# "driver ability".
# ============================================================

method_start_32hl = patched_32hl.find(
    '<div id="method-section"'
)


footer_start_32hl = patched_32hl.find(
    "<footer>",
    method_start_32hl,
)


if (
    method_start_32hl < 0
    or
    footer_start_32hl < 0
):

    raise RuntimeError(
        "Could not locate methodology section."
    )


method_segment_32hl = patched_32hl[
    method_start_32hl:
    footer_start_32hl
]


new_measurement_paragraph_32hl = r"""
<p>

The season-level latent value

<strong>α</strong>

is the model's estimate of a driver's

<strong>
realised season-level on-track performance
</strong>

relative to the average occupied F1 seat.

For modern seasons it is identified primarily from
same-lap race pace and timed qualifying, using teammate
contrasts and whole-field team-level information to separate
driver performance from constructor performance and
organisational familiarity.

It is therefore broader than a fastest-lap statistic, but it
does not directly award credit for championships or points,
nor does it separately score skills such as overtaking,
starts, error avoidance or racecraft.

For 1982–1995, qualifying grid position and classified race
order provide the available ordinal evidence.

Age-related decline remains part of realised season
performance.

</p>
"""


# First ordinary paragraph after the Method heading.
method_segment_32hl, measurement_count_32hl = re.subn(

    r'(<h2>\s*How to read the model\s*</h2>)\s*<p>.*?</p>',

    (
        r'\1'
        +
        "\n\n"
        +
        new_measurement_paragraph_32hl
    ),

    method_segment_32hl,

    count=1,

    flags=re.S,
)


if measurement_count_32hl != 1:

    raise RuntimeError(
        "Could not replace the main alpha explanation."
    )


# ============================================================
# 16. CLARIFY THE OVERALL RANKING'S INTERPRETATION
# ============================================================

ranking_measurement_note_32hl = r"""
<div class="measurement-note">

<strong>
What does the overall ranking mean?
</strong>

Pantheon ranks

<strong>
model-estimated driving performance
</strong>,

not raw speed alone and not championship achievement.

Peak captures how highly the model rates a driver's strongest
seasons; Career value rewards sustained positive performance
across the career.

</div>
"""


# Insert immediately before eligibility paragraph.
eligibility_marker_32hl = (
    "<p>\n\nEligibility requires at least"
)


if eligibility_marker_32hl not in method_segment_32hl:

    raise RuntimeError(
        "Could not locate methodology eligibility paragraph."
    )


method_segment_32hl = method_segment_32hl.replace(

    eligibility_marker_32hl,

    (
        ranking_measurement_note_32hl
        +
        "\n\n"
        +
        eligibility_marker_32hl
    ),

    1,
)


patched_32hl = (
    patched_32hl[
        :method_start_32hl
    ]
    +
    method_segment_32hl
    +
    patched_32hl[
        footer_start_32hl:
    ]
)


# ============================================================
# 17. FINAL FOOTER LINE
# ============================================================

footer_pattern_32hl = r'(<footer>.*?)(</footer>)'


footer_addition_32hl = r"""

<br><br>

Pantheon ranks model-estimated driving performance rather
than championship success.

"""


patched_32hl = regex_once_32hl(

    patched_32hl,

    footer_pattern_32hl,

    (
        r'\1'
        +
        footer_addition_32hl
        +
        r'\2'
    ),

    "footer clarification",
)


# ============================================================
# 18. NEW OUTPUT FILENAMES
# ============================================================

LIVE_DASHBOARD_FILENAME_32HL = (
    "f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "FINAL_POLISHED.html"
)


LIVE_DASHBOARD_BUILDER_FILENAME_32HL = (
    "build_f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "FINAL_POLISHED.py"
)


output_dir_32hl = Path(
    CFG.output_dir
)


LIVE_DASHBOARD_PATH_32HL = (
    output_dir_32hl
    /
    LIVE_DASHBOARD_FILENAME_32HL
)


LIVE_DASHBOARD_BUILDER_PATH_32HL = (
    output_dir_32hl
    /
    LIVE_DASHBOARD_BUILDER_FILENAME_32HL
)


# ============================================================
# 19. REFUSE SILENT OVERWRITE
# ============================================================

existing_32hl = [

    path

    for path
    in [
        LIVE_DASHBOARD_PATH_32HL,
        LIVE_DASHBOARD_BUILDER_PATH_32HL,
    ]

    if path.exists()
]


if existing_32hl:

    raise RuntimeError(
        "Stage 32H-L output already exists:\n  "
        +
        "\n  ".join(
            str(
                path
            )
            for path
            in existing_32hl
        )
        +
        "\n\nDelete or rename it explicitly if you want "
        "to rebuild this exact release."
    )


# ============================================================
# 20. SAVE POLISHED BUILDER
# ============================================================

LIVE_DASHBOARD_BUILDER_PATH_32HL.write_text(

    patched_32hl,

    encoding="utf-8",
)


print()
print(
    "Saved polished dashboard builder ->"
)

print(
    LIVE_DASHBOARD_BUILDER_PATH_32HL
)


# ============================================================
# 21. EXECUTION NAMESPACE
# ============================================================

dashboard_namespace_32hl = dict(
    dashboard_namespace_32hk
)


dashboard_namespace_32hl.update(
    {
        "LIVE_DASHBOARD_FILENAME_32HH":
            LIVE_DASHBOARD_FILENAME_32HL,

        "OPEN_LIVE_DASHBOARD_AFTER_BUILD_32HH":
            True,
    }
)


# ============================================================
# 22. BUILD POLISHED DASHBOARD
# ============================================================

print()
print("=" * 80)
print("BUILDING FINAL POLISHED DASHBOARD")
print("=" * 80)


compiled_32hl = compile(

    patched_32hl,

    str(
        LIVE_DASHBOARD_BUILDER_PATH_32HL
    ),

    "exec",
)


exec(

    compiled_32hl,

    dashboard_namespace_32hl,

    dashboard_namespace_32hl,
)


# ============================================================
# 23. RETRIEVE OUTPUT
# ============================================================

generated_path_32hl = Path(

    dashboard_namespace_32hl[
        "PANTHEON_DASHBOARD_PATH"
    ]
)


if not generated_path_32hl.exists():

    raise RuntimeError(
        "Polished dashboard was not generated:\n"
        f"{generated_path_32hl}"
    )


if (
    generated_path_32hl.resolve()
    !=
    LIVE_DASHBOARD_PATH_32HL.resolve()
):

    raise RuntimeError(
        "Dashboard builder wrote to unexpected path:\n"
        f"{generated_path_32hl}"
    )


# ============================================================
# 24. FINAL HTML VALIDATION
# ============================================================

html_32hl = generated_path_32hl.read_text(
    encoding="utf-8"
)


checks_32hl = {

    "performance-ranking intro":
        (
            "model-based ranking of Formula One driver performance"
            in html_32hl
        ),

    "metric key":
        (
            "metric-key-grid"
            in html_32hl
        ),

    "Pantheon score terminology":
        (
            "Pantheon score"
            in html_32hl
        ),

    "alpha definition":
        (
            "realised season-level on-track performance"
            in html_32hl
        ),

    "modern evidence explanation":
        (
            "same-lap race pace and timed qualifying"
            in html_32hl
        ),

    "not championship success":
        (
            "not championship success"
            in html_32hl
        ),

    "current drivers tab":
        (
            'id="current-driver-tab"'
            in html_32hl
        ),

    "current teams tab":
        (
            'id="current-team-tab"'
            in html_32hl
        ),

    "overall ranking heading":
        (
            "Pantheon — overall ranking"
            in html_32hl
        ),

    "single 95% rank column":
        (
            "95% rank interval"
            in html_32hl
        ),

    "old rank-LB heading removed":
        (
            "95% Rank LB"
            not in html_32hl
        ),

    "old rank-UB heading removed":
        (
            "95% Rank UB"
            not in html_32hl
        ),

    "Career value":
        (
            "Career value"
            in html_32hl
        ),

    "95% uncertainty":
        (
            "95% credible interval"
            in html_32hl
        ),

    "no 89% labels":
        (
            "89%"
            not in html_32hl
        ),

    "Compare drivers":
        (
            "Compare drivers"
            in html_32hl
        ),

    "Driver explorer":
        (
            "driver explorer"
            in html_32hl.lower()
        ),

    "Teammate network":
        (
            "teammate network"
            in html_32hl.lower()
        ),

    "Teammate Chain":
        (
            "Teammate Chain"
            in html_32hl
        ),
}


validation_32hl = pd.DataFrame(
    [
        {
            "check":
                check,

            "passed":
                passed,
        }

        for check, passed
        in checks_32hl.items()
    ]
)


print()
print("=" * 80)
print("STAGE 32H-L FINAL VALIDATION")
print("=" * 80)


display(
    validation_32hl
)


if not all(
    checks_32hl.values()
):

    failed_32hl = [

        check

        for check, passed
        in checks_32hl.items()

        if not passed
    ]


    raise RuntimeError(
        "Final polished dashboard failed validation:\n  "
        +
        "\n  ".join(
            failed_32hl
        )
    )


# ============================================================
# 25. CONFIRM PAYLOAD COUNTS UNCHANGED
# ============================================================

payload_32hl = (
    dashboard_namespace_32hl[
        "payload"
    ]
)


if len(
    payload_32hl[
        "leaderboard"
    ]
) != 98:

    raise RuntimeError(
        "Pantheon leaderboard size changed unexpectedly."
    )


if len(
    payload_32hl[
        "current_rankings"
    ]
) != 23:

    raise RuntimeError(
        "Current-driver ranking size changed unexpectedly."
    )


if len(
    payload_32hl[
        "current_team_rankings"
    ]
) != 11:

    raise RuntimeError(
        "Current-team ranking size changed unexpectedly."
    )


# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("STAGE 32H-L COMPLETE")
print("=" * 80)


print()
print(
    "Final polished dashboard:"
)

print(
    generated_path_32hl.resolve()
)


print()
print(
    "Presentation:"
)

print(
    "  Pantheon score = 60% Peak + 40% Career value"
)

print(
    "  uncertainty = central 95% credible intervals"
)

print(
    "  alpha = realised season-level driver performance"
)

print(
    "  beta = constructor-season performance context"
)

print(
    "  leaderboard rank bounds combined into one interval"
)


print()
print(
    "Payload:"
)

print(
    "  Pantheon drivers:",
    len(
        payload_32hl[
            "leaderboard"
        ]
    ),
)

print(
    "  current drivers:",
    len(
        payload_32hl[
            "current_rankings"
        ]
    ),
)

print(
    "  current teams:",
    len(
        payload_32hl[
            "current_team_rankings"
        ]
    ),
)


print()
print(
    "No model, posterior, Pantheon score or ranking changed."
)

In [ ]:
# ============================================================
# STAGE 32H-M — FINAL MICRO-CLEANUP
# ============================================================
#
# PRESENTATION ONLY.
#
# Changes
# -------
# 1. Remove remaining public-facing "JAWS" terminology.
# 2. "Posterior mean #1" -> "Top-ranked driver".
# 3. Remove internal vertical scrolling from the two primary
#    ranking tables.
# 4. Combine Driver Explorer rank LB / UB cards.
# 5. Shorten the introductory paragraph.
# 6. Make remaining α labels consistently say "performance"
#    rather than "ability".
#
# NO MODEL CHANGE.
# NO POSTERIOR CHANGE.
# NO PANTHEON SCORE CHANGE.
# NO RANKING CHANGE.
# ============================================================

from pathlib import Path
import re
import pandas as pd


# ============================================================
# 0. REQUIRE STAGE 32H-L
# ============================================================

required_32hm = [

    "LIVE_DASHBOARD_BUILDER_PATH_32HL",
    "dashboard_namespace_32hl",
    "CFG",
]


missing_32hm = [

    name

    for name
    in required_32hm

    if name not in globals()
]


if missing_32hm:

    raise RuntimeError(
        "Stage 32H-M requires:\n  "
        +
        "\n  ".join(
            missing_32hm
        )
        +
        "\n\nRun Stage 32H-L first."
    )


print("=" * 80)
print("STAGE 32H-M — FINAL MICRO-CLEANUP")
print("=" * 80)


# ============================================================
# 1. LOAD POLISHED BUILDER
# ============================================================

base_builder_32hm = Path(
    LIVE_DASHBOARD_BUILDER_PATH_32HL
)


if not base_builder_32hm.exists():

    raise RuntimeError(
        "Stage 32H-L builder does not exist:\n"
        f"{base_builder_32hm}"
    )


source_32hm = (
    base_builder_32hm
    .read_text(
        encoding="utf-8"
    )
)


patched_32hm = source_32hm


print()
print("Base builder:")
print(
    base_builder_32hm.resolve()
)


# ============================================================
# 2. HELPER
# ============================================================

def regex_once_32hm(
    source,
    pattern,
    replacement,
    label,
    flags=re.S,
):

    new_source, count = re.subn(

        pattern,

        replacement,

        source,

        count=1,

        flags=flags,
    )


    if count != 1:

        raise RuntimeError(
            f"Patch '{label}' expected exactly one match, "
            f"found {count}."
        )


    return new_source


# ============================================================
# 3. SHORTEN TOP INTRODUCTION
# ============================================================

new_intro_32hm = r"""<div class="pantheon-intro">

Pantheon ranks

<strong>
model-estimated F1 driver performance
</strong>

rather than championship success, combining

<strong>
60% Peak
</strong>

with

<strong>
40% Career value
</strong>.

</div>"""


patched_32hm = regex_once_32hm(

    patched_32hm,

    r'<div class="pantheon-intro">.*?</div>',

    new_intro_32hm,

    "short top introduction",
)


# ============================================================
# 4. CLEAN LIVE BANNER WORDING
# ============================================================

patched_32hm = (
    patched_32hm
    .replace(
        "JAWS gives the incomplete 2026 season",
        "The incomplete 2026 season receives",
    )
    .replace(
        "of one season of peak/career-value exposure",
        "of a full season's Peak/Career value exposure",
    )
)


# ============================================================
# 5. SUMMARY LABEL
# ============================================================

if (
    "Posterior mean #1"
    not in patched_32hm
):

    raise RuntimeError(
        "Could not find 'Posterior mean #1' summary label."
    )


patched_32hm = patched_32hm.replace(

    "Posterior mean #1",

    "Top-ranked driver",
)


# ============================================================
# 6. REMOVE INTERNAL SCROLLBARS FROM PRIMARY RANKING TABLES
# ============================================================
#
# Keep horizontal scrolling available on narrow screens,
# but allow the tables to expand vertically with the page.
# ============================================================

ranking_scroll_css_32hm = r"""

/* =========================================================
   Stage 32H-M — primary ranking tables expand vertically
========================================================= */

#current-rankings-section .table-wrap,
#leaderboard-section .table-wrap {

    max-height:
        none !important;

    overflow-y:
        visible !important;

    overflow-x:
        auto;

}


/* Keep the second Driver Explorer row visually balanced. */

.career-sample-card {

    grid-column:
        span 2;

}


@media (
    max-width:
        560px
) {

    .career-sample-card {

        grid-column:
            auto;

    }

}

"""


if (
    "</style>"
    not in patched_32hm
):

    raise RuntimeError(
        "Could not locate dashboard </style> tag."
    )


patched_32hm = patched_32hm.replace(

    "</style>",

    (
        ranking_scroll_css_32hm
        +
        "\n</style>"
    ),

    1,
)


# ============================================================
# 7. COMBINE DRIVER EXPLORER 95% RANK CARDS
# ============================================================

rank_cards_pattern_32hm = r"""
<div\s+class="small-card">
\s*
<div\s+class="small-label">
\s*
95%\s+rank\s+interval\s*[—-]\s*LB
\s*
</div>
\s*
<div\s+class="small-value">
\s*
\#\$\{num\(row\.rank_lb,\s*0\)\}
\s*
</div>
\s*
<div\s+class="note">
\s*
2\.5th\s+percentile
\s*
</div>
\s*
</div>

\s*

<div\s+class="small-card">
\s*
<div\s+class="small-label">
\s*
95%\s+rank\s+interval\s*[—-]\s*UB
\s*
</div>
\s*
<div\s+class="small-value">
\s*
\#\$\{num\(row\.rank_ub,\s*0\)\}
\s*
</div>
\s*
<div\s+class="note">
\s*
97\.5th\s+percentile
\s*
</div>
\s*
</div>
"""


combined_rank_card_32hm = r"""<div class="small-card">

            <div class="small-label">
                95% rank interval
            </div>

            <div class="small-value">
                #${num(row.rank_lb, 0)}
                –
                #${num(row.rank_ub, 0)}
            </div>

            <div class="note">
                2.5th–97.5th percentiles
            </div>

        </div>"""


patched_32hm = regex_once_32hm(

    patched_32hm,

    rank_cards_pattern_32hm,

    combined_rank_card_32hm,

    "combine Driver Explorer rank interval cards",

    flags=(
        re.S
        |
        re.X
        |
        re.I
    ),
)


# ============================================================
# 8. MAKE CAREER SAMPLE CARD SPAN THE REMAINING TWO COLUMNS
# ============================================================

career_sample_pattern_32hm = r"""
<div\s+class="small-card">
(\s*
<div\s+class="small-label">
\s*
Career\s+sample
\s*
</div>
)
"""


def career_sample_replacement_32hm(
    match,
):

    return (
        '<div class="small-card career-sample-card">'
        +
        match.group(1)
    )


patched_32hm, career_sample_count_32hm = re.subn(

    career_sample_pattern_32hm,

    career_sample_replacement_32hm,

    patched_32hm,

    count=1,

    flags=(
        re.S
        |
        re.X
        |
        re.I
    ),
)


if career_sample_count_32hm != 1:

    raise RuntimeError(
        "Could not identify the Driver Explorer "
        "Career sample card."
    )


# ============================================================
# 9. REMOVE REMAINING PUBLIC-FACING JAWS WORDING
# ============================================================
#
# Internal variable names are lower-case, e.g. jaws_mean,
# and are therefore untouched.
# ============================================================

patched_32hm = (
    patched_32hm
    .replace(
        "JAWS score",
        "Pantheon score",
    )
    .replace(
        "JAWS scores",
        "Pantheon scores",
    )
    .replace(
        "· JAWS ${",
        "· Pantheon ${",
    )
    .replace(
        "JAWS",
        "Pantheon",
    )
)


# ============================================================
# 10. PERFORMANCE TERMINOLOGY CONSISTENCY
# ============================================================
#
# We now describe alpha publicly as realised on-track
# performance rather than leaving mixed ability/performance
# wording around the dashboard.
# ============================================================

terminology_replacements_32hm = {

    "season-level realised ability":
        "season-level realised performance",

    "Latent driver ability α":
        "Latent driver performance α",

    "that year's driver ability":
        "that year's driver performance",

    "driver ability, constructor/car performance":
        "driver performance, constructor/car performance",

    "mean ability across the best five equivalent season-units":
        "mean performance across the best five equivalent season-units",
}


for old, new in terminology_replacements_32hm.items():

    patched_32hm = patched_32hm.replace(
        old,
        new,
    )


# ============================================================
# 11. NEW OUTPUT FILES
# ============================================================

LIVE_DASHBOARD_FILENAME_32HM = (
    "f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "FINAL.html"
)


LIVE_DASHBOARD_BUILDER_FILENAME_32HM = (
    "build_f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "FINAL.py"
)


output_dir_32hm = Path(
    CFG.output_dir
)


LIVE_DASHBOARD_PATH_32HM = (
    output_dir_32hm
    /
    LIVE_DASHBOARD_FILENAME_32HM
)


LIVE_DASHBOARD_BUILDER_PATH_32HM = (
    output_dir_32hm
    /
    LIVE_DASHBOARD_BUILDER_FILENAME_32HM
)


# ============================================================
# 12. REFUSE SILENT OVERWRITE
# ============================================================

existing_32hm = [

    path

    for path
    in [
        LIVE_DASHBOARD_PATH_32HM,
        LIVE_DASHBOARD_BUILDER_PATH_32HM,
    ]

    if path.exists()
]


if existing_32hm:

    raise RuntimeError(
        "Stage 32H-M output already exists:\n  "
        +
        "\n  ".join(
            str(
                path
            )
            for path
            in existing_32hm
        )
        +
        "\n\nDelete or rename it explicitly if you want "
        "to rebuild this exact release."
    )


# ============================================================
# 13. SAVE FINAL BUILDER
# ============================================================

LIVE_DASHBOARD_BUILDER_PATH_32HM.write_text(

    patched_32hm,

    encoding="utf-8",
)


print()
print(
    "Saved final dashboard builder ->"
)

print(
    LIVE_DASHBOARD_BUILDER_PATH_32HM
)


# ============================================================
# 14. EXECUTION NAMESPACE
# ============================================================

dashboard_namespace_32hm = dict(
    dashboard_namespace_32hl
)


dashboard_namespace_32hm.update(
    {
        "LIVE_DASHBOARD_FILENAME_32HH":
            LIVE_DASHBOARD_FILENAME_32HM,

        "OPEN_LIVE_DASHBOARD_AFTER_BUILD_32HH":
            True,
    }
)


# ============================================================
# 15. BUILD FINAL DASHBOARD
# ============================================================

print()
print("=" * 80)
print("BUILDING FINAL CLEAN DASHBOARD")
print("=" * 80)


compiled_32hm = compile(

    patched_32hm,

    str(
        LIVE_DASHBOARD_BUILDER_PATH_32HM
    ),

    "exec",
)


exec(

    compiled_32hm,

    dashboard_namespace_32hm,

    dashboard_namespace_32hm,
)


# ============================================================
# 16. RETRIEVE OUTPUT
# ============================================================

generated_path_32hm = Path(

    dashboard_namespace_32hm[
        "PANTHEON_DASHBOARD_PATH"
    ]
)


if not generated_path_32hm.exists():

    raise RuntimeError(
        "Final dashboard was not generated:\n"
        f"{generated_path_32hm}"
    )


if (
    generated_path_32hm.resolve()
    !=
    LIVE_DASHBOARD_PATH_32HM.resolve()
):

    raise RuntimeError(
        "Dashboard builder wrote to an unexpected path:\n"
        f"{generated_path_32hm}"
    )


# ============================================================
# 17. FINAL HTML VALIDATION
# ============================================================

html_32hm = (
    generated_path_32hm
    .read_text(
        encoding="utf-8"
    )
)


checks_32hm = {

    # --------------------------------------------------------
    # Top-level wording
    # --------------------------------------------------------

    "short introduction":
        (
            "model-estimated F1 driver performance"
            in html_32hm
        ),

    "top-ranked driver label":
        (
            "Top-ranked driver"
            in html_32hm
        ),

    "old posterior mean label removed":
        (
            "Posterior mean #1"
            not in html_32hm
        ),


    # --------------------------------------------------------
    # Public terminology
    # --------------------------------------------------------

    "Pantheon score":
        (
            "Pantheon score"
            in html_32hm
        ),

    "no public JAWS wording":
        (
            "JAWS"
            not in html_32hm
        ),

    "Career value":
        (
            "Career value"
            in html_32hm
        ),


    # --------------------------------------------------------
    # 95% intervals
    # --------------------------------------------------------

    "95% credible intervals":
        (
            "95% credible interval"
            in html_32hm
        ),

    "no 89%":
        (
            "89%"
            not in html_32hm
        ),


    # --------------------------------------------------------
    # Ranking table behaviour
    # --------------------------------------------------------

    "ranking scrollbar override":
        (
            "Stage 32H-M — primary ranking tables expand vertically"
            in html_32hm
        ),

    "current ranking section preserved":
        (
            'id="current-rankings-section"'
            in html_32hm
        ),

    "Pantheon ranking section preserved":
        (
            'id="leaderboard-section"'
            in html_32hm
        ),


    # --------------------------------------------------------
    # Driver Explorer cleanup
    # --------------------------------------------------------

    "combined Driver Explorer rank interval":
        (
            "2.5th–97.5th percentiles"
            in html_32hm
        ),

    "old LB card removed":
        (
            "95% rank interval — LB"
            not in html_32hm
        ),

    "old UB card removed":
        (
            "95% rank interval — UB"
            not in html_32hm
        ),

    "career sample span class":
        (
            "career-sample-card"
            in html_32hm
        ),

    "realised performance terminology":
        (
            "season-level realised performance"
            in html_32hm
        ),


    # --------------------------------------------------------
    # Features preserved
    # --------------------------------------------------------

    "Drivers tab":
        (
            'id="current-driver-tab"'
            in html_32hm
        ),

    "Teams tab":
        (
            'id="current-team-tab"'
            in html_32hm
        ),

    "Driver explorer":
        (
            "driver explorer"
            in html_32hm.lower()
        ),

    "Compare drivers":
        (
            "Compare drivers"
            in html_32hm
        ),

    "Teammate network":
        (
            "teammate network"
            in html_32hm.lower()
        ),

    "Teammate Chain":
        (
            "Teammate Chain"
            in html_32hm
        ),

    "Method":
        (
            "How to read the model"
            in html_32hm
        ),
}


validation_32hm = pd.DataFrame(
    [
        {
            "check":
                check,

            "passed":
                passed,
        }

        for check, passed
        in checks_32hm.items()
    ]
)


print()
print("=" * 80)
print("STAGE 32H-M FINAL VALIDATION")
print("=" * 80)


display(
    validation_32hm
)


if not all(
    checks_32hm.values()
):

    failed_32hm = [

        check

        for check, passed
        in checks_32hm.items()

        if not passed
    ]


    raise RuntimeError(
        "Final dashboard cleanup failed validation:\n  "
        +
        "\n  ".join(
            failed_32hm
        )
    )


# ============================================================
# 18. PAYLOAD COUNTS MUST REMAIN IDENTICAL
# ============================================================

payload_32hm = (
    dashboard_namespace_32hm[
        "payload"
    ]
)


if len(
    payload_32hm[
        "leaderboard"
    ]
) != 98:

    raise RuntimeError(
        "Pantheon leaderboard size changed unexpectedly."
    )


if len(
    payload_32hm[
        "current_rankings"
    ]
) != 23:

    raise RuntimeError(
        "Current-driver ranking size changed unexpectedly."
    )


if len(
    payload_32hm[
        "current_team_rankings"
    ]
) != 11:

    raise RuntimeError(
        "Current-team ranking size changed unexpectedly."
    )


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("STAGE 32H-M COMPLETE")
print("=" * 80)


print()
print(
    "Final dashboard:"
)

print(
    generated_path_32hm.resolve()
)


print()
print(
    "Final public terminology:"
)

print(
    "  Pantheon score"
)

print(
    "  Peak"
)

print(
    "  Career value"
)

print(
    "  driver alpha = realised season-level performance"
)

print(
    "  constructor beta = constructor-season context"
)

print(
    "  central 95% credible intervals"
)


print()
print(
    "UI cleanup:"
)

print(
    "  ranking tables expand vertically"
)

print(
    "  Driver Explorer rank interval combined"
)

print(
    "  remaining JAWS labels removed"
)

print(
    "  top summary says Top-ranked driver"
)

print(
    "  opening explanation shortened"
)


print()
print(
    "No model, posterior, score or ranking changed."
)

In [ ]:
# ============================================================
# STAGE 32H-N — FIX LIVE BANNER / INTRO SPACING
# ============================================================

from pathlib import Path


required_32hn = [
    "LIVE_DASHBOARD_BUILDER_PATH_32HM",
    "dashboard_namespace_32hm",
    "CFG",
]


missing_32hn = [
    name
    for name in required_32hn
    if name not in globals()
]


if missing_32hn:

    raise RuntimeError(
        "Stage 32H-N requires:\n  "
        +
        "\n  ".join(missing_32hn)
        +
        "\n\nRun Stage 32H-M first."
    )


base_builder_32hn = Path(
    LIVE_DASHBOARD_BUILDER_PATH_32HM
)


source_32hn = base_builder_32hn.read_text(
    encoding="utf-8"
)


spacing_css_32hn = r"""

/* =========================================================
   Stage 32H-N — LIVE banner spacing
========================================================= */

.pantheon-intro {

    margin:
        12px 0 18px 0 !important;

}

"""


if "</style>" not in source_32hn:

    raise RuntimeError(
        "Could not locate closing </style> tag."
    )


patched_32hn = source_32hn.replace(

    "</style>",

    spacing_css_32hn
    +
    "\n</style>",

    1,
)


# ============================================================
# OUTPUT
# ============================================================

LIVE_DASHBOARD_FILENAME_32HN = (
    "f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "FINAL.html"
)


LIVE_DASHBOARD_BUILDER_FILENAME_32HN = (
    "build_f1_pantheon_1982_2026_"
    "v2_2_LIVE_"
    "through_r15_15gp_cf1f82aaab32_"
    "15of23gp_"
    "FINAL.py"
)


output_dir_32hn = Path(
    CFG.output_dir
)


LIVE_DASHBOARD_PATH_32HN = (
    output_dir_32hn
    /
    LIVE_DASHBOARD_FILENAME_32HN
)


LIVE_DASHBOARD_BUILDER_PATH_32HN = (
    output_dir_32hn
    /
    LIVE_DASHBOARD_BUILDER_FILENAME_32HN
)


# We are deliberately replacing the Stage 32H-M "FINAL"
# presentation artifact only. The statistical release files
# remain untouched.

LIVE_DASHBOARD_BUILDER_PATH_32HN.write_text(
    patched_32hn,
    encoding="utf-8",
)


dashboard_namespace_32hn = dict(
    dashboard_namespace_32hm
)


dashboard_namespace_32hn.update(
    {
        "LIVE_DASHBOARD_FILENAME_32HH":
            LIVE_DASHBOARD_FILENAME_32HN,

        "OPEN_LIVE_DASHBOARD_AFTER_BUILD_32HH":
            True,
    }
)


compiled_32hn = compile(
    patched_32hn,
    str(LIVE_DASHBOARD_BUILDER_PATH_32HN),
    "exec",
)


exec(
    compiled_32hn,
    dashboard_namespace_32hn,
    dashboard_namespace_32hn,
)


generated_path_32hn = Path(
    dashboard_namespace_32hn[
        "PANTHEON_DASHBOARD_PATH"
    ]
)


print("=" * 80)
print("STAGE 32H-N COMPLETE")
print("=" * 80)

print()
print("Final dashboard:")
print(generated_path_32hn.resolve())

print()
print(
    "LIVE banner / introductory text spacing fixed."
)

## Release state

For the current Round-15 snapshot, the statistical release is already frozen. Re-running the expensive production-fit cell should normally load the existing checkpoint rather than sample a new posterior.

The next data update should create a new snapshot/hash rather than overwrite the Round-15 release.
